# Soportes: modelo por planilla Excel

Notebook nuevo (no modifica el ejemplo `agente_estructural.ipynb`). El modelo se define en una planilla, como en RAM Element: no hay que dibujar.

## Flujo

1. **Bloque 6**: carga la plantilla `plantilla_soportes.xlsx` (incrustada) y muestra sus hojas. Descárguela desde la pestaña Output.
2. Llénela en Excel:
   - **NODOS**: ID, X, Y, Z (m) y UX, UY, UZ, RX, RY, RZ (`SI` = grado restringido; vacío o NO = libre). Empotrado = los 6 en SI; rotulado = UX, UY, UZ.
   - **BARRAS**: ID, NODO_I, NODO_J, ELEMENTO, PERFIL (lista Gerdau), EJE_FUERTE_X/Y/Z (direccion del peralte; OBLIGATORIO en cada barra, no tiene valor por defecto), CONEXION_I y CONEXION_J (`empotrado` o `rotulado` en cada extremo) y DUCTILIDAD (ALTA o MEDIA, lista; vacío = no verifica la Tabla 12.1.2).
   - **VISTA_VIVA**: planta y alzado que cambian al editar NODOS o BARRAS. Puede elegir el nivel Z (C3) y la cota Y (C4).
   - **LONGITUDES**: por barra, LX y LY (m) longitud no arriostrada en X e Y, KX y KY (factor K), LB (m) para pandeo lateral y CB. Vacío = supuesto, y queda marcado en el diseño.
   - **Ángulos (LI)**: en LONGITUDES, marque SI en las cuatro columnas E5 solo si la barra las cumple; si no, el diseño usa K·L/rz y H2. LI CONEXION (UNA PATA o DOS PATAS) y LI LONGITUD CONEXION (m) fijan el factor U de tracción.
   - **CONEXIONES**, **LISTAS**, **CALCULO**: no editar (CALCULO sostiene los gráficos).
3. Súbala a Kaggle con **Add data**. El **bloque 7** la lee y valida.
4. **Bloque 8**: dibuja planta, alzado y 3D (secciones y ejes fuertes) en el notebook y regenera la planilla como `planilla_con_vista.xlsx`, con las vistas en vivo de VISTA_VIVA. Vuelva a ejecutar este bloque después de cambiar la planilla.
5. **Bloque 9**: análisis con OpenSees. Usa E y NU de MATERIALES, la sección de la base Gerdau, los apoyos y rótulas, las cargas por estado, el peso propio (ESTADOS: PESO PROPIO = SI) y las combinaciones de COMBINACIONES. Escribe `resultados_soportes.xlsx` con las barras, nodos y reacciones marcados REPORTAR = SI.
6. **Bloque 10**: diseño por AISC 360-22 (ASD) de IR, CE y LI, con las combinaciones de servicio del bloque 9 (sin las que incluyen viento: para el diseño solo aplica el sismo) y las longitudes de LONGITUDES. Los perfiles macizos (CS, OS y SOL) fueron eliminados de la base y no se diseñan; LM no se incluye. Escribe `diseno_asd.xlsx` (CAPACIDADES, RESULTADOS, DETALLE, LISTAS_GRUPO) y `planilla_con_diseno.xlsx`, con las listas desplegables de perfiles que cumplen por grupo.

**Cambio de perfil por grupo (Excel):** en `planilla_con_diseno.xlsx`, la hoja **PERFIL_GRUPO** tiene un desplegable por grupo. Al elegir un perfil, se aplica a todas las barras de ese grupo (todas las columnas, todas las vigas...) y el FU y la esbeltez local y global se recalculan en la hoja **FU_BARRAS** sin volver a ejecutar nada. La hoja **FU_MATRIZ** guarda el FU de cada barra con cada perfil del catalogo. Si cambia ELEMENTO, longitudes o cargas, vuelva a ejecutar el bloque 10. Suba la planilla a Kaggle y ejecute el bloque 7: toma el perfil elegido por grupo. **Cada grupo tiene una sola sección:** si BARRAS trae perfiles distintos dentro de un grupo, el bloque 10 se detiene y lista cuáles son.

**Lo que se actualiza solo en Excel:** la vista en planta y alzado (hoja VISTA_VIVA).
**Lo que requiere ejecutar el bloque 8:** la vista 3D con secciones y ejes fuertes (Excel no dibuja en 3D por sí solo).

**Reglas:** coordenadas en metros; un nodo no puede repetir coordenadas; una barra no puede unir un nodo consigo mismo; el eje fuerte, si se da, no puede ser paralelo a la barra. Basta una componente: las vacias valen 0.

## Índice de bloques

1. Motor de dibujo (dibujo_directo.py)
2. Datos de perfiles (gerdau_datos.py)
3. Base de perfiles y lista ordenada (perfiles_gerdau.py)
4. Orientación y secciones (orientacion_gerdau.py)
5. Uso (sin código)
6. Cargar plantilla Excel
7. Leer, validar y armar el modelo
8. Dibujo de revisión y planilla regenerada
9. Análisis con OpenSees y Excel de resultados (analisis_opensees.py, celda9_analisis.py)


## Bloque 1. Motor de dibujo (dibujo_directo.py)

In [ ]:
%%writefile dibujo_directo.py
"""
MODULO DE DIBUJO DIRECTO (reemplaza la lectura de DXF).

Permite construir el modelo estructural dentro del notebook: nodos (puntos x, y, z en m),
barras (polilineas entre nodos), apoyos y ejes fuertes. Produce EXACTAMENTE el mismo
diccionario que entrega lector_validador.leer_dxf(), para que el motor de analisis
(motor_analisis.construir_y_resolver) y el validador funcionen sin cambios.

Formato de salida (igual al DXF):
  {'NODOS': [(x,y,z) en m], 'BARRAS': [{'n1','n2','elemento','perfil','capa','eje_fuerte'}],
   'NODOS_APOYO': [indices], 'ruta': str}

Unidades: metros (la conversion a cm la hace el motor).
"""
import json
import numpy as np

TOL_NODO = 1e-3                 # m: nodos a menos de esta distancia son el mismo nodo
ELEMENTOS = ['COLUMNA', 'VIGA', 'ARRIOSTRE', 'DIAGONAL']
TIPOS_APOYO = ['empotrado', 'rotulado']   # atajo para apoyo(nodo, tipo): empotrado = 6 GDL; rotulado = UX, UY, UZ
GRADOS = ['UX', 'UY', 'UZ', 'RX', 'RY', 'RZ']   # grados de libertad restringidos en un nodo (columnas de NODOS)
TIPOS_CONEXION = ['empotrado', 'rotulado']  # extremo de barra: empotrado = momento transmitido; rotulado = momento liberado


def restricciones_de(tipo_o_lista):
    """(UX, UY, UZ, RX, RY, RZ) como 6 bools. Acepta 'empotrado', 'rotulado' o una lista de 6 valores."""
    if isinstance(tipo_o_lista, str):
        if tipo_o_lista == 'empotrado':
            return (True,) * 6
        if tipo_o_lista == 'rotulado':
            return (True, True, True, False, False, False)
        raise ValueError(f'tipo de apoyo {tipo_o_lista!r} no reconocido (use empotrado, rotulado o 6 valores)')
    valores = tuple(bool(v) for v in tipo_o_lista)
    if len(valores) != 6:
        raise ValueError('un apoyo tiene 6 grados: UX, UY, UZ, RX, RY, RZ')
    return valores


def marca_apoyo(restr):
    """'empotrado' si restringe los 6 GDL; 'rotulado' si restringe alguna traslacion; 'giro' si solo giros."""
    r = restricciones_de(restr)
    if all(r):
        return 'empotrado'
    if any(r[:3]):
        return 'rotulado'
    return 'giro'


def etiqueta_restricciones(restr):
    r = restricciones_de(restr)
    if all(r):
        return 'todos (6 GDL)'
    return ', '.join(g for g, on in zip(GRADOS, r) if on) or 'ninguno'


class DibujoModelo:
    def __init__(self, tol=TOL_NODO, ruta='dibujo_directo'):
        self.nodos = []         # lista de tuplas (x, y, z)
        self.barras = []        # lista de dicts con el mismo formato que el DXF
        self.apoyos = {}        # {indice_nodo: tipo}
        self.tol = tol
        self.ruta = ruta

    # ---------- nodos ----------
    def nodo(self, x, y, z):
        """Agrega un nodo (o devuelve el existente a menos de la tolerancia). Devuelve su indice."""
        p = (round(float(x), 4), round(float(y), 4), round(float(z), 4))
        for i, q in enumerate(self.nodos):
            if all(abs(p[k] - q[k]) < self.tol for k in range(3)):
                return i
        self.nodos.append(p)
        return len(self.nodos) - 1

    # ---------- barras ----------
    def _existe(self, n1, n2):
        return any({b['n1'], b['n2']} == {n1, n2} for b in self.barras)

    def barra(self, n1, n2, elemento, perfil, conexion_i='empotrado', conexion_j='empotrado', material=None):
        """Agrega una barra entre dos nodos (indices). Si ya existe, no la duplica. Devuelve su indice o None.
        conexion_i / conexion_j: 'empotrado' o 'rotulado' en cada extremo de la barra."""
        if n1 == n2:
            raise ValueError(f'barra degenerada en el nodo {n1}')
        if self._existe(n1, n2):
            return None
        elemento = str(elemento).upper()
        if elemento not in ELEMENTOS:
            raise ValueError(f'elemento {elemento!r} no reconocido; usar {ELEMENTOS}')
        for nombre, c in [('conexion_i', conexion_i), ('conexion_j', conexion_j)]:
            if c not in TIPOS_CONEXION:
                raise ValueError(f'{nombre} {c!r} no reconocida; usar {TIPOS_CONEXION}')
        self.barras.append({'n1': n1, 'n2': n2, 'elemento': elemento, 'perfil': perfil,
                            'capa': f'{elemento}-{perfil}', 'eje_fuerte': None,
                            'conexion_i': conexion_i, 'conexion_j': conexion_j, 'material': material})
        return len(self.barras) - 1

    def polilinea(self, puntos, elemento, perfil):
        """Une una secuencia de puntos con barras consecutivas (estilo 'dibujar con lineas').
        puntos: lista de indices de nodo o de coordenadas (x, y, z). Devuelve indices de barras creadas."""
        ids = [p if isinstance(p, (int, np.integer)) else self.nodo(*p) for p in puntos]
        creadas = []
        for a, b in zip(ids[:-1], ids[1:]):
            r = self.barra(a, b, elemento, perfil)
            if r is not None:
                creadas.append(r)
        return creadas

    def eje_fuerte(self, indice_barra, direccion):
        """Fija la direccion del eje fuerte de una barra (vector en coordenadas globales)."""
        d = np.array(direccion, float)
        n = np.linalg.norm(d)
        if n < 1e-9:
            raise ValueError('direccion nula')
        self.barras[indice_barra]['eje_fuerte'] = tuple(float(v) for v in d / n)

    # ---------- apoyos ----------
    def apoyo(self, nodo_idx, restr='empotrado'):
        """Apoyo de un nodo: 6 bools (UX, UY, UZ, RX, RY, RZ) o 'empotrado' / 'rotulado'."""
        r = restricciones_de(restr)
        if not any(r):
            self.apoyos.pop(int(nodo_idx), None)
            return
        self.apoyos[int(nodo_idx)] = r

    # ---------- generadores ----------
    def marco_grilla(self, xs, ys, zs, perfil_columna, perfil_viga_x, perfil_viga_y, eje_fuerte_columna=None):
        """Marco ortogonal a partir de ejes: nodos en todas las intersecciones (x, y, z).
        Columnas en cada (x, y) entre niveles consecutivos; vigas X y vigas Y en cada nivel superior.
        eje_fuerte_columna: direccion del eje fuerte para las columnas (opcional, ej. (1,0,0))."""
        xs, ys, zs = sorted(xs), sorted(ys), sorted(zs)
        idx = {}
        for x in xs:
            for y in ys:
                for z in zs:
                    idx[(x, y, z)] = self.nodo(x, y, z)
        for x in xs:
            for y in ys:
                for za, zb in zip(zs[:-1], zs[1:]):
                    i = self.barra(idx[(x, y, za)], idx[(x, y, zb)], 'COLUMNA', perfil_columna)
                    if i is not None and eje_fuerte_columna is not None:
                        self.eje_fuerte(i, eje_fuerte_columna)
        for y in ys:
            for z in zs[1:]:
                for xa, xb in zip(xs[:-1], xs[1:]):
                    self.barra(idx[(xa, y, z)], idx[(xb, y, z)], 'VIGA', perfil_viga_x)
        for x in xs:
            for z in zs[1:]:
                for ya, yb in zip(ys[:-1], ys[1:]):
                    self.barra(idx[(x, ya, z)], idx[(x, yb, z)], 'VIGA', perfil_viga_y)
        return idx

    # ---------- salida y persistencia ----------
    def a_modelo(self):
        """Diccionario con el formato de leer_dxf(): listo para motor_analisis y validar_modelo."""
        return {'NODOS': list(self.nodos), 'BARRAS': [dict(b) for b in self.barras],
                'NODOS_APOYO': sorted(self.apoyos.keys()), 'TIPO_APOYO': dict(self.apoyos), 'ruta': self.ruta}

    def guardar_json(self, ruta):
        datos = {'nodos': self.nodos, 'barras': self.barras,
                 'apoyos': {str(k): v for k, v in self.apoyos.items()}}
        with open(ruta, 'w', encoding='utf-8') as f:
            json.dump(datos, f, ensure_ascii=False, indent=1)

    @classmethod
    def cargar_json(cls, ruta):
        with open(ruta, encoding='utf-8') as f:
            datos = json.load(f)
        m = cls()
        m.nodos = [tuple(p) for p in datos['nodos']]
        m.barras = datos['barras']
        m.apoyos = {int(k): restricciones_de(v) for k, v in datos['apoyos'].items()}
        return m


def comparar_modelos(m1, m2, tol=1e-3):
    """Compara dos modelos (formato leer_dxf) sin depender del orden de nodos ni barras.
    Devuelve (ok, lista_de_diferencias)."""
    def claves_nodos(m):
        return {tuple(round(c, 3) for c in p) for p in m['NODOS']}

    def claves_barras(m):
        out = []
        for b in m['BARRAS']:
            p1 = tuple(round(c, 3) for c in m['NODOS'][b['n1']])
            p2 = tuple(round(c, 3) for c in m['NODOS'][b['n2']])
            ext = tuple(sorted([p1, p2]))
            ef = None if b['eje_fuerte'] is None else tuple(round(c, 3) for c in b['eje_fuerte'])
            out.append((ext, b['elemento'], b['perfil'], ef))
        return sorted(out, key=str)

    def claves_apoyos(m):
        return sorted(tuple(round(c, 3) for c in m['NODOS'][i]) for i in m['NODOS_APOYO'])

    dif = []
    if claves_nodos(m1) != claves_nodos(m2):
        dif.append(f"nodos distintos: {len(claves_nodos(m1))} vs {len(claves_nodos(m2))}")
    b1, b2 = claves_barras(m1), claves_barras(m2)
    if b1 != b2:
        s1, s2 = set(map(str, b1)), set(map(str, b2))
        dif.append(f"barras distintas: {len(b1)} vs {len(b2)} | solo en 1: {len(s1 - s2)} | solo en 2: {len(s2 - s1)}")
    if claves_apoyos(m1) != claves_apoyos(m2):
        dif.append("apoyos distintos")
    return (len(dif) == 0), dif


## Bloque 2. Datos de perfiles (gerdau_datos.py)

Los 280 perfiles (164 Gerdau rev04 y 116 HSS del catálogo ATC) en un texto comprimido. No hay que editar este bloque.

In [ ]:
%%writefile gerdau_datos.py
_DATOS_TEXTO = "..."


## Bloque 3. Base de perfiles y lista ordenada (perfiles_gerdau.py)

In [ ]:
%%writefile perfiles_gerdau.py
"""Base de perfiles: Gerdau rev04 (datos en gerdau_datos.py) y HSS del catalogo ATC (en la misma base)."""
import base64
import json
import zlib
from gerdau_datos import _DATOS_TEXTO

_DATOS = json.loads(zlib.decompress(base64.b64decode(_DATOS_TEXTO)).decode('utf-8'))   # texto comprimido (zlib + base64)
PERFILES = {p['perfil_comercial']: p for p in _DATOS}

FAMILIA_ORDEN = ['IR', 'CE', 'LI', 'HSS']   # CS, OS y SOL (macizos) eliminados de la base


def obtener_perfil(nombre):
    """Dimensiones en cm: H = d (peralte o diametro), B = bf, e = tw, t = tf. None = dato pendiente."""
    p = PERFILES.get(nombre)
    if p is None:
        return None
    return {'familia': p['familia'], 'H': p['d_ala_diametro_cm'], 'B': p['bf_b_cm'],
            'e': p['tw_cm'], 't': p['tf_t_cm'], 'A': p['area_cm2'], 'Ix': p['ix_cm4'], 'Iy': p['iy_cm4'],
            'Sx': p['sx_cm3'], 'Sy': p['sy_cm3'], 'rx': p['rx_cm'], 'ry': p['ry_cm'], 'J': p['j_cm4'],
            'Cw': p['cw_cm6'], 'Zx': p.get('zx_cm3'), 'Zy': p.get('zy_cm3'), 'ro': p.get('ro_cm'),
            'estado': p['estado'], 'masa': p['masa_kg_m'], 'pagina_manual': p.get('pagina_manual'),
            'fuente_completado': p.get('fuente_completado'), 'seccion': p.get('seccion'),
            'pagina_catalogo_atc': p.get('pagina_catalogo_atc'), 'referencia_catalogo': p.get('referencia_catalogo'),
            'fuente_propiedades': p.get('fuente_propiedades')}


def _clave_seccion(p):
    """Familia, luego peralte (o diametro/lado), masa (mas liviano primero), ancho. Nulos al final."""
    def k(v):
        return (v is None, v or 0)
    return (FAMILIA_ORDEN.index(p['familia']),
            k(p['d_ala_diametro_cm']), k(p['masa_kg_m']), k(p['bf_b_cm']),
            p['perfil_comercial'])


def lista_perfiles_ordenada():
    """[(etiqueta, nombre)] para la lista desplegable. Muestra masa y estado si no es Revisado."""
    out = []
    for p in sorted(_DATOS, key=_clave_seccion):
        masa = '' if p['masa_kg_m'] is None else f"   {p['masa_kg_m']:g} kg/m"
        estado = '' if p['estado'] == 'Revisado' else f"   [{p['estado']}]"
        out.append((f"{p['perfil_comercial']}{masa}{estado}", p['perfil_comercial']))
    return out


## Bloque 4. Orientación y secciones (orientacion_gerdau.py)

In [ ]:
%%writefile orientacion_gerdau.py
"""
ORIENTACION DEL EJE FUERTE (peralte) EN PANTALLA, con la base Gerdau Mexico rev04.

Convencion del motor (copiada de motor_analisis.py del otro chat, sin depender de ese paquete):
  xAxis = (p2 - p1)/L ; yAxis = vecxz x xAxis ; zAxis = xAxis x yAxis.
  El eje fuerte (Iz) es el eje local y => su direccion es el PERALTE d del perfil.

Defaults del motor (sin marca):
  - vigas/diagonales: peralte VERTICAL (eje fuerte resiste gravedad)
  - columnas:         peralte en Y global (vecxz = (1,0,0))
"""
import math
import numpy as np
import perfiles_gerdau as G


def vector_vecxz(p1, p2, elemento):
    """vecxz por defecto segun el tipo de elemento (igual que el motor)."""
    v = np.array(p2, float) - np.array(p1, float)
    L = np.linalg.norm(v)
    x = v / L
    if abs(x[2]) > 0.9:                       # columna (casi vertical)
        return (1.0, 0.0, 0.0)
    vx = np.cross(x, np.array([0.0, 0.0, 1.0]))   # viga/diagonal: yAxis vertical
    n = np.linalg.norm(vx)
    if n < 1e-6:
        return (1.0, 0.0, 0.0)
    vx = vx / n
    return (float(vx[0]), float(vx[1]), float(vx[2]))


def vecxz_desde_direccion(p1, p2, s):
    """vecxz que hace que el eje local y (= peralte) apunte segun s (proyectado perpendicular a la barra)."""
    x = np.array(p2, float) - np.array(p1, float); x = x / np.linalg.norm(x)
    s = np.array(s, float)
    u = s - x * np.dot(s, x)
    nu = np.linalg.norm(u)
    if nu < 1e-6:
        return None
    u = u / nu
    v = np.cross(x, u)
    return (float(v[0]), float(v[1]), float(v[2]))


def vecxz_de_barra(nodos, b):
    """vecxz de una barra a partir de su eje fuerte, que es obligatorio (hoja BARRAS: EJE_FUERTE_X/Y/Z).
    Sin valor por defecto: el modelo depende de este dato, y un valor implicito lo ocultaria."""
    p1 = nodos[b['n1']]; p2 = nodos[b['n2']]
    s = b.get('eje_fuerte')
    if not s:
        raise ValueError('barra sin eje fuerte: indique EJE_FUERTE_X/Y/Z en la hoja BARRAS (direccion del peralte)')
    v = vecxz_desde_direccion(p1, p2, s)
    if v is None:
        raise ValueError('el eje fuerte de la barra es paralelo a la barra: indique una direccion perpendicular')
    return v


def ejes_locales(dib, i):
    """Ejes locales (x, y, z) de la barra i."""
    b = dib.barras[i]
    p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
    vecxz = np.array(vecxz_de_barra(dib.nodos, b), float)
    x = np.array(p2, float) - np.array(p1, float); x = x / np.linalg.norm(x)
    y = np.cross(vecxz, x); y = y / np.linalg.norm(y)
    z = np.cross(x, y)
    return x, y, z


def dimensiones(perfil):
    """(d, bf) en metros desde Gerdau (H = d, B = bf en cm). None si el perfil no existe o no tiene dato."""
    p = G.obtener_perfil(perfil)
    if p is None or p['H'] is None or p['B'] is None:
        return None
    return float(p['H']) / 100.0, float(p['B']) / 100.0


def contorno(dib, i, escala=1.0):
    """4 vertices globales del rectangulo del perfil en el punto medio de la barra i.
    Alto = d a lo largo del eje local y (eje fuerte); ancho = bf a lo largo de z."""
    b = dib.barras[i]
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    pm = (p1 + p2) / 2
    dims = dimensiones(b['perfil'])
    if dims is None:
        return None
    d, bf = dims[0] * escala, dims[1] * escala
    x, y, z = ejes_locales(dib, i)
    c = [pm + sy * (d / 2) * y + sz * (bf / 2) * z for sy, sz in [(1, 1), (-1, 1), (-1, -1), (1, -1)]]
    return [tuple(float(v) for v in q) for q in c]


PRESETS = {'peralte vertical (Z)': (0.0, 0.0, 1.0),
           'peralte en X': (1.0, 0.0, 0.0),
           'peralte en Y': (0.0, 1.0, 0.0)}


def orientar(dib, i, preset):
    """Fija el peralte (eje fuerte) de la barra i con un preset o un vector. Rechaza direccion paralela a la barra."""
    b = dib.barras[i]
    s = np.array(PRESETS[preset] if isinstance(preset, str) else preset, float)
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    x = (p2 - p1) / np.linalg.norm(p2 - p1)
    if abs(np.dot(s, x)) > 0.999:
        raise ValueError(f'el peralte no puede ser paralelo a la barra {i}; elija otra direccion')
    dib.eje_fuerte(i, tuple(s))
    return b['eje_fuerte']


def ortogonalidad(dib, i):
    """Angulo (grados) entre el eje fuerte asignado y la barra; 90 = perpendicular (correcto)."""
    b = dib.barras[i]
    if b['eje_fuerte'] is None:
        return None
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    x = (p2 - p1) / np.linalg.norm(p2 - p1)
    s = np.array(b['eje_fuerte'], float)
    return math.degrees(math.acos(max(-1.0, min(1.0, abs(np.dot(s, x))))))


In [ ]:
%%writefile vista3d_plotly.py
"""
FIGURA 3D INTERACTIVA (plotly): gira, acerca y muestra datos al pasar el cursor. Funciona en Kaggle porque
plotly embebe su propio motor en la salida; no requiere widgets de matplotlib.

Contenido:
  - barras por tipo de elemento (color), con ID, perfil, conexiones y eje fuerte al pasar el cursor
  - seccion envolvente de cada perfil (caja de d x bf) en la barra, ampliada ESCALA_SECCION
  - flecha = eje fuerte (direccion del peralte) en el punto medio de cada barra
  - circulo abierto = extremo rotulado; cuadrado = apoyo empotrado; rombo = apoyo rotulado
"""
import numpy as np
import plotly.graph_objects as go
from orientacion_gerdau import contorno, ejes_locales, dimensiones
from dibujo_directo import marca_apoyo

COLOR_ELEMENTO = {'COLUMNA': '#1f77b4', 'VIGA': '#2ca02c', 'ARRIOSTRE': '#d62728', 'DIAGONAL': '#ff7f0e'}


def _flecha(fig, origen, vec, color, nombre, tam_cono):
    """Flecha 3D: linea con etiqueta y cono en la punta."""
    tip = origen + vec
    fig.add_trace(go.Scatter3d(x=[origen[0], tip[0]], y=[origen[1], tip[1]], z=[origen[2], tip[2]],
                               mode='lines+text', line=dict(color=color, width=7), text=['', nombre],
                               textposition='top center', textfont=dict(color=color, size=12),
                               name=nombre, hoverinfo='skip'))
    fig.add_trace(go.Cone(x=[tip[0]], y=[tip[1]], z=[tip[2]], u=[vec[0]], v=[vec[1]], w=[vec[2]],
                          anchor='tip', sizemode='absolute', sizeref=tam_cono,
                          colorscale=[[0, color], [1, color]], showscale=False, hoverinfo='skip',
                          showlegend=False))


def figura_3d(dib, id_de_indice, escala_seccion=2.5, titulo=None, ejes=True, barra_ejemplo=None):
    """Devuelve una go.Figure 3D del modelo. escala_seccion solo amplia la seccion dibujada, no el calculo."""
    fig = go.Figure()
    nodos = np.array(dib.nodos, float)
    id_barra = getattr(dib, 'id_barra_de_indice', {}) or {}

    def rotulada(b):
        return b['conexion_i'] == 'rotulado' or b['conexion_j'] == 'rotulado'

    def texto_barra(i, b):
        return (f"Barra ID {id_barra.get(i, i)} (nodos N{id_de_indice.get(b['n1'], '?')}-N{id_de_indice.get(b['n2'], '?')})"
                f"<br>{b['elemento']} {b['perfil']}"
                f"<br>conexion I: {b['conexion_i']}, J: {b['conexion_j']}"
                f"<br>eje fuerte: {'por defecto' if b['eje_fuerte'] is None else np.round(b['eje_fuerte'], 3).tolist()}")

    # --- barras con extremo rotulado: linea discontinua morada (se ven distintas de las continuas) ---
    xs, ys, zs, textos = [], [], [], []
    for i, b in enumerate(dib.barras):
        if not rotulada(b):
            continue
        p1, p2 = nodos[b['n1']], nodos[b['n2']]
        info = texto_barra(i, b)
        xs += [p1[0], p2[0], None]; ys += [p1[1], p2[1], None]; zs += [p1[2], p2[2], None]
        textos += [info, info, None]
    if xs:
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                                   line=dict(color='#6a1b9a', width=9, dash='dash'),
                                   name='barra con extremo rotulado (linea discontinua)',
                                   text=textos, hoverinfo='text'))

    # --- barras continuas por tipo de elemento (una traza por tipo, con huecos entre barras) ---
    for elem, color in COLOR_ELEMENTO.items():
        xs, ys, zs, textos = [], [], [], []
        for i, b in enumerate(dib.barras):
            if b['elemento'] != elem or rotulada(b):
                continue
            p1, p2 = nodos[b['n1']], nodos[b['n2']]
            info = texto_barra(i, b)
            xs += [p1[0], p2[0], None]; ys += [p1[1], p2[1], None]; zs += [p1[2], p2[2], None]
            textos += [info, info, None]
        if xs:
            fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', line=dict(color=color, width=7),
                                       name=elem, text=textos, hoverinfo='text'))

    # --- secciones envolventes (caja d x bf en ambos extremos) y eje fuerte ---
    for i, b in enumerate(dib.barras):
        caja = contorno(dib, i, escala_seccion)
        if not caja:
            continue
        p1, p2 = nodos[b['n1']], nodos[b['n2']]
        x_loc, y_loc, z_loc = ejes_locales(dib, i)
        d, bf = dimensiones(b['perfil'])
        d *= escala_seccion; bf *= escala_seccion
        vertices = []
        for p in (p1, p2):
            for sy, sz in [(1, 1), (-1, 1), (-1, -1), (1, -1)]:
                q = p + sy * (d / 2) * y_loc + sz * (bf / 2) * z_loc
                vertices.append(q)
        vertices = np.array(vertices)
        fig.add_trace(go.Mesh3d(x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2], alphahull=0,
                                color=COLOR_ELEMENTO[b['elemento']], opacity=0.12 if rotulada(b) else 0.35, flatshading=True,
                                name=f'seccion {b["perfil"]}', showlegend=False, hoverinfo='skip'))
        pm = (p1 + p2) / 2
        largo = 0.6 * d
        tip = pm + y_loc * largo
        fig.add_trace(go.Scatter3d(x=[pm[0], tip[0]], y=[pm[1], tip[1]], z=[pm[2], tip[2]], mode='lines',
                                   line=dict(color='black', width=4), showlegend=False, hoverinfo='skip'))
        fig.add_trace(go.Cone(x=[tip[0]], y=[tip[1]], z=[tip[2]], u=[y_loc[0]], v=[y_loc[1]], w=[y_loc[2]],
                              anchor='tip', sizemode='absolute', sizeref=0.12,
                              colorscale=[[0, 'black'], [1, 'black']], showscale=False, hoverinfo='skip',
                              showlegend=False))

    # --- extremos rotulados (circulo abierto) ---
    xr, yr, zr = [], [], []
    for b in dib.barras:
        for idx, c in [(b['n1'], b['conexion_i']), (b['n2'], b['conexion_j'])]:
            if c == 'rotulado':
                xr.append(nodos[idx][0]); yr.append(nodos[idx][1]); zr.append(nodos[idx][2])
    if xr:
        fig.add_trace(go.Scatter3d(x=xr, y=yr, z=zr, mode='markers',
                                   marker=dict(symbol='circle-open', size=9, color='black', line=dict(width=3)),
                                   name='extremo rotulado (circulo abierto)', hoverinfo='skip'))

    # --- nodos con IDs y apoyos ---
    nodos_id = [(i, nodos[i]) for i in range(len(nodos))]
    # IDs de barra (B) en el punto medio de cada barra; se pueden ocultar desde la leyenda
    mx = [(nodos[b['n1']][0] + nodos[b['n2']][0]) / 2 for b in dib.barras]
    my = [(nodos[b['n1']][1] + nodos[b['n2']][1]) / 2 for b in dib.barras]
    mz = [(nodos[b['n1']][2] + nodos[b['n2']][2]) / 2 for b in dib.barras]
    fig.add_trace(go.Scatter3d(x=mx, y=my, z=mz, mode='text',
                               text=[f"B{id_barra.get(i, i)}" for i in range(len(dib.barras))],
                               textfont=dict(size=12, color='#6a1b9a'), name='ID de barra (B)',
                               hoverinfo='skip'))
    fig.add_trace(go.Scatter3d(x=nodos[:, 0], y=nodos[:, 1], z=nodos[:, 2], mode='markers+text',
                               marker=dict(size=3, color='black'),
                               text=[f"N{id_de_indice.get(i, i)}" for i in range(len(nodos))],
                               textposition='top center', textfont=dict(size=12, color='black'),
                               name='ID de nodo (N)', hoverinfo='text',
                               hovertext=[f'nodo ID {id_de_indice.get(i, i)}<br>X {p[0]:g}  Y {p[1]:g}  Z {p[2]:g}'
                                          for i, p in nodos_id]))
    fig.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='black', width=4),
                               name='eje fuerte (flecha)', hoverinfo='skip'))
    for tipo, simbolo, color, nombre in [('empotrado', 'square', '#e67e22', 'apoyo empotrado (cuadrado)'),
                                         ('rotulado', 'diamond', '#e67e22', 'apoyo rotulado (rombo)')]:
        idx = [i for i, t in dib.apoyos.items() if marca_apoyo(t) == tipo or (tipo == 'rotulado' and marca_apoyo(t) == 'giro')]
        if idx:
            fig.add_trace(go.Scatter3d(x=nodos[idx, 0], y=nodos[idx, 1], z=nodos[idx, 2], mode='markers',
                                       marker=dict(symbol=simbolo, size=7, color=color), name=nombre,
                                       hoverinfo='skip'))

    # --- ejes globales X, Y, Z y ejes locales de una barra de ejemplo (x = de I a J, y = eje fuerte, z) ---
    if ejes and len(nodos):
        lo, hi = nodos.min(axis=0), nodos.max(axis=0)
        tam = max(float(np.max(hi - lo)), 1.0)
        o = lo - 0.15 * tam
        for vec, nombre in [((1, 0, 0), 'X global'), ((0, 1, 0), 'Y global'), ((0, 0, 1), 'Z global')]:
            _flecha(fig, o, np.array(vec, float) * 0.25 * tam, '#555555', nombre, 0.06 * tam)
        if barra_ejemplo is None:
            vigas = [i for i, b in enumerate(dib.barras) if b['elemento'] == 'VIGA']
            barra_ejemplo = vigas[0] if vigas else 0
        if dib.barras:
            b = dib.barras[barra_ejemplo]
            p1, p2 = nodos[b['n1']], nodos[b['n2']]
            x_loc, y_loc, z_loc = ejes_locales(dib, barra_ejemplo)
            c = (p1 + p2) / 2
            l = 0.2 * tam
            _flecha(fig, c, x_loc * l, '#d62728', 'x local (I a J)', 0.06 * tam)
            _flecha(fig, c, y_loc * l, '#2ca02c', 'y local (eje fuerte)', 0.06 * tam)
            _flecha(fig, c, z_loc * l, '#1f77b4', 'z local (eje debil)', 0.06 * tam)
            for pt, etiqueta in [(p1, 'I'), (p2, 'J')]:
                fig.add_trace(go.Scatter3d(x=[pt[0]], y=[pt[1]], z=[pt[2]], mode='text', text=[etiqueta],
                                           textfont=dict(size=14, color='black'), hoverinfo='skip',
                                           showlegend=False))

    fig.update_layout(
        title=titulo or f'Modelo 3D: secciones x{escala_seccion:g}; flecha = eje fuerte (ver leyenda)',
        scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)', aspectmode='data',
                   camera=dict(eye=dict(x=1.5, y=-1.7, z=0.9))),
        margin=dict(l=0, r=0, t=50, b=0),
        legend=dict(itemsizing='constant', title=dict(text='Leyenda'), orientation='h',
                    yanchor='top', y=-0.02, xanchor='left', x=0, font=dict(size=11)),
        height=760,
    )
    return fig


## Bloque 5. Uso

Ejecutar los bloques 1 a 4 una sola vez. Luego trabajar con los bloques 6 a 8.

## Bloque 6. Cargar plantilla Excel

Carga `plantilla_soportes.xlsx` y muestra sus hojas. Llénela en Excel y súbala a Kaggle con Add data.

In [ ]:
# Plantilla Excel: escribe el libro completo (NODOS, BARRAS, CARGAS, ESTADOS, CONEXIONES, LISTAS, VISTA_VIVA).
# La misma funcion regenera el Excel con sus datos en el bloque 8, asi nada se pierde al guardar.
import os
import base64
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
COLS_MATRIZ_XL = ['CLAVE', 'ID', 'PERFIL', 'ELEMENTO DISENO', 'FU MAXIMO', 'FU DEMANDA',
                  'FU ESBELTEZ LOCAL (B4.1)', 'FU ESBELTEZ GLOBAL (KL/r<=200)',
                  'FU DUCTILIDAD (12.1.2, 12.1.3.1)', 'CASO QUE GOBIERNA', 'ESTADO',
                  'FU ESFUERZOS', 'FU ESBELTEZ', 'ESBELTEZ QUE GOBIERNA']
from openpyxl.chart import ScatterChart, Reference, Series
from openpyxl.drawing.image import Image as XLImage
from openpyxl.workbook.properties import CalcProperties
from dibujo_directo import DibujoModelo, ELEMENTOS, TIPOS_APOYO, TIPOS_CONEXION
import perfiles_gerdau as G
import pandas as pd

NODOS_COLS = ['ID', 'X (m)', 'Y (m)', 'Z (m)', 'UX', 'UY', 'UZ', 'RX', 'RY', 'RZ', 'NOTA', 'REPORTAR']
GRADOS_COLS = ['UX', 'UY', 'UZ', 'RX', 'RY', 'RZ']       # columnas E..J de NODOS: SI = grado restringido
BARRAS_COLS = ['ID', 'NODO_I', 'NODO_J', 'ELEMENTO', 'PERFIL', 'EJE_FUERTE_X', 'EJE_FUERTE_Y', 'EJE_FUERTE_Z',
               'CONEXION_I', 'CONEXION_J', 'MATERIAL', 'NOTA', 'REPORTAR', 'DUCTILIDAD']
MAT_MAX = 10
MAT_FIN = MAT_MAX + 1
# Valores de partida de la plantilla. Son datos de ENTRADA en la hoja MATERIALES: el notebook los lee de ahi
# (no los fija). E = 200 000 MPa = 2 038 900 kgf/cm2 y NU = 0.3, igual que A-36 en el modelo SAP CMZ-5746.
E_ACERO_PARTIDA = 2038900
NU_ACERO_PARTIDA = 0.3
MATERIALES_DEFAULT = [
    ['A-36', 'Acero', 2530, 4078, E_ACERO_PARTIDA, NU_ACERO_PARTIDA,
     'Manual Gerdau, pie de pagina 22 (fy). Fu y E: modelo SAP CMZ-5746 (A36).'],
    ['A-529 G 50', 'Acero', 3515, 4570, E_ACERO_PARTIDA, NU_ACERO_PARTIDA,
     'Manual Gerdau, pie de pagina 22 (fy). Fu 65 ksi: referencia Gerdau Corsa (confirmar). E y NU: confirmar.'],
    ['A-572 G 50', 'Acero', 3515, 4570, E_ACERO_PARTIDA, NU_ACERO_PARTIDA,
     'Manual Gerdau, pie de pagina 22. Perfiles IR (pag. 6): A992 / A572 G50. Fu 4570 kgf/cm2: dato del usuario (confirmar). E y NU: confirmar.'],
    ['A-572 G 60', 'Acero', 4200, None, E_ACERO_PARTIDA, NU_ACERO_PARTIDA,
     'Manual Gerdau, pie de pagina 22. Fu pendiente. E y NU: confirmar.'],
    ['A500 Gr C', 'Acero', 3234, 4359, E_ACERO_PARTIDA, NU_ACERO_PARTIDA,
     'HSS (ATC). Referencia ASTM A500 Gr C: Fy 46 ksi, Fu 62 ksi (1 ksi = 70.31 kgf/cm2). Dato de entrada: confirmar con el proveedor del tubo. E y NU: confirmar.'],
]
NODOS_MAX = 98          # filas 2..99 de NODOS
BARRAS_MAX = 106        # filas 2..107 de BARRAS
FIN_N = NODOS_MAX + 1
FIN_B = BARRAS_MAX + 1
NIV = 'VISTA_VIVA!$C$3'     # nivel Z para planta (celda editable)
COTA = 'VISTA_VIVA!$C$4'    # cota Y para alzado (celda editable)
PERFILES_ORDEN = G.lista_perfiles_ordenada()
ENCABEZADO = PatternFill('solid', fgColor='DDEBF7')
CABECERA_CALC = PatternFill('solid', fgColor='EDEDED')
ENTRADA = PatternFill('solid', fgColor='FFF2CC')
NEGRITA = Font(bold=True)


def _encabezado(ws, columnas, anchos, relleno=ENCABEZADO):
    for c, nombre in enumerate(columnas, start=1):
        celda = ws.cell(row=1, column=c, value=nombre)
        celda.font = NEGRITA; celda.fill = relleno
        celda.alignment = Alignment(horizontal='center', wrap_text=True)
        ws.column_dimensions[celda.column_letter].width = anchos[c - 1]
    ws.freeze_panes = 'A2'


def _filas(df, columnas):
    """Filas (listas) de un DataFrame leido de la planilla, en el orden de columnas; NaN -> None."""
    salida = []
    for _, r in df.iterrows():
        fila = []
        for c in columnas:
            v = r[c] if c in r.index else None
            if v is None or (isinstance(v, float) and pd.isna(v)):
                v = None
            fila.append(v)
        salida.append(fila)
    return salida


def ejemplo_filas():
    """Marco de ejemplo: 18 nodos, 26 barras, 6 apoyos empotrados."""
    ejemplo = DibujoModelo(ruta='ejemplo')
    ejemplo.marco_grilla([0, 5, 10], [0, 4], [0, 3.5, 7], 'IR 10 × 49', 'IR 12 × 16', 'IR 12 × 16')
    # Eje fuerte explicito de cada barra (queda en la hoja BARRAS): la direccion que daba la regla anterior por
    # defecto (columnas: eje fuerte hacia Y global; vigas y diagonales: peralte vertical). Con este valor el modelo
    # es el mismo que antes y ya no depende de una regla oculta.
    import numpy as np
    for i, b in enumerate(ejemplo.barras):
        p1, p2 = ejemplo.nodos[b['n1']], ejemplo.nodos[b['n2']]
        x = np.array(p2, float) - np.array(p1, float); x = x / np.linalg.norm(x)
        if abs(x[2]) > 0.9:
            vecxz = np.array([1.0, 0.0, 0.0])
        else:
            vx = np.cross(x, np.array([0.0, 0.0, 1.0])); vecxz = vx / np.linalg.norm(vx)
        y = np.cross(vecxz, x); y = y / np.linalg.norm(y)
        ejemplo.eje_fuerte(i, tuple(float(c) for c in y))
    for i, q in enumerate(ejemplo.nodos):
        if abs(q[2]) < 1e-9:
            ejemplo.apoyo(i, 'empotrado')
    nodos = []
    for i, (x, y, z) in enumerate(ejemplo.nodos):
        restr = ejemplo.apoyos.get(i, (False,) * 6)
        nodos.append([i + 1, x, y, z, *['SI' if v else None for v in restr], None, 'SI'])
    barras = [[k + 1, b['n1'] + 1, b['n2'] + 1, b['elemento'], b['perfil'], *(b['eje_fuerte'] or (None, None, None)),
               b['conexion_i'], b['conexion_j'], (b.get('material') or 'A-572 G 50'), None, 'SI',
               ('MEDIA' if b['elemento'] in ('COLUMNA', 'VIGA') else None)]
              for k, b in enumerate(ejemplo.barras)]
    return nodos, barras


def _hoja_calculo(wb):
    """Columnas que alimentan los graficos. Regla: la coordenada X de cada serie es siempre numerica (asi el
    grafico no usa numeros de fila); solo la coordenada que se oculta (la de la vista que no corresponde) es #N/A."""
    ws = wb.create_sheet('CALCULO')
    _encabezado(ws, ['', 'nodo I (fila)', 'nodo J (fila)', 'X1', 'X2', 'Y1', 'Y2', 'Z1', 'Z2',
                     'planta X1', 'planta X2', 'planta Y1', 'planta Y2',
                     'alzado X1', 'alzado X2', 'alzado Z1', 'alzado Z2', '',
                     'nodo X (planta)', 'nodo Y (planta)', 'nodo X (alzado)', 'nodo Z (alzado)',
                     'apoyo X (planta)', 'apoyo Y (planta)', 'apoyo X (alzado)', 'apoyo Z (alzado)'],
                [4] + [11] * 16 + [3] + [11] * 8, relleno=CABECERA_CALC)
    for r in range(2, FIN_B + 1):
        ws[f'B{r}'] = f'=IFERROR(MATCH(BARRAS!B{r},NODOS!$A$2:$A${FIN_N},0),NA())'
        ws[f'C{r}'] = f'=IFERROR(MATCH(BARRAS!C{r},NODOS!$A$2:$A${FIN_N},0),NA())'
        for col, ref, idx in [('D', 'B', 'B'), ('E', 'B', 'C'), ('F', 'C', 'B'), ('G', 'C', 'C'),
                              ('H', 'D', 'B'), ('I', 'D', 'C')]:
            ws[f'{col}{r}'] = f'=IFERROR(INDEX(NODOS!${ref}$2:${ref}${FIN_N},{idx}{r}),0)'
        vis_p = f'AND(ISNUMBER(B{r}),H{r}={NIV},I{r}={NIV})'
        vis_a = f'AND(ISNUMBER(B{r}),F{r}={COTA},G{r}={COTA})'
        ws[f'J{r}'] = f'=D{r}'
        ws[f'K{r}'] = f'=E{r}'
        ws[f'L{r}'] = f'=IF({vis_p},F{r},NA())'
        ws[f'M{r}'] = f'=IF({vis_p},G{r},NA())'
        ws[f'N{r}'] = f'=D{r}'
        ws[f'O{r}'] = f'=E{r}'
        ws[f'P{r}'] = f'=IF({vis_a},H{r},NA())'
        ws[f'Q{r}'] = f'=IF({vis_a},I{r},NA())'
    CX = 'VISTA_VIVA!$C$8'          # cota X de la vista lateral Y-Z
    for col_, titulo in [(32, 'lat Z1 (barra)'), (33, 'lat Z2 (barra)'), (34, 'lat Y nodo'), (35, 'lat Z nodo'),
                         (36, 'lat Z apoyo'), (37, 'lat rotula I (Z)'), (38, 'lat rotula J (Z)')]:
        ws.cell(row=1, column=col_, value=titulo).font = NEGRITA
        ws.column_dimensions[_L(col_)].width = 14
    for r in range(2, FIN_B + 1):
        ws[f'AF{r}'] = f'=IF(AND(ISNUMBER(B{r}),D{r}={CX},E{r}={CX}),H{r},NA())'
        ws[f'AG{r}'] = f'=IF(AND(ISNUMBER(B{r}),D{r}={CX},E{r}={CX}),I{r},NA())'
        ws[f'AK{r}'] = f'=IF(AND(LOWER(TRIM(BARRAS!I{r}))="rotulado",ISNUMBER(B{r}),D{r}={CX}),H{r},NA())'
        ws[f'AL{r}'] = f'=IF(AND(LOWER(TRIM(BARRAS!J{r}))="rotulado",ISNUMBER(C{r}),E{r}={CX}),I{r},NA())'
    for r in range(2, FIN_N + 1):
        ws[f'AH{r}'] = f'=NODOS!$C{r}'
        ws[f'AI{r}'] = f'=IF(AND(NODOS!$A{r}<>"",NODOS!$B{r}={CX}),NODOS!$D{r},NA())'
        ws[f'AJ{r}'] = f'=IF(AND(COUNTIF(NODOS!$E{r}:$J{r},"SI")>0,NODOS!$B{r}={CX}),NODOS!$D{r},NA())'
    for col_, titulo in [(28, 'rotula I planta (Y)'), (29, 'rotula J planta (Y)'),
                         (30, 'rotula I alzado (Z)'), (31, 'rotula J alzado (Z)')]:
        ws.cell(row=1, column=col_, value=titulo).font = NEGRITA
        ws.column_dimensions[_L(col_)].width = 14
    for r in range(2, FIN_B + 1):
        rot_i = f'LOWER(TRIM(BARRAS!I{r}))="rotulado"'
        rot_j = f'LOWER(TRIM(BARRAS!J{r}))="rotulado"'
        ws[f'AB{r}'] = f'=IF(AND({rot_i},ISNUMBER(B{r}),H{r}={NIV}),F{r},NA())'
        ws[f'AC{r}'] = f'=IF(AND({rot_j},ISNUMBER(C{r}),I{r}={NIV}),G{r},NA())'
        ws[f'AD{r}'] = f'=IF(AND({rot_i},ISNUMBER(B{r}),F{r}={COTA}),H{r},NA())'
        ws[f'AE{r}'] = f'=IF(AND({rot_j},ISNUMBER(C{r}),G{r}={COTA}),I{r},NA())'
    for r in range(2, FIN_N + 1):
        ok = f'NODOS!$A{r}<>""'
        ws[f'S{r}'] = f'=NODOS!$B{r}'
        ws[f'T{r}'] = f'=IF(AND({ok},NODOS!$D{r}={NIV}),NODOS!$C{r},NA())'
        ws[f'U{r}'] = f'=NODOS!$B{r}'
        ws[f'V{r}'] = f'=IF(AND({ok},NODOS!$C{r}={COTA}),NODOS!$D{r},NA())'
        ws[f'W{r}'] = f'=NODOS!$B{r}'
        ws[f'X{r}'] = f'=IF(AND(COUNTIF(NODOS!$E{r}:$J{r},"SI")>0,NODOS!$D{r}={NIV}),NODOS!$C{r},NA())'
        ws[f'Y{r}'] = f'=NODOS!$B{r}'
        ws[f'Z{r}'] = f'=IF(AND(COUNTIF(NODOS!$E{r}:$J{r},"SI")>0,NODOS!$C{r}={COTA}),NODOS!$D{r},NA())'
    return ws


from openpyxl.utils import get_column_letter as _L, column_index_from_string as _CI

CALC3D_BARRAS = ['bx1', 'bxm', 'bx2', 'bxm2', 'bxt',
                 'by1', 'bym', 'by2', 'bym2', 'byt',
                 'L', 'xu', 'yu', 'zu', 'nb', 'nc', 'hasU', 'dot',
                 'ux', 'uy', 'uz', 'un', 'ex', 'ey', 'ez', 'mx', 'my', 'mz', 'rotI', 'rotJ', 'lblB',
                 'rot', 'nA', 'nM', 'nB', 'rA', 'rM', 'rB']
CALC3D_NODOS = ['nsx', 'nsy', 'apE', 'apR', 'lblN']


def _hoja_3d(wb):
    """Vista 3D esquematica en vivo (proyeccion ortogonal). Azimut (C5), elevacion (C6) y largo de la flecha (C7)
    de VISTA_VIVA. Cada barra es una serie de 5 puntos: inicio, medio, fin, medio, punta de la flecha del eje
    fuerte (vector del usuario o por defecto del motor, igual que ejes_locales())."""
    ws = wb.create_sheet('CALCULO_3D')
    col = {}
    for k, nombre in enumerate(CALC3D_BARRAS, start=2):
        col[nombre] = _L(k)
    ini_n = len(CALC3D_BARRAS) + 4
    for k, nombre in enumerate(CALC3D_NODOS, start=ini_n):
        col[nombre] = _L(k)
    KC = _L(ini_n + len(CALC3D_NODOS) + 2)
    VC = _L(ini_n + len(CALC3D_NODOS) + 3)
    col['_VC'] = VC
    _encabezado(ws, [''] + CALC3D_BARRAS + [''] * 2 + CALC3D_NODOS,
                [4] + [9] * len(CALC3D_BARRAS) + [3, 3] + [9] * len(CALC3D_NODOS), relleno=CABECERA_CALC)
    cA, sA, cE, sE, largo = f'${VC}$2', f'${VC}$3', f'${VC}$4', f'${VC}$5', f'${VC}$6'
    ws[f'{KC}2'] = 'cos(azimut)'; ws[f'{VC}2'] = f'=COS(RADIANS(VISTA_VIVA!$C$5))'
    ws[f'{KC}3'] = 'sen(azimut)'; ws[f'{VC}3'] = f'=SIN(RADIANS(VISTA_VIVA!$C$5))'
    ws[f'{KC}4'] = 'cos(elevacion)'; ws[f'{VC}4'] = f'=COS(RADIANS(VISTA_VIVA!$C$6))'
    ws[f'{KC}5'] = 'sen(elevacion)'; ws[f'{VC}5'] = f'=SIN(RADIANS(VISTA_VIVA!$C$6))'
    ws[f'{KC}6'] = 'largo flecha (m)'; ws[f'{VC}6'] = '=VISTA_VIVA!$C$7'
    ws[f'{KC}7'] = 'x de leyenda (no se dibuja)'; ws[f'{VC}7'] = 0
    ws[f'{KC}8'] = 'y de leyenda (no se dibuja)'; ws[f'{VC}8'] = '=NA()'
    for r in range(2, FIN_B + 1):
        c = lambda n: f'{col[n]}{r}'
        vis = f'ISNUMBER(CALCULO!B{r})'
        mx, my, mz = c('mx'), c('my'), c('mz')
        f = {
            'bx1': f'=CALCULO!D{r}*{cA}-CALCULO!F{r}*{sA}',
            'bx2': f'=CALCULO!E{r}*{cA}-CALCULO!G{r}*{sA}',
            'bxm': f'={mx}*{cA}-{my}*{sA}',
            'bxm2': f'={c("bxm")}',
            'bxt': f'=({mx}+{largo}*{c("ex")})*{cA}-({my}+{largo}*{c("ey")})*{sA}',
            'by1': f'=IF({vis},CALCULO!H{r}*{cE}+(CALCULO!D{r}*{sA}+CALCULO!F{r}*{cA})*{sE},NA())',
            'by2': f'=IF(ISNUMBER(CALCULO!C{r}),CALCULO!I{r}*{cE}+(CALCULO!E{r}*{sA}+CALCULO!G{r}*{cA})*{sE},NA())',
            'bym': f'=IF({vis},{mz}*{cE}+({mx}*{sA}+{my}*{cA})*{sE},NA())',
            'bym2': f'={c("bym")}',
            'byt': f'=IF({vis},({mz}+{largo}*{c("ez")})*{cE}+(({mx}+{largo}*{c("ex")})*{sA}+({my}+{largo}*{c("ey")})*{cA})*{sE},NA())',
            'L': f'=IF({vis},SQRT((CALCULO!E{r}-CALCULO!D{r})^2+(CALCULO!G{r}-CALCULO!F{r})^2+(CALCULO!I{r}-CALCULO!H{r})^2),1)',
            'xu': f'=(CALCULO!E{r}-CALCULO!D{r})/{c("L")}',
            'yu': f'=(CALCULO!G{r}-CALCULO!F{r})/{c("L")}',
            'zu': f'=(CALCULO!I{r}-CALCULO!H{r})/{c("L")}',
            'nb': f'=MAX(SQRT({c("xu")}^2+{c("yu")}^2),0.000000001)',
            'nc': f'=MAX(SQRT({c("zu")}^2+{c("yu")}^2),0.000000001)',
            'hasU': f'=COUNT(BARRAS!F{r}:H{r})>0',
            'dot': f'=N(BARRAS!F{r})*{c("xu")}+N(BARRAS!G{r})*{c("yu")}+N(BARRAS!H{r})*{c("zu")}',
            'ux': f'=N(BARRAS!F{r})-{c("dot")}*{c("xu")}',
            'uy': f'=N(BARRAS!G{r})-{c("dot")}*{c("yu")}',
            'uz': f'=N(BARRAS!H{r})-{c("dot")}*{c("zu")}',
            'un': f'=MAX(SQRT({c("ux")}^2+{c("uy")}^2+{c("uz")}^2),0.000000001)',
            'ex': f'=IFERROR(IF({c("hasU")},{c("ux")}/{c("un")},IF(ABS({c("zu")})>0.9,0,-{c("xu")}*{c("zu")}/{c("nb")})),0)',
            'ey': f'=IFERROR(IF({c("hasU")},{c("uy")}/{c("un")},IF(ABS({c("zu")})>0.9,-{c("zu")}/{c("nc")},-{c("yu")}*{c("zu")}/{c("nb")})),0)',
            'ez': f'=IFERROR(IF({c("hasU")},{c("uz")}/{c("un")},IF(ABS({c("zu")})>0.9,{c("yu")}/{c("nc")},({c("xu")}^2+{c("yu")}^2)/{c("nb")})),0)',
            'mx': f'=(CALCULO!D{r}+CALCULO!E{r})/2',
            'my': f'=(CALCULO!F{r}+CALCULO!G{r})/2',
            'mz': f'=(CALCULO!H{r}+CALCULO!I{r})/2',
            'rotI': f'=IF(AND({vis},LOWER(TRIM(BARRAS!I{r}))="rotulado"),{c("by1")},NA())',
            'rotJ': f'=IF(AND(ISNUMBER(CALCULO!C{r}),LOWER(TRIM(BARRAS!J{r}))="rotulado"),{c("by2")},NA())',
            'lblB': f'=IF(BARRAS!A{r}="","","B"&BARRAS!A{r})',
            'rot': f'=IF(OR(LOWER(TRIM(BARRAS!I{r}))="rotulado",LOWER(TRIM(BARRAS!J{r}))="rotulado"),1,0)',
            'nA': f'=IF(AND({vis},{c("rot")}=0),{c("by1")},NA())',
            'nM': f'=IF(AND({vis},{c("rot")}=0),{c("bym")},NA())',
            'nB': f'=IF(AND(ISNUMBER(CALCULO!C{r}),{c("rot")}=0),{c("by2")},NA())',
            'rA': f'=IF(AND({vis},{c("rot")}=1),{c("by1")},NA())',
            'rM': f'=IF(AND({vis},{c("rot")}=1),{c("bym")},NA())',
            'rB': f'=IF(AND(ISNUMBER(CALCULO!C{r}),{c("rot")}=1),{c("by2")},NA())',
        }
        for nombre, formula in f.items():
            ws[c(nombre)] = formula
    for r in range(2, FIN_N + 1):
        ws[f"{col['nsx']}{r}"] = f'=NODOS!B{r}*{cA}-NODOS!C{r}*{sA}'
        ws[f"{col['nsy']}{r}"] = f'=IF(NODOS!A{r}<>"",NODOS!D{r}*{cE}+(NODOS!B{r}*{sA}+NODOS!C{r}*{cA})*{sE},NA())'
        cuenta = f'COUNTIF(NODOS!$E{r}:$J{r},"SI")'          # grados restringidos del nodo (0 a 6)
        condiciones = {'apE': f'{cuenta}=6', 'apR': f'AND({cuenta}>0,{cuenta}<6)'}   # empotrado; parcial o rotulado
        for nombre, cond in condiciones.items():
            ws[f"{col[nombre]}{r}"] = (f'=IF(AND(NODOS!A{r}<>"",{cond}),'
                                       f'NODOS!D{r}*{cE}+(NODOS!B{r}*{sA}+NODOS!C{r}*{cA})*{sE},NA())')
        ws[f"{col['lblN']}{r}"] = f'=IF(NODOS!A{r}="","","N"&NODOS!A{r})'
    return ws, col


def _grafico_3d(ws_v, ws3, col):
    """Vista 3D esquematica en vivo. Cada barra y cada nodo es una serie cuyo nombre sale de una celda
    (B14, N3): asi las etiquetas funcionan en todas las versiones de Excel. Barra gris; flecha negra (ultimo
    tramo de la serie); extremo rotulado = circulo blanco; apoyo empotrado = cuadrado; rotulado = rombo.
    Limite de Excel: 255 series por grafico; aqui hay 214."""
    from openpyxl.chart.legend import Legend, LegendEntry
    from openpyxl.chart.series import SeriesLabel
    from openpyxl.chart.data_source import StrRef
    from openpyxl.chart.label import DataLabel, DataLabelList
    from openpyxl.chart.marker import DataPoint
    from openpyxl.chart.shapes import GraphicalProperties
    from openpyxl.drawing.line import LineProperties

    ch = ScatterChart()
    ch.title = '3D esquematico (C5 azimut, C6 elevacion)'
    ch.style = 13
    ch.height, ch.width = 13, 17
    ch.x_axis.title, ch.y_axis.title = ('horizontal', 'vertical')
    ch.x_axis.delete = False; ch.y_axis.delete = False

    def ci(nombre):
        return _CI(col[nombre])

    def ref_nombre(nombre, r):
        return f"'CALCULO_3D'!${col[nombre]}${r}"

    def etiqueta(idx):
        return DataLabelList(
            dLbl=[DataLabel(idx=idx, showSerName=True, showVal=False, showCatName=False,
                            showLegendKey=False, showPercent=False, showBubbleSize=False, dLblPos='t')],
            showSerName=False, showVal=False, showCatName=False, showLegendKey=False,
            showPercent=False, showBubbleSize=False)

    for r in range(2, FIN_B + 1):
        s = Series(Reference(ws3, min_col=ci('by1'), max_col=ci('byt'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci('bx1'), max_col=ci('bxt'), min_row=r, max_row=r))
        s.tx = SeriesLabel(strRef=StrRef(ref_nombre('lblB', r)))
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = '7F7F7F'
        s.graphicalProperties.line.width = 28000
        s.smooth = False
        s.smooth = False
        s.dPt = [DataPoint(idx=4, spPr=GraphicalProperties(ln=LineProperties(solidFill='000000', w=15000)))]
        s.dLbls = etiqueta(1)
        ch.series.append(s)
    for nombre_x, nombre_y in [('bx1', 'rotI'), ('bx2', 'rotJ')]:
        s = Series(Reference(ws3, min_col=ci(nombre_y), min_row=2, max_row=FIN_B),
                   Reference(ws3, min_col=ci(nombre_x), min_row=2, max_row=FIN_B), title='Extremo rotulado')
        s.marker.symbol = 'circle'; s.marker.size = 8
        s.marker.graphicalProperties.solidFill = 'FFFFFF'
        s.marker.graphicalProperties.line.solidFill = '000000'
        s.graphicalProperties.line.noFill = True
        ch.series.append(s)
    for r in range(2, FIN_N + 1):
        n = Series(Reference(ws3, min_col=ci('nsy'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci('nsx'), min_row=r, max_row=r))
        n.tx = SeriesLabel(strRef=StrRef(ref_nombre('lblN', r)))
        n.marker.symbol = 'circle'; n.marker.size = 4
        n.marker.graphicalProperties.solidFill = '000000'
        n.graphicalProperties.line.noFill = True
        n.smooth = False
        n.dLbls = etiqueta(0)
        ch.series.append(n)
    for nombre_y, simbolo, etiq in [('apE', 'square', 'Apoyo empotrado'), ('apR', 'diamond', 'Apoyo rotulado')]:
        a = Series(Reference(ws3, min_col=ci(nombre_y), min_row=2, max_row=FIN_N),
                   Reference(ws3, min_col=ci('nsx'), min_row=2, max_row=FIN_N), title=etiq)
        a.marker.symbol = simbolo; a.marker.size = 9
        a.marker.graphicalProperties.solidFill = 'E67E22'
        a.graphicalProperties.line.noFill = True
        ch.series.append(a)
    # series de leyenda: no dibujan nada (y = #N/A); solo muestran el simbolo de cada tipo
    x_leyenda = f"{col['_VC']}7"
    y_leyenda = f"{col['_VC']}8"
    leyenda = [('Barra', 'linea', '7F7F7F'), ('Eje fuerte (flecha)', 'linea', '000000'),
               ('Extremo rotulado (circulo blanco)', 'circulo', 'FFFFFF'), ('Nodo (N)', 'circulo', '000000'),
               ('Apoyo empotrado (cuadrado)', 'square', 'E67E22'), ('Apoyo rotulado (rombo)', 'diamond', 'E67E22')]
    primera_leyenda = len(ch.series)
    for etiq, tipo, color in leyenda:
        s = Series(Reference(ws3, min_col=_CI(col['_VC']), min_row=8, max_row=8),
                   Reference(ws3, min_col=_CI(col['_VC']), min_row=7, max_row=7), title=etiq)
        if tipo == 'linea':
            s.marker.symbol = 'none'
            s.graphicalProperties.line.solidFill = color
            s.graphicalProperties.line.width = 28000 if color == '7F7F7F' else 15000
        else:
            s.marker.symbol = {'circulo': 'circle', 'square': 'square', 'diamond': 'diamond'}[tipo]
            s.marker.size = 8 if tipo != 'circulo' or color == 'FFFFFF' else 5
            s.marker.graphicalProperties.solidFill = color
            s.marker.graphicalProperties.line.solidFill = '000000'
            s.graphicalProperties.line.noFill = True
        ch.series.append(s)
    for serie in ch.series:
        serie.smooth = False          # lineas rectas: sin suavizado las flechas no se curvan
    assert len(ch.series) <= 255, f'grafico 3D con {len(ch.series)} series (limite de Excel: 255)'
    ch.legend = Legend()
    ch.legend.position = 'b'
    ch.legend.legendEntry = [LegendEntry(idx=i, delete=True) for i in range(primera_leyenda)]
    ws_v.add_chart(ch, 'A31')


def _grafico_3d_rotulas(ws_v, ws3, col):
    """3D con rotulas de los elementos (sin IDs): barras normales (gris), barras con extremo rotulado (morada
    discontinua) y circulo blanco en cada extremo rotulado. Usa la misma proyeccion que la 3D principal."""
    from openpyxl.chart.legend import Legend, LegendEntry
    ch = ScatterChart()
    ch.title = '3D con rotulas de los elementos'
    ch.style = 13
    ch.height, ch.width = 13, 17
    ch.x_axis.title, ch.y_axis.title = ('horizontal', 'vertical')
    ch.x_axis.delete = False; ch.y_axis.delete = False

    def ci(nombre):
        return _CI(col[nombre])

    for r in range(2, FIN_B + 1):
        s = Series(Reference(ws3, min_col=ci('nA'), max_col=ci('nB'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci('bx1'), max_col=ci('bx2'), min_row=r, max_row=r), title='Barra')
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = '7F7F7F'
        s.graphicalProperties.line.width = 28000
        s.smooth = False
        ch.series.append(s)
    for r in range(2, FIN_B + 1):
        s = Series(Reference(ws3, min_col=ci('rA'), max_col=ci('rB'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci('bx1'), max_col=ci('bx2'), min_row=r, max_row=r),
                   title='Barra con extremo rotulado')
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = '6A1B9A'
        s.graphicalProperties.line.width = 28000
        s.graphicalProperties.line.prstDash = 'dash'
        s.smooth = False
        ch.series.append(s)
    for nombre_x, nombre_y in [('bx1', 'rotI'), ('bx2', 'rotJ')]:
        s = Series(Reference(ws3, min_col=ci(nombre_y), min_row=2, max_row=FIN_B),
                   Reference(ws3, min_col=ci(nombre_x), min_row=2, max_row=FIN_B), title='Extremo rotulado')
        s.marker.symbol = 'circle'; s.marker.size = 8
        s.marker.graphicalProperties.solidFill = '6A1B9A'
        s.marker.graphicalProperties.line.solidFill = '6A1B9A'
        s.graphicalProperties.line.noFill = True
        s.smooth = False
        ch.series.append(s)
    n = Series(Reference(ws3, min_col=ci('nsy'), min_row=2, max_row=FIN_N),
               Reference(ws3, min_col=ci('nsx'), min_row=2, max_row=FIN_N), title='Nodo')
    n.marker.symbol = 'circle'; n.marker.size = 4
    n.marker.graphicalProperties.solidFill = '000000'
    n.graphicalProperties.line.noFill = True
    n.smooth = False
    ch.series.append(n)
    for nombre_y, simbolo, etiq in [('apE', 'square', 'Apoyo empotrado (cuadrado)'),
                                    ('apR', 'diamond', 'Apoyo rotulado (rombo)')]:
        a = Series(Reference(ws3, min_col=ci(nombre_y), min_row=2, max_row=FIN_N),
                   Reference(ws3, min_col=ci('nsx'), min_row=2, max_row=FIN_N), title=etiq)
        a.marker.symbol = simbolo; a.marker.size = 9
        a.marker.graphicalProperties.solidFill = 'E67E22'
        a.graphicalProperties.line.noFill = True
        a.smooth = False
        ch.series.append(a)
    primera_leyenda = len(ch.series)
    for etiq, tipo, color in [('Barra', 'linea', '7F7F7F'), ('Barra con extremo rotulado', 'linea', '6A1B9A'),
                              ('Extremo rotulado (circulo morado)', 'circulo', '6A1B9A'),
                              ('Nodo', 'circulo', '000000'), ('Apoyo empotrado (cuadrado)', 'square', 'E67E22'),
                              ('Apoyo rotulado (rombo)', 'diamond', 'E67E22')]:
        s = Series(Reference(ws3, min_col=_CI(col['_VC']), min_row=8, max_row=8),
                   Reference(ws3, min_col=_CI(col['_VC']), min_row=7, max_row=7), title=etiq)
        if tipo == 'linea':
            s.marker.symbol = 'none'
            s.graphicalProperties.line.solidFill = color
            s.graphicalProperties.line.width = 28000
        else:
            s.marker.symbol = {'circulo': 'circle', 'square': 'square', 'diamond': 'diamond'}[tipo]
            s.marker.size = 7
            s.marker.graphicalProperties.solidFill = color
            s.marker.graphicalProperties.line.solidFill = '000000'
            s.graphicalProperties.line.noFill = True
        ch.series.append(s)
    for serie in ch.series:
        serie.smooth = False
    assert len(ch.series) <= 255, f'grafico 3D con rotulas: {len(ch.series)} series'
    ch.legend = Legend()
    ch.legend.position = 'b'
    ch.legend.legendEntry = [LegendEntry(idx=i, delete=True) for i in range(primera_leyenda)]
    ws_v.add_chart(ch, 'A60')


def _grafico_lateral(ws_v, ws_c):
    """Alzado lateral Y-Z (plano X = cota X de C8): barras (gris), nodos, apoyos y extremos rotulados."""
    ch = ScatterChart()
    ch.title = 'Alzado Y-Z (cota X de C8)'
    ch.style = 13
    ch.legend = None
    ch.height, ch.width = 10, 16
    ch.x_axis.title, ch.y_axis.title = ('Y (m)', 'Z (m)')
    ch.x_axis.delete = False; ch.y_axis.delete = False
    for r in range(2, FIN_B + 1):
        s_ = Series(Reference(ws_c, min_col=_CI('AF'), max_col=_CI('AG'), min_row=r, max_row=r),
                    Reference(ws_c, min_col=_CI('F'), max_col=_CI('G'), min_row=r, max_row=r))
        s_.marker.symbol = 'none'
        s_.graphicalProperties.line.solidFill = '000000'
        s_.graphicalProperties.line.width = 22000
        s_.smooth = False
        ch.series.append(s_)
    n = Series(Reference(ws_c, min_col=_CI('AI'), min_row=2, max_row=FIN_N),
               Reference(ws_c, min_col=_CI('AH'), min_row=2, max_row=FIN_N), title='Nodos')
    n.marker.symbol = 'circle'; n.marker.size = 5
    n.marker.graphicalProperties.solidFill = '000000'
    n.graphicalProperties.line.noFill = True
    n.smooth = False
    ch.series.append(n)
    a = Series(Reference(ws_c, min_col=_CI('AJ'), min_row=2, max_row=FIN_N),
               Reference(ws_c, min_col=_CI('AH'), min_row=2, max_row=FIN_N), title='Apoyos')
    a.marker.symbol = 'square'; a.marker.size = 9
    a.marker.graphicalProperties.solidFill = 'E67E22'
    a.graphicalProperties.line.noFill = True
    a.smooth = False
    ch.series.append(a)
    for xcol, ycol in [('F', 'AK'), ('G', 'AL')]:
        s_ = Series(Reference(ws_c, min_col=_CI(ycol), min_row=2, max_row=FIN_B),
                    Reference(ws_c, min_col=_CI(xcol), min_row=2, max_row=FIN_B), title='Rotula')
        s_.marker.symbol = 'circle'; s_.marker.size = 8
        s_.marker.graphicalProperties.solidFill = '6A1B9A'
        s_.marker.graphicalProperties.line.solidFill = '6A1B9A'
        s_.graphicalProperties.line.noFill = True
        s_.smooth = False
        ch.series.append(s_)
    assert len(ch.series) <= 255
    ws_v.add_chart(ch, 'K31')


def _graficos(ws_v, ws_c):
    """Planta (X-Y) y alzado (X-Z) en VISTA_VIVA: barras, nodos y apoyos vinculados a las celdas."""
    def grafico(titulo, ejes, x_nodo, y_nodo, x_apoyo, y_apoyo, rango_barras, etiquetas):
        ch = ScatterChart()
        ch.title = titulo
        ch.style = 13
        ch.legend = None
        ch.height, ch.width = 10, 16
        ch.x_axis.title, ch.y_axis.title = etiquetas
        ch.x_axis.delete = False; ch.y_axis.delete = False
        for r in range(2, FIN_B + 1):
            s = Series(Reference(ws_c, min_col=rango_barras[1], max_col=rango_barras[1] + 1, min_row=r, max_row=r),
                       Reference(ws_c, min_col=rango_barras[0], max_col=rango_barras[0] + 1, min_row=r, max_row=r))
            s.marker.symbol = 'none'
            s.graphicalProperties.line.solidFill = '555555'
            s.graphicalProperties.line.width = 22000
            s.smooth = False
            ch.series.append(s)
        n = Series(Reference(ws_c, min_col=y_nodo, min_row=2, max_row=FIN_N),
                   Reference(ws_c, min_col=x_nodo, min_row=2, max_row=FIN_N), title='Nodos')
        n.marker.symbol = 'circle'; n.marker.size = 5
        n.marker.graphicalProperties.solidFill = '000000'
        n.graphicalProperties.line.noFill = True
        ch.series.append(n)
        a = Series(Reference(ws_c, min_col=y_apoyo, min_row=2, max_row=FIN_N),
                   Reference(ws_c, min_col=x_apoyo, min_row=2, max_row=FIN_N), title='Apoyos')
        a.marker.symbol = 'square'; a.marker.size = 9
        a.marker.graphicalProperties.solidFill = 'E67E22'
        a.graphicalProperties.line.noFill = True
        ch.series.append(a)
        return ch

    # planta: barras X1:X2 (J:K) y Y1:Y2 (L:M); nodos S (x) y T (y); apoyos W, X
    p = grafico('Planta (nivel Z de C3)', None, 19, 20, 23, 24, (10, 12),
                ('X (m)', 'Y (m)'))
    # alzado: barras X1:X2 (N:O) y Z1:Z2 (P:Q); nodos U (x) y V (z); apoyos Y, Z
    z = grafico('Alzado X-Z (cota Y de C4)', None, 21, 22, 25, 26, (14, 16),
                ('X (m)', 'Z (m)'))
    for ch_obj, ycols in [(p, ('AB', 'AC')), (z, ('AD', 'AE'))]:
        for xcol, ycol in [('D', ycols[0]), ('E', ycols[1])]:
            s_ = Series(Reference(ws_c, min_col=_CI(ycol), min_row=2, max_row=FIN_B),
                        Reference(ws_c, min_col=_CI(xcol), min_row=2, max_row=FIN_B), title='Rotula')
            s_.marker.symbol = 'circle'; s_.marker.size = 8
            s_.marker.graphicalProperties.solidFill = '6A1B9A'
            s_.marker.graphicalProperties.line.solidFill = '6A1B9A'
            s_.graphicalProperties.line.noFill = True
            s_.smooth = False
            ch_obj.series.append(s_)
    ws_v.add_chart(p, 'A10')
    ws_v.add_chart(z, 'J10')


# ---------------- CARGAS EN NODOS ----------------
CARGAS_COLS = ['ID NODO', 'ESTADO', 'FX (kgf)', 'FY (kgf)', 'FZ (kgf)', 'MX (kgf·m)', 'MY (kgf·m)', 'MZ (kgf·m)', 'NOTA']
CARGA_FILA0 = 9                       # primera fila de datos de la hoja CARGAS
CARGA_MAX = 300
CARGA_FIN = CARGA_FILA0 + CARGA_MAX - 1
ESTADOS_MAX = 30                      # filas 2..31 de la hoja ESTADOS
ESTADOS_FIN = ESTADOS_MAX + 1


def ejemplo_estados():
    """Estados de carga de ejemplo (el usuario los nombra y agrega en la hoja ESTADOS)."""
    return [['Peso propio y muerta', 'D', 'Peso propio, tabiqueria y cubierta', 'SI'],
            ['Sobrecarga', 'L', 'Sobrecarga de uso en el nivel superior', 'NO'],
            ['Viento X', 'W', 'Viento en direccion X sobre fachada izquierda', 'NO']]


def ejemplo_cargas():
    """Ejemplo: peso propio sobre los 6 nudos del nivel superior y viento X sobre 2 de ellos."""
    filas = [[id_, 'Peso propio y muerta', 0, 0, -1500, 0, 0, 0, None] for id_ in (3, 6, 9, 12, 15, 18)]
    filas += [[id_, 'Viento X', 800, 0, 0, 0, 0, 0, None] for id_ in (3, 6)]
    return filas


def _hoja_estados(wb, estados):
    """Hoja ESTADOS: lista de estados de carga que define el usuario (nombre, tipo y descripcion)."""
    ws = wb.create_sheet('ESTADOS')
    _encabezado(ws, ['NOMBRE DEL ESTADO', 'TIPO (D, L, W, E u otro)', 'DESCRIPCION', 'PESO PROPIO (SI/NO)'],
                [34, 22, 60, 20])
    for f in estados:
        ws.append(list(f) + [None] * (4 - len(f)))
    for _ in range(ESTADOS_MAX - len(estados)):
        ws.append([None] * 4)
    dv_pp = DataValidation(type='list', formula1='"SI,NO"', allow_blank=True)
    ws.add_data_validation(dv_pp); dv_pp.add(f'D2:D{ESTADOS_FIN}')
    return ws


def _hoja_calc_cargas(wb, col3d):
    """Calculo de la vista 3D de cargas: suma de cargas por nodo para el estado elegido en CARGAS!C3, y
    posicion en pantalla (misma proyeccion que CALCULO_3D) de cada nodo y de la punta de su flecha."""
    ws = wb.create_sheet('CALCULO_CARGAS')
    cols = ['id', 'fx', 'fy', 'fz', 'mx', 'my', 'mz', 'any', 'mag', 'tx3', 'ty3', 'tz3',
            'ax1', 'ax2', 'ay1', 'ay2', 'lbl']
    col = {nombre: _L(k) for k, nombre in enumerate(cols, start=1)}
    constantes = ['fmax', 'escala', 'dx', 'dy']
    KC = _L(len(cols) + 2)
    VC = _L(len(cols) + 3)
    for k, nombre in enumerate(constantes, start=2):
        col[nombre] = f'${VC}${k}'
    _encabezado(ws, [n for n in cols] + [''] + ['constante', 'valor'] + [''] * 0,
                [7] * 6 + [9] * 11 + [3, 14, 10], relleno=CABECERA_CALC)
    cA = f"CALCULO_3D!${col3d['_VC']}$2"; sA = f"CALCULO_3D!${col3d['_VC']}$3"
    cE = f"CALCULO_3D!${col3d['_VC']}$4"; sE = f"CALCULO_3D!${col3d['_VC']}$5"
    rango = lambda c: f"CARGAS!${c}${CARGA_FILA0}:${c}${CARGA_FIN}"
    ws[f'{KC}2'] = 'fuerza maxima (tf)'; ws[f'{VC}2'] = f'=MAX({col["mag"]}2:{col["mag"]}{FIN_N})'
    ws[f'{KC}3'] = 'escala flecha (m/tf)'; ws[f'{VC}3'] = f'=IF({col["fmax"]}>0,CARGAS!$C$4/{col["fmax"]},0)'
    ws[f'{KC}4'] = 'x de leyenda'; ws[f'{VC}4'] = 0
    ws[f'{KC}5'] = 'y de leyenda'; ws[f'{VC}5'] = '=NA()'
    for r in range(2, FIN_N + 1):
        c = lambda n: f'{col[n]}{r}'
        ws[c('id')] = f'=IF(NODOS!A{r}="","",NODOS!A{r})'
        for nombre, letra in [('fx', 'C'), ('fy', 'D'), ('fz', 'E'), ('mx', 'F'), ('my', 'G'), ('mz', 'H')]:
            ws[c(nombre)] = (f'=IF({c("id")}="",0,SUMIFS({rango(letra)},{rango("A")},{c("id")},'
                             f'{rango("B")},CARGAS!$C$3))')
        ws[c('any')] = (f'=IF(OR({c("fx")}<>0,{c("fy")}<>0,{c("fz")}<>0,{c("mx")}<>0,{c("my")}<>0,'
                        f'{c("mz")}<>0),1,0)')
        ws[c('mag')] = f'=SQRT({c("fx")}^2+{c("fy")}^2+{c("fz")}^2)'
        ws[c('tx3')] = f'=NODOS!B{r}+{col["escala"]}*{c("fx")}'
        ws[c('ty3')] = f'=NODOS!C{r}+{col["escala"]}*{c("fy")}'
        ws[c('tz3')] = f'=NODOS!D{r}+{col["escala"]}*{c("fz")}'
        ws[c('ax1')] = f'=CALCULO_3D!{col3d["nsx"]}{r}'
        ws[c('ax2')] = f'={c("tx3")}*{cA}-{c("ty3")}*{sA}'
        ws[c('ay1')] = f'=IF({c("any")}=1,CALCULO_3D!{col3d["nsy"]}{r},NA())'
        ws[c('ay2')] = (f'=IF(AND({c("any")}=1,{c("mag")}>0),{c("tz3")}*{cE}+({c("tx3")}*{sA}+{c("ty3")}*{cA})*{sE},'
                        f'NA())')
        texto = '&'.join(f'IF({c(n)}<>0,"{e} "&ROUND({c(n)},2)&" ","")'
                        for n, e in [('fx', 'Fx'), ('fy', 'Fy'), ('fz', 'Fz'), ('mx', 'Mx'), ('my', 'My'), ('mz', 'Mz')])
        ws[c('lbl')] = f'=IF({c("any")}=1,"N"&{c("id")}&": "&{texto},"")'
    return ws, col


# ---------------- LISTAS DESPLEGABLES POR GRUPO DE ELEMENTO (perfiles que cumplen el diseno) ----------------
GRUPOS_LISTA = ['COLUMNA', 'VIGA', 'ARRIOSTRE', 'DIAGONAL']


def _hoja_listas_grupo(wb, ws_l, listas=None):
    """Columnas F..I de LISTAS: perfiles por grupo de elemento, con nombres definidos LISTA_<GRUPO>. Sin diseno
    ejecutado (listas None) se listan todos los perfiles; si ninguno cumple, se indica."""
    from openpyxl.workbook.defined_name import DefinedName
    from openpyxl.utils import get_column_letter
    for j, g in enumerate(GRUPOS_LISTA):
        col = 6 + j
        letra = get_column_letter(col)
        if listas is None or g not in listas:
            nombres = [n for _, n in PERFILES_ORDEN]
        elif listas[g] is None:
            nombres = ['(sin barras de este grupo)']
        else:
            nombres = listas[g] or ['(ningun perfil cumple: revise el diseno)']
        cab = ws_l.cell(row=1, column=col, value=f'PERFILES {g}')
        cab.font = NEGRITA
        cab.fill = ENCABEZADO
        for k, n in enumerate(nombres):
            ws_l.cell(row=k + 2, column=col, value=n)
        ws_l.column_dimensions[letra].width = 24
        wb.defined_names[f'LISTA_{g}'] = DefinedName(f'LISTA_{g}',
                                                     attr_text=f"LISTAS!${letra}$2:${letra}${len(nombres) + 1}")


# ---------------- LONGITUDES POR BARRA (K y longitudes no arriostradas, como en RAM Element) ----------------
LONG_COLS = ['ID', 'LX (m)', 'LY (m)', 'KX', 'KY', 'LB (m)', 'CB',
             'E5 UNA PATA', 'E5 UNION SOLDADA O 2 PERNOS', 'E5 SIN CARGA INTERMEDIA',
             'E5 INDIVIDUAL O ALMA DE ARMADURA', 'LI CONEXION', 'LI LONGITUD CONEXION (m)', 'NOTA']
LONG_FIN = BARRAS_MAX + 1          # filas 2..107, alineadas fila a fila con BARRAS


def ejemplo_longitudes():
    """Ejemplo: LX, LY y LB = longitud de cada barra (m) desde la geometria del marco; KX = KY = 1.0 y CB = 1.0.
    Son datos obligatorios para el diseno: cambielos por los de su modelo (LX, LY y LB no se suponen)."""
    ej = DibujoModelo(ruta='ejemplo')
    ej.marco_grilla([0, 5, 10], [0, 4], [0, 3.5, 7], 'IR 10 × 49', 'IR 12 × 16', 'IR 12 × 16')
    filas = []
    for b in ej.barras:
        p1, p2 = ej.nodos[b['n1']], ej.nodos[b['n2']]
        L = round(sum((q - p) ** 2 for p, q in zip(p1, p2)) ** 0.5, 4)
        filas.append([L, L, 1.0, 1.0, L, 1.0] + [None] * 7)     # LX, LY, KX, KY, LB, CB, E5 x4, LI x2, NOTA
    return filas


def _hoja_longitudes(wb, filas=None):
    """Hoja LONGITUDES: por barra, longitud no arriostrada en X (LX) y en Y (LY), factores K, LB (pandeo lateral
    torsional) y CB. El ID sale de BARRAS. Vacio = valor supuesto por el diseño (se marca en la salida)."""
    ws = wb.create_sheet('LONGITUDES')
    _encabezado(ws, LONG_COLS, [8, 11, 11, 9, 9, 11, 9, 13, 14, 14, 14, 12, 14, 44])
    filas = filas or []
    for k in range(BARRAS_MAX):
        r = k + 2
        fila = filas[k] if k < len(filas) else [None] * (len(LONG_COLS) - 1)
        ws.cell(row=r, column=1, value=f'=IF(BARRAS!$A{r}="","",BARRAS!$A{r})')
        for c, v in enumerate(fila[:len(LONG_COLS) - 1], start=2):
            ws.cell(row=r, column=c, value=v)
    dv = DataValidation(type='decimal', operator='greaterThan', formula1='0', allow_blank=True)
    ws.add_data_validation(dv); dv.add(f'B2:G{LONG_FIN}'); dv.add(f'M2:M{LONG_FIN}')
    dv_si = DataValidation(type='list', formula1='"SI,NO"', allow_blank=True)
    ws.add_data_validation(dv_si); dv_si.add(f'H2:K{LONG_FIN}')
    dv_con = DataValidation(type='list', formula1='"UNA PATA,DOS PATAS"', allow_blank=True)
    ws.add_data_validation(dv_con); dv_con.add(f'L2:L{LONG_FIN}')
    return ws


def _hoja_cargas(wb, filas_cargas, estado_inicial=''):
    """Hoja CARGAS: estado de carga a mostrar (C3), largo de la flecha (C4) y la tabla de cargas por nodo."""
    ws = wb.create_sheet('CARGAS')
    ws['A1'] = 'Cargas aplicadas en los nodos (kgf, kgf·m)'
    ws['A1'].font = Font(bold=True, size=12)
    ws['A3'] = 'Estado de carga a mostrar en el 3D'
    ws['C3'] = estado_inicial
    ws['A4'] = 'Largo maximo de la flecha de carga (m)'
    ws['C4'] = 1.5
    ws['A5'] = 'Azimut y elevacion de la vista: se cambian en VISTA_VIVA (C5 y C6).'
    ws['A6'] = 'Unidades: fuerzas en kgf y momentos en kgf·m. Cada fila es un nodo con un estado de carga (hoja ESTADOS).'
    for celda in ('C3', 'C4'):
        ws[celda].fill = ENTRADA
    for k, nombre in enumerate(CARGAS_COLS, start=1):
        celda = ws.cell(row=CARGA_FILA0 - 1, column=k, value=nombre)
        celda.font = NEGRITA; celda.fill = ENCABEZADO
        celda.alignment = Alignment(horizontal='center', wrap_text=True)
    for k, ancho in enumerate([10, 28, 11, 11, 11, 12, 12, 12, 30], start=1):
        ws.column_dimensions[_L(k)].width = ancho
    for i, f in enumerate(filas_cargas):
        for k, v in enumerate(f, start=1):
            ws.cell(row=CARGA_FILA0 + i, column=k, value=v)
    dv_nodo = DataValidation(type='list', formula1=f'NODOS!$A$2:$A${FIN_N}', allow_blank=True)
    dv_estado = DataValidation(type='list', formula1=f'ESTADOS!$A$2:$A${ESTADOS_FIN}', allow_blank=True)
    dv_num = DataValidation(type='decimal', operator='between', formula1='-1000000', formula2='1000000',
                            allow_blank=True)
    for dv, rango_ in [(dv_nodo, f'A{CARGA_FILA0}:A{CARGA_FIN}'), (dv_estado, f'B{CARGA_FILA0}:B{CARGA_FIN}'),
                       (dv_num, f'C{CARGA_FILA0}:H{CARGA_FIN}')]:
        ws.add_data_validation(dv); dv.add(rango_)
    dv_caso = DataValidation(type='list', formula1=f'ESTADOS!$A$2:$A${ESTADOS_FIN}', allow_blank=True)
    ws.add_data_validation(dv_caso); dv_caso.add('C3')
    ws.freeze_panes = f'A{CARGA_FILA0}'
    return ws


def _grafico_cargas(ws_car, ws_cc, col_cc, ws3, col3d):
    """3D de cargas: barras (gris), nodos (negro), cada nodo con su etiqueta (N y carga) y su flecha de carga
    (roja, punta triangular). Apoyos: cuadrado (empotrado) y rombo (rotulado). Sin IDs de barras."""
    from openpyxl.chart.legend import Legend, LegendEntry
    from openpyxl.chart.series import SeriesLabel
    from openpyxl.chart.data_source import StrRef
    from openpyxl.chart.label import DataLabel, DataLabelList
    from openpyxl.chart.marker import DataPoint, Marker
    from openpyxl.chart.shapes import GraphicalProperties
    from openpyxl.drawing.line import LineProperties

    ch = ScatterChart()
    ch.title = '3D de cargas por nodo (estado en CARGAS!C3)'
    ch.style = 13
    ch.height, ch.width = 14, 18
    ch.x_axis.title, ch.y_axis.title = ('horizontal', 'vertical')
    ch.x_axis.delete = False; ch.y_axis.delete = False

    def ci3(n):
        return _CI(col3d[n])

    def cci(n):
        return _CI(col_cc[n])

    def etiqueta(idx):
        return DataLabelList(
            dLbl=[DataLabel(idx=idx, showSerName=True, showVal=False, showCatName=False, showLegendKey=False,
                            showPercent=False, showBubbleSize=False, dLblPos='t')],
            showSerName=False, showVal=False, showCatName=False, showLegendKey=False,
            showPercent=False, showBubbleSize=False)

    for r in range(2, FIN_B + 1):                     # barras: 3 puntos (inicio, medio, fin), sin etiqueta
        s = Series(Reference(ws3, min_col=ci3('by1'), max_col=ci3('by2'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci3('bx1'), max_col=ci3('bx2'), min_row=r, max_row=r))
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = '7F7F7F'
        s.graphicalProperties.line.width = 28000
        s.smooth = False
        ch.series.append(s)
    for r in range(2, FIN_N + 1):                     # una serie por nodo: su nombre es la etiqueta N + carga
        s = Series(Reference(ws_cc, min_col=cci('ay1'), max_col=cci('ay2'), min_row=r, max_row=r),
                   Reference(ws_cc, min_col=cci('ax1'), max_col=cci('ax2'), min_row=r, max_row=r))
        s.tx = SeriesLabel(strRef=StrRef(f"'CALCULO_CARGAS'!${col_cc['lbl']}${r}"))
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = 'C62828'
        s.graphicalProperties.line.width = 28000
        s.smooth = False
        s.dPt = [DataPoint(idx=1, marker=Marker(symbol='triangle', size=8),
                           spPr=GraphicalProperties(solidFill='C62828', ln=LineProperties(solidFill='C62828')))]
        s.dLbls = etiqueta(0)
        ch.series.append(s)
    n = Series(Reference(ws3, min_col=ci3('nsy'), min_row=2, max_row=FIN_N),
               Reference(ws3, min_col=ci3('nsx'), min_row=2, max_row=FIN_N), title='Nodo (N)')
    n.marker.symbol = 'circle'; n.marker.size = 4
    n.marker.graphicalProperties.solidFill = '000000'
    n.graphicalProperties.line.noFill = True
    n.smooth = False
    ch.series.append(n)
    for nombre_y, simbolo, etiq in [('apE', 'square', 'Apoyo empotrado (cuadrado)'),
                                    ('apR', 'diamond', 'Apoyo rotulado (rombo)')]:
        a = Series(Reference(ws3, min_col=ci3(nombre_y), min_row=2, max_row=FIN_N),
                   Reference(ws3, min_col=ci3('nsx'), min_row=2, max_row=FIN_N), title=etiq)
        a.marker.symbol = simbolo; a.marker.size = 9
        a.marker.graphicalProperties.solidFill = 'E67E22'
        a.graphicalProperties.line.noFill = True
        a.smooth = False
        ch.series.append(a)
    primera_leyenda = len(ch.series)
    x_leg = Reference(ws_cc, min_col=_CI(col_cc['dx'].split('$')[1]), min_row=4, max_row=4)
    y_leg = Reference(ws_cc, min_col=_CI(col_cc['dy'].split('$')[1]), min_row=5, max_row=5)
    for etiq, tipo, color in [('Barra', 'linea', '7F7F7F'), ('Carga aplicada (flecha roja)', 'linea', 'C62828'),
                              ('Nodo (N)', 'circulo', '000000'), ('Apoyo empotrado (cuadrado)', 'square', 'E67E22'),
                              ('Apoyo rotulado (rombo)', 'diamond', 'E67E22')]:
        s = Series(y_leg, x_leg, title=etiq)
        if tipo == 'linea':
            s.marker.symbol = 'none'
            s.graphicalProperties.line.solidFill = color
            s.graphicalProperties.line.width = 28000
        else:
            s.marker.symbol = {'circulo': 'circle', 'square': 'square', 'diamond': 'diamond'}[tipo]
            s.marker.size = 7
            s.marker.graphicalProperties.solidFill = color
            s.graphicalProperties.line.noFill = True
        ch.series.append(s)
    for serie in ch.series:
        serie.smooth = False
    assert len(ch.series) <= 255, f'grafico de cargas con {len(ch.series)} series'
    ch.legend = Legend()
    ch.legend.position = 'b'
    ch.legend.legendEntry = [LegendEntry(idx=i, delete=True) for i in range(primera_leyenda)]
    ws_car.add_chart(ch, 'K3')


def _hoja_materiales(wb, materiales):
    """Hoja MATERIALES: aceros del proyecto (nombre, tipo, Fy y Fu en kgf/cm2, fuente)."""
    ws = wb.create_sheet('MATERIALES')
    _encabezado(ws, ['NOMBRE', 'TIPO', 'FY (kgf/cm2)', 'FU (kgf/cm2)', 'E (kgf/cm2)', 'NU (Poisson)', 'FUENTE'],
                [14, 12, 14, 14, 16, 13, 80])
    for f in materiales:
        ws.append(list(f) + [None] * (7 - len(f)))
    for _ in range(MAT_MAX - len(materiales)):
        ws.append([None] * 7)
    return ws


# ---------------- COMBINACIONES DE CARGA ----------------
COMB_FILA0 = 3                       # primera fila de combinaciones
COMB_MAX = 40
COMB_FIN = COMB_FILA0 + COMB_MAX - 1
COMBINACIONES_DEFAULT = [
    [1] + [None] * (ESTADOS_MAX - 1),
    [1, 1] + [None] * (ESTADOS_MAX - 2),
    [1, 0.75, 0.75] + [None] * (ESTADOS_MAX - 3),
    [1, None, 1] + [None] * (ESTADOS_MAX - 3),
    [0.6, None, 1] + [None] * (ESTADOS_MAX - 3),
]


def _hoja_combinaciones(wb, combinaciones):
    """Hoja COMBINACIONES: columnas = estados (de la hoja ESTADOS), filas = combinaciones. Los factores son
    entradas del usuario (vacio = 0). La columna LC se arma sola: 'ASDn = f1·estado1 + f2·estado2 ...'."""
    ws = wb.create_sheet('COMBINACIONES')
    ws['A1'] = 'Combinaciones de carga: escriba los factores en las celdas amarillas (vacio = 0).'
    ws['A1'].font = NEGRITA
    cols = [_L(2 + j) for j in range(ESTADOS_MAX)]
    ws.cell(row=2, column=1, value='LC (calculada)').font = NEGRITA
    ws.cell(row=2, column=1).fill = ENCABEZADO
    ws.column_dimensions['A'].width = 70
    for j, col in enumerate(cols):
        ws[f'{col}2'] = f'=IF(ESTADOS!A{2 + j}="","",ESTADOS!A{2 + j})'
        ws[f'{col}2'].font = NEGRITA
        ws[f'{col}2'].fill = ENCABEZADO
        ws[f'{col}2'].alignment = Alignment(horizontal='center', wrap_text=True)
        ws.column_dimensions[col].width = 14
    ws.row_dimensions[2].height = 45
    for k in range(COMB_MAX):
        r = COMB_FILA0 + k
        terminos = '&'.join(
            f'IF(AND(ISNUMBER({c}{r}),{c}{r}<>0,{c}$2<>""),"+ "&ROUND({c}{r},2)&"·"&{c}$2&" ","")' for c in cols)
        ws[f'A{r}'] = f'=IF(LEN({terminos})=0,"","ASD{k + 1} = "&TRIM(MID({terminos},3,999)))'
    filas = list(combinaciones) if combinaciones else []
    for k in range(COMB_MAX):
        fila = filas[k] if k < len(filas) else [None] * ESTADOS_MAX
        for j, col in enumerate(cols):
            v = fila[j] if j < len(fila) else None
            celda = ws[f'{col}{COMB_FILA0 + k}']
            celda.value = None if v is None or (isinstance(v, str) and v.strip() == '') else v
            celda.fill = ENTRADA
            celda.number_format = '0.00'
    dv = DataValidation(type='decimal', operator='between', formula1='-100', formula2='100', allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f'B{COMB_FILA0}:{cols[-1]}{COMB_FIN}')
    ws.freeze_panes = f'B{COMB_FILA0}'
    return ws



# ---------------------------------------------------------------- PARAMETROS y NORMA (hojas de la planilla)
# PARAMETROS: lo que usted elige o ingresa (listas desplegables). NORMA: valores de la norma con su referencia
# (solo lectura). El notebook lee ambas hojas: no hay valores de norma ni supuestos dentro del codigo.
PARAM_CATALOGO = [
    # CLAVE, concepto, valor (ejemplo), unidad, opciones, referencia, explicacion
    ('SISTEMA_SISMICO', 'Sistema estructural sismico', 'Marcos de acero de ductilidad media', '',
     'lista (hoja NORMA, tabla 4.3.2)', 'NTC Diseno por Sismo 2023, tabla 4.3.2, pag. 53',
     'Define Q, gamma SV, gamma OI y R0 del sistema.'),
    ('CRUJIAS_AMBAS', 'Tres o mas crujias resistentes en las dos direcciones', 'SÍ', '', 'SÍ / NO',
     'NTC Diseno por Sismo 2023, §3.3.1, pag. 42-43',
     'SÍ: k1 = 1.0. NO: k1 = 0.8 (menos de tres crujias en alguna direccion).'),
    ('ZONA_SISMICA', 'Zona sismica del sitio', 'B', '', 'A / B / C',
     'NTC Diseno por Sismo 2023, §3.1 (eq. 3.1.1)', 'Se usa si KS_METODO = SEGUN ZONA. Dato obligatorio.'),
    ('TS', 'Ts del sitio (SASID)', 0.5, 's', 'numero > 0', 'NTC Diseno por Sismo 2023, eq. 3.1.1 y SASID',
     'Periodo caracteristico del sitio. Dato del SASID. Obligatorio.'),
    ('TA', 'Ta del sitio (SASID)', 0.2, 's', 'numero > 0', 'NTC Diseno por Sismo 2023, §3.3.1b y SASID',
     'Limite de la meseta. k2 = 0 si T >= Ta (T de §6.4.1). Obligatorio.'),
    ('TB', 'Tb del sitio (SASID)', 1.0, 's', 'numero > Ta', 'NTC Diseno por Sismo 2023, §3.2 y SASID',
     'Limite superior de la meseta. Obligatorio.'),
    ('K_SASID', 'Factor k del SASID', 1.0, '', 'numero > 0', 'NTC Diseno por Sismo 2023, §3.2, eq. 3.2.1',
     "Q' = 1 + (Q-1)/k. Con k = 1, Q' = Q. Dato del SASID. Obligatorio."),
    ('C_SASID', 'Coeficiente sismico c (SASID)', 0.25, '', 'numero > 0',
     'NTC Diseno por Sismo 2023, §6.3, eq. 6.3.1', "Base de la fuerza de diseno V0 = c W0/(Q' R'). Obligatorio."),
    ('KS_METODO', 'Factor Ks para la distorsion LDNE', 'CONSERVADOR (Ks=1)', '',
     'CONSERVADOR (Ks=1) / SEGUN ZONA (ec. 3.1.1)', 'NTC Diseno por Sismo 2023, tabla 3.1.1, nota [3]; eq. 3.1.1',
     'CONSERVADOR: Ks = 1. SEGUN ZONA: Ks de la zona y de Ts.'),
    ('DEFORMABLES', 'Elementos no estructurales capaces de deformarse o separados', 'SÍ', '', 'SÍ / NO',
     'NTC Diseno por Sismo 2023, §1.7, pag. 26', 'SÍ: limite LDNE 0.004. NO: 0.002.'),
    ('FRAGILES', 'Elementos no estructurales fragiles unidos', 'NO', '', 'SÍ / NO',
     'NTC Criterios y Acciones, §4.1.1 b), pag. 8', 'SÍ: distorsion h/500. NO: h/250.'),
    ('VIENTO', 'Viento considerado en el diseno', 'NO', '', 'NO',
     'Criterio del proyecto (version actual)',
     'Esta version no considera viento. Los estados y combinaciones con W se omiten.'),
    ('KZ_TORSION', 'Factor Kz de pandeo torsional (extremos)', 1.0, '', '1.0 / 0.5',
     'AISC 360-22, E4 (Kz, factor de longitud efectiva para pandeo torsional)',
     '1.0: extremos sin restriccion al alabeo. 0.5: extremos con restriccion al alabeo y a la rotacion. Confirme segun sus conexiones.'),
    ('METODO', 'Metodo de analisis sismico', 'ESTÁTICO', '', 'ESTÁTICO',
     'NTC Diseno por Sismo 2023, §6.1-6.2',
     'Solo analisis estatico. El modal espectral no se usa en esta aplicacion.'),
]

# DECISION D15 (abierta): solo marcos de acero (tabla 4.3.2). Alternativa: agregar contraventeos y sistemas duales.
NORMA_SISTEMAS = [
    # NOMBRE, Q, gamma SV, gamma OI, R0, referencia
    ('Marcos de acero de ductilidad alta', 4.0, 0.030, 0.0075, 2.0,
     'NTC Diseno por Sismo 2023, tabla 4.3.2, i) Marcos, pag. 53; R0 de §3.3.1a'),
    ('Marcos de acero de ductilidad media', 3.0, 0.020, 0.0075, 2.0,
     'NTC Diseno por Sismo 2023, tabla 4.3.2, i) Marcos, pag. 53; R0 de §3.3.1a'),
    ('Marcos de acero de ductilidad baja', 2.0, 0.015, 0.0075, 1.75,
     'NTC Diseno por Sismo 2023, tabla 4.3.2, i) Marcos, pag. 53; R0 de §3.3.1a (Q < 3)'),
]

NORMA_VALORES = [
    # CLAVE, valor, unidad, referencia, explicacion
    ('LDNE_DEFORMABLES', 0.004, '', 'NTC Diseno por Sismo 2023, §1.7, pag. 26',
     'Limite de distorsion LDNE si los elementos no estructurales son deformables o estan separados.'),
    ('LDNE_NO', 0.002, '', 'NTC Diseno por Sismo 2023, §1.7, pag. 26', 'Limite LDNE en los demas casos.'),
    ('NTC_CA_H_FRAGILES', 500, '', 'NTC Criterios y Acciones, §4.1.1 b), pag. 8',
     'Distorsion horizontal h/500 con elementos no estructurales fragiles unidos.'),
    ('NTC_CA_H_OTROS', 250, '', 'NTC Criterios y Acciones, §4.1.1 b), pag. 8', 'Distorsion h/250 en otros casos.'),
    ('SEG2_FACTOR', 0.08, '', 'NTC Diseno por Sismo 2023, §2.4, eq. 2.4.1, pag. 32',
     'Efectos de segundo orden despreciables si la distorsion no excede 0.08 V/W.'),
    ('FLECHA_DIVISOR', 300, '', 'AISC 360-22, Comentario L2, pag. 16.1-543',
     'Flecha vertical maxima L/300 en vigas. El comentario indica que 1/300 del claro es visible y puede causar dano arquitectonico. La NTC-Acero no fija un numero (§11.4 remite a la NTC-CA).'),
    ('K1_SI', 1.0, '', 'NTC Diseno por Sismo 2023, §3.3.1a, pag. 42-43', 'k1 con tres o mas crujias en ambas direcciones.'),
    ('K1_NO', 0.8, '', 'NTC Diseno por Sismo 2023, §3.3.1a, pag. 42-43', 'k1 con menos de tres crujias en alguna direccion.'),
    ('K2_COEF', 0.5, '', 'NTC Diseno por Sismo 2023, §3.3.1b, pag. 43', 'k2 = 0.5 (1 - (T/Ta)^2) si T < Ta.'),
    ('K2_EXP', 2, '', 'NTC Diseno por Sismo 2023, §3.3.1b, pag. 43', 'Exponente de T/Ta en k2.'),
    ('RP_SV', 1.0, '', 'NTC Diseno por Sismo 2023, §3.3.2a', 'R-prima para Seguridad de Vida = R.'),
    ('RP_OI', 0.75, '', 'NTC Diseno por Sismo 2023, §3.3.2b', 'R-prima para Ocupacion Inmediata = 0.75 R.'),
    ('KS_A_DEN', 6, '', 'NTC Diseno por Sismo 2023, eq. 3.1.1 (zona A)', 'Ks = 1/6 en zona A.'),
    ('KS_C_DEN', 4, '', 'NTC Diseno por Sismo 2023, eq. 3.1.1 (zona C)', 'Ks = 1/4 en zona C.'),
    ('KS_B_A', 6, '', 'NTC Diseno por Sismo 2023, eq. 3.1.1 (zona B)', 'Zona B: Ks = 1/[6 - 4(Ts - 0.5)].'),
    ('KS_B_B', 4, '', 'NTC Diseno por Sismo 2023, eq. 3.1.1 (zona B)', 'Coeficiente de Ts en la zona B.'),
    ('KS_B_C', 0.5, 's', 'NTC Diseno por Sismo 2023, eq. 3.1.1 (zona B)', 'Valor de Ts en la zona B.'),
    ('G', 9.81, 'm/s2', 'NTC Diseno por Sismo 2023, §6.4.1', 'Aceleracion de la gravedad para el periodo T.'),
    ('NTC_LR_PATIN', 0.56, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.1, pag. 57', 'lambda_r de patin (I y canal) en compresion.'),
    ('NTC_LR_ALMA', 1.49, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.1, pag. 57', 'lambda_r de alma en compresion.'),
    ('NTC_LR_PATA', 0.45, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.1, pag. 57', 'lambda_r de ala de angulo en compresion.'),
    ('NTC_LIM_PATIN_PL', 0.30, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, pag. 58', 'Patin tipo 1 (flexion).'),
    ('NTC_LIM_PATIN_P', 0.38, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, pag. 58', 'Patin tipo 2 (flexion) y limite media de Tabla 12.1.2.'),
    ('NTC_LIM_PATIN_R', 1.00, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, pag. 58', 'Patin tipo 3 (flexion).'),
    ('NTC_LIM_ALMA_PL', 2.45, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, pag. 58', 'Alma tipo 1 (flexion).'),
    ('NTC_LIM_ALMA_P', 3.76, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, pag. 58', 'Alma tipo 2 (flexion).'),
    ('NTC_LIM_ALMA_R', 5.70, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, pag. 58', 'Alma tipo 3 (flexion).'),
    ('NTC_LIM_PATA_P', 0.54, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, pag. 58', 'Ala de angulo tipo 2 (flexion).'),
    ('NTC_LIM_PATA_R', 0.91, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, pag. 58', 'Ala de angulo tipo 3 (flexion).'),
    ('NTC_KLR_COMP', 200, '', 'NTC-Acero 2020, §2.3.2, pag. 41', 'KL/r maximo en compresion (obligatorio).'),
    ('NTC_KLR_REF_PRINC', 240, '', 'NTC-Acero 2020, §2.3.2 (comentario)', 'L/r de referencia en tension, miembros principales.'),
    ('NTC_KLR_REF_CONTR', 300, '', 'NTC-Acero 2020, §2.3.2 (comentario)', 'L/r de referencia en tension, contraventeos.'),
    ('NTC_KLR_DUCTIL', 60, '', 'NTC-Acero 2020, §12.1.3.1, pag. 207', 'KL/r maximo en columnas ductiles.'),
    ('AISC_OMEGA_TFLU', 1.67, '', 'AISC 360-22, D2 (fluencia en traccion)', 'Factor de seguridad ASD, traccion por fluencia.'),
    ('AISC_OMEGA_TFRA', 2.00, '', 'AISC 360-22, D2 (fractura en traccion)', 'Factor de seguridad ASD, traccion por fractura.'),
    ('AISC_OMEGA_C', 1.67, '', 'AISC 360-22, E1 (compresion)', 'Factor de seguridad ASD en compresion.'),
    ('AISC_OMEGA_B', 1.67, '', 'AISC 360-22, F1 (flexion)', 'Factor de seguridad ASD en flexion.'),
    ('AISC_OMEGA_V', 1.67, '', 'AISC 360-22, G1 (cortante)', 'Factor de seguridad ASD en cortante.'),
    ('AISC_OMEGA_V_LAM', 1.50, '', 'AISC 360-22, G2.1(a)', 'Cortante en alma de I laminada con h/tw <= 2.24 sqrt(E/Fy).'),
    ('AISC_KV_I', 5.34, '', 'AISC 360-22, G2.2', 'kv de alma sin atiesadores.'),
    ('AISC_KV_ANG', 1.2, '', 'AISC 360-22, G3', 'kv de la pata de angulo.'),
    ('AISC_CB_MAX_ANG', 1.5, '', 'AISC 360-22, F10-4', 'Cb maximo para angulos.'),
    ('AISC_LP_PATIN', 0.38, 'sqrt(E/Fy)', 'AISC 360-22, Tabla B4.1b', 'lambda_p de ala (flexion, I y canal).'),
    ('AISC_LR_PATIN', 1.00, 'sqrt(E/Fy)', 'AISC 360-22, Tabla B4.1b', 'lambda_r de ala (flexion, I y canal).'),
    ('AISC_CV_LIM', 1.10, '', 'AISC 360-22, G2.2', 'Limite de h/tw para Cv = 1 (factor 1.10 sqrt(kv E/Fy)).'),
    ('AISC_HTW_LAM', 2.24, '', 'AISC 360-22, G2.1(a)', 'Limite h/tw de I laminada con Ωv = 1.50.'),
    ('AISC_LIM_ANG_E4', 0.71, '', 'AISC 360-22, E4 y F10.3 (angulos)', 'b/t de angulo sin pandeo flexotorsional.'),
    ('AISC_E5_RAT_LIM', 80, '', 'AISC 360-22, E5', 'L/ra de corte entre E5-1 y E5-2.'),
    ('AISC_B41A_C1_ALMA', 0.18, '', 'AISC 360-22, Tabla B4.1a', 'Coeficiente c1 de alma (E7).'),
    ('AISC_B41A_C2_ALMA', 1.31, '', 'AISC 360-22, Tabla B4.1a', 'Coeficiente c2 de alma (E7).'),
    ('AISC_B41A_C1_PATA', 0.22, '', 'AISC 360-22, Tabla B4.1a', 'Coeficiente c1 de pata (E7).'),
    ('AISC_B41A_C2_PATA', 1.49, '', 'AISC 360-22, Tabla B4.1a', 'Coeficiente c2 de pata (E7).'),
    ('AISC_LR_PATA_B41A', 0.45, 'sqrt(E/Fy)', 'AISC 360-22, Tabla B4.1a', 'lambda_r de pata de angulo (E7).'),
    # HSS rectangulares y cuadradas (agregadas para el diseno de HSS del catalogo ATC)
    ('AISC_HSS_LP_PATIN', 1.12, 'sqrt(E/Fy)', 'AISC 360-22, Tabla B4.1b, caso 17 (F7.2)', 'lambda_p de patin de HSS en flexion (compacto).'),
    ('AISC_HSS_LR_PATIN', 1.40, 'sqrt(E/Fy)', 'AISC 360-22, Tabla B4.1b, caso 17 (F7.2)', 'lambda_r de patin de HSS en flexion (no compacto a esbelto).'),
    ('AISC_HSS_LP_ALMA', 2.42, 'sqrt(E/Fy)', 'AISC 360-22, Tabla B4.1b, caso 19 (F7.3)', 'lambda_p de alma de HSS en flexion.'),
    ('AISC_HSS_LR_ALMA', 5.70, 'sqrt(E/Fy)', 'AISC 360-22, Tabla B4.1b, caso 19 (F7.3)', 'lambda_r de alma de HSS en flexion.'),
    ('AISC_HSS_LR_COMP', 1.40, 'sqrt(E/Fy)', 'AISC 360-22, Tabla B4.1a, caso 6 (E7)', 'lambda_r de paredes de HSS rectangular en compresion.'),
    ('AISC_HSS_C1', 0.20, '', 'AISC 360-22, Tabla E7.1, caso (b)', 'Coeficiente c1 de paredes de HSS (E7-3).'),
    ('AISC_HSS_C2', 1.38, '', 'AISC 360-22, Tabla E7.1, caso (b)', 'Coeficiente c2 de paredes de HSS (E7-5).'),
    ('NTC_HSS_LR_COMP', 1.40, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.1, elementos atiesados, paredes de HSS, pag. 58',
     'lambda_r de paredes de HSS rectangulares en compresion (clasificacion de seccion).'),
    ('NTC_HSS_LIM_PATIN_PL', 1.12, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, patines de HSS, pag. 59', 'Patin de HSS: lambda_pl (seccion tipo 1).'),
    ('NTC_HSS_LIM_PATIN_P', 1.12, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, patines de HSS, pag. 59', 'Patin de HSS: lambda_p (seccion tipo 2).'),
    ('NTC_HSS_LIM_PATIN_R', 1.40, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, patines de HSS, pag. 59', 'Patin de HSS: lambda_r (seccion tipo 3).'),
    ('NTC_HSS_LIM_ALMA_P', 2.42, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, almas de HSS, pag. 59', 'Alma de HSS: lambda_p (seccion tipo 2).'),
    ('NTC_HSS_LIM_ALMA_R', 5.70, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 3.2.2, almas de HSS, pag. 59', 'Alma de HSS: lambda_r (seccion tipo 3).'),
    ('AISC_KV_HSS', 5, '', 'AISC 360-22, G4 y G2.2 (kv = 5 para HSS rectangulares y cajon)', 'kv de la alma de HSS rectangular (Cv2).'),
    ('AISC_CV2_LIM2', 1.37, 'sqrt(kv E/Fy)', 'AISC 360-22, G2.2, ec. G2-10 (pagina 16.1-77)', 'Limite superior de h/t para Cv2 (G2-10).'),
    ('AISC_CV2_K', 1.51, '', 'AISC 360-22, G2.2, ec. G2-11 (pagina 16.1-77)', 'Constante 1.51 de Cv2 para h/t > 1.37 sqrt(kv E/Fy) (G2-11).'),
    ('AISC_OMEGA_T_HSS', 1.67, '', 'AISC 360-22, H3.1 (ΩT, ASD)', 'Factor de seguridad de la torsion de HSS: Tc = Tn/ΩT.'),
    ('AISC_H3_LIM1', 2.45, 'sqrt(E/Fy)', 'AISC 360-22, H3.1(b)(1), ec. H3-3', 'Limite de h/t con Fcr = 0.6 Fy en torsion de HSS rectangular.'),
    ('AISC_H3_LIM2', 3.07, 'sqrt(E/Fy)', 'AISC 360-22, H3.1(b)(2) y (b)(3), ec. H3-4', 'Limite de h/t entre H3-4 y H3-5 en torsion de HSS rectangular.'),
    ('AISC_H3_LIM3', 260, '', 'AISC 360-22, H3.1(b)(3), ec. H3-5', 'Limite superior de h/t para la torsion de HSS rectangular.'),
    ('AISC_H3_K', 0.458, '', 'AISC 360-22, H3.1(b)(3), ec. H3-5', 'Constante 0.458 pi^2 E/(h/t)^2 del Fcr elastico de torsion.'),
    ('AISC_H3_FCR_Y', 0.6, '', 'AISC 360-22, H3.1, ec. H3-3', 'Factor de fluencia de la torsion: Fcr = 0.6 Fy.'),
    ('AISC_H3_20PCT', 0.2, '', 'AISC 360-22, H3.2', 'Fraccion Tr/Tc a partir de la cual se aplica H3-6 en lugar de H1.'),
    ('NTC_HSS_DUC_ALTA', 0.55, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 12.1.2, paredes de HSS rectangulares, pag. 212',
     'lambda_da de paredes de HSS (ductilidad alta). La nota [b] fija 0.6 para columnas de cajon: no reduce este valor.'),
    ('NTC_HSS_DUC_MEDIA', 0.64, 'sqrt(E/Fy)', 'NTC-Acero 2020, tabla 12.1.2, paredes de HSS rectangulares, pag. 212',
     'lambda_dm de paredes de HSS (ductilidad media). POR CONFIRMAR: la nota [c] (1.12 sqrt(E/Fy)) se interpreta como '
     'limite de patines de vigas, no de paredes comprimidas.'),
]


def _cabecera_en(ws, fila, columnas, anchos):
    """Cabecera de tabla en la fila indicada (la cabecera de _encabezado va siempre en la fila 1)."""
    for c, nombre in enumerate(columnas, start=1):
        celda = ws.cell(row=fila, column=c, value=nombre)
        celda.font = NEGRITA
        celda.fill = ENCABEZADO
        celda.alignment = Alignment(horizontal='center', wrap_text=True)
        ws.column_dimensions[celda.column_letter].width = anchos[c - 1]


def _hoja_normas_y_parametros(wb):
    """Crea las hojas NORMA (solo lectura) y PARAMETROS (listas desplegables y datos a ingresar)."""
    from openpyxl.worksheet.datavalidation import DataValidation as DV
    ws_n = wb.create_sheet('NORMA')
    ws_n['A1'] = 'NORMA: valores de la norma usados por el calculo (solo lectura). Cada valor tiene su referencia.'
    ws_n['A1'].font = NEGRITA
    ws_n['A3'] = 'SISTEMAS ESTRUCTURALES (NTC Diseno por Sismo 2023, tabla 4.3.2)'
    ws_n['A3'].font = NEGRITA
    _cabecera_en(ws_n, 4, ['NOMBRE DEL SISTEMA', 'Q', 'gamma SV', 'gamma OI', 'R0', 'REFERENCIA'],
                 [44, 8, 11, 11, 8, 70])
    for k, (nombre, q, gsv, goi, r0, ref) in enumerate(NORMA_SISTEMAS):
        fila = 5 + k
        for col, valor in zip('ABCDEF', [nombre, q, gsv, goi, r0, ref]):
            ws_n[f'{col}{fila}'] = valor
    fila_v = 5 + len(NORMA_SISTEMAS) + 2
    ws_n[f'A{fila_v}'] = 'VALORES DE NORMA (usados por el calculo)'
    ws_n[f'A{fila_v}'].font = NEGRITA
    _cabecera_en(ws_n, fila_v + 1, ['CLAVE', 'VALOR', 'UNIDAD', 'REFERENCIA', 'EXPLICACION'], [44, 12, 12, 60, 90])
    for k, (clave, valor, unidad, ref, expl) in enumerate(NORMA_VALORES):
        fila = fila_v + 2 + k
        for col, dato in zip('ABCDE', [clave, valor, unidad, ref, expl]):
            ws_n[f'{col}{fila}'] = dato
    ws_n.protection.sheet = True
    ws_n.freeze_panes = 'A4'

    ws_p = wb.create_sheet('PARAMETROS')
    ws_p['A1'] = 'PARAMETROS DE DISENO: elija en las listas (columna C, amarilla). Sin estos datos no se hace el diseno.'
    ws_p['A1'].font = NEGRITA
    ws_p['A2'] = ('Los datos del SASID (Ts, Ta, Tb, k, c) y la zona vienen de su estudio de sitio. Los valores de '
                  'ejemplo son para probar: reemplacelos por los suyos.')
    _cabecera_en(ws_p, 4, ['CLAVE', 'CONCEPTO', 'VALOR', 'UNIDAD', 'OPCIONES', 'REFERENCIA NORMATIVA', 'EXPLICACION'],
                 [22, 44, 40, 8, 40, 52, 70])
    dv_lista = {
        'CRUJIAS_AMBAS': DV(type='list', formula1='"SÍ,NO"', allow_blank=False),
        'ZONA_SISMICA': DV(type='list', formula1='"A,B,C"', allow_blank=False),
        'KS_METODO': DV(type='list', formula1='"CONSERVADOR (Ks=1),SEGUN ZONA (ec. 3.1.1)"', allow_blank=False),
        'DEFORMABLES': DV(type='list', formula1='"SÍ,NO"', allow_blank=False),
        'FRAGILES': DV(type='list', formula1='"SÍ,NO"', allow_blank=False),
        'VIENTO': DV(type='list', formula1='"NO"', allow_blank=False),
        'METODO': DV(type='list', formula1='"ESTÁTICO"', allow_blank=False),
        'KZ_TORSION': DV(type='list', formula1='"1.0,0.5"', allow_blank=False),
        'SISTEMA_SISMICO': DV(type='list', formula1=f'NORMA!$A$5:$A${4 + len(NORMA_SISTEMAS)}', allow_blank=False),
    }
    for dv in dv_lista.values():
        ws_p.add_data_validation(dv)
    dv_num = DV(type='decimal', operator='greaterThan', formula1='0', allow_blank=False)
    ws_p.add_data_validation(dv_num)
    for k, (clave, concepto, valor, unidad, opciones, ref, expl) in enumerate(PARAM_CATALOGO):
        fila = 5 + k
        ws_p[f'A{fila}'] = clave
        ws_p[f'B{fila}'] = concepto
        ws_p[f'C{fila}'] = valor
        ws_p[f'C{fila}'].fill = ENTRADA
        ws_p[f'D{fila}'] = unidad
        ws_p[f'E{fila}'] = opciones
        ws_p[f'F{fila}'] = ref
        ws_p[f'G{fila}'] = expl
        if clave in dv_lista:
            dv_lista[clave].add(f'C{fila}')
        if clave in ('TS', 'TA', 'TB', 'K_SASID', 'C_SASID'):
            dv_num.add(f'C{fila}')
    ws_p.freeze_panes = 'A5'
    return ws_p, ws_n


def escribir_planilla(ruta, filas_nodos, filas_barras, imagen=None, filas_cargas=None, estados=None,
                      materiales=None, combinaciones=None, longitudes=None, listas=None,
                      grupos_perfil=None, fu_matriz=None, perfiles_analizados=None):
    """Escribe el libro completo con los datos dados. Cada fila de NODOS: NODOS_COLS; de BARRAS: BARRAS_COLS."""
    wb = openpyxl.Workbook()
    ws_i = wb.active; ws_i.title = 'INSTRUCCIONES'
    texto = [
        'PLANTILLA DE SOPORTES: como llenarla',
        '',
        'PARAMETROS: elija en las listas desplegables (columna C, amarilla) e ingrese Ts, Ta, Tb, k y c del SASID.',
        'NORMA: valores de la norma con su referencia. Solo lectura: el calculo los toma de aqui.',
        'LONGITUDES: LX, LY, LB, KX, KY y CB son obligatorios en cada barra. No se suponen valores.',
        'BARRAS: DUCTILIDAD (ALTA o MEDIA) es obligatoria en COLUMNA y VIGA.',
        'VIENTO: esta version no considera viento; no se usan los estados ni las combinaciones con W.',
        '',
        'NODOS: una fila por nodo. X, Y, Z en metros. Capacidad: 98 nodos.',
        'UX, UY, UZ, RX, RY, RZ: SI = ese grado de libertad queda restringido en el nodo. Vacio o NO = libre.',
        '   Empotrado = los 6 en SI. Rotulado = UX, UY, UZ en SI (giros libres). Rodillo = solo UZ, etc.',
        'BARRAS: una fila por barra. NODO_I y NODO_J son IDs de la hoja NODOS. Capacidad: 106 barras.',
        'ELEMENTO: COLUMNA, VIGA, ARRIOSTRE o DIAGONAL (lista).',
        'DUCTILIDAD: ALTA o MEDIA (lista). Vacio = no se verifica la Tabla 12.1.2 de la NTC para esta barra.',
        'PERFIL: nombre Gerdau rev04 (lista ordenada por familia y seccion).',
        'MATERIAL: elija la calidad de la lista (base de datos de la hoja MATERIALES). Es obligatorio: vacio da error.',
        'EJE_FUERTE_X, _Y, _Z: direccion del eje fuerte (peralte) en coordenadas globales. OBLIGATORIA en cada barra: no hay valor por defecto.',
        '   Vacio = direccion por defecto del motor. Basta una componente (las vacias valen 0). No puede ser paralelo a la barra.',
        'CONEXION_I y CONEXION_J: empotrado o rotulado en cada extremo de la barra. Vacio = empotrado.',
        'Ver la hoja CONEXIONES para el significado de cada tipo.',
        'REPORTAR (NODOS y BARRAS): SI = el resultado de ese nodo o barra sale en el Excel de resultados. Vacio = no sale.',
        'MATERIALES: E (modulo de elasticidad, kgf/cm2) y NU (Poisson) de cada calidad. Son datos suyos: el analisis los lee de aqui.',
        'ESTADOS: PESO PROPIO = SI en el estado donde OpenSees aplica el peso de la seccion (masa Gerdau, kgf/m).',
        'LONGITUDES: por barra, LX y LY (m) longitud no arriostrada en X e Y, KX y KY (factor K), LB (m) para pandeo lateral, CB. Vacio = supuesto.',
        'E5 (angulos LI): SI en las cuatro columnas solo si la pata cumple: (1) carga en los extremos por una pata; (2) union soldada o con 2 pernos o mas; '
        '(3) sin cargas transversales intermedias; (4) miembro individual o alma de armadura plana. Vacio o NO = K L/rz y H2.',
        'LI CONEXION: UNA PATA o DOS PATAS (ambas patas conectadas). LI LONGITUD CONEXION (m): longitud de la conexion; vacio = longitud de la barra.',
        'VISTA_VIVA: planta, alzado y 3D esquematico que se actualizan al cambiar NODOS y BARRAS.',
        '   Cambie el nivel Z (C3), la cota Y (C4), el azimut (C5), la elevacion (C6) y el largo de la flecha (C7).',
        '',
        'No cambiar los nombres de las hojas ni de las columnas. Las filas vacias se ignoran.',
    ]
    for linea in texto:
        ws_i.append([linea])
    ws_i['A1'].font = Font(bold=True, size=13)
    ws_i.column_dimensions['A'].width = 115

    ws_n = wb.create_sheet('NODOS')
    _encabezado(ws_n, NODOS_COLS, [8, 12, 12, 12, 7, 7, 7, 7, 7, 7, 36, 12])
    for f in filas_nodos:
        ws_n.append(f)
    for _ in range(FIN_N - 1 - len(filas_nodos)):
        ws_n.append([None] * len(NODOS_COLS))
    dv_grados = DataValidation(type='list', formula1='"SI,NO"', allow_blank=True)
    ws_n.add_data_validation(dv_grados); dv_grados.add(f'E2:J{FIN_N}')
    dv_rep_n = DataValidation(type='list', formula1='"SI,NO"', allow_blank=True)
    ws_n.add_data_validation(dv_rep_n); dv_rep_n.add(f'L2:L{FIN_N}')

    ws_b = wb.create_sheet('BARRAS')
    _encabezado(ws_b, BARRAS_COLS, [8, 10, 10, 14, 18, 13, 13, 13, 14, 14, 12, 30, 12, 14])
    for f in filas_barras:
        ws_b.append(f)
    for _ in range(FIN_B - 1 - len(filas_barras)):
        ws_b.append([None] * len(BARRAS_COLS))
    dv_elem = DataValidation(type='list', formula1='"' + ','.join(ELEMENTOS) + '"', allow_blank=True)
    # PERFIL: lista que depende del ELEMENTO de la fila (LISTA_COLUMNA, LISTA_VIGA, ...). Sin diseno, todos los perfiles.
    dv_perf = DataValidation(type='list', formula1='INDIRECT("LISTA_"&$D2)', allow_blank=True)
    dv_vec = DataValidation(type='decimal', operator='between', formula1='-1000000', formula2='1000000',
                            allow_blank=True)
    dv_ci = DataValidation(type='list', formula1='"' + ','.join(TIPOS_CONEXION) + '"', allow_blank=True)
    dv_cj = DataValidation(type='list', formula1='"' + ','.join(TIPOS_CONEXION) + '"', allow_blank=True)
    dv_mat = DataValidation(type='list', formula1=f'MATERIALES!$A$2:$A${MAT_FIN}', allow_blank=True)
    dv_rep_b = DataValidation(type='list', formula1='"SI,NO"', allow_blank=True)
    dv_duc_b = DataValidation(type='list', formula1='"ALTA,MEDIA"', allow_blank=True)
    lista_dv = [(dv_elem, f'D2:D{FIN_B}'), (dv_vec, f'F2:H{FIN_B}'),
                (dv_ci, f'I2:I{FIN_B}'), (dv_cj, f'J2:J{FIN_B}'), (dv_mat, f'K2:K{FIN_B}'),
                (dv_rep_b, f'M2:M{FIN_B}'), (dv_duc_b, f'N2:N{FIN_B}')]
    if grupos_perfil is None:
        lista_dv.insert(1, (dv_perf, f'E2:E{FIN_B}'))      # sin diseno: perfil por barra
    else:
        for r in range(2, FIN_B + 1):                       # con diseno: el perfil lo da su grupo (PERFIL_GRUPO)
            ws_b.cell(row=r, column=5, value=f'=IFERROR(INDEX(PERFIL_GRUPO!$B$2:$B$5,MATCH(D{r},PERFIL_GRUPO!$A$2:$A$5,0)),"")')
    for dv, rango in lista_dv:
        ws_b.add_data_validation(dv); dv.add(rango)

    _hoja_longitudes(wb, longitudes)
    ws_c = wb.create_sheet('CONEXIONES')
    _encabezado(ws_c, ['TIPO', 'EN NODOS (UX a RZ = SI)', 'EN EXTREMOS DE BARRA (CONEXION_I / CONEXION_J)',
                       'USO TIPICO'], [14, 44, 50, 46])
    ws_c.append(['empotrado', 'SI en UX, UY, UZ, RX, RY y RZ (base de columna empotrada).',
                 'El extremo transmite momento: la barra sigue el giro del nodo (union rigida).',
                 'Base de columna, nudo rigido de marco.'])
    ws_c.append(['rotulado', 'SI en UX, UY y UZ; RX, RY y RZ en NO (giros libres).',
                 'El momento en el extremo es cero (articulado).',
                 'Apoyo articulado, viga simple, extremo de arriostre.'])
    ws_c.append(['(vacio)', 'En NODOS: NO o vacio en los 6 grados: nodo libre.', 'En BARRAS: se toma empotrado.', '-'])
    for fila in ws_c.iter_rows(min_row=2):
        for celda in fila:
            celda.alignment = Alignment(wrap_text=True, vertical='top')

    ws_l = wb.create_sheet('LISTAS')
    _encabezado(ws_l, ['PERFIL', 'ESTADO', 'MASA (kg/m)', 'ELEMENTO'], [18, 20, 14, 14])
    for k, (_, nombre) in enumerate(PERFILES_ORDEN):
        p = G.PERFILES[nombre]
        ws_l.cell(row=k + 2, column=1, value=nombre)
        ws_l.cell(row=k + 2, column=2, value=p['estado'])
        ws_l.cell(row=k + 2, column=3, value=p['masa_kg_m'])
    for k, e in enumerate(ELEMENTOS):
        ws_l.cell(row=k + 2, column=4, value=e)
    _hoja_listas_grupo(wb, ws_l, listas)
    if grupos_perfil is not None and fu_matriz is not None:
        _hojas_diseno_vivo(wb, grupos_perfil, fu_matriz, perfiles_analizados)

    ws_calc = _hoja_calculo(wb)
    ws_3d, col_3d = _hoja_3d(wb)
    _hoja_estados(wb, estados or [])
    _hoja_materiales(wb, materiales if materiales is not None else MATERIALES_DEFAULT)
    _hoja_combinaciones(wb, combinaciones if combinaciones is not None else COMBINACIONES_DEFAULT)
    _hoja_normas_y_parametros(wb)
    ws_cc, col_cc = _hoja_calc_cargas(wb, col_3d)
    primer_estado = estados[0][0] if estados else ''
    ws_car = _hoja_cargas(wb, filas_cargas or [], primer_estado)
    _grafico_cargas(ws_car, ws_cc, col_cc, ws_3d, col_3d)

    ws_v = wb.create_sheet('VISTA_VIVA')
    ws_v['A1'] = 'Vista en vivo de la planilla: se actualiza al cambiar NODOS o BARRAS.'
    ws_v['A1'].font = Font(bold=True, size=12)
    ws_v['A3'] = 'Nivel Z para planta (m)'
    ws_v['C3'] = 3.5
    ws_v['A4'] = 'Cota Y para alzado (m)'
    ws_v['C4'] = 0
    for celda in ('C3', 'C4'):
        ws_v[celda].fill = ENTRADA
    ws_v['A5'] = 'Azimut (grados)'
    ws_v['C5'] = 30
    ws_v['A6'] = 'Elevacion (grados)'
    ws_v['C6'] = 20
    ws_v['A7'] = 'Largo flecha eje fuerte (m)'
    ws_v['C7'] = 0.8
    for celda in ('C5', 'C6', 'C7'):
        ws_v[celda].fill = ENTRADA
    ws_v['A8'] = 'Cota X para alzado lateral Y-Z (m)'
    ws_v['C8'] = 0
    ws_v['C8'].fill = ENTRADA
    ws_v['A9'] = 'Cambie los angulos para girar la vista 3D esquematica (abajo).'
    ws_v.column_dimensions['A'].width = 26
    _graficos(ws_v, ws_calc)
    _grafico_lateral(ws_v, ws_calc)
    _grafico_3d(ws_v, ws_3d, col_3d)
    _grafico_3d_rotulas(ws_v, ws_3d, col_3d)

    wb.calculation = CalcProperties(fullCalcOnLoad=True)
    wb.save(ruta)
    return ruta


def _sin_suavizado(ruta_xlsx):
    """LibreOffice guarda las series con suavizado (smooth=1): las flechas del eje fuerte se verian curvas.
    Se reescriben los graficos con smooth=0 (lineas rectas)."""
    import zipfile
    tmp = ruta_xlsx + '.tmp'
    with zipfile.ZipFile(ruta_xlsx) as origen, zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as destino:
        for item in origen.infolist():
            datos = origen.read(item.filename)
            if item.filename.startswith('xl/charts/chart') and item.filename.endswith('.xml'):
                datos = datos.replace(b'<c:smooth val="1"/>', b'<c:smooth val="0"/>')
            destino.writestr(item, datos)
    os.replace(tmp, ruta_xlsx)


def guardar_con_valores(ruta):
    """Recalcula el libro con LibreOffice (si esta instalado) para que las formulas y graficos tengan valores
    guardados: asi se ven tambien en visores que no calculan. Sin LibreOffice, Excel calcula al abrir."""
    import shutil, subprocess, tempfile
    soffice = shutil.which('soffice') or shutil.which('libreoffice')
    if not soffice:
        print('LibreOffice no disponible: Excel calcula la hoja al abrirla.')
        return
    tmp = tempfile.mkdtemp()
    subprocess.run([soffice, '--headless', '--calc', '--convert-to', 'xlsx', '--outdir', tmp, ruta],
                   capture_output=True, timeout=300)
    salida = os.path.join(tmp, os.path.basename(ruta))
    if os.path.exists(salida):
        _sin_suavizado(salida)
        shutil.move(salida, ruta)
        print('Valores calculados guardados en', ruta)
    else:
        print('No se pudo recalcular con LibreOffice; Excel calcula la hoja al abrirla.')


def crear_plantilla(ruta=None, sobrescribir=False):
    ruta = ruta or os.path.join(os.getcwd(), 'plantilla_soportes.xlsx')
    if os.path.exists(ruta) and not sobrescribir:
        print('Ya existe', ruta, '- no se sobrescribe la planilla llenada.')
        return ruta
    nodos, barras = ejemplo_filas()
    escribir_planilla(ruta, nodos, barras, filas_cargas=ejemplo_cargas(), estados=ejemplo_estados(),
                      materiales=MATERIALES_DEFAULT, combinaciones=COMBINACIONES_DEFAULT,
                      longitudes=ejemplo_longitudes())
    guardar_con_valores(ruta)
    print('Plantilla creada:', ruta)
    return ruta


def cargar_plantilla(ruta=None):
    ruta = ruta or os.path.join(os.getcwd(), 'plantilla_soportes.xlsx')
    if os.path.exists(ruta):
        print('Ya existe, se usa tal cual:', ruta)
    else:
        with open(ruta, 'wb') as f:
            f.write(base64.b64decode(PLANTILLA_B64))
        print('Plantilla escrita en:', ruta)
    return ruta


# Plantilla incrustada en el notebook (mismo archivo que plantilla_soportes.xlsx del repositorio)
PLANTILLA_B64 = '...'
RUTA_PLANTILLA = cargar_plantilla() if PLANTILLA_B64 else crear_plantilla()
_hojas = pd.read_excel(RUTA_PLANTILLA, sheet_name=None, dtype=object)
print('Hojas cargadas:', list(_hojas))
display(_hojas['NODOS'].head(8))
display(_hojas['BARRAS'].head(8))


# ---------------- DISENO POR GRUPO EN EXCEL: el perfil se elige por grupo y el FU se recalcula ----------------
def _hojas_diseno_vivo(wb, grupos_perfil, fu_matriz, perfiles_analizados=None):
    """PERFIL_GRUPO: un perfil por grupo (desplegable), con FU y esbeltez maximos de sus barras.
    FU_BARRAS: FU y esbeltez de cada barra con su perfil (formulas). FU_MATRIZ: FU de cada barra con cada perfil
    del catalogo, calculado al ejecutar el bloque 10. Al cambiar el perfil de un grupo, Excel recalcula todo."""
    from openpyxl.formatting.rule import FormulaRule
    from openpyxl.styles import PatternFill as _PF
    rojo, verde = _PF('solid', fgColor='F8CBAD'), _PF('solid', fgColor='C6EFCE')

    ws_g = wb.create_sheet('PERFIL_GRUPO')
    _encabezado(ws_g, ['GRUPO', 'PERFIL ELEGIDO', 'FU MAXIMO', 'FU ESBELTEZ LOCAL', 'FU ESBELTEZ GLOBAL',
                       'ESTADO', 'BARRAS', 'PERFIL ANALIZADO', 'EJECUCION'], [14, 26, 14, 18, 20, 14, 10, 26, 52])
    for k, g in enumerate(GRUPOS_LISTA):
        r = k + 2
        ws_g.cell(row=r, column=1, value=g)
        ws_g.cell(row=r, column=2, value=grupos_perfil.get(g))
        ws_g.cell(row=r, column=7, value=f'=COUNTIF(FU_BARRAS!$B$2:$B${FIN_B},A{r})')
        # Perfil con que corrio el analisis (snapshot al ejecutar el bloque 9): si cambia B, el resultado queda desactualizado
        ws_g.cell(row=r, column=8, value=(perfiles_analizados or {}).get(g, 'sin analisis'))
        ws_g.cell(row=r, column=9, value=(f'=IF(B{r}="","SIN PERFIL",IF(B{r}=H{r},"vigente: el analisis uso este perfil",'
                                          f'"DESACTUALIZADO: el perfil cambio; vuelva a ejecutar el analisis"))'))
        for col, letra in ((3, 'E'), (4, 'F'), (5, 'G')):
            rango = f'FU_BARRAS!${letra}$2:${letra}${FIN_B}'
            ws_g.cell(row=r, column=col, value=(
                f'=IF(G{r}=0,"sin barras",IF(COUNTIFS(FU_BARRAS!$B$2:$B${FIN_B},A{r},{rango},">=0")=0,"no aplica",'
                f'_xlfn.MAXIFS({rango},FU_BARRAS!$B$2:$B${FIN_B},A{r})))'))
        ws_g.cell(row=r, column=6, value=f'=IF(ISNUMBER(C{r}),IF(C{r}<=1,"CUMPLE","NO CUMPLE"),C{r})')
        dv = DataValidation(type='list', formula1=f'LISTA_{g}', allow_blank=True, errorStyle='warning',
                            showErrorMessage=True, errorTitle='Perfil fuera de la lista',
                            error='Este perfil no cumple para el grupo. Se usara igual: revise el FU.')
        ws_g.add_data_validation(dv); dv.add(f'B{r}')
    ws_g.cell(row=7, column=1, value='Elija el perfil de cada grupo: se aplica a todas sus barras y el FU se recalcula. '
                                     'Verde = FU <= 1 (cumple), rojo = FU > 1.')
    ws_g.conditional_formatting.add('C2:E5', FormulaRule(formula=['AND(ISNUMBER(C2),C2>1)'], fill=rojo))
    ws_g.conditional_formatting.add('C2:E5', FormulaRule(formula=['AND(ISNUMBER(C2),C2<=1)'], fill=verde))
    ws_g.conditional_formatting.add('I2:I5', FormulaRule(formula=['LEFT(I2,4)="DESA"'], fill=rojo))
    ws_g.conditional_formatting.add('I2:I5', FormulaRule(formula=['LEFT(I2,4)="vige"'], fill=verde))

    ws_f = wb.create_sheet('FU_BARRAS')
    _encabezado(ws_f, ['ID', 'ELEMENTO', 'PERFIL', 'FILA (auxiliar)', 'FU MAXIMO', 'FU ESBELTEZ LOCAL',
                       'FU ESBELTEZ GLOBAL', 'FU DUCTILIDAD', 'CASO QUE GOBIERNA', 'ESTADO', 'AVISO'],
                [8, 13, 22, 12, 12, 16, 16, 15, 26, 16, 36])
    for r in range(2, FIN_B + 1):
        ws_f.cell(row=r, column=1, value=f'=IF(BARRAS!A{r}="","",BARRAS!A{r})')
        ws_f.cell(row=r, column=2, value=f'=IF(A{r}="","",BARRAS!D{r})')
        ws_f.cell(row=r, column=3, value=f'=IF(A{r}="","",BARRAS!E{r})')
        ws_f.cell(row=r, column=4, value=f'=IF(A{r}="","",IFERROR(MATCH(A{r}&"|"&C{r},FU_MATRIZ!$A:$A,0),""))')
        for col, letra in ((5, 'E'), (6, 'G'), (7, 'H'), (8, 'I'), (9, 'J')):
            ws_f.cell(row=r, column=col, value=f'=IF(D{r}="","",INDEX(FU_MATRIZ!${letra}:${letra},D{r}))')
        ws_f.cell(row=r, column=10, value=(f'=IF(A{r}="","",IF(D{r}="","SIN DATO: perfil fuera del diseno",'
                                           f'INDEX(FU_MATRIZ!$K:$K,D{r})))'))
        ws_f.cell(row=r, column=11, value=(f'=IF(D{r}="","",IF(B{r}<>INDEX(FU_MATRIZ!$D:$D,D{r}),'
                                           f'"ELEMENTO cambio: ejecute el bloque 10",""))'))
    ws_f.conditional_formatting.add(f'E2:G{FIN_B}', FormulaRule(formula=['AND(ISNUMBER(E2),E2>1)'], fill=rojo))
    ws_f.conditional_formatting.add(f'E2:G{FIN_B}', FormulaRule(formula=['AND(ISNUMBER(E2),E2<=1)'], fill=verde))
    ws_f.freeze_panes = 'A2'

    ws_m = wb.create_sheet('FU_MATRIZ')
    _encabezado(ws_m, COLS_MATRIZ_XL, [26, 6, 22, 14, 12, 12, 16, 18, 20, 24, 14, 12, 12, 30])
    for fila in fu_matriz.itertuples(index=False):
        ws_m.append([None if (isinstance(v, float) and v != v) else (v.item() if hasattr(v, 'item') else v)
                     for v in fila])
    ws_m.freeze_panes = 'A2'


In [ ]:
%%writefile parametros.py
"""Parametros de diseno y de norma. Vienen de las hojas PARAMETROS y NORMA de la planilla (no hay valores de norma
ni supuestos en este modulo). resolver() valida cada dato, convierte las opciones en numeros y devuelve la
configuracion del calculo. Si falta un dato o una opcion no es valida, devuelve errores y el diseno no se hace."""
import math

CAMPOS_REQUERIDOS = ['SISTEMA_SISMICO', 'CRUJIAS_AMBAS', 'ZONA_SISMICA', 'TS', 'TA', 'TB', 'K_SASID', 'C_SASID',
                     'KS_METODO', 'DEFORMABLES', 'FRAGILES', 'VIENTO', 'METODO', 'KZ_TORSION']
KS_CONSERVADOR = 'CONSERVADOR (Ks=1)'
KS_SEGUN_ZONA = 'SEGUN ZONA (ec. 3.1.1)'
METODO_ESTATICO = 'ESTÁTICO'
VIENTO_NO = 'NO'


def _vacio(v):
    return v is None or (isinstance(v, str) and not v.strip()) or (isinstance(v, float) and math.isnan(v))


def _numero(v):
    try:
        x = float(v)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(x) else x


def _si_no(v):
    t = str(v).strip().upper()
    return {'SÍ': True, 'SI': True, 'NO': False}.get(t)


def resolver(par, nor):
    """par: {CLAVE: valor} de la hoja PARAMETROS. nor: {'SISTEMAS': {nombre: {Q, gSV, gOI, R0}}, 'VALORES': {CLAVE: n}}.
    Devuelve (conf, errores). errores vacio = datos completos."""
    errores = []
    for k in CAMPOS_REQUERIDOS:
        if _vacio(par.get(k)):
            errores.append(f'PARAMETROS {k}: falta el dato (elija de la lista o ingrese el valor)')
    if errores:
        return None, errores
    V = nor['VALORES']
    sistemas = nor['SISTEMAS']
    sistema = str(par['SISTEMA_SISMICO']).strip()
    if sistema not in sistemas:
        errores.append(f'PARAMETROS SISTEMA_SISMICO: "{sistema}" no esta en la lista de la hoja NORMA')
        return None, errores
    s = sistemas[sistema]
    crujias = _si_no(par['CRUJIAS_AMBAS'])
    if crujias is None:
        errores.append('PARAMETROS CRUJIAS_AMBAS: elija SÍ o NO')
    zona = str(par['ZONA_SISMICA']).strip().upper()
    if zona not in ('A', 'B', 'C'):
        errores.append('PARAMETROS ZONA_SISMICA: elija A, B o C')
    num = {}
    for k, nombre in [('TS', 'Ts'), ('TA', 'Ta'), ('TB', 'Tb'), ('K_SASID', 'k'), ('C_SASID', 'c')]:
        x = _numero(par[k])
        if x is None or x <= 0:
            errores.append(f'PARAMETROS {k}: {nombre} debe ser un numero mayor que cero (valor del SASID)')
        num[k] = x
    if not errores and num['TB'] <= num['TA']:
        errores.append('PARAMETROS TB: Tb debe ser mayor que Ta (valores del SASID)')
    ks_metodo = str(par['KS_METODO']).strip()
    if ks_metodo not in (KS_CONSERVADOR, KS_SEGUN_ZONA):
        errores.append(f'PARAMETROS KS_METODO: elija "{KS_CONSERVADOR}" o "{KS_SEGUN_ZONA}"')
    # DECISION D10 (abierta): DEFORMABLES = SÍ da LDNE 0.004 (NTC 1.7). Se eligio SÍ porque el proyecto no tiene frageles; confirmar.
    deformables = _si_no(par['DEFORMABLES'])
    fragiles = _si_no(par['FRAGILES'])
    if deformables is None:
        errores.append('PARAMETROS DEFORMABLES: elija SÍ o NO')
    if fragiles is None:
        errores.append('PARAMETROS FRAGILES: elija SÍ o NO')
    if str(par['VIENTO']).strip().upper() != VIENTO_NO:
        errores.append('PARAMETROS VIENTO: en esta version el viento no se considera (elija NO)')
    # DECISION D4 (abierta): Kz = 1.0 o 0.5 (AISC E4). El significado de cada opcion no esta confirmado.
    kz = _numero(par['KZ_TORSION'])
    if kz not in (1.0, 0.5):
        errores.append('PARAMETROS KZ_TORSION: elija 1.0 o 0.5 (factor Kz de pandeo torsional, AISC E4)')
    if str(par['METODO']).strip().upper() != METODO_ESTATICO:
        errores.append('PARAMETROS METODO: en esta version solo se usa el metodo estatico')
    if errores:
        return None, errores

    k1 = V['K1_SI'] if crujias else V['K1_NO']
    # DECISION D6 (abierta): Q' = 1 + (Q-1)/k con k del SASID. Con k = 1, Q' = Q. Si k < 1, Q' > Q (NTC 3.2, comentario).
    qp = 1.0 + (s['Q'] - 1.0) / num['K_SASID']                       # NTC 3.2.1 (rama Ta-Tb), con k del SASID
    if ks_metodo == KS_CONSERVADOR:
        # DECISION D7 (abierta): Ks = 1 por defecto (conservador). Alternativa: Ks por zona (tabla 3.1.1, eq. 3.1.1).
        ks, ks_txt = 1.0, 'Ks = 1 (conservador)'
    elif zona == 'A':
        ks, ks_txt = 1.0 / V['KS_A_DEN'], f'Ks = 1/{V["KS_A_DEN"]:g} (zona A)'
    elif zona == 'C':
        ks, ks_txt = 1.0 / V['KS_C_DEN'], f'Ks = 1/{V["KS_C_DEN"]:g} (zona C)'
    else:
        den = V['KS_B_A'] - V['KS_B_B'] * (num['TS'] - V['KS_B_C'])
        if den <= 0:
            errores.append('PARAMETROS TS: con Ts de la zona B, 1/[6-4(Ts-0.5)] no es valido; revise Ts')
            return None, errores
        ks, ks_txt = 1.0 / den, f'Ks = 1/[{V["KS_B_A"]:g}-{V["KS_B_B"]:g}(Ts-{V["KS_B_C"]:g})] (zona B)'
    conf = dict(
        sistema=sistema, Q=s['Q'], gSV=s['gSV'], gOI=s['gOI'], R0=s['R0'], k1=k1,
        crujias_ambas=crujias, zona=zona, ts=num['TS'], ta=num['TA'], tb=num['TB'],
        k_sasid=num['K_SASID'], c_sasid=num['C_SASID'], qp=qp,
        ks=ks, ks_metodo=ks_metodo, ks_txt=ks_txt,
        deformables=deformables, gLDNE=V['LDNE_DEFORMABLES'] if deformables else V['LDNE_NO'],
        fragiles=fragiles, lim_h=V['NTC_CA_H_FRAGILES'] if fragiles else V['NTC_CA_H_OTROS'],
        seg2=V['SEG2_FACTOR'], flecha_divisor=V['FLECHA_DIVISOR'],
        rp_sv=V['RP_SV'], rp_oi=V['RP_OI'], k2_coef=V['K2_COEF'], k2_exp=V['K2_EXP'], g=V['G'],
        viento=VIENTO_NO, metodo=METODO_ESTATICO, kz=kz)
    conf['R'] = k1 * s['R0']                                          # k2 se suma con el periodo (ver k2_de_T)
    return conf, []


def k2_de_T(T, conf):
    """k2 = 0.5 (1 - (T/Ta)^2) si T < Ta; 0 si T >= Ta (NTC 3.3.1b)."""
    if T is None or T >= conf['ta']:
        return 0.0
    return conf['k2_coef'] * (1.0 - (T / conf['ta']) ** conf['k2_exp'])


## Bloque 7. Leer, validar y armar el modelo

Lee la planilla subida a Kaggle o, si no hay ninguna, la de ejemplo. Si hay ERROR, no se arma el modelo.

In [ ]:
# Leer la planilla llenada, validar y armar el modelo
import glob
import os
import numpy as np
import pandas as pd
from dibujo_directo import DibujoModelo, ELEMENTOS, TIPOS_CONEXION, GRADOS

TOL = 1e-3
EJE_COLS = ['EJE_FUERTE_X', 'EJE_FUERTE_Y', 'EJE_FUERTE_Z']


def _vacio(v):
    return v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip() == ''


def _col(fila, nombre):
    """Valor de una columna; None si la planilla no tiene esa columna (planillas anteriores)."""
    return fila[nombre] if nombre in fila.index else None


def _tipo(v):
    return '' if _vacio(v) else str(v).strip().lower()


def buscar_planilla():
    """Busca la planilla llenada en /kaggle/input: cualquier .xlsx que tenga las hojas NODOS y BARRAS (el nombre del
    archivo no importa). Si hay mas de una se detiene: no se elige una al azar (R-09). Si hay .xlsx subidos que no son
    planillas, se detiene. Si no hay ninguno, usa la plantilla de ejemplo y lo dice en voz alta: ese no es su modelo."""
    xlsx = sorted(glob.glob('/kaggle/input/**/*.xlsx', recursive=True))
    planillas = []
    for f in xlsx:
        try:
            hojas = set(pd.ExcelFile(f).sheet_names)
        except Exception:
            hojas = set()
        if {'NODOS', 'BARRAS'} <= hojas:
            planillas.append(f)
    if len(planillas) > 1:
        raise RuntimeError(f'Hay {len(planillas)} planillas de soportes en /kaggle/input: deje solo una. '
                           + '; '.join(planillas))
    if planillas:
        return planillas[0]
    if xlsx:
        raise RuntimeError('En /kaggle/input hay .xlsx que no son planillas de soportes (falta la hoja NODOS o BARRAS): '
                           + '; '.join(xlsx))
    print('ATENCION: no hay planilla subida en /kaggle/input. Se usa la plantilla de ejemplo: NO es su modelo.')
    return RUTA_PLANTILLA


def leer_perfil_grupo(ruta):
    """Perfil elegido por grupo (hoja PERFIL_GRUPO de la planilla con diseno): {GRUPO: PERFIL}. {} si no existe."""
    try:
        g = pd.read_excel(ruta, sheet_name='PERFIL_GRUPO', dtype=object)
    except Exception:
        return {}
    salida = {}
    for _, r in g.iterrows():
        if not _vacio(r.get('GRUPO')) and not _vacio(r.get('PERFIL ELEGIDO')):
            salida[str(r['GRUPO']).strip().upper()] = str(r['PERFIL ELEGIDO']).strip()
    return salida


def leer_planilla(ruta):
    nodos = pd.read_excel(ruta, sheet_name='NODOS', dtype=object)
    barras = pd.read_excel(ruta, sheet_name='BARRAS', dtype=object)
    nodos = nodos[[not all(_vacio(v) for v in fila) for fila in nodos[['ID', 'X (m)', 'Y (m)', 'Z (m)']].values]]
    barras = barras[[not all(_vacio(v) for v in fila) for fila in barras[['ID', 'NODO_I', 'NODO_J']].values]]
    if 'DUCTILIDAD' not in barras.columns:          # planilla anterior: sin la columna DUCTILIDAD
        barras = barras.assign(DUCTILIDAD=None)
    # Con diseno, el perfil de cada barra lo da su grupo (PERFIL_GRUPO); la celda PERFIL de BARRAS es una formula.
    grupos = leer_perfil_grupo(ruta)
    if grupos:
        barras = barras.copy()
        barras['PERFIL'] = [grupos.get(str(e).strip().upper(), p) for e, p in zip(barras['ELEMENTO'], barras['PERFIL'])]
    return nodos, barras


def leer_materiales(ruta):
    """Hoja MATERIALES de la planilla, normalizada a [NOMBRE, TIPO, FY, FU, E, NU, FUENTE]. None si la planilla no
    tiene la hoja. Una columna que no exista (planilla anterior) queda vacia: el analisis la marca como error."""
    try:
        m = pd.read_excel(ruta, sheet_name='MATERIALES', dtype=object)
    except Exception:
        return None
    cols = {str(c).split(' (')[0].strip().upper(): c for c in m.columns}
    orden = ['NOMBRE', 'TIPO', 'FY', 'FU', 'E', 'NU', 'FUENTE']
    salida = []
    for _, r in m.iterrows():
        nombre = r[cols['NOMBRE']] if 'NOMBRE' in cols else r.iloc[0]
        if _vacio(nombre):
            continue
        salida.append([None if (k not in cols or _vacio(r[cols[k]])) else r[cols[k]] for k in orden])
    return salida


def _letra(n):
    from openpyxl.utils import get_column_letter
    return get_column_letter(n)


def leer_combinaciones(ruta, estados):
    """Hoja COMBINACIONES (informacion para el notebook). Devuelve (filas_crudas, combinaciones, mensajes):
    filas_crudas = factores tal como estan en la hoja (40 x 30); combinaciones = {nombre: {estado: factor}}."""
    msgs = []
    try:
        import openpyxl
        ws = openpyxl.load_workbook(ruta, data_only=False)['COMBINACIONES']
    except Exception:
        return [], {}, msgs
    nombres_estado = [str(v).strip() if not _vacio(v) else None for v in estados['NOMBRE'].values]
    filas_crudas, combinaciones = [], {}
    for k in range(40):
        fila_excel = 3 + k
        crudo = [ws.cell(row=fila_excel, column=2 + j).value for j in range(30)]
        filas_crudas.append(crudo)
        factores = {}
        for j, v in enumerate(crudo):
            if _vacio(v):
                continue
            try:
                f = float(v)
            except (TypeError, ValueError):
                msgs.append(('COMBINACIONES', fila_excel, 'ERROR', f'factor no numerico en la columna {_letra(2 + j)}'))
                continue
            if f == 0:
                continue
            if j >= len(nombres_estado) or nombres_estado[j] is None:
                msgs.append(('COMBINACIONES', fila_excel, 'ERROR', f'factor {f} en la columna {_letra(2 + j)}, que no tiene estado en ESTADOS'))
                continue
            factores[nombres_estado[j]] = f
        if factores:
            combinaciones[f'ASD{k + 1}'] = factores
    return filas_crudas, combinaciones, msgs


def validar(nodos, barras):
    """Lista de (hoja, fila_excel, tipo, mensaje). ERROR = no se arma el modelo; AVISO = informativo."""
    msgs = []
    def avisar(hoja, fila, tipo, texto):
        msgs.append((hoja, fila, tipo, texto))

    ids, coords = {}, []
    for k, r in nodos.iterrows():
        fila = k + 2
        try:
            id_ = int(float(r['ID']))
        except (TypeError, ValueError):
            avisar('NODOS', fila, 'ERROR', 'ID no es un entero'); continue
        if id_ in ids:
            avisar('NODOS', fila, 'ERROR', f'ID {id_} repetido'); continue
        try:
            p = (float(r['X (m)']), float(r['Y (m)']), float(r['Z (m)']))
        except (TypeError, ValueError):
            avisar('NODOS', fila, 'ERROR', 'coordenada X, Y o Z no numerica'); continue
        for q_id, q in coords:
            if all(abs(p[c] - q[c]) < TOL for c in range(3)):
                avisar('NODOS', fila, 'ERROR', f'coincide con el nodo {q_id}')
        ids[id_] = p
        coords.append((id_, p))
        for g in GRADOS:
            if _tipo(_col(r, g)) not in ('', 'si', 'no'):
                avisar('NODOS', fila, 'ERROR', f"{g} '{_col(r, g)}' no reconocido (use SI o NO; vacio = libre)")
        if _tipo(_col(r, 'REPORTAR')) not in ('', 'si', 'no'):
            avisar('NODOS', fila, 'AVISO', "REPORTAR debe ser SI o vacio: este nodo no sale en el Excel de resultados")

    vistos = set()
    for k, r in barras.iterrows():
        fila = k + 2
        try:
            n1, n2 = int(float(r['NODO_I'])), int(float(r['NODO_J']))
        except (TypeError, ValueError):
            avisar('BARRAS', fila, 'ERROR', 'NODO_I o NODO_J no es un entero'); continue
        if n1 not in ids or n2 not in ids:
            avisar('BARRAS', fila, 'ERROR', f'nodo inexistente en NODOS ({n1} o {n2})'); continue
        if n1 == n2:
            avisar('BARRAS', fila, 'ERROR', 'barra de un nodo consigo mismo'); continue
        par = frozenset((n1, n2))
        if par in vistos:
            avisar('BARRAS', fila, 'AVISO', f'barra {n1}-{n2} repetida: se ignora')
        vistos.add(par)
        elem = str(r['ELEMENTO']).strip().upper()
        if elem not in ELEMENTOS:
            avisar('BARRAS', fila, 'ERROR', f"ELEMENTO '{r['ELEMENTO']}' no reconocido")
        perfil = str(r['PERFIL']).strip()
        if perfil not in G.PERFILES:
            avisar('BARRAS', fila, 'ERROR', f"PERFIL '{perfil}' no existe en la base Gerdau")
        else:
            p = G.PERFILES[perfil]
            if p['d_ala_diametro_cm'] is None or p['bf_b_cm'] is None:
                avisar('BARRAS', fila, 'AVISO', f"perfil {perfil} sin dato geometrico (estado: {p['estado']})")
        # eje fuerte: vector libre. Una componente basta: las vacias valen 0. Todas vacias = por defecto.
        vec = [_col(r, c) for c in EJE_COLS]
        if any(not _vacio(v) for v in vec):
            try:
                v = np.array([0.0 if _vacio(x) else float(x) for x in vec])
            except (TypeError, ValueError):
                avisar('BARRAS', fila, 'ERROR', 'componente del eje fuerte no numerica'); v = None
            if v is not None:
                nv = np.linalg.norm(v)
                if nv < 1e-9:
                    avisar('BARRAS', fila, 'ERROR', 'eje fuerte nulo: indique una direccion o deje las tres vacias')
                elif n1 in ids and n2 in ids:
                    d = np.array(ids[n2]) - np.array(ids[n1])
                    if abs(np.dot(v / nv, d / np.linalg.norm(d))) > 0.999:
                        avisar('BARRAS', fila, 'ERROR', 'eje fuerte paralelo a la barra: elija otra direccion')
        material = '' if _vacio(_col(r, 'MATERIAL')) else str(_col(r, 'MATERIAL')).strip()
        if not material:
            avisar('BARRAS', fila, 'ERROR', 'falta MATERIAL: elija una calidad de la lista (hoja MATERIALES)')
        elif material not in MATERIALES_VALIDOS:
            avisar('BARRAS', fila, 'ERROR', f"MATERIAL '{material}' no existe en la hoja MATERIALES")
        duct_b = _col(r, 'DUCTILIDAD')
        if not _vacio(duct_b) and str(duct_b).strip().upper() not in ('ALTA', 'MEDIA'):
            avisar('BARRAS', fila, 'ERROR', 'DUCTILIDAD debe ser ALTA, MEDIA o vacio')
        for col in ('CONEXION_I', 'CONEXION_J'):
            c = _tipo(_col(r, col))
            if c and c not in TIPOS_CONEXION:
                avisar('BARRAS', fila, 'ERROR', f"{col} '{_col(r, col)}' no reconocida (use empotrado, rotulado o vacio)")
        if _tipo(_col(r, 'REPORTAR')) not in ('', 'si', 'no'):
            avisar('BARRAS', fila, 'AVISO', "REPORTAR debe ser SI o vacio: esta barra no sale en el Excel de resultados")
    return msgs


def construir_modelo(nodos, barras):
    """Arma el DibujoModelo con los IDs de la planilla. Devuelve (dib, id_de_indice, mensajes)."""
    dib = DibujoModelo(ruta=RUTA_PLANILLA_LEIDA)
    mapa, msgs = {}, []
    id_barra = {}                       # indice de barra en el modelo -> ID de la hoja BARRAS
    for _, r in nodos.iterrows():
        mapa[int(float(r['ID']))] = dib.nodo(float(r['X (m)']), float(r['Y (m)']), float(r['Z (m)']))
    for _, r in barras.iterrows():
        n1, n2 = mapa[int(float(r['NODO_I']))], mapa[int(float(r['NODO_J']))]
        ci = _tipo(_col(r, 'CONEXION_I')) or 'empotrado'
        cj = _tipo(_col(r, 'CONEXION_J')) or 'empotrado'
        material = '' if _vacio(_col(r, 'MATERIAL')) else str(_col(r, 'MATERIAL')).strip()
        try:
            idx = dib.barra(n1, n2, str(r['ELEMENTO']).strip().upper(), str(r['PERFIL']).strip(),
                            conexion_i=ci, conexion_j=cj, material=material)
        except ValueError as e:
            msgs.append(('BARRAS', int(float(r['ID'])), 'ERROR', str(e))); continue
        vec = [_col(r, c) for c in EJE_COLS]
        if idx is not None:
            id_barra[idx] = int(float(r['ID']))
        if idx is not None and not any(not _vacio(v) for v in vec):
            msgs.append(('BARRAS', int(float(r['ID'])), 'ERROR',
                         'falta el EJE FUERTE: indique EJE_FUERTE_X/Y/Z (direccion del peralte) en esta barra'))
        if idx is not None and any(not _vacio(v) for v in vec):
            try:
                dib.eje_fuerte(idx, tuple(0.0 if _vacio(v) else float(v) for v in vec))
            except ValueError as e:
                msgs.append(('BARRAS', int(float(r['ID'])), 'ERROR', str(e)))
    for _, r in nodos.iterrows():
        restr = [_tipo(_col(r, g)) == 'si' for g in GRADOS]          # UX, UY, UZ, RX, RY, RZ marcados SI
        if any(restr):
            dib.apoyo(mapa[int(float(r['ID']))], restr)
    id_de_indice = {idx: id_ for id_, idx in mapa.items()}
    dib.id_barra_de_indice = id_barra
    return dib, id_de_indice, msgs


CARGA_FILA0 = 9
CARGA_COLS_LEER = ['ID', 'ESTADO', 'FX', 'FY', 'FZ', 'MX', 'MY', 'MZ', 'NOTA']


def leer_cargas(ruta):
    """Hoja CARGAS: una fila por nodo y estado. FX, FY, FZ en kgf; MX, MY, MZ en kgf·m (ejes globales X, Y, Z)."""
    try:
        c = pd.read_excel(ruta, sheet_name='CARGAS', header=CARGA_FILA0 - 2, dtype=object)
    except Exception:
        return pd.DataFrame(columns=CARGA_COLS_LEER)
    c = c.iloc[:, :9]
    c.columns = CARGA_COLS_LEER
    return c[[not all(_vacio(v) for v in fila) for fila in c[['ID', 'ESTADO']].values]]


def leer_estados(ruta):
    """Hoja ESTADOS: nombre del estado (lo define el usuario), tipo, descripcion y PESO PROPIO (SI/NO).
    Si la planilla no tiene la columna PESO PROPIO (planilla anterior), queda vacia: el analisis la marca."""
    columnas = ['NOMBRE', 'TIPO', 'DESCRIPCION', 'PESO PROPIO']
    try:
        e = pd.read_excel(ruta, sheet_name='ESTADOS', dtype=object)
    except Exception:
        return pd.DataFrame(columns=columnas)
    filas = []
    for _, r in e.iterrows():
        if _vacio(r.iloc[0]):
            continue
        pp = r.iloc[3] if e.shape[1] > 3 and str(e.columns[3]).strip().upper().startswith('PESO') else None
        filas.append([r.iloc[0], r.iloc[1] if e.shape[1] > 1 else None,
                      r.iloc[2] if e.shape[1] > 2 else None, pp])
    return pd.DataFrame(filas, columns=columnas)


def leer_longitudes(ruta):
    """Hoja LONGITUDES: filas alineadas con BARRAS (fila 2 = primera barra). Vacia si la planilla no la tiene.
    Las columnas se leen por nombre: una planilla anterior sin las columnas nuevas las toma vacias (NO / supuesto)."""
    try:
        l = pd.read_excel(ruta, sheet_name='LONGITUDES', dtype=object)
    except Exception:
        return pd.DataFrame(columns=LONG_COLS)
    l = l.reindex(columns=LONG_COLS)        # por nombre; columnas que falten quedan vacias
    # el ID sale de BARRAS (formula). Si el archivo no trae el valor calculado, se toma por posicion de fila.
    try:
        ids_barras = pd.read_excel(ruta, sheet_name='BARRAS', dtype=object)['ID'].tolist()
    except Exception:
        ids_barras = []
    ids = []
    for k in range(len(l)):
        v = l['ID'].iloc[k]
        ids.append(v if not _vacio(v) else (ids_barras[k] if k < len(ids_barras) else None))
    l['ID'] = ids
    return l


def leer_parametros(ruta):
    """Hoja PARAMETROS: {CLAVE: valor} (columna VALOR). None si la planilla no tiene la hoja (plantilla anterior)."""
    import openpyxl
    try:
        ws = openpyxl.load_workbook(ruta, data_only=True)['PARAMETROS']
    except KeyError:
        return None
    par = {}
    for fila in range(5, ws.max_row + 1):
        clave = ws.cell(row=fila, column=1).value
        if _vacio(clave):
            continue
        par[str(clave).strip()] = ws.cell(row=fila, column=3).value
    return par


def leer_norma(ruta):
    """Hoja NORMA: {'SISTEMAS': {nombre: {Q, gSV, gOI, R0}}, 'VALORES': {CLAVE: valor}}. None si falta la hoja."""
    import openpyxl
    try:
        ws = openpyxl.load_workbook(ruta, data_only=True)['NORMA']
    except KeyError:
        return None
    sistemas, valores, modo = {}, {}, None
    for fila in range(1, ws.max_row + 1):
        a = ws.cell(row=fila, column=1).value
        if _vacio(a):
            continue
        t = str(a).strip()
        if t == 'NOMBRE DEL SISTEMA':
            modo = 'S'
            continue
        if t == 'CLAVE':
            modo = 'V'
            continue
        if t.startswith(('NORMA:', 'SISTEMAS ESTRUCTURALES', 'VALORES DE NORMA')):
            modo = None
            continue
        if modo == 'S':
            sistemas[t] = {'Q': float(ws.cell(row=fila, column=2).value), 'gSV': float(ws.cell(row=fila, column=3).value),
                           'gOI': float(ws.cell(row=fila, column=4).value), 'R0': float(ws.cell(row=fila, column=5).value)}
        elif modo == 'V':
            valores[t] = float(ws.cell(row=fila, column=2).value)
    return {'SISTEMAS': sistemas, 'VALORES': valores}


def validar_datos_diseno(barras, longitudes):
    """Datos obligatorios para el diseno de cada barra: LX, LY, LB, KX, KY, CB en LONGITUDES y DUCTILIDAD en
    columnas y vigas (BARRAS). Sin estos datos no se supone nada: aparece como ERROR."""
    msgs = []
    lon = {int(float(r['ID'])): r for _, r in longitudes.iterrows() if not _vacio(r['ID'])}
    for k, r in barras.iterrows():
        if _vacio(r['ID']):
            continue
        id_ = int(float(r['ID']))
        fila = k + 2
        lr = lon.get(id_)
        # DECISION D17 (abierta): LX, LY, LB, KX, KY y CB obligatorios; alternativa: valores por defecto marcados.
        for campo in ['LX (m)', 'LY (m)', 'LB (m)', 'KX', 'KY', 'CB']:
            if lr is None or _vacio(lr[campo]):
                msgs.append(('LONGITUDES', fila, 'ERROR', f'barra {id_}: falta {campo} (dato obligatorio para el diseno)'))
        elemento = str(r['ELEMENTO']).strip().upper() if not _vacio(r['ELEMENTO']) else ''
        # DECISION D19 (abierta): DUCTILIDAD obligatoria solo en COLUMNA y VIGA; ARRIOSTRE y DIAGONAL opcionales.
        if elemento in ('COLUMNA', 'VIGA') and _vacio(r['DUCTILIDAD']):
            msgs.append(('BARRAS', fila, 'ERROR', f'barra {id_} ({elemento}): falta DUCTILIDAD (ALTA o MEDIA)'))
    return msgs


def leer_filas_parametros(ruta):
    """Filas de PARAMETROS para la memoria: (concepto, clave, valor, unidad, referencia)."""
    import openpyxl
    try:
        ws = openpyxl.load_workbook(ruta, data_only=True)['PARAMETROS']
    except KeyError:
        return []
    filas = []
    for fila in range(5, ws.max_row + 1):
        clave = ws.cell(row=fila, column=1).value
        if _vacio(clave):
            continue
        vacio_a = lambda v: '' if v is None else v
        filas.append((vacio_a(ws.cell(row=fila, column=2).value), str(clave).strip(),
                      vacio_a(ws.cell(row=fila, column=3).value), vacio_a(ws.cell(row=fila, column=4).value),
                      vacio_a(ws.cell(row=fila, column=6).value)))
    return filas


def leer_filas_norma(ruta):
    """Filas de NORMA (valores de norma) para la memoria: (clave, valor, unidad, referencia)."""
    import openpyxl
    try:
        ws = openpyxl.load_workbook(ruta, data_only=True)['NORMA']
    except KeyError:
        return []
    filas, en_valores = [], False
    for fila in range(1, ws.max_row + 1):
        a = ws.cell(row=fila, column=1).value
        if _vacio(a):
            continue
        t = str(a).strip()
        if t == 'CLAVE':
            en_valores = True
            continue
        v = ws.cell(row=fila, column=2).value
        if en_valores and isinstance(v, (int, float)):
            filas.append((t, float(v), ws.cell(row=fila, column=3).value or '', ws.cell(row=fila, column=4).value or ''))
    return filas


def validar_longitudes(longitudes):
    """Valores de LONGITUDES: numericos y mayores que cero. Vacio no es error: el diseño lo marca como supuesto."""
    msgs = []
    campos = ['LX (m)', 'LY (m)', 'KX', 'KY', 'LB (m)', 'CB']
    for k, r in longitudes.iterrows():
        fila = k + 2
        if _vacio(r['ID']):
            if any(not _vacio(r[c]) for c in campos):
                msgs.append(('LONGITUDES', fila, 'AVISO', 'hay valores en una fila sin barra en BARRAS: no se usan'))
            continue
        for c in ['E5 UNA PATA', 'E5 UNION SOLDADA O 2 PERNOS', 'E5 SIN CARGA INTERMEDIA', 'E5 INDIVIDUAL O ALMA DE ARMADURA']:
            if not _vacio(r[c]) and str(r[c]).strip().upper() not in ('SI', 'NO'):
                msgs.append(('LONGITUDES', fila, 'ERROR', f'{c} debe ser SI, NO o vacio'))
        if not _vacio(r['LI CONEXION']) and str(r['LI CONEXION']).strip().upper() not in ('UNA PATA', 'DOS PATAS'):
            msgs.append(('LONGITUDES', fila, 'ERROR', 'LI CONEXION debe ser UNA PATA, DOS PATAS o vacio'))
        for c in campos:
            if _vacio(r[c]):
                continue
            try:
                v = float(r[c])
            except (TypeError, ValueError):
                msgs.append(('LONGITUDES', fila, 'ERROR', f'{c} no es numerico'))
                continue
            if v <= 0:
                msgs.append(('LONGITUDES', fila, 'ERROR', f'{c} debe ser mayor que cero'))
    return msgs


def validar_cargas(cargas, estados, ids_nodo):
    """Valida CARGAS contra NODOS y ESTADOS. Devuelve (mensajes, sumas) con sumas[(id_nodo, estado)] = [6 valores]."""
    msgs, sumas = [], {}
    nombres = {str(v).strip() for v in estados['NOMBRE'].values if not _vacio(v)}
    for k, r in cargas.iterrows():
        fila = k + CARGA_FILA0
        try:
            id_ = int(float(r['ID']))
        except (TypeError, ValueError):
            msgs.append(('CARGAS', fila, 'ERROR', 'ID NODO no es un entero'))
            continue
        if id_ not in ids_nodo:
            msgs.append(('CARGAS', fila, 'ERROR', f'el nodo {id_} no existe en NODOS'))
            continue
        est = str(r['ESTADO']).strip()
        if est not in nombres:
            msgs.append(('CARGAS', fila, 'ERROR', f"el estado '{est}' no existe en la hoja ESTADOS"))
            continue
        try:
            valores = [0.0 if _vacio(r[c]) else float(r[c]) for c in ['FX', 'FY', 'FZ', 'MX', 'MY', 'MZ']]
        except (TypeError, ValueError):
            msgs.append(('CARGAS', fila, 'ERROR', 'fuerza o momento no numerico'))
            continue
        clave = (id_, est)
        if clave in sumas:
            msgs.append(('CARGAS', fila, 'AVISO', f'el nodo {id_} ya tiene el estado {est}: se suman'))
            sumas[clave] = [a + b for a, b in zip(sumas[clave], valores)]
        else:
            sumas[clave] = valores
    return msgs, sumas


def adjuntar_cargas(dib, sumas, id_de_indice):
    """dib.cargas = {estado: {indice_nodo: [Fx, Fy, Fz (kgf), Mx, My, Mz (kgf·cm)]}}.
    Los momentos se pasan de kgf·m a kgf·cm porque el motor de OpenSees trabaja en kgf y kgf·cm."""
    idx_de_id = {v: k for k, v in id_de_indice.items()}
    dib.cargas = {}
    for (id_, est), v in sumas.items():
        fx, fy, fz, mx, my, mz = v
        dib.cargas.setdefault(est, {})[idx_de_id[id_]] = [fx, fy, fz, mx * 100.0, my * 100.0, mz * 100.0]


def filas_longitudes_para_excel(longitudes):
    """Filas de la hoja LONGITUDES (todas las columnas salvo ID, que es formula), para regenerar el Excel."""
    filas = []
    for _, r in longitudes.iterrows():
        filas.append([None if _vacio(r[c]) else r[c] for c in LONG_COLS[1:]])
    return filas


def filas_cargas_para_excel(cargas):
    """Filas de la hoja CARGAS para regenerar el Excel con sus datos."""
    filas = []
    for _, r in cargas.iterrows():
        try:
            id_ = int(float(r['ID']))
        except (TypeError, ValueError):
            id_ = None
        vals = [0.0 if _vacio(r[c]) else float(r[c]) for c in ['FX', 'FY', 'FZ', 'MX', 'MY', 'MZ']]
        filas.append([id_, None if _vacio(r['ESTADO']) else str(r['ESTADO']).strip(), *vals,
                      None if _vacio(r['NOTA']) else r['NOTA']])
    return filas


RUTA_PLANILLA_LEIDA = buscar_planilla()   # o escriba la ruta exacta, por ejemplo '/kaggle/input/mi-dataset/mis_soportes.xlsx'
print('Planilla que se lee:', RUTA_PLANILLA_LEIDA)
nodos, barras = leer_planilla(RUTA_PLANILLA_LEIDA)
materiales_leidos = leer_materiales(RUTA_PLANILLA_LEIDA)     # None = la planilla no tiene la hoja MATERIALES
materiales_filas = materiales_leidos or MATERIALES_DEFAULT   # solo para dibujar y validar nombres; el analisis usa materiales_leidos
MATERIALES_VALIDOS = {str(f[0]).strip() for f in materiales_filas}
mensajes = validar(nodos, barras)
cargas = leer_cargas(RUTA_PLANILLA_LEIDA)
estados_df = leer_estados(RUTA_PLANILLA_LEIDA)
longitudes_df = leer_longitudes(RUTA_PLANILLA_LEIDA)
# DUCTILIDAD (ALTA o MEDIA) se marca en BARRAS; el diseno la lee por ID desde la tabla de longitudes
_duct_por_id = {int(float(r['ID'])): r['DUCTILIDAD'] for _, r in barras.iterrows() if not _vacio(r['ID'])}
longitudes_df['DUCTILIDAD'] = [None if _vacio(v) else _duct_por_id.get(int(float(v)), None)
                               for v in longitudes_df['ID']]
ids_nodo = set()
for _v in nodos['ID'].values:
    try:
        ids_nodo.add(int(float(_v)))
    except (TypeError, ValueError):
        pass
msgs_cargas, sumas_cargas = validar_cargas(cargas, estados_df, ids_nodo)
mensajes += msgs_cargas
comb_crudas, combinaciones_modelo, msgs_comb = leer_combinaciones(RUTA_PLANILLA_LEIDA, estados_df)
mensajes += msgs_comb
mensajes += validar_longitudes(longitudes_df)
mensajes += validar_datos_diseno(barras, longitudes_df)
import parametros as P
PAR_LEIDA = leer_parametros(RUTA_PLANILLA_LEIDA)     # hoja PARAMETROS (listas desplegables del Excel)
NOR_LEIDA = leer_norma(RUTA_PLANILLA_LEIDA)          # hoja NORMA (valores de norma con su referencia)
CONF = None
if PAR_LEIDA is None or NOR_LEIDA is None:
    mensajes.append(('PARAMETROS', '', 'ERROR', 'la planilla no tiene las hojas PARAMETROS y NORMA: use la plantilla nueva'))
else:
    CONF, errores_par = P.resolver(PAR_LEIDA, NOR_LEIDA)
    mensajes += [('PARAMETROS', '', 'ERROR', e) for e in errores_par]
    if CONF is not None:
        CONF['par_filas'] = leer_filas_parametros(RUTA_PLANILLA_LEIDA)
        CONF['nor_filas'] = leer_filas_norma(RUTA_PLANILLA_LEIDA)
dib, id_de_indice, extra = (None, None, [])
if not any(t == 'ERROR' for _, _, t, _ in mensajes):
    dib, id_de_indice, extra = construir_modelo(nodos, barras)
    mensajes += extra
    if any(t == 'ERROR' for _, _, t, _ in extra):
        dib = None                  # un error al construir deja el modelo incompleto: no se entrega
if dib is not None:
    adjuntar_cargas(dib, sumas_cargas, id_de_indice)
    dib.combinaciones = combinaciones_modelo          # informacion para el notebook: {nombre: {estado: factor}}
    print(f'Combinaciones de carga: {len(combinaciones_modelo)}')
    print('Estados de carga:', [str(v) for v in estados_df['NOMBRE'].values])
    print(f'Asignaciones nodo-estado: {len(sumas_cargas)}')
resumen = pd.DataFrame(mensajes, columns=['hoja', 'fila_excel', 'tipo', 'mensaje'])
if resumen.empty:
    print('Planilla sin observaciones.')
else:
    display(resumen)
if dib is not None:
    print(f'Modelo armado: {len(dib.nodos)} nodos, {len(dib.barras)} barras, {len(dib.apoyos)} apoyos')
    modelo = dib.a_modelo()        # formato leer_dxf(): listo para el motor de analisis
else:
    print('Modelo NO armado: corrija los ERROR de la tabla y vuelva a ejecutar este bloque.')


## Bloque 8. Dibujo de revisión y planilla regenerada

Figura planta, alzado y 3D, y `planilla_con_vista.xlsx`. Ejecutar después de cada cambio.

In [ ]:
# Dibujo de revision (planta, alzado, 3D con secciones y ejes fuertes) y planilla regenerada con la vista
import math
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from orientacion_gerdau import contorno, ejes_locales, dimensiones
from dibujo_directo import marca_apoyo
from vista3d_plotly import figura_3d

COLOR = {'COLUMNA': '#1f77b4', 'VIGA': '#2ca02c', 'ARRIOSTRE': '#d62728', 'DIAGONAL': '#ff7f0e'}
ESCALA_SECCION = 2.5    # las secciones se dibujan ampliadas en la figura (no cambia el calculo)


def plano_con_mas_barras(dib, planta):
    """Nivel Z (planta) o cota Y (alzado) con mas barras en el plano."""
    from collections import Counter
    cuenta = Counter()
    for b in dib.barras:
        p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
        if planta and abs(p1[2] - p2[2]) < 1e-9:
            cuenta[round(p1[2], 6)] += 1
        if not planta and abs(p1[1] - p2[1]) < 1e-9:
            cuenta[round(p1[1], 6)] += 1
    return cuenta.most_common(1)[0][0] if cuenta else min(p[2] if planta else p[1] for p in dib.nodos)


def dibujar_vista(ax, dib, id_de_indice, vista='planta', plano=None):
    planta = vista == 'planta'
    if plano is None:
        plano = plano_con_mas_barras(dib, planta)

    def en_plano(p):
        return abs(p[2] - plano) < 1e-9 if planta else abs(p[1] - plano) < 1e-9

    def coord(p):
        return (p[0], p[1]) if planta else (p[0], p[2])

    for b in dib.barras:
        p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
        if not (en_plano(p1) and en_plano(p2)):
            continue
        (x1, y1), (x2, y2) = coord(p1), coord(p2)
        ax.plot([x1, x2], [y1, y2], color=COLOR[b['elemento']], lw=3.5, zorder=2, solid_capstyle='round')
        for p, c in [(p1, b['conexion_i']), (p2, b['conexion_j'])]:
            if c == 'rotulado':
                ax.plot(*coord(p), 'o', mfc='white', mec='k', ms=7, zorder=4)
    for i, q in enumerate(dib.nodos):
        if en_plano(q):
            ax.plot(*coord(q), 'ko', ms=4, zorder=5)
            ax.text(coord(q)[0], coord(q)[1], f' {id_de_indice[i]}', fontsize=8, zorder=6)
            if i in dib.apoyos:
                marca = 's' if marca_apoyo(dib.apoyos[i]) == 'empotrado' else '^'
                ax.plot(*coord(q), marca, color='#e67e22', ms=10, zorder=5)
    ax.set_aspect('equal'); ax.margins(0.12); ax.grid(True, color='#dddddd', lw=0.6)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)' if planta else 'Z (m)')
    ax.set_title(f"{'Planta' if planta else 'Alzado'} en {'Z' if planta else 'Y'} = {plano:g} m", fontsize=10)


def dibujar_3d(ax, dib, id_de_indice):
    for i, b in enumerate(dib.barras):
        p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color=COLOR[b['elemento']], lw=2.5)
        for p, c in [(p1, b['conexion_i']), (p2, b['conexion_j'])]:
            if c == 'rotulado':
                ax.scatter(*p, s=40, facecolors='white', edgecolors='k', zorder=5)
        # seccion (envolvente del perfil) en el punto medio, y eje fuerte = eje local y (peralte)
        caja = contorno(dib, i, ESCALA_SECCION)
        if caja:
            ax.add_collection3d(Poly3DCollection([caja], facecolor=COLOR[b['elemento']], alpha=0.35,
                                                 edgecolor='k', linewidths=0.4))
            pm = [(p1[k] + p2[k]) / 2 for k in range(3)]
            d = dimensiones(b['perfil'])[0] * ESCALA_SECCION
            _, ey, _ = ejes_locales(dib, i)
            ax.quiver(*pm, *(ey * 0.6 * d), color='k', arrow_length_ratio=0.3, linewidth=1.2)
    for i, q in enumerate(dib.nodos):
        ax.text(q[0], q[1], q[2], f' {id_de_indice[i]}', fontsize=7)
        if i in dib.apoyos:
            ax.scatter(*q, s=60, marker='s' if marca_apoyo(dib.apoyos[i]) == 'empotrado' else '^', color='#e67e22', zorder=6)
        else:
            ax.scatter(*q, s=8, color='k', zorder=6)
    rangos = [max(max(p[k] for p in dib.nodos) - min(p[k] for p in dib.nodos), 1.0) for k in range(3)]
    ax.set_box_aspect(rangos)
    from matplotlib.ticker import MaxNLocator
    for eje in (ax.xaxis, ax.yaxis, ax.zaxis):
        eje.set_major_locator(MaxNLocator(4))
    ax.view_init(elev=22, azim=-60)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
    ax.set_title(f'3D: secciones x{ESCALA_SECCION:g}, flecha = eje fuerte, circulo blanco = extremo rotulado, '
                 'cuadrado = empotrado, triangulo = apoyo rotulado', fontsize=8)


if dib is None:
    print('Sin modelo para dibujar: corrija la planilla (bloque 7).')
else:
    fig = plt.figure(figsize=(16, 5.8))
    dibujar_vista(fig.add_subplot(1, 3, 1), dib, id_de_indice, 'planta')
    dibujar_vista(fig.add_subplot(1, 3, 2), dib, id_de_indice, 'alzado')
    dibujar_3d(fig.add_subplot(1, 3, 3, projection='3d'), dib, id_de_indice)
    fig.tight_layout()
    plt.show()

    # 3D interactiva (gira con el mouse; al pasar el cursor muestra ID, perfil y eje fuerte de cada barra)
    figura_3d(dib, id_de_indice, escala_seccion=2.5).show(renderer='iframe')

    # Planilla regenerada con sus datos (las hojas de calculo y graficos se rehacen)
    escribir_planilla('planilla_con_vista.xlsx', _filas(nodos, NODOS_COLS), _filas(barras, BARRAS_COLS),
                      filas_cargas=filas_cargas_para_excel(cargas), materiales=materiales_filas,
                      combinaciones=comb_crudas,
                      longitudes=filas_longitudes_para_excel(longitudes_df),
                      estados=[[None if _vacio(v) else v for v in f] for f in estados_df.values.tolist()])
    guardar_con_valores('planilla_con_vista.xlsx')
    print('Planilla regenerada: planilla_con_vista.xlsx (hoja VISTA_VIVA)')


## Bloque 9. Análisis con OpenSees y Excel de resultados

Los datos vienen de la planilla: E y NU (MATERIALES), sección (base Gerdau), apoyos y rótulas, cargas, peso propio (ESTADOS) y combinaciones (COMBINACIONES). Solo sale lo marcado REPORTAR = SI. Escribe `resultados_soportes.xlsx`.

In [ ]:
%%writefile analisis_opensees.py
"""
ANALISIS LINEAL 3D CON OPENSEES (openseespy) A PARTIR DE LA PLANILLA. El Excel manda:
  - E y nu por calidad: hoja MATERIALES (columnas E y NU).
  - Seccion (A, Ix fuerte, Iy debil, J, masa): base Gerdau rev04 (no ICHA).
  - Apoyos: NODOS (UX, UY, UZ, RX, RY, RZ = SI restringe ese grado) y rotulas de extremo: BARRAS (CONEXION_I/J).
  - Cargas por nodo y estado: hoja CARGAS (kgf, kgf·m).
  - Peso propio: hoja ESTADOS (PESO PROPIO = SI), calculado con la masa Gerdau (kgf/m).
  - Combinaciones: hoja COMBINACIONES (superposicion lineal de los estados).
  - Que reportar: columna REPORTAR (SI) en NODOS y BARRAS.

Unidades internas: cm, kgf, kgf·cm. Salida: kgf, kgf·m, mm, rad.

Convencion de ejes (verificada en OpenSees 3.8 con una viga de prueba):
  x = del nodo I al J; y = vecxz x x (peralte del eje fuerte); z = x x y.
  El eje fuerte es el local y: Iz (OpenSees) = Ix Gerdau; Iy = Iy Gerdau.
  Cortante fuerte = Vy; momento fuerte = Mz; cortante debil = Vz; momento debil = My.
"""
import math
import os
import tempfile
import numpy as np
import pandas as pd
import openseespy.opensees as ops
from orientacion_gerdau import vecxz_de_barra, ejes_locales
import perfiles_gerdau as G
from dibujo_directo import etiqueta_restricciones

GRAVEDAD = np.array([0.0, 0.0, -1.0])        # el peso propio actua hacia -Z global
ETIQUETA = ['N', 'Vy', 'Vz', 'T', 'My', 'Mz']


class ErrorAnalisis(Exception):
    pass


def _num(v):
    """Numero o None (vacio, NaN o texto no numerico)."""
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def _marcado(v):
    """True si la celda REPORTAR dice SI."""
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return False
    return str(v).strip().upper() == 'SI'


def materiales_desde_filas(filas):
    """{nombre: {'E', 'nu'}} desde las filas de MATERIALES (NOMBRE, TIPO, FY, FU, E, NU, FUENTE)."""
    out = {}
    for f in filas:
        nombre = str(f[0]).strip()
        out[nombre] = {'E': _num(f[4]) if len(f) > 4 else None, 'nu': _num(f[5]) if len(f) > 5 else None}
    return out


def estados_desde_df(df):
    """[(nombre, peso_propio)] en el orden de la hoja ESTADOS. PESO PROPIO debe ser SI o NO."""
    salida = []
    for _, r in df.iterrows():
        nombre = str(r['NOMBRE']).strip()
        v = r.get('PESO PROPIO')
        v = None if v is None or (isinstance(v, float) and math.isnan(v)) else str(v).strip().upper()
        if v not in ('SI', 'NO'):
            raise ErrorAnalisis(f"ESTADOS, estado '{nombre}': PESO PROPIO debe ser SI o NO "
                                f"(si la columna no existe, use la plantilla nueva del bloque 6)")
        salida.append((nombre, v == 'SI'))
    return salida


def propiedades_barras(dib, materiales, id_barra):
    """Propiedades de cada barra: seccion desde la base Gerdau y E, nu desde MATERIALES. Lanza ErrorAnalisis
    con la lista completa de datos faltantes (no sustituye valores por cero)."""
    props, faltas = [], []
    for i, b in enumerate(dib.barras):
        etiqueta = f"barra {id_barra.get(i, i)} ({b['perfil']})"
        p = G.obtener_perfil(b['perfil']) or {}
        falta = [nombre for nombre, clave in [('area A', 'A'), ('Ix', 'Ix'), ('Iy', 'Iy'), ('J', 'J'),
                                               ('masa', 'masa')] if p.get(clave) is None]
        m = materiales.get(b.get('material'))
        if m is None:
            falta.append('material en MATERIALES')
        else:
            if m['E'] is None:
                falta.append(f"E del material {b.get('material')}")
            if m['nu'] is None:
                falta.append(f"NU del material {b.get('material')}")
        if falta:
            faltas.append(f"{etiqueta}: falta {', '.join(falta)}")
            props.append(None)
            continue
        props.append({'A': p['A'], 'Iz': p['Ix'], 'Iy': p['Iy'], 'J': p['J'], 'masa': p['masa'],
                      'E': m['E'], 'nu': m['nu']})
    if faltas:
        raise ErrorAnalisis('Datos incompletos para el analisis (no se sustituye ningun valor):\n  '
                            + '\n  '.join(faltas))
    return props


def _verificar_conectividad(dib):
    usados = set()
    for b in dib.barras:
        usados.update((b['n1'], b['n2']))
    sueltos = [i for i in range(len(dib.nodos)) if i not in usados]
    if sueltos:
        raise ErrorAnalisis(f'nodos sin barras (no forman parte del modelo): {sueltos}. '
                            'Quitelos de NODOS o conectelos.')


ALFA_PENALIZACION = 1e12   # con rotulas, las igualdades se imponen por penalizacion (error relativo ~1e-7)


def _restricciones(dib):
    """Transformacion exacta, salvo que una barra rotulada sea oblicua al sistema global: su restriccion de giro
    es lineal y se impone por penalizacion (error relativo ~1e-7)."""
    oblicua = False
    for b in dib.barras:
        if b.get('conexion_i') == 'rotulado' or b.get('conexion_j') == 'rotulado':
            p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
            e = (p2 - p1) / np.linalg.norm(p2 - p1)
            oblicua = oblicua or abs(e).max() < 1.0 - 1e-9
    if oblicua:
        ops.constraints('Penalty', ALFA_PENALIZACION, ALFA_PENALIZACION)
    else:
        ops.constraints('Transformation')


def _analizar():
    """Analisis con captura del aviso del solver. OpenSees devuelve 0 aunque la factorizacion falle con matriz
    singular: el aviso sale por la salida de error y se lee aqui."""
    with tempfile.TemporaryFile(mode='w+b') as aviso_f:
        guardado = os.dup(2)
        os.dup2(aviso_f.fileno(), 2)
        try:
            rc = ops.analyze(1)
        finally:
            os.dup2(guardado, 2)
            os.close(guardado)
        aviso_f.seek(0)
        aviso = aviso_f.read().decode('utf-8', 'replace')
    return rc, aviso


def _hay_singular(aviso):
    return 'singular' in aviso.lower() or 'failed' in aviso.lower()


def _nudos_rotulados(dib):
    """Conteo de conexiones por nudo y eje de cada barra rotulada. Devuelve (directas, rotulas, e_rotula, fijos).
    fijos: giros en el plano de nudos con una sola barra rotulada y ninguna otra conexion. Esos giros no pueden
    definirse (no hay elemento que los sujete): se fijan, salvo el giro alrededor de la barra."""
    directas, rotulas, e_rotula = {}, {}, {}
    for b in dib.barras:
        p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
        e = (p2 - p1) / np.linalg.norm(p2 - p1)
        for clave, nodo in (('conexion_i', b['n1']), ('conexion_j', b['n2'])):
            if b.get(clave) == 'rotulado':
                rotulas[nodo] = rotulas.get(nodo, 0) + 1
                e_rotula[nodo] = e
            else:
                directas[nodo] = directas.get(nodo, 0) + 1
    fijos = {}
    for nodo, e in e_rotula.items():
        apo = dib.apoyos.get(nodo, (False,) * 6)
        if nodo not in directas and rotulas.get(nodo) == 1 and not any(apo[3:]):
            k = int(np.argmax(np.abs(e)))
            fijos[nodo] = [3 + j for j in range(3) if j != k and abs(e[j]) < 1e-12]
    return directas, rotulas, e_rotula, fijos


def _modelo(dib, props, cargas_nodales, pp):
    """Construye el modelo OpenSees del caso: nodos, apoyos, barras (con rotulas), cargas nodales y peso propio."""
    ops.wipe()
    # DECISION D12 (abierta): analisis lineal de primer orden (sin P-Delta).
    ops.model('basic', '-ndm', 3, '-ndf', 6)
    coords = [tuple(float(c) * 100.0 for c in p) for p in dib.nodos]          # m -> cm
    for i, (x, y, z) in enumerate(coords):
        ops.node(i + 1, x, y, z)
    directas, rotulas, e_rotula, fijos = _nudos_rotulados(dib)
    for i in sorted(set(dib.apoyos) | set(fijos)):     # restr = (UX, UY, UZ, RX, RY, RZ) de la hoja NODOS
        restr = list(dib.apoyos.get(i, (False,) * 6))
        for d in fijos.get(i, []):                     # giros en el plano de un nudo sin otra conexion
            restr[d] = True
        ops.fix(i + 1, *[int(v) for v in restr])
    siguiente = len(coords) + 1
    reac_aux = {}          # nodo de la planilla -> etiquetas de extremos articulados apoyados en ese nodo
    def extremo_rotulado(nodo_orig, e):
        """Extremo de barra articulado (rotula en el plano): nodo auxiliar. Las traslaciones siguen al nudo (las
        restringidas del nudo se aplican al auxiliar; su reaccion se suma a la del nudo). El giro alrededor de la barra
        se transmite del nudo al auxiliar; los giros en el plano quedan libres. Barra alineada con un eje global: el
        giro se liga por igualdad (exacto). Barra oblicua: restriccion lineal (penalizacion, ver _restricciones)."""
        nonlocal siguiente
        ops.node(siguiente, *coords[nodo_orig])
        tr = dib.apoyos[nodo_orig][:3] if nodo_orig in dib.apoyos else (False, False, False)
        libres = [d + 1 for d in range(3) if not tr[d]]          # traslaciones que siguen al nudo
        k = int(np.argmax(np.abs(e)))
        alineada = abs(e[k]) >= 1.0 - 1e-9
        ligados = libres + ([4 + k] if alineada else [])
        if ligados:
            ops.equalDOF(nodo_orig + 1, siguiente, *ligados)
        flags = [int(v) for v in tr] + [0, 0, 0]
        if not alineada:
            # (giro auxiliar . e) - (giro del nudo . e) = 0 escrito sobre el giro k; los giros del nudo fijados valen cero
            fijo = dib.apoyos.get(nodo_orig, (False,) * 6)[3:]
            termina = [(siguiente, 4 + j, float(e[j] / e[k])) for j in range(3) if j != k and abs(e[j]) >= 1e-12]
            termina += [(nodo_orig + 1, 4 + j, -float(e[j] / e[k])) for j in range(3)
                        if abs(e[j]) >= 1e-12 and not fijo[j]]
            if termina:
                ops.equationConstraint(siguiente, 4 + k, 1.0, *[v for t in termina for v in t])
            else:                                                # el giro alrededor de la barra lo fija el apoyo
                flags[3 + k] = 1
        if any(flags):
            ops.fix(siguiente, *flags)
        if any(tr):
            reac_aux.setdefault(nodo_orig, []).append(siguiente)
        siguiente += 1
        return siguiente - 1

    for i, b in enumerate(dib.barras):
        p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
        e = (p2 - p1) / np.linalg.norm(p2 - p1)
        ni, nj = b['n1'] + 1, b['n2'] + 1
        if b.get('conexion_i') == 'rotulado':
            ni = extremo_rotulado(b['n1'], e)
            e_rotula[b['n1']] = e
        if b.get('conexion_j') == 'rotulado':
            nj = extremo_rotulado(b['n2'], e)
            e_rotula[b['n2']] = e
        p = props[i]
        G_mod = p['E'] / (2.0 * (1.0 + p['nu']))
        ops.geomTransf('Linear', i + 1, *vecxz_de_barra(dib.nodos, b))
        ops.element('elasticBeamColumn', i + 1, ni, nj, float(p['A']), float(p['E']), float(G_mod),
                    float(p['J']), float(p['Iy']), float(p['Iz']), i + 1)
    for nodo, e in e_rotula.items():                       # giro en el plano de un nudo sin otra conexion
        if nodo in fijos:
            k = int(np.argmax(np.abs(e)))
            for j in range(3):
                if j != k and abs(e[j]) >= 1e-12:
                    ops.equationConstraint(nodo + 1, 4 + j, 1.0, nodo + 1, 4 + k, -float(e[j] / e[k]))
    ops.timeSeries('Constant', 1)
    ops.pattern('Plain', 1, 1)
    for idx, v in cargas_nodales.items():
        ops.load(idx + 1, *[float(c) for c in v])
    if pp:
        for i, b in enumerate(dib.barras):
            x_loc, y_loc, z_loc = ejes_locales(dib, i)
            w = props[i]['masa'] / 100.0 * GRAVEDAD                 # kgf/m -> kgf/cm, hacia -Z
            # componentes del peso en ejes locales: Wy, Wz (transversal) y Wx (axial: columnas verticales)
            ops.eleLoad('-ele', i + 1, '-type', '-beamUniform',
                        float(w @ y_loc), float(w @ z_loc), float(w @ x_loc))
    return reac_aux


LIMITE_INESTABLE = 1e4   # cm o rad: con una carga de prueba de ~1 kgf, un desplazamiento mayor indica un mecanismo
GDL_NOMBRE = ['UX', 'UY', 'UZ', 'RX', 'RY', 'RZ']


def _probar_estabilidad(dib, props):
    """Un mecanismo o un grado de libertad sin rigidez puede dar numeros sin error en OpenSees (la matriz queda casi
    singular y el pivote no es exactamente cero). Se aplica una carga de prueba fija en cada nudo: si el modelo es
    estable, los desplazamientos son pequenos; si no, son enormes. Lanza ErrorAnalisis con el nudo y el GDL."""
    rng = np.random.default_rng(20261116)
    cargas = {i: tuple(float(v) for v in rng.standard_normal(6)) for i in range(len(dib.nodos))}
    _modelo(dib, props, cargas, False)
    ops.system('BandGeneral')
    ops.numberer('RCM')
    _restricciones(dib)
    ops.integrator('LoadControl', 1.0)
    ops.algorithm('Linear')
    ops.analysis('Static')
    rc, aviso = _analizar()
    desp = {i: np.array(ops.nodeDisp(i + 1), float) for i in range(len(dib.nodos))}
    ops.wipe()
    if rc != 0 or _hay_singular(aviso) or any(not np.all(np.isfinite(v)) for v in desp.values()):
        raise ErrorAnalisis('MODELO INESTABLE: OpenSees no resuelve la estructura (mecanismo o grado de libertad sin '
                            'rigidez). Revise apoyos, rotulas y conexiones de los nudos.')
    mayor, nudo, gdl = max((abs(v[d]), i, d) for i, v in desp.items() for d in range(6))
    if mayor > LIMITE_INESTABLE:
        raise ErrorAnalisis(f'MODELO INESTABLE: la carga de prueba produce un desplazamiento de {mayor:.3g} en el nudo '
                            f'{nudo + 1} (GDL {GDL_NOMBRE[gdl]}). Hay un mecanismo o un grado de libertad sin rigidez: '
                            'revise apoyos y rotulas. Un pasador en el plano no restringe el giro alrededor de la barra.')


def _resolver(dib, props, cargas_nodales, pp):
    """Un caso (estado): desplazamientos de nodos, fuerzas en extremos de barras y reacciones (cm, kgf)."""
    reac_aux = _modelo(dib, props, cargas_nodales, pp)
    ops.system('BandGeneral')
    ops.numberer('RCM')
    _restricciones(dib)
    ops.integrator('LoadControl', 1.0)
    ops.algorithm('Linear')
    ops.analysis('Static')
    rc, aviso = _analizar()
    if rc != 0 or _hay_singular(aviso):
        ops.wipe()
        raise ErrorAnalisis('OpenSees no pudo resolver el caso: matriz singular (mecanismo o grado de libertad sin rigidez). '
                            'Revise apoyos, rotulas y conexiones.')
    ops.reactions()
    reac = {}
    for i in dib.apoyos:
        r = np.array(ops.nodeReaction(i + 1), float)
        for tag in reac_aux.get(i, []):                 # extremos articulados apoyados en este nodo
            r = r + np.array(ops.nodeReaction(tag), float)
        reac[i] = r
    res = {'nodos': {i: np.array(ops.nodeDisp(i + 1), float) for i in range(len(dib.nodos))},
           'barras': {i: np.array(ops.eleResponse(i + 1, 'localForces'), float) for i in range(len(dib.barras))},
           'reac': reac}
    res['extremos'] = {}
    for i in range(len(dib.barras)):
        ni, nj = ops.eleNodes(i + 1)
        res['extremos'][i] = (np.array(ops.nodeDisp(ni), float), np.array(ops.nodeDisp(nj), float))
    ops.wipe()
    for grupo in res.values():
        for v in grupo.values():
            if not np.all(np.isfinite(v)):
                raise ErrorAnalisis('resultado no finito: revise el modelo (mecanismo o nodo mal apoyado)')
    # Un mecanismo puede dar numeros sin error en OpenSees: las reacciones deben equilibrar las cargas.
    escala = max(1.0, float(np.linalg.norm(_fuerza_aplicada(dib, props, cargas_nodales, pp))))
    if _residuo(dib, props, cargas_nodales, pp, res) > 1e-6 * escala:
        raise ErrorAnalisis('el modelo no equilibra las cargas (residuo de fuerzas mayor que 1e-6 de la carga): '
                            'revise apoyos y rotulas; es un mecanismo o un nodo mal apoyado.')
    return res


def _combinar(partes):
    """Superposicion lineal: partes = [(factor, resultado)]."""
    out = {'nodos': {}, 'barras': {}, 'reac': {}, 'extremos': {}}
    for f, res in partes:
        for clave in ('nodos', 'barras', 'reac'):
            for i, v in res[clave].items():
                out[clave][i] = out[clave].get(i, 0.0) + f * v
        for i, (ui, uj) in res['extremos'].items():
            a, b = out['extremos'].get(i, (0.0, 0.0))
            out['extremos'][i] = (a + f * ui, b + f * uj)
    return out


def _fuerza_aplicada(dib, props, cargas_nodales, pp):
    """Suma de las cargas nodales y del peso propio (kgf), vector de 3 componentes."""
    F = np.zeros(3)
    for v in cargas_nodales.values():
        F += np.array(v[:3], float)
    if pp:
        L = [np.linalg.norm(np.array(dib.nodos[b['n2']]) - np.array(dib.nodos[b['n1']])) * 100.0
             for b in dib.barras]
        F += sum(props[i]['masa'] / 100.0 * L[i] for i in range(len(L))) * GRAVEDAD
    return F


def _residuo(dib, props, cargas_nodales, pp, res):
    """Equilibrio global de fuerzas: suma de reacciones + cargas aplicadas + peso propio (debe ser ~0)."""
    F = _fuerza_aplicada(dib, props, cargas_nodales, pp)
    R = np.zeros(3)
    for v in res['reac'].values():
        R += v[:3]
    return float(np.linalg.norm(R + F))


def _fuerzas_extremo(f):
    """Fuerzas de un extremo en unidades de salida: kgf y kgf·m."""
    return [f[0], f[1], f[2], f[3] / 100.0, f[4] / 100.0, f[5] / 100.0]


def _tablas(dib, casos, rep_barras, rep_nodos, id_barra, id_de_indice, props=None, fpp=None):
    filas_b, filas_n, filas_r = [], [], []
    for nombre, tipo, res in casos:
        for i in sorted(rep_barras):
            b = dib.barras[i]
            f = res['barras'][i]
            fila = [nombre, tipo, id_barra.get(i, i), id_de_indice[b['n1']], id_de_indice[b['n2']],
                    b['elemento'], b['perfil']] + _fuerzas_extremo(f[:6]) + _fuerzas_extremo(f[6:])
            if props is not None:
                mz, my = momentos_maximos_viga(dib, props, i, res, fpp[nombre])
                fila += [mz / 100.0, my / 100.0]                       # kgf cm -> kgf m
            filas_b.append(fila)
        for i in sorted(rep_nodos):
            d = res['nodos'][i]
            filas_n.append([nombre, tipo, id_de_indice[i], d[0] * 10, d[1] * 10, d[2] * 10,
                            d[3], d[4], d[5]])
            if i in dib.apoyos:
                r = res['reac'][i]
                filas_r.append([nombre, tipo, id_de_indice[i], etiqueta_restricciones(dib.apoyos[i]), r[0], r[1], r[2],
                                r[3] / 100.0, r[4] / 100.0, r[5] / 100.0])
    cab_b = ['CASO', 'TIPO', 'BARRA', 'NODO I', 'NODO J', 'ELEMENTO', 'PERFIL'] + \
            [f'{e} I ({u})' for e, u in zip(ETIQUETA, ['kgf', 'kgf', 'kgf', 'kgf·m', 'kgf·m', 'kgf·m'])] + \
            [f'{e} J ({u})' for e, u in zip(ETIQUETA, ['kgf', 'kgf', 'kgf', 'kgf·m', 'kgf·m', 'kgf·m'])] + \
            (['MZ MAX EN EL TRAMO (kgf·m)', 'MY MAX EN EL TRAMO (kgf·m)'] if props is not None else [])
    cab_n = ['CASO', 'TIPO', 'NODO', 'UX (mm)', 'UY (mm)', 'UZ (mm)', 'RX (rad)', 'RY (rad)', 'RZ (rad)']
    cab_r = ['CASO', 'TIPO', 'NODO', 'APOYO', 'FX (kgf)', 'FY (kgf)', 'FZ (kgf)', 'MX (kgf·m)', 'MY (kgf·m)',
             'MZ (kgf·m)']
    return (pd.DataFrame(filas_b, columns=cab_b), pd.DataFrame(filas_n, columns=cab_n),
            pd.DataFrame(filas_r, columns=cab_r))


UNIDAD_ESF = dict(zip(ETIQUETA, ['kgf', 'kgf', 'kgf', 'kgf·m', 'kgf·m', 'kgf·m']))


def tabla_maximos(barras):
    """Maximo y minimo de cada esfuerzo entre los extremos I y J, por barra y caso (solo barras REPORTAR = SI)."""
    cab = ['CASO', 'TIPO', 'BARRA', 'ELEMENTO', 'PERFIL']
    for e in ETIQUETA:
        cab += [f'{e} MAX ({UNIDAD_ESF[e]})', f'{e} MIN ({UNIDAD_ESF[e]})']
    filas = []
    for _, r in barras.iterrows():
        fila = [r['CASO'], r['TIPO'], r['BARRA'], r['ELEMENTO'], r['PERFIL']]
        for e in ETIQUETA:
            v = [r[f'{e} I ({UNIDAD_ESF[e]})'], r[f'{e} J ({UNIDAD_ESF[e]})']]
            fila += [max(v), min(v)]
        filas.append(fila)
    return pd.DataFrame(filas, columns=cab)


def tabla_deformada(dib, casos, id_de_indice):
    """Coordenadas de todos los nodos por caso: original, desplazamientos (mm, rad) y posicion deformada (m)."""
    cab = ['CASO', 'TIPO', 'NODO', 'X (m)', 'Y (m)', 'Z (m)', 'UX (mm)', 'UY (mm)', 'UZ (mm)',
           'RX (rad)', 'RY (rad)', 'RZ (rad)', 'X def (m)', 'Y def (m)', 'Z def (m)']
    filas = []
    for nombre, tipo, res in casos:
        for i, p in enumerate(dib.nodos):
            d = res['nodos'][i]
            filas.append([nombre, tipo, id_de_indice[i], p[0], p[1], p[2],
                          d[0] * 10, d[1] * 10, d[2] * 10, d[3], d[4], d[5],
                          p[0] + d[0] / 100, p[1] + d[1] / 100, p[2] + d[2] / 100])
    return pd.DataFrame(filas, columns=cab)


def _arco(ax, centro, a, b, r, color, texto):
    """Arco de momento: positivo alrededor de a x b (regla de la mano derecha)."""
    th = np.linspace(0.3, 2.4, 40)
    pts = centro + r * (np.outer(np.cos(th), a) + np.outer(np.sin(th), b))
    ax.plot(pts[:, 0], pts[:, 1], pts[:, 2], color=color, lw=2.2)
    ax.quiver(*pts[-2], *(pts[-1] - pts[-2]), color=color, arrow_length_ratio=0.6, lw=2.2)
    ax.text(*(centro + 1.45 * r * (a + b) / 2), texto, color=color, fontsize=9)


def figura_ejes_locales(ruta):
    """PNG: ejes locales de una barra de ejemplo (viga a lo largo de X con peralte vertical), fuerzas en los
    extremos I y J y signos. Sirve de referencia para leer la hoja BARRAS."""
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(13, 5.8))
    ax = fig.add_axes([0.0, 0.0, 0.56, 1.0], projection='3d')
    txt = fig.add_axes([0.58, 0.02, 0.42, 0.96]); txt.axis('off')
    I, J = np.array([0.0, 0.0, 0.0]), np.array([2.0, 0.0, 0.0])
    x_loc, y_loc, z_loc = np.array([1.0, 0, 0]), np.array([0, 0, 1.0]), np.array([0, -1.0, 0])
    ax.plot(*zip(I, J), color='#9e9e9e', lw=9, solid_capstyle='round')
    ax.text(-0.15, 0.0, -0.3, 'I', fontsize=13, weight='bold')
    ax.text(2.15, 0.0, -0.3, 'J', fontsize=13, weight='bold')
    L = 0.75
    for vec, color, texto in [(x_loc, '#d62728', 'x local'), (y_loc, '#2ca02c', 'y local (eje fuerte)'),
                              (z_loc, '#1f77b4', 'z local (eje debil)')]:
        ax.quiver(*I, *vec * L, color=color, lw=2.6, arrow_length_ratio=0.18)
        pos = I + vec * (L + 0.1) + (np.array([-0.45, 0.0, -0.22]) if vec[2] < 0 else
                                      (np.array([0.0, 0.0, 0.12]) if vec[1] > 0 else np.array([0.12, -0.2, 0.0])))
        ax.text(*pos, texto, color=color, fontsize=10, weight='bold')
    O = np.array([-1.9, -1.0, -0.95])
    for vec, texto in [((1, 0, 0), 'X'), ((0, 1, 0), 'Y'), ((0, 0, 1), 'Z')]:
        ax.quiver(*O, *(np.array(vec) * 0.6), color='#555555', lw=1.8, arrow_length_ratio=0.2)
        ax.text(*(O + np.array(vec) * 0.7), texto + ' global', color='#555555', fontsize=10)
    # fuerzas en los extremos (flecha = sentido positivo del eje local)
    ax.quiver(*I, 0.45, 0, 0, color='#444444', lw=1.4, arrow_length_ratio=0.3)
    ax.text(0.2, 0.0, 0.22, 'N(I)', fontsize=9, color='#444444')
    ax.quiver(*J, 0.45, 0, 0, color='#444444', lw=1.4, arrow_length_ratio=0.3)
    ax.text(2.62, 0.0, 0.28, 'N(J)', fontsize=9, color='#444444')
    ax.quiver(*J, 0, 0.45, 0, color='#444444', lw=1.4, arrow_length_ratio=0.3)
    ax.text(2.1, 0.5, 0.0, 'Vy(J)', fontsize=9, color='#444444')
    # momentos a mitad de la barra: Mz alrededor de z (fuerte), My alrededor de y (debil)
    M = np.array([1.0, 0.0, 0.0])
    _arco(ax, M, x_loc, y_loc, 0.3, '#6a1b9a', 'Mz (fuerte)')
    _arco(ax, M, z_loc, x_loc, 0.3, '#ef6c00', 'My (debil)')
    ax.set_xlim(-1.5, 3.1); ax.set_ylim(-1.0, 0.9); ax.set_zlim(-0.8, 1.1)
    ax.set_box_aspect((4.6, 1.9, 1.9)); ax.set_axis_off(); ax.view_init(elev=28, azim=-38)
    lineas = [
        ('EJES LOCALES Y SIGNOS (hoja BARRAS)', True),
        ('x: eje de la barra, de I a J.', False),
        ('y: eje fuerte (peralte del perfil): cortante fuerte Vy, momento fuerte Mz.', False),
        ('z: eje debil: cortante debil Vz, momento debil My.', False),
        ('T: torsion, alrededor de x.     N: axial.', False),
        ('', False),
        ('SIGNOS DE LOS EXTREMOS I y J', True),
        ('Un valor positivo es la fuerza o el momento que el nudo ejerce sobre la barra', False),
        ('en ese extremo, en el sentido del eje local (momentos: regla de la mano derecha).', False),
        ('', False),
        ('N(I) positivo = compresion.     N(J) positivo = traccion.', True),
        ('Barra traccionada: N(I) < 0 y N(J) > 0.', False),
        ('', False),
        ('Ejes globales X, Y, Z (gris). Los desplazamientos van en ejes globales.', False),
    ]
    for k, (linea, negrita) in enumerate(lineas):
        txt.text(0.0, 0.95 - k * 0.068, linea, fontsize=10.5 if negrita else 10,
                 weight='bold' if negrita else 'normal', transform=txt.transAxes)
    fig.savefig(ruta, dpi=110)
    plt.close(fig)
    return ruta


def _lineas(barras, P):
    xs, ys, zs = [], [], []
    for b in barras:
        a, c = P[b['n1']], P[b['n2']]
        xs += [a[0], c[0], None]; ys += [a[1], c[1], None]; zs += [a[2], c[2], None]
    return xs, ys, zs


def figura_deformada(dib, casos, id_de_indice, escala=None):
    """Figura 3D interactiva: deformada por estado y combinacion (botones). Sin escala dada, la escala hace que el
    mayor desplazamiento mida el 10 % de la dimension del modelo. Las barras se dibujan entre extremos deformados."""
    import plotly.graph_objects as go
    P0 = np.array(dib.nodos, float)
    tam = max(max(float(np.ptp(P0[:, k])) for k in range(3)), 1.0)
    U = {nombre: np.array([res['nodos'][i][:3] for i in range(len(P0))], float) / 100.0 for nombre, _, res in casos}
    dmax = max((float(np.max(np.linalg.norm(u, axis=1))) for u in U.values()), default=0.0)
    if escala is None:
        escala = 0.1 * tam / dmax if dmax > 0 else 1.0
    fig = go.Figure()
    xs, ys, zs = _lineas(dib.barras, P0)
    fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', line=dict(color='#bbbbbb', width=3, dash='dash'),
                               name='sin carga (original)', hoverinfo='skip'))
    n = len(casos)
    for k, (nombre, tipo, _) in enumerate(casos):
        P = P0 + escala * U[nombre]
        xs, ys, zs = _lineas(dib.barras, P)
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', line=dict(color='#1f77b4', width=6),
                                   name=f'{nombre} ({tipo})', visible=(k == 0), hoverinfo='skip'))
        d_mm = U[nombre] * 1000.0          # desplazamiento real del nodo (sin la escala grafica), en mm
        textos = [f'nodo {id_de_indice[i]} · {nombre} ({tipo})<br>Ux = {d_mm[i, 0]:.2f} mm<br>Uy = {d_mm[i, 1]:.2f} mm'
                  f'<br>Uz = {d_mm[i, 2]:.2f} mm<br>|U| = {np.linalg.norm(d_mm[i]):.2f} mm' for i in range(len(P))]
        fig.add_trace(go.Scatter3d(x=P[:, 0], y=P[:, 1], z=P[:, 2], mode='markers+text',
                                   marker=dict(size=5, color='black'),
                                   text=[f'N{id_de_indice[i]}' for i in range(len(P))], textposition='top center',
                                   hovertext=textos, hoverinfo='text',
                                   name=f'nodos {nombre}', visible=(k == 0)))
    botones = []
    for k, (nombre, tipo, _) in enumerate(casos):
        vis = [True] + [(t // 2) == k for t in range(2 * n)]
        botones.append(dict(label=nombre, method='update',
                            args=[{'visible': vis},
                                  {'title': f'Deformada: {nombre} ({tipo}), escala x{escala:.3g}'}]))
    titulo = (f'Deformada: {casos[0][0]} ({casos[0][1]}), escala x{escala:.3g}' if casos
              else 'Sin casos para mostrar')
    fig.update_layout(title=titulo,
                      updatemenus=[dict(buttons=botones, direction='down', x=0.0, xanchor='left', y=1.14,
                                        yanchor='top')],
                      scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)', aspectmode='data'),
                      margin=dict(l=0, r=0, t=70, b=0), height=760)
    return fig


def _escribir(ruta, bloques, resumen_filas, png):
    from openpyxl.drawing.image import Image as XLImage
    with pd.ExcelWriter(ruta, engine='openpyxl') as w:
        pd.DataFrame(resumen_filas, columns=['DATO', 'VALOR']).to_excel(w, sheet_name='RESUMEN', index=False)
        for nombre in ['BARRAS', 'NODOS', 'REACCIONES', 'MAXIMOS_BARRAS', 'DEFORMADA']:
            bloques[nombre].to_excel(w, sheet_name=nombre, index=False)
        hoja_res = w.book['RESUMEN']
        if png and os.path.exists(png):
            img = XLImage(png)
            img.width, img.height = 1040, 448
            hoja_res.add_image(img, 'D2')
        for hoja in w.book.worksheets:
            for col in hoja.columns:
                largo = max(len(str(c.value)) if c.value is not None else 0 for c in col)
                hoja.column_dimensions[col[0].column_letter].width = min(max(10, largo + 2), 40)
            hoja.freeze_panes = 'A2'
            if hoja.title == 'RESUMEN':
                hoja.column_dimensions['A'].width = 48
                hoja.column_dimensions['B'].width = 90


def clase_combinacion(nombre, combinaciones, tipos):
    """Clase de una combinacion segun el TIPO (hoja ESTADOS) de sus estados con factor no nulo: 'viento' si incluye W
    (no entra al diseno: para el diseno solo aplica el sismo), 'sismo' si incluye E, 'gravitacional' si solo D y L,
    'otra' si algun estado no tiene un TIPO reconocido."""
    t = [tipos.get(e) for e, f in combinaciones.get(nombre, {}).items() if f != 0]
    if 'W' in t:
        return 'viento'
    if 'E' in t:
        return 'sismo'
    if t and all(x in ('D', 'L') for x in t):
        return 'gravitacional'
    return 'otra'


def _deflexion_viga(dib, props, i, res, fpp, xi):
    """Desviacion vertical (Z global, cm) de la viga i a lo largo de su eje (xi = 0 a 1), respecto a la cuerda entre
    extremos. Euler-Bernoulli exacto entre extremos: giros de extremo del elemento (res['extremos']) mas la carga propia
    uniforme, con factor fpp (suma de factores de los estados con PESO PROPIO = SI en la combinacion)."""
    p1 = np.array(dib.nodos[dib.barras[i]['n1']], float)
    p2 = np.array(dib.nodos[dib.barras[i]['n2']], float)
    L = float(np.linalg.norm(p2 - p1)) * 100.0
    _, yl, zl = ejes_locales(dib, i)
    w = props[i]['masa'] / 100.0 * GRAVEDAD                       # kgf/cm, igual que en _modelo
    E, Iy, Iz = props[i]['E'], props[i]['Iy'], props[i]['Iz']
    h1 = xi - 2.0 * xi ** 2 + xi ** 3                              # forma de Hermite: giro en el extremo i
    h2 = -xi ** 2 + xi ** 3                                        # forma de Hermite: giro en el extremo j
    burbuja = xi ** 2 * (1.0 - xi) ** 2                            # solucion particular de la carga uniforme
    ui, uj = res['extremos'][i]
    ti, tj = ui[3:6], uj[3:6]
    # plano local x-y (v): giro theta_z = dv/dx, rigidez E*Iz.  plano local x-z (w): theta_y = -dw/dx, rigidez E*Iy
    # Hermite: el termino de traslacion de los extremos no es lineal. Respecto a la cuerda queda g(xi)*(v_i - v_j),
    # con g = xi - 3 xi^2 + 2 xi^3 (corrige H-02: antes solo se usaban los giros).
    g = xi - 3.0 * xi ** 2 + 2.0 * xi ** 3
    v_ext = (ui[:3] @ yl) - (uj[:3] @ yl)
    w_ext = (ui[:3] @ zl) - (uj[:3] @ zl)
    v = L * ((ti @ zl) * h1 + (tj @ zl) * h2) + g * v_ext + fpp * (w @ yl) * L ** 4 * burbuja / (24.0 * E * Iz)
    wz = L * (-(ti @ yl) * h1 - (tj @ yl) * h2) + g * w_ext + fpp * (w @ zl) * L ** 4 * burbuja / (24.0 * E * Iy)
    return v * yl[2] + wz * zl[2]


def _curvatura_max(v_i, v_j, s_i, s_j, q, EI, L):
    """Maximo |M| = EI * v'' a lo largo de una viga (Euler-Bernoulli exacto entre extremos, sin muestreo).
    v_i, v_j: desplazamientos transversales de los extremos (cm); s_i, s_j: pendientes dv/dx; q: carga uniforme (kgf/cm)
    en el mismo plano; EI en kgf cm2. Devuelve kgf cm.
    Con carga uniforme, EI*v'' = A0 + A1*x + A2*x^2 (polinomio de grado 2). El maximo de |M| en [0, L] esta en un
    extremo o en el vertice de la parabola: se evaluan esos puntos (cierra el error de muestreo de rev01, A2)."""
    A0 = EI * (-6.0 * v_i / L ** 2 + 6.0 * v_j / L ** 2 - 4.0 * s_i / L - 2.0 * s_j / L + q * L ** 2 / (12.0 * EI))
    A1 = EI * (12.0 * v_i / L ** 3 - 12.0 * v_j / L ** 3 + 6.0 * s_i / L ** 2 + 6.0 * s_j / L ** 2 - q * L / (2.0 * EI))
    A2 = EI * (q / (2.0 * EI))
    candidatos = [0.0, L]
    if A2 != 0.0:
        xv = -A1 / (2.0 * A2)
        if 0.0 < xv < L:
            candidatos.append(xv)
    return float(max(abs(A0 + A1 * x + A2 * x * x) for x in candidatos))


def momentos_maximos_viga(dib, props, i, res, fpp):
    """Momento maximo en el tramo (no solo en los extremos) de la barra i para una combinacion (corrige H-01).
    Devuelve (Mz_max, My_max) en kgf cm, con el peso propio de la combinacion (factor fpp)."""
    p1 = np.array(dib.nodos[dib.barras[i]['n1']], float)
    p2 = np.array(dib.nodos[dib.barras[i]['n2']], float)
    L = float(np.linalg.norm(p2 - p1)) * 100.0
    _, yl, zl = ejes_locales(dib, i)
    w = props[i]['masa'] / 100.0 * GRAVEDAD
    E, Iy, Iz = props[i]['E'], props[i]['Iy'], props[i]['Iz']
    ui, uj = res['extremos'][i]
    ti, tj = ui[3:6], uj[3:6]
    # plano x-y: flexion Mz con EI = E Iz; v = u.yl, pendiente dv/dx = theta_z = t.zl
    Mz = _curvatura_max(ui[:3] @ yl, uj[:3] @ yl, ti @ zl, tj @ zl, fpp * (w @ yl), E * Iz, L)
    # plano x-z: flexion My con EI = E Iy; w = u.zl, pendiente dw/dx = -theta_y = -(t.yl)
    My = _curvatura_max(ui[:3] @ zl, uj[:3] @ zl, -(ti @ yl), -(tj @ yl), fpp * (w @ zl), E * Iy, L)
    return Mz, My


def flechas_vigas(dib, props, casos, combinaciones, estados_pp, clase):
    """Flecha vertical maxima de cada viga horizontal (desviacion respecto a la cuerda entre sus extremos) sobre las
    combinaciones gravitacionales. Devuelve {barra: {'L_cm', 'max_mm', 'comb', 'xi'}}; xi = posicion del maximo (0 a 1)."""
    xi = np.linspace(0.0, 1.0, 401)
    out = {}
    for i, b in enumerate(dib.barras):
        p1 = np.array(dib.nodos[b['n1']], float)
        p2 = np.array(dib.nodos[b['n2']], float)
        L = float(np.linalg.norm(p2 - p1)) * 100.0
        if abs(p2[2] - p1[2]) * 100.0 > 0.1 * L:
            continue                                               # columnas y elementos inclinados: no es viga
        mejor = None
        for nombre, tipo, res in casos:
            # DECISION D2 (abierta): la flecha se evalua solo en combinaciones gravitacionales (D y L, total). Alternativa: solo carga viva.
            if tipo != 'COMBINACION' or clase.get(nombre) != 'gravitacional':
                continue
            fpp = sum(f for e, f in combinaciones[nombre].items() if estados_pp.get(e))
            d = np.abs(_deflexion_viga(dib, props, i, res, fpp, xi))
            k = int(np.argmax(d))
            if mejor is None or d[k] > mejor[0]:
                mejor = (float(d[k]), nombre, float(xi[k]))
        if mejor is not None:
            out[i] = {'L_cm': L, 'max_mm': mejor[0] * 10.0, 'comb': mejor[1], 'xi': mejor[2]}
    return out


def correr(dib, materiales, estados, rep_barras, rep_nodos, id_barra, id_de_indice, ruta_salida):
    """Resuelve todos los estados y combinaciones y escribe el Excel de resultados.
    materiales: {nombre: {'E', 'nu'}}; estados: [(nombre, peso_propio)]; rep_barras / rep_nodos: indices a reportar.
    Devuelve (bloques, equilibrio, casos): tablas, residuo de equilibrio por estado (kgf) y resultados por caso
    (todos los nodos, para la deformada)."""
    _verificar_conectividad(dib)
    props = propiedades_barras(dib, materiales, id_barra)
    cargas = getattr(dib, 'cargas', {}) or {}
    combinaciones = getattr(dib, 'combinaciones', {}) or {}
    tipos = getattr(dib, 'estados_tipo', {}) or {}
    # DECISION D13 (abierta): viento fuera de esta version: se omiten los estados W y las combinaciones que lo incluyen.
    estados = [(n, pp) for n, pp in estados if tipos.get(n) != 'W']
    clase_todas = {n: clase_combinacion(n, combinaciones, tipos) for n in combinaciones}
    combinaciones = {n: f for n, f in combinaciones.items() if clase_todas[n] != 'viento'}
    resultados, equilibrio, casos, resumen = {}, [], [], []
    if estados:
        _probar_estabilidad(dib, props)     # una vez por modelo, antes de resolver cualquier estado
    for nombre, pp in estados:
        cn = cargas.get(nombre, {})
        resultados[nombre] = _resolver(dib, props, cn, pp)
        equilibrio.append((nombre, _residuo(dib, props, cn, pp, resultados[nombre])))
        casos.append((nombre, 'ESTADO', resultados[nombre]))
        resumen.append([f'Estado {nombre}', f'peso propio {"SI" if pp else "NO"}; cargas en {len(cn)} nodos'])
    for nombre, factores in combinaciones.items():
        partes = [(f, resultados[e]) for e, f in factores.items() if e in resultados]
        if not partes:
            continue
        casos.append((nombre, 'COMBINACION', _combinar(partes)))
        resumen.append([f'Combinacion {nombre}', ' + '.join(f'{f:g}·{e}' for e, f in factores.items())])
    clase = {nombre: clase_combinacion(nombre, combinaciones, tipos) for nombre in combinaciones}
    dib.clase_combinacion = clase_todas                 # clase de TODAS las combinaciones (la memoria excluye las de viento)
    # DECISION D11 (abierta): el diseno usa las combinaciones de servicio ASD definidas por el usuario (sin viento); E entra si esta definido.
    casos_diseno = [c for c in casos if not (c[1] == 'COMBINACION' and clase.get(c[0]) == 'viento')]
    dib.flechas = flechas_vigas(dib, props, casos, combinaciones, dict(estados), clase)
    pp_map = dict(estados)
    fpp_caso = {n: (1.0 if pp_map.get(n) else 0.0) for n, _ in estados}
    fpp_caso.update({n: sum(f for e, f in combinaciones[n].items() if pp_map.get(e)) for n in combinaciones})
    cab_b, cab_n, cab_r = _tablas(dib, casos, rep_barras, rep_nodos, id_barra, id_de_indice, props, fpp_caso)
    bloques = {'BARRAS': cab_b, 'NODOS': cab_n, 'REACCIONES': cab_r,
               'MAXIMOS_BARRAS': tabla_maximos(cab_b),
               'DEFORMADA': tabla_deformada(dib, casos, id_de_indice)}
    cabecera = [
        ['Unidades', 'fuerzas kgf; momentos kgf·m; desplazamientos mm; giros rad'],
        ['Barras', 'N axial; Vy cortante fuerte; Vz cortante debil; T torsion; My momento debil; '
                   'Mz momento fuerte. Ejes locales y signos: figura a la derecha'],
        ['Signos', 'fuerzas que el nudo ejerce sobre la barra en cada extremo, ejes locales (OpenSees localForces). '
                   'Barra traccionada: N en I < 0 y N en J > 0'],
        ['Reacciones', 'fuerzas que el apoyo ejerce sobre la estructura (FZ positivo hacia arriba); con las cargas aplicadas suman cero'],
        ['MAXIMOS_BARRAS', 'maximo y minimo de cada esfuerzo entre los extremos I y J, por barra y caso'],
        ['DEFORMADA', 'coordenadas de todos los nodos por caso: original, desplazamientos y posicion deformada'],
        ['Barras y nodos reportados', f'{len(rep_barras)} barras y {len(rep_nodos)} nodos marcados REPORTAR = SI'],
        ['Viento', 'no se considera en esta version: se omiten los estados y combinaciones con W'],
    ]
    for nombre, residuo in equilibrio:
        cabecera.append([f'Equilibrio estado {nombre}', f'residuo de fuerzas {residuo:.3g} kgf'])
    bloques['DISENO_BARRAS'] = _tablas(dib, casos_diseno, set(range(len(dib.barras))), set(), id_barra, id_de_indice,
                                       props, fpp_caso)[0]
    png = figura_ejes_locales('ejes_locales.png')
    _escribir(ruta_salida, bloques, cabecera + resumen, png)
    return bloques, equilibrio, casos


def correr_desde_planilla(dib, materiales_filas, estados_df, nodos, barras, id_barra, id_de_indice, ruta_salida):
    """Punto de entrada del notebook: toma los datos de la planilla ya leida y valida (bloque 7).
    materiales_filas es None si la planilla no tiene la hoja MATERIALES (no se usan valores de ejemplo)."""
    if materiales_filas is None:
        raise ErrorAnalisis('la planilla no tiene la hoja MATERIALES con las columnas E y NU: '
                            'use la plantilla nueva (bloque 6) y vuelva a llenarla.')
    materiales = materiales_desde_filas(materiales_filas)
    estados = estados_desde_df(estados_df)
    marcados_b = {int(float(r['ID'])) for _, r in barras.iterrows() if _marcado(r.get('REPORTAR'))}
    marcados_n = {int(float(r['ID'])) for _, r in nodos.iterrows() if _marcado(r.get('REPORTAR'))}
    rep_barras = {i for i, id_ in id_barra.items() if id_ in marcados_b}
    rep_nodos = {i for i, id_ in id_de_indice.items() if id_ in marcados_n}
    dib.estados_tipo = {str(r['NOMBRE']).strip(): str(r['TIPO']).strip().upper()
                        for _, r in estados_df.iterrows()
                        if isinstance(r['NOMBRE'], str) and isinstance(r['TIPO'], str)}
    return correr(dib, materiales, estados, rep_barras, rep_nodos, id_barra, id_de_indice, ruta_salida)


In [ ]:
# Analisis lineal con OpenSees y Excel de resultados. Todo sale de la planilla (bloques 6 y 7).
import subprocess
import sys
from IPython.display import display, Image

try:
    import openseespy.opensees  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openseespy==3.8.0.0'], check=True)
import analisis_opensees as AO

RUTA_RESULTADOS = 'resultados_soportes.xlsx'
if dib is None:
    print('Sin modelo para analizar: corrija la planilla (bloque 7).')
else:
    try:
        bloques, equilibrio, casos = AO.correr_desde_planilla(dib, materiales_leidos, estados_df, nodos, barras,
                                                              dib.id_barra_de_indice, id_de_indice, RUTA_RESULTADOS)
    except AO.ErrorAnalisis as error:
        print('NO se analizo. Corrija lo siguiente en la planilla y vuelva a ejecutar este bloque:')
        print(error)
    else:
        print('Equilibrio global por estado (residuo de fuerzas, debe ser ~0 kgf):')
        for nombre, residuo in equilibrio:
            print(f'  {nombre}: {residuo:.3g} kgf')
        if bloques['BARRAS'].empty and bloques['NODOS'].empty:
            print('Ninguna barra ni nodo marcado REPORTAR = SI: el Excel de resultados no tiene datos.')
        print(f"Excel de resultados: {RUTA_RESULTADOS}")
        print('Hojas: RESUMEN (con ejes locales), BARRAS, NODOS, REACCIONES, MAXIMOS_BARRAS, DEFORMADA')
        print(f"Filas: barras {len(bloques['BARRAS'])}, nodos {len(bloques['NODOS'])}, "
              f"reacciones {len(bloques['REACCIONES'])}, maximos {len(bloques['MAXIMOS_BARRAS'])}")
        print('Ejes locales y signos:')
        display(Image(filename='ejes_locales.png', width=1000))
        print('Maximos y minimos por barra y caso (primeras filas):')
        display(bloques['MAXIMOS_BARRAS'].head(12))
        print('Deformada: ver la celda siguiente (escala editable, no vuelve a resolver).')


## Bloque 9b. Deformada con escala editable

Cambie `ESCALA_DEFORMADA` y ejecute solo esta celda: no vuelve a resolver el análisis.

In [ ]:
# Deformada por estado y combinacion. Escala editable: cambie ESCALA_DEFORMADA y ejecute solo esta celda.
# None = automatica (el mayor desplazamiento mide el 10 % de la dimension del modelo).
# Ejemplo: ESCALA_DEFORMADA = 50  (multiplica los desplazamientos por 50 en el dibujo).
# No modifica el calculo ni el Excel de resultados (DEFORMADA guarda los desplazamientos sin escalar).
ESCALA_DEFORMADA = None

import analisis_opensees as AO
if dib is None or 'casos' not in globals():
    print('Ejecute primero el bloque 9 (analisis).')
else:
    AO.figura_deformada(dib, casos, id_de_indice, escala=ESCALA_DEFORMADA).show(renderer='iframe')


## Bloque 10. Diseño por AISC 360-22 (ASD)

Capacidades de IR, CE y LI (tensión, compresión, flexión, cortante, interacción H1 o H2) con las combinaciones de servicio del bloque 9 (sin las que incluyen viento: para el diseño solo aplica el sismo) y las longitudes de la hoja LONGITUDES. Escribe `diseno_asd.xlsx` y `planilla_con_diseno.xlsx`. Los perfiles macizos (CS, OS y SOL) fueron eliminados de la base y no se diseñan. Las barras CS quedan fuera del diseño. Los resultados salen **por grupo**: elija la sección de cada grupo en su desplegable (ordenado por tipo y sección); la tabla se recalcula y el botón **Aplicar** actualiza el Excel por elemento y la planilla (hoja PERFIL_GRUPO).

In [ ]:
%%writefile diseno_ntc.py
"""
DISEÑO DE ELEMENTOS DE ACERO CON LAS NTC PARA DISEÑO Y CONSTRUCCION DE ESTRUCTURAS DE ACERO
(Gaceta Oficial CDMX, 7 de julio de 2020). Unidades internas: cm, kgf, kgf·cm.

ALCANCE DE ESTA VERSION
  - Perfiles I de doble simetria (familia IR): tension, compresion, flexion en ambos ejes, cortante en ambos
    ejes, flexocompresion, flexotension y flexion con cortante.
  - Otras familias: solo tension (si tienen area). El resto aparece como NO REVISADO.
  - Acciones: las COMBINACIONES del analisis (bloque 9). Deben venir ya factoradas segun la norma.
  - Analisis de primer orden: no incluye efectos P-Delta ni P-delta (la norma los pide en el diseño).

DATOS DE ENTRADA (hoja BARRAS, columnas por barra):
  K EJE FUERTE, K EJE DEBIL: factor de longitud efectiva para pandeo alrededor de cada eje. Vacio = 1.0 (se marca).
  LB (m): longitud no soportada lateralmente. Vacio = longitud de la barra (se marca).

CRITERIOS CONSERVADORES (se marcan en la salida):
  Cb = 1.0 (6.2.1 permite un valor mayor); area neta Ae = A (sin agujeros; las conexiones no se modelan).
  Patin no compacto (tipo 3): Mn = My (6.4); el valor interpolado de 6.4.3a no se usa.
"""
import math
import numpy as np
import pandas as pd
import perfiles_gerdau as G

FR_FLUENCIA_T = 0.9        # 4.2.1
FR_FRACTURA_T = 0.75       # 4.2.2
FR = 0.9                   # capitulos 5, 6 y 8
FR_CORTANTE_LAMINADA = 1.0  # 7.2.2 (vigas I laminadas, h/tw <= 2.24 sqrt(E/Fy))
FR_CORTANTE_OTRAS = 0.9    # 7.1 y 7.7
CB_CONSERVADOR = 1.0
K_SUPUESTO = 1.0


class ErrorDiseno(Exception):
    pass


# ---------------------------------------------------------------- tablas y clasificacion
def n_compresion(fy):
    """Tabla 5.2.1 (perfiles I laminados estandar, tp <= 51 mm): (n eje mayor, n eje menor) segun Fy (kgf/cm2)."""
    if fy < 3513:               # <= 2530 kgf/cm2 (A-36) y 2535-3510 kgf/cm2
        return 1.4, 1.4
    if fy <= 3517:              # 3515 kgf/cm2 (Gr. 50)
        return 2.0, 1.4
    return 2.0, 2.0             # 3520-6325 kgf/cm2


def clasificar_i(p, fy, E):
    """Clasificacion de la seccion I: patines (b = bf/2) y alma (h = d - 2 tf). Tablas 3.2.1 y 3.2.2."""
    r = math.sqrt(E / fy)
    b = p['B'] / 2.0
    tf = p['t']
    tw = p['e']
    h = p['H'] - 2.0 * tf
    lf, lw = b / tf, h / tw
    lpf, lrf = 0.38 * r, 1.00 * r
    lpw, lrw = 3.76 * r, 5.70 * r
    f = 'C' if lf <= lpf else ('NC' if lf <= lrf else 'E')
    w = 'C' if lw <= lpw else ('NC' if lw <= lrw else 'E')
    if f == 'C' and w == 'C':
        tipo = 'tipo 2'
    elif f != 'E' and w != 'E':
        tipo = 'tipo 3'
    else:
        tipo = 'tipo 4'
    comp_ok = (lf <= 0.56 * r) and (lw <= 1.49 * r)          # tabla 3.2.1, compresion axial
    return dict(r=r, b=b, h=h, lf=lf, lw=lw, lpf=lpf, lrf=lrf, lpw=lpw, lrw=lrw,
                f=f, w=w, tipo=tipo, comp_ok=comp_ok)


# ---------------------------------------------------------------- tension (capitulo 4)
def resistencia_traccion(p, fy, fu):
    """Rt = min(FR Fy A, FR Fu Ae) con Ae = A. Si no hay Fu, solo fluencia (se marca)."""
    rt_f = FR_FLUENCIA_T * fy * p['A']
    if fu is None:
        return rt_f, 'fractura NO REVISADA: falta Fu del material'
    return min(rt_f, FR_FRACTURA_T * fu * p['A']), ''


# ---------------------------------------------------------------- compresion (capitulo 5)
def resistencia_compresion(p, fy, E, Kx, Ky, Lx, Ly, c):
    """Rc = FR chi Fy A con n por eje (tabla 5.2.1). Devuelve (Rc, detalle por eje)."""
    nx, ny = n_compresion(fy)
    out = {}
    for eje, n, K, L, rr in [('X', nx, Kx, Lx, p['rx']), ('Y', ny, Ky, Ly, p['ry'])]:
        kl_r = K * L / rr
        fe = math.pi ** 2 * E / kl_r ** 2                             # 5.2.4
        lam = math.sqrt(fy / fe)                                      # 5.2.3
        chi = (1.0 + (fy / fe) ** n) ** (-1.0 / n)                    # 5.2.2
        out[eje] = dict(n=n, K=K, L=L, KL_r=kl_r, Fe=fe, lam=lam, chi=chi, Rc=FR * chi * fy * p['A'])
    rc = min(out['X']['Rc'], out['Y']['Rc'])
    return rc, out


# ---------------------------------------------------------------- flexion (capitulo 6)
def _Lu_Lr(Z, fy, Cb, G_, J, Cw, Iy, E):
    """Longitudes Lu y Lr (6.3.5 a 6.3.8). Z = Zx (compacta) o Sx (no compacta, 6.4)."""
    xu = 4.293 * (Z * fy) / (Cb * G_ * J) * math.sqrt(Cw / Iy)
    xr = (4.0 / 3.0) * (Z * fy) / (Cb * G_ * J) * math.sqrt(Cw / Iy)
    k = math.sqrt(E * Cw / (G_ * J))
    lu = math.sqrt(2) * math.pi / xu * k * math.sqrt(1 + math.sqrt(1 + xu ** 2))
    lr = math.sqrt(2) * math.pi / xr * k * math.sqrt(1 + math.sqrt(1 + xr ** 2))
    return lu, lr


def _Me(Cb, Lb, E, G_, Iy, J, Cw):
    """Momento elastico de pandeo lateral (6.3.4)."""
    return (Cb * math.pi / Lb) * math.sqrt(E * Iy * G_ * J + (math.pi * E / Lb) ** 2 * Iy * Cw)


def mn_fuerte(p, fy, E, G_, Lb, Cb, c):
    """Mn alrededor del eje fuerte. Patin compacto: 6.3 (ref. Mp). Patin no compacto: 6.4 (ref. My, sin
    interpolar). Patin esbelto: no revisado. Devuelve (Mn, detalle)."""
    Mp = p['Zx'] * fy
    My = p['Sx'] * fy
    if c['f'] == 'E':
        return None, {'nota': 'patin esbelto (tipo 4): no revisado en esta version'}
    if c['f'] == 'C':
        ref, Z_lim = Mp, p['Zx']
    else:
        ref, Z_lim = My, p['Sx']
    lu, lr = _Lu_Lr(Z_lim, fy, Cb, G_, p['J'], p['Cw'], p['Iy'], E)
    me = _Me(Cb, Lb, E, G_, p['Iy'], p['J'], p['Cw'])
    if Lb <= lu:
        mn = ref                                                     # 6.3.1 / 6.4.1
    elif me > (2.0 / 3.0) * ref:
        mn = min(1.15 * ref * (1.0 - 0.28 * ref / me), ref)          # 6.3.2 / 6.4.2
    else:
        mn = me                                                      # 6.3.3 / 6.4.3
    return mn, dict(Lu=lu, Lr=lr, Me=me, Mp=Mp, My=My)


def mn_debil(p, fy, c):
    """Mn alrededor del eje debil (6.6). Patin compacto: min(Zy Fy, 1.6 Sy Fy). Patin no compacto: Sy Fy
    (conservador; 6.6.2b permite interpolar). Patin esbelto: no revisado."""
    if c['f'] == 'E':
        return None, 'patin esbelto (tipo 4): no revisado en esta version'
    if c['f'] == 'NC':
        return p['Sy'] * fy, 'patin no compacto: Mn = Sy Fy (conservador)'
    return min(p['Zy'] * fy, 1.6 * p['Sy'] * fy), ''


# ---------------------------------------------------------------- cortante (capitulo 7)
def _cv(ht, kv, E, fy):
    """Coeficiente Cv (7.2.3 a 7.2.5)."""
    lim1 = 1.10 * math.sqrt(kv * E / fy)
    lim2 = 1.37 * math.sqrt(kv * E / fy)
    if ht <= lim1:
        return 1.0
    if ht <= lim2:
        return lim1 / ht
    return 1.51 * kv * E / (ht ** 2 * fy)


def vn_fuerte(p, fy, E, c, forma='I'):
    """Cortante en el plano del alma (7.2.1): Aa = h tw. Vigas I laminadas (h/tw <= 2.24 sqrt(E/Fy)): Cv = 1, FR = 1."""
    aa = c['h'] * p['e']
    ht = c['h'] / p['e']
    if forma == 'I' and ht <= 2.24 * math.sqrt(E / fy):
        cv, fr = 1.0, FR_CORTANTE_LAMINADA
    else:
        cv, fr = _cv(ht, 5.0, E, fy), FR_CORTANTE_OTRAS
    vn = 0.6 * fy * aa * cv
    return fr * vn, dict(Aa=aa, ht=ht, Cv=cv, FR=fr)


def vn_debil(p, fy, E, c):
    """Cortante normal al alma (7.7): Aa = bf tf, b/tp con b = bf/2, kv = 1.2, FR = 0.9."""
    aa = p['B'] * p['t']
    ht = c['b'] / p['t']
    cv = _cv(ht, 1.2, E, fy)
    vn = 0.6 * fy * aa * cv
    return FR_CORTANTE_OTRAS * vn, dict(Aa=aa, ht=ht, Cv=cv, FR=FR_CORTANTE_OTRAS)


# ---------------------------------------------------------------- tabla 12.1.2 (ductilidad, NTC 12.1.7)
def ductilidad_12_1_2(p, fy, E, forma, elemento, nivel):
    """Tabla 12.1.2: limites de ancho/grueso de elementos en compresion de miembros de ductilidad alta (lambda_da)
    y media (lambda_dm). Patin de I o canal (no atiesado): b/t <= 0.30 (alta) o 0.38 (media) sqrt(E/Fy).
    Alma de I de contraventeo (ARRIOSTRE o DIAGONAL): h/t <= 1.49 sqrt(E/Fy). nivel = 'ALTA', 'MEDIA' o None
    (None = solo informativo). U <= 1 cumple. Devuelve None para secciones macizas."""
    if p is None or forma not in ('I', 'canal'):
        return None
    r = math.sqrt(E / fy)
    b = p['B'] / 2.0 if forma == 'I' else p['B']
    lf = b / p['t']
    lda_f, ldm_f = 0.30 * r, 0.38 * r
    out = {'lf': lf, 'lda_f': lda_f, 'ldm_f': ldm_f}
    u = None
    if nivel == 'ALTA':
        u = lf / lda_f
    elif nivel == 'MEDIA':
        u = lf / ldm_f
    if forma == 'I' and elemento in ('ARRIOSTRE', 'DIAGONAL'):
        h = p['H'] - 2.0 * p['t']
        lw = h / p['e']
        lim_w = 1.49 * r
        out.update({'lw': lw, 'lda_w': lim_w})
        if nivel is not None:
            uw = lw / lim_w
            u = max(u, uw) if u is not None else uw
    out['U'] = u
    return out


# ---------------------------------------------------------------- canales CE (un eje de simetria)
def clasificar_canal(p, fy, E):
    """Canal: patin de ancho total B (tabla 3.2.2, canales) y alma h = d - 2 tf."""
    r = math.sqrt(E / fy)
    b = p['B']
    tf = p['t']
    tw = p['e']
    h = p['H'] - 2.0 * tf
    lf, lw = b / tf, h / tw
    lpf, lrf = 0.38 * r, 1.00 * r
    lpw, lrw = 3.76 * r, 5.70 * r
    f = 'C' if lf <= lpf else ('NC' if lf <= lrf else 'E')
    w = 'C' if lw <= lpw else ('NC' if lw <= lrw else 'E')
    if f == 'C' and w == 'C':
        tipo = 'tipo 2'
    elif f != 'E' and w != 'E':
        tipo = 'tipo 3'
    else:
        tipo = 'tipo 4'
    comp_ok = (lf <= 0.56 * r) and (lw <= 1.49 * r)
    return dict(r=r, b=b, h=h, lf=lf, lw=lw, lpf=lpf, lrf=lrf, lpw=lpw, lrw=lrw,
                f=f, w=w, tipo=tipo, comp_ok=comp_ok)


def resistencia_compresion_canal(p, fy, E, G_, Kx, Ky, Lx, Ly, Lz, Kz=K_SUPUESTO):
    """Canal (5.2.2.2): Fe = min(Fex, Fe acoplada), Fe acoplada = flexion alrededor del eje debil (no simetrico) con
    torsion (5.2.6). xo = sqrt(ro^2 - (Ix+Iy)/A) (centro de cortante), yo = 0. n = 1.4 y FR = 0.9 (5.2.2).
    Interpretacion: la nota de 5.2.6 dice 'eje de simetria X'; la flexion acoplada se toma alrededor de Y (conservador)."""
    A = p['A']
    ro = p['ro']
    xo = math.sqrt(max(ro ** 2 - (p['Ix'] + p['Iy']) / A, 0.0))
    H = 1.0 - xo ** 2 / ro ** 2
    fex = math.pi ** 2 * E / (Kx * Lx / p['rx']) ** 2
    fey = math.pi ** 2 * E / (Ky * Ly / p['ry']) ** 2
    fez = (G_ * p['J'] + math.pi ** 2 * E * p['Cw'] / (Kz * Lz) ** 2) / (A * ro ** 2)
    fe_c = (fey + fez) / (2.0 * H) * (1.0 - math.sqrt(max(1.0 - 4.0 * fey * fez * H / (fey + fez) ** 2, 0.0)))
    fe = min(fex, fe_c)
    n = 1.4
    lam = math.sqrt(fy / fe)
    chi = (1.0 + (fy / fe) ** n) ** (-1.0 / n)
    rc = FR * chi * fy * A
    det = {'X': dict(n=n, K=Kx, L=Lx, KL_r=Kx * Lx / p['rx'], Fe=fex, lam=None, chi=None, Rc=None),
           'Y': dict(n=n, K=Ky, L=Ly, KL_r=Ky * Ly / p['ry'], Fe=fey, lam=None, chi=None, Rc=None),
           'flexotorsion': dict(n=n, xo=xo, H=H, Fe=fe_c, Fez=fez, Fe_min=fe, lam=lam, chi=chi, Rc=rc)}
    return rc, det


# ---------------------------------------------------------------- por barra
def _props(perfil):
    p = G.obtener_perfil(perfil)
    if p is None:
        return None, f'perfil {perfil} no existe en la base Gerdau'
    faltan = [k for k in ('H', 'B', 'e', 't', 'A', 'Ix', 'Iy', 'Sx', 'Sy', 'rx', 'ry', 'J', 'Cw', 'Zx', 'Zy')
              if p.get(k) is None]
    return p, faltan


def _diseno_cuadrada(out, p, fy, fu, E, nu, Lx, Ly, Lb, Kx, Ky, Cb):
    """Barra maciza cuadrada (CS): compresion 5.2.1(b) con n = 1.4; flexion 6.11 con d = t = lado;
    cortante no cubierto por la NTC (NO REVISADO)."""
    req = ['H', 'A', 'Ix', 'Iy', 'Sx', 'Sy', 'rx', 'ry', 'J', 'Zx', 'Zy']
    falta = [k for k in req if p.get(k) is None]
    if falta:
        out['notas'].append('faltan datos de seccion: ' + ', '.join(falta))
        return out
    G_ = E / (2.0 * (1.0 + nu))
    a = p['H']
    A = p['A']
    out.update({'tipo': 'tipo 2', 'f': 'C', 'w': 'C', 'comp_ok': True, 'forma': 'maciza'})
    res = {}
    for eje, K, L, rr in [('X', Kx, Lx, p['rx']), ('Y', Ky, Ly, p['ry'])]:
        kl_r = K * L / rr
        fe = math.pi ** 2 * E / kl_r ** 2
        chi = (1.0 + (fy / fe) ** 1.4) ** (-1.0 / 1.4)
        res[eje] = dict(n=1.4, K=K, L=L, KL_r=kl_r, Fe=fe, chi=chi, Rc=FR * chi * fy * A)
    out['Rc'] = min(res['X']['Rc'], res['Y']['Rc'])
    out['pandeo'] = res
    mp = p['Zx'] * fy
    my = p['Sx'] * fy
    rel = Lb * a / (a * a)                       # L d / t^2 con d = t = a
    lim1, lim2 = 0.08 * E / fy, 1.9 * E / fy
    if rel <= lim1:
        mn = min(mp, 1.6 * fy * p['Sx'])                                  # 6.11.1
    elif rel <= lim2:
        mn = min(Cb * (1.52 - 0.274 * rel * fy / E) * my, mp)             # 6.11.2
    else:
        mn = min((1.9 * E * Cb / rel) * p['Sx'], mp)                      # 6.11.3 y 6.11.4
    out['MRx'] = FR * mn
    out['MRy'] = FR * min(p['Zy'] * fy, 1.6 * fy * p['Sy'])               # 6.11.1 eje debil
    out['VRy'], out['VRz'] = None, None
    out['notas'].append('NO REVISADO: cortante de barra maciza (la NTC no da regla para esta seccion)')
    out['det_x'] = dict(rel=rel, Mp=mp, My=my)
    out['Mpx'] = p['Zx'] * fy
    out['Mpy'] = p['Zy'] * fy
    out['Py'] = A * fy
    out['Lb'] = Lb
    return out


def diseno_barra(perfil, familia, fy, fu, E, nu, Lx, Ly, Lb, Kx, Ky, Cb=CB_CONSERVADOR):
    """Capacidades de una barra (cm, kgf). IR (perfil I) y CE (canal). Otras familias: solo tension (ver notas)."""
    out = {'perfil': perfil, 'notas': []}
    p, falta = _props(perfil)
    if p is None:
        out['notas'].append(falta)
        return out
    if p.get('A') is None:
        out['notas'].append('falta el area A de la base Gerdau')
        return out
    G_ = E / (2.0 * (1.0 + nu))
    out['Rt'], nota_t = resistencia_traccion(p, fy, fu)
    if nota_t:
        out['notas'].append(nota_t)
    if familia == 'CS':
        return _diseno_cuadrada(out, p, fy, fu, E, nu, Lx, Ly, Lb, Kx, Ky, Cb)
    if familia not in ('IR', 'CE'):
        out['notas'].append(f'NO REVISADO: familia {familia} (en esta version solo tension)')
        return out
    if falta:
        out['notas'].append('faltan datos de seccion: ' + ', '.join(falta))
        return out
    if familia == 'IR':
        c = clasificar_i(p, fy, E)
        forma = 'I'
    else:
        c = clasificar_canal(p, fy, E)
        forma = 'canal'
    out.update({k: c[k] for k in ('tipo', 'f', 'w', 'lf', 'lpf', 'lrf', 'lw', 'lpw', 'lrw', 'comp_ok')})
    out['forma'] = forma
    if forma == 'I':
        out['Rc'], out['pandeo'] = resistencia_compresion(p, fy, E, Kx, Ky, Lx, Ly, c)
    else:
        if p.get('ro') is None:
            out['notas'].append('falta ro (centro de cortante) de la base: compresion NO REVISADA')
            out['Rc'], out['pandeo'] = None, None
        else:
            out['Rc'], out['pandeo'] = resistencia_compresion_canal(p, fy, E, G_, Kx, Ky, Lx, Ly, max(Lx, Ly))
            out['notas'].append('canal: Kz = 1.0 y Lz = mayor longitud no arriostrada (conservador)')
    if not c['comp_ok']:
        out['notas'].append('elemento esbelto en compresion (tabla 3.2.1): compresion NO REVISADA')
    out['MRx'], det_x = mn_fuerte(p, fy, E, G_, Lb, Cb, c)
    if out['MRx'] is not None:
        out['MRx'] = FR * out['MRx']
    else:
        out['notas'].append(det_x['nota'])
    out['det_x'] = det_x
    mn_y, nota_y = mn_debil(p, fy, c)
    out['MRy'] = None if mn_y is None else FR * mn_y
    if nota_y:
        out['notas'].append(nota_y)
    out['VRy'], out['det_vy'] = vn_fuerte(p, fy, E, c, forma)
    out['VRz'], out['det_vz'] = vn_debil(p, fy, E, c)
    out['Mpx'] = p['Zx'] * fy
    out['Mpy'] = p['Zy'] * fy
    out['Py'] = p['A'] * fy
    out['Lb'] = Lb
    return out


def _mx(valores):
    v = [abs(x) for x in valores if x is not None and not (isinstance(x, float) and math.isnan(x))]
    return max(v) if v else 0.0


def chequeos_caso(r, b):
    """Revisa un caso (fila de demandas) contra las capacidades de la barra. r: dict con N, V, M por extremo.
    Devuelve dict con C1..C10 (None = no revisado)."""
    fy_ok = b.get('Rt') is not None
    pc_I, pc_J = max(r['N_I'], 0.0), max(-r['N_J'], 0.0)
    pt_I, pt_J = max(-r['N_I'], 0.0), max(r['N_J'], 0.0)
    pc, pt = max(pc_I, pc_J), max(pt_I, pt_J)
    mx_I, mx_J = abs(r['Mz_I']) * 100.0, abs(r['Mz_J']) * 100.0      # kgf·m -> kgf·cm (eje fuerte)
    my_I, my_J = abs(r['My_I']) * 100.0, abs(r['My_J']) * 100.0     # eje debil
    mx, my = max(mx_I, mx_J), max(my_I, my_J)
    vy, vz = max(abs(r['Vy_I']), abs(r['Vy_J'])), max(abs(r['Vz_I']), abs(r['Vz_J']))
    res = {}
    res['C1 tension'] = (pt / b['Rt']) if (fy_ok and pt > 0) else (0.0 if pt == 0 else None)
    if pc > 0:
        res['C2 compresion'] = (pc / b['Rc']) if (b.get('Rc') and b.get('comp_ok')) else None
    else:
        res['C2 compresion'] = 0.0
    if b.get('MRx') is not None:
        res['C3 flexion eje fuerte'] = mx / b['MRx']
    else:
        res['C3 flexion eje fuerte'] = None
    res['C4 flexion eje debil'] = (my / b['MRy']) if b.get('MRy') else None
    res['C5 cortante eje fuerte'] = (vy / b['VRy']) if b.get('VRy') else None
    res['C6 cortante eje debil'] = (vz / b['VRz']) if b.get('VRz') else None
    # C7: flexocompresion en extremos (8.1.1 y 8.1.3 si tipo 2; 8.1.5 si tipo 3)
    c7 = None
    if pc > 0 and b.get('tipo') in ('tipo 2', 'tipo 3') and b.get('MRx') is not None and b.get('MRy') is not None:
        vals = []
        for pe, mxe, mye in [(pc_I, mx_I, my_I), (pc_J, mx_J, my_J)]:
            if pe <= 0:
                continue
            if b.get('forma') in ('canal', 'maciza') and b['tipo'] == 'tipo 2':
                if not b.get('Mpy'):
                    continue
                vals.append(pe / (FR * b['Py']) + mxe / (FR * b['Mpx']) + mye / (FR * b['Mpy']))   # 8.1.4
                continue
            if b['tipo'] == 'tipo 2':
                if not b.get('Mpy'):
                    continue
                u1 = pe / (FR * b['Py']) + 0.85 * mxe / (FR * b['Mpx']) + 0.60 * mye / (FR * b['Mpy'])
                u2 = mxe / (FR * b['Mpx']) + mye / (FR * b['Mpy'])
                vals.append(max(u1, u2))
            else:
                vals.append(pe / (FR * b['Py']) + mxe / b['MRx'] + mye / b['MRy'])
        c7 = max(vals) if vals else 0.0
    res['C7 flexocompresion extremos'] = c7 if pc > 0 else 0.0
    # C8: columna completa (8.1.6 o 8.1.7)
    if pc > 0 and b.get('Rc') is not None and b.get('MRx') is not None and b.get('MRy') is not None and b.get('comp_ok'):
        if b['tipo'] == 'tipo 2' and b.get('Mpy'):
            res['C8 flexocompresion columna'] = pc / b['Rc'] + mx / b['MRx'] + my / (FR * b['Mpy'])
        elif b['tipo'] == 'tipo 3':
            res['C8 flexocompresion columna'] = pc / b['Rc'] + mx / b['MRx'] + my / b['MRy']
        else:
            res['C8 flexocompresion columna'] = None
    else:
        res['C8 flexocompresion columna'] = None if pc > 0 else 0.0
    # C9: flexotension (8.2.1)
    if pt > 0 and (mx > 0 or my > 0):
        if fy_ok and b.get('MRx') is not None and b.get('MRy') is not None:
            res['C9 flexotension'] = pt / b['Rt'] + mx / b['MRx'] + my / b['MRy']
        else:
            res['C9 flexotension'] = None
    else:
        res['C9 flexotension'] = 0.0
    # C10: flexion y cortante (8.3.1), eje fuerte
    if b.get('MRx') and b.get('VRy'):
        res['C10 flexion y cortante'] = (mx / b['MRx']) ** 2 + (vy / b['VRy']) ** 2
    else:
        res['C10 flexion y cortante'] = None
    res['C11 ductilidad 12.1.2'] = b.get('C11')
    res['C12 esbeltez global KL/r<=200'] = b.get('C12')
    res['C13 columna ductil KL/r<=60'] = b.get('C13')
    return res


def _columnas_caso():
    return ['C1 tension', 'C2 compresion', 'C3 flexion eje fuerte', 'C4 flexion eje debil',
            'C5 cortante eje fuerte', 'C6 cortante eje debil', 'C7 flexocompresion extremos',
            'C8 flexocompresion columna', 'C9 flexotension', 'C10 flexion y cortante',
            'C11 ductilidad 12.1.2', 'C12 esbeltez global KL/r<=200', 'C13 columna ductil KL/r<=60']


# ---------------------------------------------------------------- planilla -> diseño
def _num(v, defecto=None):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return defecto
    try:
        return float(v)
    except (TypeError, ValueError):
        return defecto


def diseno_desde_planilla(dib, materiales_filas, longitudes_df, demandas, id_barra, ruta_salida):
    raise NotImplementedError('Sin uso en la version actual: sus demandas leen solo momentos de extremo (auditoria H-01). No usar.')
    """Diseña todas las barras con las combinaciones del analisis. Las longitudes y factores K vienen de la hoja
    LONGITUDES; lo vacio se toma como supuesto y se marca en la salida. demandas: tabla BARRAS del analisis
    (todas las barras y casos). Devuelve (capacidades, resultados, detalle) y escribe el Excel."""
    if materiales_filas is None:
        raise ErrorDiseno('la planilla no tiene la hoja MATERIALES: no hay fy, fu, E ni nu.')
    mat = {}
    for f in materiales_filas:
        mat[str(f[0]).strip()] = dict(fy=_num(f[2]), fu=_num(f[3]), E=_num(f[4]), nu=_num(f[5]))
    lon = {}
    for _, r in longitudes_df.iterrows():
        id_ = _num(r.get('ID'))
        if id_ is not None:
            lon[int(id_)] = r
    combinaciones = demandas[demandas['TIPO'] == 'COMBINACION']
    if combinaciones.empty:
        raise ErrorDiseno('no hay combinaciones de carga: defina COMBINACIONES (acciones factoradas).')

    caps, res_filas, det_filas = [], [], []
    for i, b in enumerate(dib.barras):
        id_ = id_barra.get(i, i)
        perfil = b['perfil']
        m = mat.get(str(b.get('material') or '').strip())
        p0, _ = _props(perfil)
        familia = p0['familia'] if p0 else '?'
        L = float(np.linalg.norm(np.array(dib.nodos[b['n2']]) - np.array(dib.nodos[b['n1']]))) * 100.0
        r = lon.get(id_)
        supuestos = []

        def campo(nombre):
            return _num(r.get(nombre)) if r is not None else None

        lx, ly, lb = campo('LX (m)'), campo('LY (m)'), campo('LB (m)')
        kx, ky, cb = campo('KX'), campo('KY'), campo('CB')
        if lx is None:
            lx = L / 100.0; supuestos.append('LX = longitud de la barra (supuesto)')
        if ly is None:
            ly = L / 100.0; supuestos.append('LY = longitud de la barra (supuesto)')
        if lb is None:
            lb = L / 100.0; supuestos.append('LB = longitud de la barra (supuesto)')
        if kx is None:
            kx = K_SUPUESTO; supuestos.append('KX = 1.0 (supuesto)')
        if ky is None:
            ky = K_SUPUESTO; supuestos.append('KY = 1.0 (supuesto)')
        if cb is None:
            cb = CB_CONSERVADOR; supuestos.append('CB = 1.0 (conservador)')
        if m is None or m['fy'] is None or m['E'] is None or m['nu'] is None:
            caps.append({'ID': id_, 'PERFIL': perfil, 'NOTA': 'material sin fy, E o nu en MATERIALES'})
            res_filas.append([id_, perfil, b.get('material'), None, None, None, 'DATOS INCOMPLETOS',
                              'material sin fy, E o nu en MATERIALES'])
            continue
        cap = diseno_barra(perfil, familia, m['fy'], m['fu'], m['E'], m['nu'],
                           lx * 100.0, ly * 100.0, lb * 100.0, kx, ky, cb)
        duct_raw = r.get('DUCTILIDAD') if r is not None else None
        nivel = None
        if duct_raw is not None and not (isinstance(duct_raw, float) and math.isnan(duct_raw)) \
                and str(duct_raw).strip() != '':
            nivel = str(duct_raw).strip().upper()
        dd = ductilidad_12_1_2(p0, m['fy'], m['E'], cap.get('forma'), dib.barras[i]['elemento'], nivel)
        elem = dib.barras[i]['elemento']
        es_comp = elem in COMPRESION_GRUPOS
        pdl = cap.get('pandeo') or {}
        kl_max = max(pdl['X']['KL_r'], pdl['Y']['KL_r']) if pdl else None
        cap['C12'] = (kl_max / 200.0 if kl_max is not None else None) if es_comp else 0.0
        cap['C13'] = ((kl_max / 60.0 if kl_max is not None else None)
                      if (elem == 'COLUMNA' and nivel in ('ALTA', 'MEDIA')) else 0.0)
        if kl_max is not None and es_comp and kl_max > 200.0:
            cap['notas'].append('NO CUMPLE KL/r <= 200 (2.3.2)')
        if 'lw' in cap:
            cap.update({'LIM ALMA compresion (3.2.1)': 1.49 * math.sqrt(m['E'] / m['fy']),
                        'LIM ALMA flexion compacta (3.2.2)': cap.get('lpw'),
                        'LIM ALMA flexion no compacta (3.2.2)': cap.get('lrw')})
        if dd is not None:
            cap.update({'DUCTILIDAD': nivel or '(informativo)', 'LAMBDA ALA': dd['lf'],
                        'LIM ALA ALTA': dd['lda_f'], 'LIM ALA MEDIA': dd['ldm_f'],
                        'LAMBDA ALMA': dd.get('lw'), 'LIM ALMA': dd.get('lda_w'), 'C11': dd['U']})
            if nivel is None:
                cap['notas'].append('tabla 12.1.2: sin ductilidad asignada (informativo)')
            elif dd['U'] is not None and dd['U'] > 1.0:
                cap['notas'].append(f'NO CUMPLE tabla 12.1.2 (ductilidad {nivel})')
        cap.update({'ID': id_, 'MATERIAL': b.get('material'), 'LX (m)': lx, 'LY (m)': ly, 'LB (m)': lb,
                    'KX': kx, 'KY': ky, 'CB': cb, 'SUPUESTOS': '; '.join(supuestos)})
        caps.append(cap)
        mejor, peor_caso = None, None
        for _, fila in combinaciones[combinaciones['BARRA'] == id_].iterrows():
            d = dict(N_I=_num(fila['N I (kgf)'], 0.0), N_J=_num(fila['N J (kgf)'], 0.0),
                     Vy_I=_num(fila['Vy I (kgf)'], 0.0), Vy_J=_num(fila['Vy J (kgf)'], 0.0),
                     Vz_I=_num(fila['Vz I (kgf)'], 0.0), Vz_J=_num(fila['Vz J (kgf)'], 0.0),
                     Mz_I=_num(fila['Mz I (kgf·m)'], 0.0), Mz_J=_num(fila['Mz J (kgf·m)'], 0.0),
                     My_I=_num(fila['My I (kgf·m)'], 0.0), My_J=_num(fila['My J (kgf·m)'], 0.0))
            ch = chequeos_caso(d, cap)
            util = [v for v in ch.values() if v is not None]
            maximo = max(util) if util else None
            det_filas.append([id_, perfil, fila['CASO']] + [ch[c] for c in _columnas_caso()] + [maximo])
            if maximo is not None and (mejor is None or maximo > mejor):
                mejor, peor_caso = maximo, fila['CASO']
        notas = cap.get('notas', []) + ([cap['SUPUESTOS']] if cap.get('SUPUESTOS') else [])
        res_filas.append([id_, perfil, b.get('material'), cap.get('tipo'), peor_caso, mejor,
                          _estado(mejor, cap), '; '.join(notas)])
    cols_res = ['ID', 'PERFIL', 'MATERIAL', 'TIPO SECCION', 'CASO QUE GOBIERNA', 'UTILIZACION MAXIMA', 'ESTADO',
                'NOTAS']
    resultados = pd.DataFrame(res_filas, columns=cols_res)
    detalle = pd.DataFrame(det_filas, columns=['ID', 'PERFIL', 'CASO'] + _columnas_caso() + ['MAXIMO'])
    capacidades = pd.DataFrame([_fila_capacidad(c) for c in caps])
    _escribir(ruta_salida, capacidades, resultados, detalle)
    return capacidades, resultados, detalle


def _estado(maximo, cap):
    """CUMPLE / NO CUMPLE segun la utilizacion maxima; REVISAR si alguna parte no se pudo revisar."""
    if cap.get('NOTA'):
        return 'DATOS INCOMPLETOS'
    if maximo is None:
        return 'NO REVISADO'
    notas = ' '.join(cap.get('notas', [])).upper()
    incompleto = 'NO REVISAD' in notas or 'FALTAN DATOS' in notas
    if maximo > 1.0:
        return 'NO CUMPLE'
    return 'REVISAR (partes no revisadas)' if incompleto else 'CUMPLE'


def _fila_capacidad(c):
    """Una fila de la hoja CAPACIDADES: capacidades y longitudes usadas de una barra."""
    rc = c.get('Rc')
    return {'ID': c.get('ID'), 'PERFIL': c.get('perfil') or c.get('PERFIL'), 'MATERIAL': c.get('MATERIAL'),
            'LX (m)': c.get('LX (m)'), 'LY (m)': c.get('LY (m)'), 'LB (m)': c.get('LB (m)'),
            'KX': c.get('KX'), 'KY': c.get('KY'), 'CB': c.get('CB'),
            'TIPO SECCION': c.get('tipo'), 'CLASE ALA': c.get('f'), 'CLASE ALMA': c.get('w'),
            'Rt (kgf)': c.get('Rt'), 'Rc (kgf)': rc, 'MRx (kgf·cm)': c.get('MRx'), 'MRy (kgf·cm)': c.get('MRy'),
            'VRy (kgf)': c.get('VRy'), 'VRz (kgf)': c.get('VRz'),
            'DUCTILIDAD': c.get('DUCTILIDAD'), 'LAMBDA ALA': c.get('LAMBDA ALA'),
            'LIM ALA ALTA': c.get('LIM ALA ALTA'), 'LIM ALA MEDIA': c.get('LIM ALA MEDIA'),
            'LAMBDA ALMA (contraventeo)': c.get('LAMBDA ALMA'), 'LIM ALMA (contraventeo)': c.get('LIM ALMA'),
            'LAMBDA ALMA h/tw': c.get('lw'), 'LIM ALMA compresion (3.2.1)': c.get('LIM ALMA compresion (3.2.1)'),
            'LIM ALMA flexion compacta (3.2.2)': c.get('LIM ALMA flexion compacta (3.2.2)'),
            'LIM ALMA flexion no compacta (3.2.2)': c.get('LIM ALMA flexion no compacta (3.2.2)'),
            'SUPUESTOS': c.get('SUPUESTOS'),
            'NOTAS': '; '.join(c.get('notas') or []) or c.get('NOTA')}


GRUPOS_ELEMENTO = ['COLUMNA', 'VIGA', 'ARRIOSTRE', 'DIAGONAL']
COMPRESION_GRUPOS = ('COLUMNA', 'ARRIOSTRE', 'DIAGONAL')
FAMILIAS_CATALOGO = ('IR', 'CE')   # CS eliminado de la base (macizo)


def _evaluar_perfil_grupo(nombre, fam, grupo, m, lx, ly, lb, kx, ky, nivel, filas_sint):
    """True si el perfil cumple todas las revisiones de la NTC para la envolvente del grupo (resistencia a tension
    con fractura y Fu, esbeltez local y global, flexion, cortante, interaccion y ductilidad).
    Devuelve (cumple, utilizacion_maxima, motivo)."""
    if m['fu'] is None:
        return False, None, 'no revisado: falta Fu del material (fractura en tension)'
    cap = diseno_barra(nombre, fam, m['fy'], m['fu'], m['E'], m['nu'], lx * 100.0, ly * 100.0, lb * 100.0,
                       kx, ky, CB_CONSERVADOR)
    if 'forma' not in cap or cap.get('NOTA'):
        return False, None, 'no revisable'
    p0, _ = _props(nombre)
    dd = ductilidad_12_1_2(p0, m['fy'], m['E'], cap.get('forma'), grupo, nivel)
    if dd is not None:
        cap['C11'] = dd['U']
    pdl = cap.get('pandeo') or {}
    kl_max = max(pdl['X']['KL_r'], pdl['Y']['KL_r']) if pdl else None
    cap['C12'] = (kl_max / 200.0 if kl_max is not None else None) if grupo in COMPRESION_GRUPOS else 0.0
    cap['C13'] = ((kl_max / 60.0 if kl_max is not None else None)
                  if (grupo == 'COLUMNA' and nivel in ('ALTA', 'MEDIA')) else 0.0)
    maciza = cap.get('forma') == 'maciza'
    util, faltan = [], []
    for d in filas_sint:
        for k, v in chequeos_caso(d, cap).items():
            if v is None:
                permitido = (maciza and k in ('C5 cortante eje fuerte', 'C6 cortante eje debil',
                                              'C10 flexion y cortante')) or (k == 'C11 ductilidad 12.1.2' and nivel is None)
                if not permitido:
                    faltan.append(k)
            else:
                util.append(v)
    if faltan:
        return False, None, 'no revisado: ' + ', '.join(sorted(set(faltan)))
    umax = max(util) if util else 0.0
    return umax <= 1.0, umax, 'cumple' if umax <= 1.0 else 'no cumple'


def listas_por_grupo(dib, materiales_filas, longitudes_df, demandas, id_barra, familias=FAMILIAS_CATALOGO):
    raise NotImplementedError('Sin uso en la version actual: sus demandas leen solo momentos de extremo (auditoria H-01). No usar.')
    """Perfiles del catalogo que cumplen (resistencia, Fu, esbeltez local y global) para cada grupo de elementos.
    Usa la envolvente de las combinaciones de las barras del grupo y la longitud, K y ductilidad mas exigentes.
    Devuelve (listas {grupo: [perfil,...] ordenados por masa}, tabla DataFrame)."""
    import collections
    import perfiles_gerdau as GG
    mat = {}
    for f in materiales_filas:
        mat[str(f[0]).strip()] = dict(fy=_num(f[2]), fu=_num(f[3]), E=_num(f[4]), nu=_num(f[5]))
    lon = {}
    for _, r in longitudes_df.iterrows():
        id_ = _num(r.get('ID'))
        if id_ is not None:
            lon[int(id_)] = r
    comb = demandas[demandas['TIPO'] == 'COMBINACION']
    catalogo = [(n, d['familia'], d['masa_kg_m']) for n, d in GG.PERFILES.items() if d['familia'] in familias]
    listas, filas_tabla = {}, []
    for g in GRUPOS_ELEMENTO:
        idx = [i for i, b in enumerate(dib.barras) if b['elemento'] == g]
        listas[g] = None                       # None = sin barras de este grupo
        if not idx:
            continue
        ids = [id_barra.get(i, i) for i in idx]
        f = comb[comb['BARRA'].isin(ids)]
        if f.empty:
            continue
        num = lambda c: f[c].astype(float).fillna(0.0)
        pc = max(num('N I (kgf)').clip(lower=0).max(), (-num('N J (kgf)')).clip(lower=0).max())
        pt = max((-num('N I (kgf)')).clip(lower=0).max(), num('N J (kgf)').clip(lower=0).max())
        mz = max(num('Mz I (kgf·m)').abs().max(), num('Mz J (kgf·m)').abs().max())
        my = max(num('My I (kgf·m)').abs().max(), num('My J (kgf·m)').abs().max())
        vy = max(num('Vy I (kgf)').abs().max(), num('Vy J (kgf)').abs().max())
        vz = max(num('Vz I (kgf)').abs().max(), num('Vz J (kgf)').abs().max())
        base = dict(Vy_I=vy, Vy_J=vy, Vz_I=vz, Vz_J=vz, Mz_I=mz, Mz_J=mz, My_I=my, My_J=my)
        filas_sint = [dict(base, N_I=pc, N_J=-pc), dict(base, N_I=-pt, N_J=pt)]
        lx = ly = lb = kx = ky = 0.0
        niveles, materiales = set(), collections.Counter()
        for i in idx:
            r = lon.get(id_barra.get(i, i))
            L = float(np.linalg.norm(np.array(dib.nodos[dib.barras[i]['n2']]) - np.array(dib.nodos[dib.barras[i]['n1']])))
            def campo(nombre):
                return _num(r.get(nombre)) if r is not None else None
            v_lx, v_ly, v_lb = campo('LX (m)'), campo('LY (m)'), campo('LB (m)')
            v_kx, v_ky = campo('KX'), campo('KY')
            lx = max(lx, v_lx if v_lx is not None else L)
            ly = max(ly, v_ly if v_ly is not None else L)
            lb = max(lb, v_lb if v_lb is not None else L)
            kx = max(kx, v_kx if v_kx is not None else K_SUPUESTO)
            ky = max(ky, v_ky if v_ky is not None else K_SUPUESTO)
            duct = r.get('DUCTILIDAD') if r is not None else None
            if duct is not None and not (isinstance(duct, float) and math.isnan(duct)) and str(duct).strip():
                niveles.add(str(duct).strip().upper())
            materiales[str(dib.barras[i].get('material') or '').strip()] += 1
        nivel = 'ALTA' if 'ALTA' in niveles else ('MEDIA' if 'MEDIA' in niveles else None)
        material = materiales.most_common(1)[0][0]
        m = mat.get(material)
        if m is None or m['fy'] is None or m['E'] is None or m['nu'] is None:
            continue
        cumplen = []
        listas[g] = []
        for nombre, fam, masa in catalogo:
            ok, umax, motivo = _evaluar_perfil_grupo(nombre, fam, g, m, lx, ly, lb, kx, ky, nivel, filas_sint)
            if ok and fam == 'CS':
                motivo = 'cumple (cortante no cubierto por la NTC para barra maciza)'
            filas_tabla.append([g, nombre, fam, masa, material, umax, motivo])
            if ok:
                cumplen.append((masa if masa is not None else 1e9, nombre, umax))
        cumplen.sort()
        listas[g] = [n for _, n, _ in cumplen]
    tabla = pd.DataFrame(filas_tabla, columns=['GRUPO', 'PERFIL', 'FAMILIA', 'MASA (kg/m)', 'MATERIAL',
                                                'UTILIZACION MAXIMA', 'RESULTADO'])
    return listas, tabla


def _escribir(ruta, capacidades, resultados, detalle):
    with pd.ExcelWriter(ruta, engine='openpyxl') as w:
        resumen = pd.DataFrame([
            ['Norma', 'NTC para Diseño y Construcción de Estructuras de Acero, CDMX (Gaceta 7-jul-2020)'],
            ['Alcance', 'Perfiles I (IR) completos. Otras familias: solo tension. El resto: NO REVISADO.'],
            ['Acciones', 'Combinaciones de COMBINACIONES del analisis; deben venir factoradas.'],
            ['Analisis', 'Primer orden: no incluye P-Delta ni P-delta.'],
            ['Criterios', 'Cb = 1.0; Ae = A (sin agujeros); patin no compacto: Mn = My; eje debil no compacto: Sy Fy.'],
            ['Unidades', 'fuerzas kgf; momentos kgf·cm en capacidades y kgf·m en acciones; longitudes cm'],
        ], columns=['DATO', 'VALOR'])
        resumen.to_excel(w, sheet_name='RESUMEN', index=False)
        capacidades.to_excel(w, sheet_name='CAPACIDADES', index=False)
        resultados.to_excel(w, sheet_name='RESULTADOS', index=False)
        detalle.to_excel(w, sheet_name='DETALLE', index=False)
        for hoja in w.book.worksheets:
            for col in hoja.columns:
                largo = max(len(str(c.value)) if c.value is not None else 0 for c in col)
                hoja.column_dimensions[col[0].column_letter].width = min(max(10, largo + 2), 40)
            hoja.freeze_panes = 'A2'


In [ ]:
%%writefile diseno_asd.py
"""
DISENO ASD SEGUN ANSI/AISC 360-22 (Specification for Structural Steel Buildings, 2022).
Familias: IR (perfil I doblemente simetrico), CE (canal, eje x de simetria), LI (angulo de lados iguales).
Fuera del diseno: LM (no se considera por instruccion del usuario). Los perfiles macizos CS y OS se eliminaron de la base.
Unidades: cm, kgf (fuerzas), kgf·cm (momentos). Demandas: ASD (combinaciones de servicio de COMBINACIONES).

Referencias (capitulo y ecuacion del texto de AISC 360-22):
  D2   Traccion: Pn = Fy Ag (Ωt = 1.67); Pn = Fu Ae (Ωt = 2.00)
  E3   Pandeo por flexion: Fe = pi^2 E/(KL/r)^2; Fn = 0.658^(Fy/Fe) Fy si Fy/Fe <= 2.25; 0.877 Fe si no. (E3-1..4)
  E4   Torsional y flexotorsional: E4-2 (doblemente simetrico); E4-3 con Fex si el eje x es de simetria (canales).
  E5   Angulos sencillos: Lc/rz = 72 + 0.75 L/ra (E5-1) o 32 + 1.25 L/ra (E5-2) si se cumplen las condiciones de
       E5(a) (patas de ARRIOSTRE y DIAGONAL). En otros casos se usa K L/rz y el caso debe ir por H2.
       Flexotorsional no requerido si b/t <= 0.71 sqrt(E/Fy); si no, no implementado (se marca NO REVISADO).
  E7   Elementos esbeltos: Ae = Ag - sum (b - be) t; be = b si lambda <= lambda_r; si no E7-3 con Fel = (c2 lambda_r/lambda)^2 Fy.
       Angulos: E3 solamente (E7); patas no atiesadas: lambda_r = 0.45 sqrt(E/Fy), c1 = 0.22, c2 = 1.49 (B4.1a, E7.1c).
  F2   Flexion eje mayor, compacta: Mn = min(Mp, LTB). Mp = Fy Zx; Lp, Lr, Fcr (F2-4..8).
  F3   Patin no compacto/esbelto (I doblemente simetrico): F3-1 y F3-2.
  F6   Flexion eje menor (I y canales): Mn = min(Mp, 1.6 Fy Sy) con F6-2 (no compacto) y F6-3 (esbelto).
  F10  Angulos sencillos, flexion sobre ejes principales: F10-1 (1.5 My); F10-2/3 (LTB con Mcr); F10-4 (Mcr eje
       principal mayor, beta_w = 0 en lados iguales); F10.3 pandeo local de pata: F10-6, F10-7, F10-8 (lambda_p =
       0.54 sqrt(E/Fy), lambda_r = 0.91 sqrt(E/Fy); Table B4.1b caso 12).
  G2   Cortante de almas (I y canales): Vn = 0.6 Fy Aw Cv1 (G2-1); Ωv = 1.50 para I laminada (G2.1a), 1.67 si no.
  G4   Cortante de HSS rectangulares: Vn = 0.6 Fy Aw Cv2 (G4-1), Aw = 2ht, Cv2 de G2.2 (G2-9 a G2-11) con kv = 5.
  G3   Angulo sencillo: Vn = 0.6 Fy b t Cv2 (G3-1), con h/tw = b/t y kv = 1.2 (una pata, conservador).
  H3   HSS: torsion H3.1 (Tn = Fcr C, ΩT = 1.67, C de la nota de usuario); H3.2: H3-6 si Tr > 0.2 Tc, si no H1.
  H1   Flexocompresion H1-1a/b (Pr/Pc >= 0.2: 8/9); flexotraccion H1-1a/b con Pc = Pn/Ωt.
  H2   Angulos (no cubiertos por H1): H2-1 con momento resultante sobre cada eje principal (conservador).

Supuestos de estimacion (se listan en SUPUESTOS de cada barra):
  IR faltantes: Zx = B t (H - t) + e h^2/4, Zy = B^2 t/2 + h e^2/4 (h = H - 2t); Cw = Iy ho^2/4 (E4, nota de usuario).
  LI: area, Ix e rx de catalogo; centroide, producto de inercia y ejes principales por geometria rectangular (sin acuerdos).
"""
import math
import re
import numpy as np
import pandas as pd
import perfiles_gerdau as G
# DECISION D18 (abierta): resistencia por AISC 360-22 ASD; la NTC-Acero se usa para esbelteces y ductilidad (requisito del proyecto).

# Valores de norma (AISC 360-22 y NTC-Acero CDMX): se leen de la hoja NORMA de la planilla con cargar_norma().
# Este modulo no trae valores de norma ni supuestos: si falta un valor, el calculo se detiene.
OMEGA_T_FLU = OMEGA_T_FRA = OMEGA_C = OMEGA_B = OMEGA_V = OMEGA_V_LAMINADA = None
KV_I = KV_ANGULO = CB_MAX_ANGULO = None
AISC_LP_PATIN = AISC_LR_PATIN = AISC_CV_LIM = AISC_HTW_LAM = AISC_LIM_ANG_E4 = AISC_E5_RAT_LIM = None
AISC_B41A = NTC_LR_COMPRESION = NTC_LIMITES_FLEXION = None
NTC_KLR_COMPRESION = NTC_KLR_TENSION_REF = LIMITE_KLR = None
KZ_TORSION = None
AISC_HSS_LP_PATIN = AISC_HSS_LR_PATIN = AISC_HSS_LP_ALMA = AISC_HSS_LR_ALMA = AISC_HSS_LR_COMP = None
AISC_HSS_C1 = AISC_HSS_C2 = NTC_HSS_LR_COMP = None
NTC_HSS_FLEX = NTC_HSS_DUC = None
AISC_KV_HSS = AISC_CV2_LIM2 = AISC_CV2_K = None
AISC_OMEGA_T_HSS = AISC_H3_LIM1 = AISC_H3_LIM2 = AISC_H3_LIM3 = AISC_H3_K = AISC_H3_FCR_Y = AISC_H3_20PCT = None


def _req(valor, campo_nombre, barra=''):
    """Dato obligatorio de LONGITUDES (KX, KY, CB, LX, LY, LB): sin valor no se supone nada, se detiene el diseno."""
    if valor is None:
        raise ErrorAsd(f'{barra}falta {campo_nombre} en LONGITUDES (dato obligatorio para el diseno)')
    return valor


def cargar_norma(v, kz):
    """Carga los valores de norma desde la hoja NORMA ({CLAVE: valor}) y el Kz elegido en PARAMETROS."""
    global KZ_TORSION
    KZ_TORSION = kz
    global OMEGA_T_FLU, OMEGA_T_FRA, OMEGA_C, OMEGA_B, OMEGA_V, OMEGA_V_LAMINADA, KV_I, KV_ANGULO, CB_MAX_ANGULO
    global AISC_LP_PATIN, AISC_LR_PATIN, AISC_CV_LIM, AISC_HTW_LAM, AISC_LIM_ANG_E4, AISC_E5_RAT_LIM, AISC_B41A
    global NTC_LR_COMPRESION, NTC_LIMITES_FLEXION, NTC_KLR_COMPRESION, NTC_KLR_TENSION_REF, LIMITE_KLR
    global AISC_HSS_LP_PATIN, AISC_HSS_LR_PATIN, AISC_HSS_LP_ALMA, AISC_HSS_LR_ALMA, AISC_HSS_LR_COMP
    global AISC_HSS_C1, AISC_HSS_C2, NTC_HSS_LR_COMP, NTC_HSS_FLEX, NTC_HSS_DUC
    global AISC_KV_HSS, AISC_CV2_LIM2, AISC_CV2_K
    global AISC_OMEGA_T_HSS, AISC_H3_LIM1, AISC_H3_LIM2, AISC_H3_LIM3, AISC_H3_K, AISC_H3_FCR_Y, AISC_H3_20PCT
    try:
        OMEGA_T_FLU, OMEGA_T_FRA = v['AISC_OMEGA_TFLU'], v['AISC_OMEGA_TFRA']
        OMEGA_C, OMEGA_B, OMEGA_V = v['AISC_OMEGA_C'], v['AISC_OMEGA_B'], v['AISC_OMEGA_V']
        OMEGA_V_LAMINADA, KV_I, KV_ANGULO = v['AISC_OMEGA_V_LAM'], v['AISC_KV_I'], v['AISC_KV_ANG']
        CB_MAX_ANGULO = v['AISC_CB_MAX_ANG']
        AISC_LP_PATIN, AISC_LR_PATIN = v['AISC_LP_PATIN'], v['AISC_LR_PATIN']
        AISC_CV_LIM, AISC_HTW_LAM = v['AISC_CV_LIM'], v['AISC_HTW_LAM']
        AISC_LIM_ANG_E4, AISC_E5_RAT_LIM = v['AISC_LIM_ANG_E4'], v['AISC_E5_RAT_LIM']
        AISC_B41A = {'alma': (v['AISC_B41A_C1_ALMA'], v['AISC_B41A_C2_ALMA']),
                     'patin': (v['AISC_B41A_C1_PATA'], v['AISC_B41A_C2_PATA']),
                     'lr_pata': v['AISC_LR_PATA_B41A']}
        NTC_LR_COMPRESION = {'patin': v['NTC_LR_PATIN'], 'alma': v['NTC_LR_ALMA'], 'pata': v['NTC_LR_PATA']}
        NTC_LIMITES_FLEXION = {'patin': (v['NTC_LIM_PATIN_PL'], v['NTC_LIM_PATIN_P'], v['NTC_LIM_PATIN_R']),
                               'alma': (v['NTC_LIM_ALMA_PL'], v['NTC_LIM_ALMA_P'], v['NTC_LIM_ALMA_R']),
                               'pata': (None, v['NTC_LIM_PATA_P'], v['NTC_LIM_PATA_R'])}
        NTC_KLR_COMPRESION = v['NTC_KLR_COMP']
        NTC_KLR_TENSION_REF = {'principal': v['NTC_KLR_REF_PRINC'], 'contraventeo': v['NTC_KLR_REF_CONTR']}
        LIMITE_KLR = v['NTC_KLR_COMP']
        AISC_HSS_LP_PATIN, AISC_HSS_LR_PATIN = v['AISC_HSS_LP_PATIN'], v['AISC_HSS_LR_PATIN']
        AISC_HSS_LP_ALMA, AISC_HSS_LR_ALMA = v['AISC_HSS_LP_ALMA'], v['AISC_HSS_LR_ALMA']
        AISC_HSS_LR_COMP, AISC_HSS_C1, AISC_HSS_C2 = v['AISC_HSS_LR_COMP'], v['AISC_HSS_C1'], v['AISC_HSS_C2']
        NTC_HSS_LR_COMP = v['NTC_HSS_LR_COMP']
        NTC_HSS_FLEX = {'patin': (v['NTC_HSS_LIM_PATIN_PL'], v['NTC_HSS_LIM_PATIN_P'], v['NTC_HSS_LIM_PATIN_R']),
                        'alma': (None, v['NTC_HSS_LIM_ALMA_P'], v['NTC_HSS_LIM_ALMA_R'])}
        NTC_HSS_DUC = {'ALTA': v['NTC_HSS_DUC_ALTA'], 'MEDIA': v['NTC_HSS_DUC_MEDIA']}
        AISC_KV_HSS, AISC_CV2_LIM2, AISC_CV2_K = v['AISC_KV_HSS'], v['AISC_CV2_LIM2'], v['AISC_CV2_K']
        AISC_OMEGA_T_HSS = v['AISC_OMEGA_T_HSS']
        AISC_H3_LIM1, AISC_H3_LIM2, AISC_H3_LIM3 = v['AISC_H3_LIM1'], v['AISC_H3_LIM2'], v['AISC_H3_LIM3']
        AISC_H3_K, AISC_H3_FCR_Y, AISC_H3_20PCT = v['AISC_H3_K'], v['AISC_H3_FCR_Y'], v['AISC_H3_20PCT']
    except KeyError as falta:
        raise ErrorAsd(f'falta el valor de norma {falta} en la hoja NORMA de la planilla')




class ErrorAsd(Exception):
    pass


def _forma(familia):
    return {'IR': 'I', 'CE': 'canal', 'LI': 'angulo', 'HSS': 'HSS'}.get(familia)


def _elementos_E7(p, forma, fy, E, fn):
    """Elementos esbeltos en compresion de I y canales (B4.1a / E7). Devuelve Ae y detalle."""
    Ag = p['A']
    sq = math.sqrt(E / fy)
    if forma == 'I':
        b_alma = p['H'] - 2.0 * p['t']
        b_pat = p['B'] / 2.0
        elems = [(b_alma, p['e'], NTC_LR_COMPRESION['alma'] * sq, AISC_B41A['alma'][0], AISC_B41A['alma'][1], 'alma'),
                 (b_pat, p['t'], NTC_LR_COMPRESION['patin'] * sq, AISC_B41A['patin'][0], AISC_B41A['patin'][1], 'patin'),
                 (b_pat, p['t'], NTC_LR_COMPRESION['patin'] * sq, AISC_B41A['patin'][0], AISC_B41A['patin'][1], 'patin')]
    else:
        b_alma = p['H'] - 2.0 * p['t']
        elems = [(b_alma, p['e'], NTC_LR_COMPRESION['alma'] * sq, AISC_B41A['alma'][0], AISC_B41A['alma'][1], 'alma'),
                 (p['B'], p['t'], NTC_LR_COMPRESION['patin'] * sq, AISC_B41A['patin'][0], AISC_B41A['patin'][1], 'patin'),
                 (p['B'], p['t'], NTC_LR_COMPRESION['patin'] * sq, AISC_B41A['patin'][0], AISC_B41A['patin'][1], 'patin')]
    Ae = Ag
    det = []
    for b, t, lr, c1, c2, nombre in elems:
        lam = b / t
        if lam <= lr * math.sqrt(fy / fn):                      # E7-2: be = b si lambda <= lambda_r sqrt(Fy/Fn)
            be = b
        else:
            fel = (c2 * lr / lam) ** 2 * fy                     # E7-5
            r = math.sqrt(fel / fn)
            be = min(b * (1.0 - c1 * r) * r, b)                # E7-3 (sin atajo: r < c2 da reduccion)
        Ae -= (b - be) * t
        det.append((nombre, round(lam, 2), round(lr, 2), round(be, 3)))
    return Ae, {'elementos': det}


def pn_compresion(p, familia, fy, E, G_, Kx, Ky, Lx, Ly, Lz, Kz=None):
    """Resistencia nominal a compresion de I y canales (E3, E4, E7). Devuelve (Pn, detalle)."""
    forma = _forma(familia)
    A = p['A']
    fex = math.pi ** 2 * E / (Kx * Lx / p['rx']) ** 2
    fey = math.pi ** 2 * E / (Ky * Ly / p['ry']) ** 2
    fes = {'Fex (E3)': fex, 'Fey (E3)': fey}
    if forma == 'I':
        fe_t = (math.pi ** 2 * E * p['Cw'] / (Kz * Lz) ** 2 + G_ * p['J']) / (p['Ix'] + p['Iy'])     # E4-2
        fes['Fe torsional (E4-2)'] = fe_t
    elif forma == 'canal':
        ro = p['ro']
        xo = math.sqrt(max(ro ** 2 - (p['Ix'] + p['Iy']) / A, 0.0))
        H = 1.0 - xo ** 2 / ro ** 2
        fez = (math.pi ** 2 * E * p['Cw'] / (Kz * Lz) ** 2 + G_ * p['J']) / (A * ro ** 2)              # E4-7
        fe_c = (fex + fez) / (2.0 * H) * (1.0 - math.sqrt(max(1.0 - 4.0 * fex * fez * H / (fex + fez) ** 2, 0.0)))  # E4-3 con Fex
        fes['Fe flexotorsional (E4-3, Fex)'] = fe_c
        fes['Fez (E4-7)'] = fez
    fe_min_nombre = min(fes, key=fes.get)
    fe = fes[fe_min_nombre]
    fn = (0.658 ** (fy / fe)) * fy if fy / fe <= 2.25 else 0.877 * fe            # E3-2 / E3-3
    if forma == 'HSS':
        Ae, det_e7 = _elementos_E7_hss(p, fy, E, fn)
    else:
        Ae, det_e7 = _elementos_E7(p, forma, fy, E, fn)
    return fn * Ae, dict(Fe=fe, gobierna=fe_min_nombre, Fn=fn, Ae=Ae, E7=det_e7,
                         KLr_x=Kx * Lx / p['rx'], KLr_y=Ky * Ly / p['ry'])


def _ltb_I(p, fy, E, G_, Lb, Cb, canal=False):
    """Pandeo lateral torsional (F2.2 y F2-4..F2-8) con Mp = Fy Zx."""
    Sx, Zx = p['Sx'], p['Zx']
    Mp = fy * Zx
    ho = p['H'] - p['t']
    if canal:
        c = (ho / 2.0) * math.sqrt(p['Iy'] / p['Cw'])               # F2-8b
    else:
        c = 1.0                                                     # F2-8a
    rts = math.sqrt(math.sqrt(p['Iy'] * p['Cw']) / Sx)              # F2-7
    jc = p['J'] * c / (Sx * ho)
    lp = 1.76 * p['ry'] * math.sqrt(E / fy)                          # F2-5
    lr = 1.95 * rts * (E / (0.7 * fy)) * math.sqrt(jc + math.sqrt(jc ** 2 + 6.76 * (0.7 * fy / E) ** 2))  # F2-6
    if Lb <= lp:
        mn = Mp
    elif Lb <= lr:
        mn = min(Cb * (Mp - (Mp - 0.7 * fy * Sx) * (Lb - lp) / (lr - lp)), Mp)    # F2-2
    else:
        fcr = Cb * math.pi ** 2 * E / (Lb / rts) ** 2 * math.sqrt(1.0 + 0.078 * jc * (Lb / rts) ** 2)  # F2-4
        mn = min(fcr * Sx, Mp)                                                                       # F2-3
    return mn, dict(Lp=lp, Lr=lr, rts=rts, Mp=Mp)


def mn_fuerte(p, familia, fy, E, G_, Lb, Cb=None):
    """Momento nominal eje mayor de I y canales (F2, F3). Devuelve (Mn, detalle)."""
    if familia == 'HSS':
        return mn_hss(p, fy, E, 'x', Lb, Cb)
    canal = _forma(familia) == 'canal'
    ltb, det = _ltb_I(p, fy, E, G_, Lb, Cb, canal=canal)
    b = p['B'] if canal else p['B'] / 2.0
    lam_f = b / p['t']
    lpf, lrf = AISC_LP_PATIN * math.sqrt(E / fy), AISC_LR_PATIN * math.sqrt(E / fy)
    Mp = det['Mp']
    if lam_f <= lpf:
        lfb = Mp
    elif lam_f <= lrf:
        lfb = Mp - (Mp - 0.7 * fy * p['Sx']) * (lam_f - lpf) / (lrf - lpf)       # F3-1
    else:
        h = p['H'] - 2.0 * p['t']
        kc = min(max(4.0 / math.sqrt(h / p['e']), 0.35), 0.76)
        lfb = 0.9 * E * kc * p['Sx'] / lam_f ** 2                                  # F3-2
    det.update({'lam_f': lam_f, 'lpf': lpf, 'lrf': lrf, 'LTB': ltb, 'LFB': lfb})
    return min(ltb, lfb), det


def mn_debil(p, familia, fy, E):
    """Momento nominal eje menor de I y canales (F6)."""
    if familia == 'HSS':
        return mn_hss(p, fy, E, 'y')
    forma = _forma(familia)
    Mp = min(fy * p['Zy'], 1.6 * fy * p['Sy'])                                    # F6-1
    b = p['B'] / 2.0 if forma == 'I' else p['B']
    lam = b / p['t']
    lpf, lrf = AISC_LP_PATIN * math.sqrt(E / fy), AISC_LR_PATIN * math.sqrt(E / fy)
    My = fy * p['Sy']
    if lam <= lpf:
        mn = Mp
    elif lam <= lrf:
        mn = Mp - (Mp - 0.70 * fy * p['Sy']) * (lam - lpf) / (lrf - lpf)         # F6-2
    else:
        mn = min(0.70 * E / lam ** 2 * p['Sy'], Mp)                              # F6-3/4
    return mn, dict(lam_f=lam, Mp=Mp, My=My)


def _cv(ht, kv, E, fy):
    lim1 = AISC_CV_LIM * math.sqrt(kv * E / fy)
    if ht <= lim1:
        return 1.0
    return lim1 / ht


def vn_fuerte(p, familia, fy, E):
    """Cortante en el alma eje mayor de I y canales (G2)."""
    if familia == 'HSS':
        return vn_hss(p, fy, E, 'x')
    h = p['H'] - 2.0 * p['t']
    ht = h / p['e']
    aw = p['H'] * p['e']                                       # Aw = d tw
    if _forma(familia) == 'I' and ht <= AISC_HTW_LAM * math.sqrt(E / fy):
        cv, omega = 1.0, OMEGA_V_LAMINADA                      # G2-2
    else:
        cv, omega = _cv(ht, KV_I, E, fy), OMEGA_V              # G2-3/4
    return 0.6 * fy * aw * cv, omega, dict(h_tw=ht, Cv1=cv)


def vn_debil(p, familia, fy, E):
    """Cortante en el eje menor (G6): dos patines, Cv2 con kv = 1.2 (nota de usuario de G6)."""
    if familia == 'HSS':
        return vn_hss(p, fy, E, 'y')
    ratio = (p['B'] / (2.0 * p['t'])) if _forma(familia) == 'I' else (p['B'] / p['t'])
    cv2 = _cv(ratio, KV_ANGULO, E, fy)
    vn = 2.0 * 0.6 * fy * p['B'] * p['t'] * cv2
    return vn, OMEGA_V, dict(Cv2=cv2, ratio=ratio)


# ---------------------------------------------------------------- HSS rectangulares y cuadradas (E3, E7, F7, G2)
def _hss_planos(p):
    """Anchos planos (B4.1b): ancho total menos 3t, porque el catalogo no trae radio de esquina.
    Devuelve (bp, hp): bp = ancho de las paredes horizontales, hp = peralte plano de las paredes verticales."""
    bp = p['B'] - 3.0 * p['t']
    hp = p['H'] - 3.0 * p['t']
    if bp <= 0 or hp <= 0:
        raise ErrorAsd('HSS con ancho plano no positivo: revisar dimensiones y espesor en la base')
    return bp, hp


def _hss_muros(p, eje):
    """(patin, alma) planos para flexion o cortante alrededor del eje 'x' (fuerte) o 'y' (debil)."""
    bp, hp = _hss_planos(p)
    return (bp, hp) if eje == 'x' else (hp, bp)


def _elementos_E7_hss(p, fy, E, fn):
    """Ae de HSS (E7.1, caso (b) de la tabla E7.1): cuatro paredes de grueso t; cada pared esbelta se reduce."""
    bp, hp = _hss_planos(p)
    t = p['t']
    lr = AISC_HSS_LR_COMP * math.sqrt(E / fy)
    Ae = p['A']
    det = []
    for nombre, b in [('pared horizontal', bp), ('pared vertical', hp)]:
        lam = b / t
        if lam <= lr * math.sqrt(fy / fn):                             # E7-2: be = b si lambda <= lambda_r sqrt(Fy/Fn)
            be = b
        else:
            fel = (AISC_HSS_C2 * lr / lam) ** 2 * fy                  # E7-5
            r = math.sqrt(fel / fn)
            be = min(b * (1.0 - AISC_HSS_C1 * r) * r, b)              # E7-3 (sin atajo: r < c2 da reduccion)
        Ae -= 2.0 * (b - be) * t                                       # cada pared aparece dos veces
        det.append((nombre, round(lam, 2), round(lr, 2), round(be, 3)))
    return Ae, {'elementos': det}


def _Se_hss(p, eje, be):
    """Modulo de seccion efectivo Se (F7-3): solo el patin comprimido se reduce a be. Modelo de lineas medias
    (sin radios de esquina): patines de ancho W - t, almas de altura D - t, grueso t. Se = Ief / c, con c la
    mayor distancia del eje neutro efectivo a una fibra extrema (conservador)."""
    t = p['t']
    W, D = (p['B'], p['H']) if eje == 'x' else (p['H'], p['B'])
    bf = W - 3.0 * t
    wc = (W - t) - (bf - be)                       # linea media del patin superior reducido
    dc = D - t
    A_top, y_top = wc * t, dc / 2.0
    A_bot, y_bot = (W - t) * t, -dc / 2.0
    A_web = 2.0 * dc * t
    yc = (A_top * y_top + A_bot * y_bot) / (A_top + A_bot + A_web)
    I = ((wc * t ** 3 / 12.0 + A_top * (y_top - yc) ** 2)
         + ((W - t) * t ** 3 / 12.0 + A_bot * (y_bot - yc) ** 2)
         + 2.0 * (t * dc ** 3 / 12.0 + dc * t * yc ** 2))
    c = max(D / 2.0 - yc, D / 2.0 + yc)
    return I / c


def mn_hss(p, fy, E, eje, Lb=None, Cb=None):
    """Momento nominal de HSS rectangular o cuadrada (F7). eje 'x' (fuerte): incluye pandeo lateral-torsional
    (F7.4); eje 'y' (debil): sin pandeo lateral-torsional (nota de usuario de F7). Devuelve (Mn, detalle)."""
    bf, hw = _hss_muros(p, eje)
    t = p['t']
    sq = math.sqrt(E / fy)
    S = p['Sx'] if eje == 'x' else p['Sy']
    Z = p['Zx'] if eje == 'x' else p['Zy']
    Mp = fy * Z                                                       # F7-1
    My = fy * S
    lam_f, lam_w = bf / t, hw / t
    lpf, lrf = AISC_HSS_LP_PATIN * sq, AISC_HSS_LR_PATIN * sq
    lpw, lrw = AISC_HSS_LP_ALMA * sq, AISC_HSS_LR_ALMA * sq
    det = dict(lam_f=lam_f, lpf=lpf, lrf=lrf, lam_w=lam_w, lpw=lpw, lrw=lrw, Mp=Mp, My=My, S=S, Z=Z, eje=eje)
    if lam_f <= lpf:
        lfb = Mp
    elif lam_f <= lrf:
        lfb = Mp - (Mp - My) * (lam_f - lpf) / (lrf - lpf)            # F7-2
    else:
        be = min(1.92 * t * sq * (1.0 - 0.38 / lam_f * sq), bf)        # F7-4
        Se = _Se_hss(p, eje, be)
        lfb = fy * Se                                                  # F7-3
        det.update(be=be, Se=Se)
    if lam_w <= lpw:
        lwb = Mp
    elif lam_w <= lrw:
        lwb = Mp - (Mp - My) * (lam_w - lpw) / (lrw - lpw)            # F7-6
    else:
        raise ErrorAsd('alma esbelta de HSS: F7.3(c) no se aplica (AISC: no hay HSS con almas esbeltas)')
    mn = min(lfb, lwb, Mp)
    if eje == 'x':
        if Lb is None or Cb is None:
            raise ErrorAsd('falta Lb o Cb para el pandeo lateral-torsional de HSS (LONGITUDES)')
        Ag, J, ry = p['A'], p['J'], p['ry']
        lp = 0.13 * E * ry * math.sqrt(J * Ag) / Mp                   # F7-10
        lr = 2.0 * E * ry * math.sqrt(J * Ag) / (0.7 * fy * S)        # F7-11
        if Lb <= lp:
            ltb = Mp
        elif Lb <= lr:
            ltb = Cb * (Mp - (Mp - 0.7 * fy * S) * (Lb - lp) / (lr - lp))   # F7-8
        else:
            ltb = 2.0 * E * Cb * math.sqrt(J * Ag) / (Lb / ry)            # F7-9
        ltb = min(ltb, Mp)
        mn = min(mn, ltb)
        det.update(LTB=ltb, Lp=lp, Lr=lr)
    det.update(LFB=lfb, LWB=lwb, Mn=mn)
    return mn, det


def torsion_hss(p, fy, E):
    """Resistencia a torsion de HSS rectangular (AISC 360-22 H3.1): Tn = Fcr C (H3-1), con
    C = 2(B-t)(H-t)t - 4.5(4-pi)t^3 (nota de usuario de H3.1) y Fcr segun h/t (H3-3, H3-4, H3-5), donde
    h = lado plano mas largo (B4.1b(d)). Devuelve (Tn en kgf·cm, detalle)."""
    bp, hp = _hss_planos(p)
    t = p['t']
    h = max(bp, hp)
    ht = h / t
    sq = math.sqrt(E / fy)
    if ht <= AISC_H3_LIM1 * sq:
        fcr, reg = AISC_H3_FCR_Y * fy, 'H3-3'
    elif ht <= AISC_H3_LIM2 * sq:
        fcr, reg = AISC_H3_FCR_Y * fy * (AISC_H3_LIM1 * sq) / ht, 'H3-4'
    elif ht <= AISC_H3_LIM3:
        fcr, reg = AISC_H3_K * math.pi ** 2 * E / ht ** 2, 'H3-5'
    else:
        raise ErrorAsd('HSS con h/t mayor que 260: fuera de H3.1 (torsion)')
    C = 2.0 * (p['B'] - t) * (p['H'] - t) * t - 4.5 * (4.0 - math.pi) * t ** 3
    tn = fcr * C
    return tn, dict(h=h, ht=ht, reg=reg, Fcr=fcr, C=C, Tn=tn)


def _cv2_hss(ht, E, fy):
    """Cv2 de G2.2 con kv = 5 (G4): G2-9 (h/t <= 1.10 sqrt(kv E/Fy)), G2-10 (hasta 1.37) y G2-11."""
    lim1 = AISC_CV_LIM * math.sqrt(AISC_KV_HSS * E / fy)
    lim2 = AISC_CV2_LIM2 * math.sqrt(AISC_KV_HSS * E / fy)
    if ht <= lim1:
        return 1.0                                                 # G2-9
    if ht <= lim2:
        return lim1 / ht                                           # G2-10
    return AISC_CV2_K * AISC_KV_HSS * E / (ht ** 2 * fy)           # G2-11


def vn_hss(p, fy, E, eje):
    """Cortante de HSS rectangular (G4-1): Vn = 0.6 Fy Aw Cv2, Aw = 2 h t, h = ancho plano (eje 'x': peralte plano
    H - 3t; eje 'y': B - 3t). Omega_v = 1.67 (OMEGA_V)."""
    _, h = _hss_muros(p, eje)
    t = p['t']
    ht = h / t
    cv2 = _cv2_hss(ht, E, fy)
    aw = 2.0 * h * t
    return 0.6 * fy * aw * cv2, OMEGA_V, dict(h_tw=ht, Cv1=cv2, Cv2=cv2, Aw=aw)


# ---------------------------------------------------------------- angulos de lados iguales (LI)
def propiedades_angulo(p):
    """Propiedades de un angulo de lados iguales: b = pata (H = B), t = espesor.
    Centroide, producto de inercia y ejes principales por geometria rectangular (sin acuerdos).
    Ix de catalogo (incluye acuerdos). Devuelve dict con ejes principales u (mayor) y v (menor)."""
    b, t, A = p['H'], p['t'], p['A']
    Ag = (2.0 * b - t) * t
    xc = (b * b * t / 2.0 + t * t * (b - t) / 2.0) / Ag                 # centroide desde el talon (x = y)
    Ixy = t * t * (2.0 * b * b - t * t) / 4.0 - Ag * xc * xc           # producto de inercia centroidal (< 0)
    I0 = p['Ix']                                                       # Ix = Iy
    Iu = I0 + abs(Ixy)                                                 # eje mayor principal (bisectriz del talon)
    Iv = I0 - abs(Ixy)                                                 # eje menor principal (rz)
    if Iv <= 0:
        raise ErrorAsd('angulo con inercia principal menor no positiva: revisar la base')
    c_u = b / math.sqrt(2.0)                                           # dedo a eje u (ambos lados)
    c_talon = math.sqrt(2.0) * xc                                      # talon a eje v
    c_dedo = (b + t - 2.0 * xc) / math.sqrt(2.0)                       # dedo a eje v
    return dict(b=b, t=t, rz=math.sqrt(Iv / A), ru=math.sqrt(Iu / A), Iu=Iu, Iv=Iv, Ixy=Ixy, xc=xc,
                Su=Iu / c_u, Sc_u=Iu / c_u, Sv=Iv / max(c_talon, c_dedo), Sc_v=Iv / c_dedo)


def _fn_E3(fy, E, lam):
    fe = math.pi ** 2 * E / lam ** 2
    fn = (0.658 ** (fy / fe)) * fy if fy / fe <= 2.25 else 0.877 * fe
    return fn, fe


def _slend_angulo(p, Lx, Ly, Kx, Ky, e5):
    """Esbeltez efectiva: z (menor, rz) por E5 o KL/rz, y u (mayor principal) por KL/ru."""
    Lmax = max(Lx, Ly)
    KL = max(Kx * Lx, Ky * Ly)
    if e5:
        rat = Lmax / p['rx']                                           # L/ra, ra = radio eje paralelo a la pata
        lam_z = (72.0 + 0.75 * rat) if rat <= AISC_E5_RAT_LIM else (32.0 + 1.25 * rat)     # E5-1 / E5-2
    else:
        lam_z = KL / p['rz']
    lam_u = KL / p['ru']
    return lam_z, lam_u


def _resistencias_angulo(p, fy, E, Lx, Ly, Lb, Kx, Ky, Cb, e5):
    sq = math.sqrt(E / fy)
    b, t, A = p['b'], p['t'], p['A']
    lam_z, lam_u = _slend_angulo(p, Lx, Ly, Kx, Ky, e5)
    fn_z, _ = _fn_E3(fy, E, lam_z)
    fn_u, _ = _fn_E3(fy, E, lam_u)
    fn = min(fn_z, fn_u)
    gob = 'Fe eje z (menor, E5)' if (fn_z <= fn_u and e5) else ('Fe eje z (menor, K L/rz)' if fn_z <= fn_u else 'Fe eje u (mayor)')
    # E7 patas esbeltas en compresion (B4.1a: 0.45 sqrt(E/Fy); c1 = 0.22, c2 = 1.49), ambas patas
    lam_leg = b / t
    lr_leg = AISC_B41A['lr_pata'] * sq
    if lam_leg <= lr_leg:
        be = b
    else:
        fel = (AISC_B41A['patin'][1] * lr_leg / lam_leg) ** 2 * fy                       # E7-5
        if fel >= fn:
            be = b
        else:
            r = math.sqrt(fel / fn)
            be = min(b * (1.0 - 0.22 * r) * r, b)                       # E7-3
    Ae = A - 2.0 * (b - be) * t
    pn_c = fn * Ae
    motivo = None
    if lam_leg > AISC_LIM_ANG_E4 * sq:
        # DECISION D14 (abierta): E4 flexotorsional de angulos no implementado; la barra queda NO REVISADO. Alternativa: implementarlo.
        motivo = 'b/t > 0.71 sqrt(E/Fy): requiere pandeo flexotorsional E4 (no implementado)'
    # F10 flexion sobre ejes principales
    Cb_u = min(Cb, CB_MAX_ANGULO)
    My_u, My_v = fy * p['Su'], fy * p['Sv']
    lp, lr = 0.54 * sq, 0.91 * sq

    def llb(Sc):
        if lam_leg <= lp:
            return None                                                  # compacta: no aplica
        if lam_leg <= lr:
            return fy * Sc * (2.43 - 1.72 * lam_leg / sq)                # F10-6 ((b/t) sqrt(Fy/E) = lam/sq)
        return AISC_LIM_ANG_E4 * E / lam_leg ** 2 * Sc                              # F10-7 / F10-8
    Mcr = 9.0 * E * A * p['rz'] * t * Cb_u / (8.0 * Lb)                  # F10-4 con beta_w = 0
    ratio = My_u / Mcr
    if ratio <= 1.0:
        mn_ltb = min((1.92 - 1.17 * ratio) * My_u, 1.5 * My_u)           # F10-2
    else:
        mn_ltb = (0.92 - 0.17 / ratio) * Mcr                             # F10-3
    lista_u = [1.5 * My_u, mn_ltb]
    lu = llb(p['Sc_u'])
    if lu is not None:
        lista_u.append(lu)
    lista_v = [1.5 * My_v]
    lv = llb(p['Sc_v'])
    if lv is not None:
        lista_v.append(lv)
    Mn_u, Mn_v = min(lista_u), min(lista_v)
    # G3 cortante de una pata
    cv2 = _cv(lam_leg, KV_ANGULO, E, fy)
    Va = 0.6 * fy * b * t * cv2 / OMEGA_V
    return dict(Pa_c=None if motivo else pn_c / OMEGA_C, Ma_x=Mn_u / OMEGA_B, Ma_y=Mn_v / OMEGA_B,
                Va_x=Va, Va_y=Va, klr=max(lam_z, lam_u), motivo=motivo,
                KLr_fuerte=lam_u, KLr_debil=lam_z,
                detalle_c=dict(gobierna=gob, Fn=fn, Ae=Ae, lam_z=lam_z, lam_u=lam_u, be=be),
                detalle_x=dict(Mn=Mn_u, Mcr=Mcr, LTB=mn_ltb, My=My_u, lam_leg=lam_leg, lp=lp, lr=lr),
                detalle_y=dict(Mn=Mn_v, My=My_v),
                detalle_vx=dict(Cv2=cv2), detalle_vy=dict(Cv2=cv2))


def _utilizacion_angulo(res, Pr_t, Pr_c, Mr_x, Mr_y, Vr_x, Vr_y):
    """LI: H2-1 con el momento resultante sobre cada eje principal (cubre cualquier orientacion)."""
    u = {}
    mres = math.hypot(Mr_x, Mr_y)
    u['T traccion'] = Pr_t / res['Pa_t'] if Pr_t > 0 else 0.0
    u['C compresion'] = Pr_c / res['Pa_c'] if Pr_c > 0 else 0.0
    u['F flexion eje mayor'] = mres / res['Ma_x'] if mres > 0 else 0.0      # eje principal u
    u['F flexion eje menor'] = mres / res['Ma_y'] if mres > 0 else 0.0      # eje principal v
    u['V cortante eje mayor'] = Vr_x / res['Va_x'] if Vr_x > 0 else 0.0
    u['V cortante eje menor'] = Vr_y / res['Va_y'] if Vr_y > 0 else 0.0
    if Pr_c > 0:
        pr = Pr_c / res['Pa_c']
    elif Pr_t > 0:
        pr = Pr_t / res['Pa_t']
    else:
        pr = 0.0
    if pr > 0 or mres > 0:
        u['H2 angulo (H2-1)'] = pr + mres / res['Ma_x'] + mres / res['Ma_y']
    return u


def resistencias_asd(p, familia, fy, fu, E, nu, Lx, Ly, Lb, Kx, Ky, Lz, Cb=None, e5=False, U=1.0):
    """Capacidades ASD (admisibles) de la barra: Pa tension, Pa compresion, Ma eje mayor y menor, Va."""
    G_ = E / (2.0 * (1.0 + nu))
    out = {'angulo': familia == 'LI', 'motivo': None}
    pa_flu = fy * p['A'] / OMEGA_T_FLU
    out['Pa_t_fluencia'] = pa_flu
    out['U'] = U                                                                     # D3: Ae = U Ag (sin agujeros)
    out['Pa_t_fractura'] = None if fu is None else fu * p['A'] * U / OMEGA_T_FRA
    out['Pa_t'] = pa_flu if fu is None else min(pa_flu, fu * p['A'] * U / OMEGA_T_FRA)
    if familia == 'LI':
        out.update(_resistencias_angulo(p, fy, E, Lx, Ly, Lb, Kx, Ky, Cb, e5))
        return out
    # DECISION D4 (abierta): Kz elegido en PARAMETROS (1.0 o 0.5).
    pn_c, det_c = pn_compresion(p, familia, fy, E, G_, Kx, Ky, Lx, Ly, Lz, Kz=KZ_TORSION)
    out['Pa_c'] = pn_c / OMEGA_C
    out['detalle_c'] = det_c
    mn_x, det_x = mn_fuerte(p, familia, fy, E, G_, Lb, Cb)
    out['Ma_x'] = mn_x / OMEGA_B
    out['detalle_x'] = det_x
    mn_y, det_y = mn_debil(p, familia, fy, E)
    out['Ma_y'] = mn_y / OMEGA_B
    out['detalle_y'] = det_y
    vn_x, om_x, det_vx = vn_fuerte(p, familia, fy, E)
    out['Va_x'] = vn_x / om_x
    out['detalle_vx'] = det_vx
    vn_y, om_y, det_vy = vn_debil(p, familia, fy, E)
    out['Va_y'] = vn_y / om_y
    out['detalle_vy'] = det_vy
    if familia == 'HSS':
        tn, det_t = torsion_hss(p, fy, E)                              # H3.1: Tc = Tn / ΩT
        out['Tc'] = tn / AISC_OMEGA_T_HSS
        out['detalle_t'] = det_t
    out['KLr_fuerte'] = Kx * Lx / p['rx']
    out['KLr_debil'] = Ky * Ly / p['ry']
    out['klr'] = max(out['KLr_fuerte'], out['KLr_debil'])
    return out


def utilizacion_asd(res, Pr_t, Pr_c, Mr_x, Mr_y, Vr_x, Vr_y, Tr=0.0):
    """Utilizaciones ASD (<= 1 cumple): tension, compresion, flexion eje mayor y menor, cortante, H1 o H2.
    HSS (H3): utilizacion de torsion Tr/Tc; si Tr > 0.2 Tc, la interaccion es H3-6 en lugar de H1 (H3.2)."""
    if res.get('angulo'):
        return _utilizacion_angulo(res, Pr_t, Pr_c, Mr_x, Mr_y, Vr_x, Vr_y)
    u = {}
    u['T traccion'] = Pr_t / res['Pa_t'] if Pr_t > 0 else 0.0
    u['C compresion'] = Pr_c / res['Pa_c'] if Pr_c > 0 else 0.0
    u['F flexion eje mayor'] = Mr_x / res['Ma_x'] if Mr_x > 0 else 0.0
    u['F flexion eje menor'] = Mr_y / res['Ma_y'] if Mr_y > 0 else 0.0
    u['V cortante eje mayor'] = Vr_x / res['Va_x'] if Vr_x > 0 else 0.0
    u['V cortante eje menor'] = Vr_y / res['Va_y'] if Vr_y > 0 else 0.0
    mx = Mr_x / res['Ma_x']
    my = Mr_y / res['Ma_y']
    if res.get('Tc') is not None:                                             # HSS: H3
        u['T torsion (H3.1)'] = Tr / res['Tc'] if Tr > 0 else 0.0
        if Tr > AISC_H3_20PCT * res['Tc']:                                    # H3.2: H3-6 reemplaza a H1
            if Pr_c > 0:
                ratio_p = Pr_c / res['Pa_c']
            elif Pr_t > 0:
                ratio_p = Pr_t / res['Pa_t']
            else:
                ratio_p = 0.0
            cand_v = [v for v in (Vr_x / res['Va_x'] if Vr_x > 0 else None,
                                  Vr_y / res['Va_y'] if Vr_y > 0 else None) if v is not None]
            vr = max(cand_v) if cand_v else 0.0                               # Vr/Vc: el mayor de x o y
            u['H3 interaccion (H3-6)'] = ratio_p + mx + my + (vr + Tr / res['Tc']) ** 2
            return u
    if Pr_c > 0:                                                              # H1-1a/b, compresion
        pc = res['Pa_c']
        ratio = Pr_c / pc
        u['H1 flexocompresion'] = (ratio + 8.0 / 9.0 * (mx + my)) if ratio >= 0.2 else (ratio / 2.0 + (mx + my))
    elif Pr_t > 0 and (mx > 0 or my > 0):                                     # H1-1a/b con Pc = Pt admisible
        pc = res['Pa_t']
        ratio = Pr_t / pc
        u['H1 flexotension'] = (ratio + 8.0 / 9.0 * (mx + my)) if ratio >= 0.2 else (ratio / 2.0 + (mx + my))
    return u


# NTC-Acero CDMX (Gaceta Oficial 7-jul-2020). El diseno de resistencia es AISC 360-22 (ASD); las esbelteces local
# y global se verifican con la NTC (requisito del proyecto en Mexico).
# (los limites NTC-Acero se cargan con cargar_norma(): tablas 3.2.1, 3.2.2 y 2.3.2)


def esbeltez_local(p, familia, fy, E, compresion):
    """Esbeltez local por elemento segun NTC: lambda / lambda_r (<= 1 cumple). Compresion: Tabla 3.2.1;
    flexion: Tabla 3.2.2 (lambda_r = limite de seccion tipo 3). Devuelve (FU, {elemento: (lambda, lambda_r)})."""
    sq = math.sqrt(E / fy)
    if familia == 'HSS':
        bf, hw = _hss_planos(p)
        t = p['t']
        if compresion:
            lims = {'patin': (bf / t, NTC_HSS_LR_COMP * sq), 'alma': (hw / t, NTC_HSS_LR_COMP * sq)}
        else:
            lr_p, lr_a = NTC_HSS_FLEX['patin'][2] * sq, NTC_HSS_FLEX['alma'][2] * sq
            lims = {'patin': (bf / t, lr_p), 'alma': (hw / t, lr_a),
                    'patin_y': (hw / t, lr_p), 'alma_y': (bf / t, lr_a)}
        return {n: lam / lr for n, (lam, lr) in lims.items()}, lims
    if familia == 'LI':
        lams = {'pata': p['H'] / p['t']}
    else:
        b_pat = p['B'] / 2.0 if familia == 'IR' else p['B']
        lams = {'patin': b_pat / p['t'], 'alma': (p['H'] - 2.0 * p['t']) / p['e']}
    elementos = {}
    for n, lam in lams.items():
        lr_norm = NTC_LR_COMPRESION[n] if compresion else NTC_LIMITES_FLEXION[n][2]
        elementos[n] = (lam, lr_norm * sq)
    fu = {n: lam / lr for n, (lam, lr) in elementos.items()}
    return fu, elementos


def tipo_seccion_flexion(p, familia, fy, E):
    """Tipo de seccion (NTC 3.2.1.2, Tabla 3.2.2): 1, 2, 3 o 4 segun el elemento mas esbelto."""
    sq = math.sqrt(E / fy)
    _, det = esbeltez_local(p, familia, fy, E, compresion=False)
    tipo = 1
    for n, (lam, _) in det.items():
        base = n.split('_')[0]
        lpl, lp, lr = (NTC_HSS_FLEX if familia == 'HSS' else NTC_LIMITES_FLEXION)[base]
        if lpl is not None and lam <= lpl * sq:
            t = 1
        elif lam <= lp * sq:
            t = 2
        elif lam <= lr * sq:
            t = 3
        else:
            t = 4
        tipo = max(tipo, t)
    return tipo


def clasificacion_compresion(p, familia, fy, E):
    """NTC 3.2.1.1 (Tabla 3.2.1): seccion no-esbelta si todos sus elementos cumplen lambda <= lambda_r."""
    fu, _ = esbeltez_local(p, familia, fy, E, compresion=True)
    return 'no-esbelta' if max(fu.values()) <= 1.0 else 'esbelta'


def _num(v, defecto=None):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return defecto
    try:
        return float(v)
    except (TypeError, ValueError):
        return defecto


E5_COLS = ['E5 UNA PATA', 'E5 UNION SOLDADA O 2 PERNOS', 'E5 SIN CARGA INTERMEDIA', 'E5 INDIVIDUAL O ALMA DE ARMADURA']


def _texto(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ''
    return str(v).strip().upper()


def e5_cumple(r):
    """E5(a): el angulo se trata con E5 solo si LONGITUDES marca SI en las cuatro condiciones de E5 (1 a 4).
    Sin marcar (vacio o NO) se usa K L/rz y H2, como exige AISC E5 cuando no se cumplen."""
    return r is not None and all(_texto(r.get(c)) == 'SI' for c in E5_COLS)


def factor_u_angulo(xc, l_cm, conexion):
    """Factor de rezago U para tension (AISC D3, Tabla D3.1). Ambas patas conectadas: U = 1.0 (caso 1).
    Una pata: U = 1 - xc/l (caso 2; en el caso 8 la tabla permite usar el valor mayor, que es este)."""
    if 'DOS' in (conexion or '').upper():
        return 1.0
    return max(0.0, min(1.0, 1.0 - xc / l_cm))


def _props_asd(nombre, familia):
    """Propiedades para ASD desde la base Gerdau. No se estima ninguna propiedad: lo que falta queda NO REVISADO."""
    p = G.obtener_perfil(nombre)
    if p is None:
        return None, 'perfil no existe en la base Gerdau'
    estimados = []
    if familia == 'LI':
        req = ['H', 't', 'A', 'Ix', 'rx']
    else:
        req = ['H', 'B', 'e', 't', 'A', 'Ix', 'Iy', 'Sx', 'Sy', 'rx', 'ry', 'J', 'Zx', 'Zy']
        if familia in ('IR', 'CE'):
            req.append('Cw')
        if familia == 'CE':
            req.append('ro')
    # DECISION D16 (abierta): sin estimaciones; si la base Gerdau no trae una propiedad, la barra queda NO REVISADO (alternativa: error).
    # Sin estimaciones: si la base Gerdau no trae una propiedad, la barra queda NO REVISADO (se reporta).
    falta = [k for k in req if p.get(k) is None]
    if falta:
        return None, 'NO REVISADO: faltan datos ' + ', '.join(falta)
    if familia == 'LI':
        p.update(propiedades_angulo(p))
    p['_estimados'] = estimados
    return p, ''


def _demanda_fila(fila):
    """Demandas ASD de una fila de BARRAS (kgf, kgf·cm)."""
    N_I, N_J = _num(fila['N I (kgf)'], 0.0), _num(fila['N J (kgf)'], 0.0)
    return dict(Pr_c=max(N_I, -N_J, 0.0), Pr_t=max(-N_I, N_J, 0.0),
                # H-01: el momento de diseno incluye el maximo en el tramo (no solo los extremos)
                Mr_x=max(abs(_num(fila['Mz I (kgf·m)'], 0.0)), abs(_num(fila['Mz J (kgf·m)'], 0.0)),
                         abs(fila['MZ MAX EN EL TRAMO (kgf·m)'])) * 100.0,
                Mr_y=max(abs(_num(fila['My I (kgf·m)'], 0.0)), abs(_num(fila['My J (kgf·m)'], 0.0)),
                         abs(fila['MY MAX EN EL TRAMO (kgf·m)'])) * 100.0,
                Vr_x=max(abs(_num(fila['Vy I (kgf)'], 0.0)), abs(_num(fila['Vy J (kgf)'], 0.0))),
                Tr=max(abs(_num(fila['T I (kgf·m)'], 0.0)), abs(_num(fila['T J (kgf·m)'], 0.0))) * 100.0,
                Vr_y=max(abs(_num(fila['Vz I (kgf)'], 0.0)), abs(_num(fila['Vz J (kgf)'], 0.0))))


# ---------------------------------------------------------------- planilla -> diseno ASD (IR, CE, LI)
GRUPOS_ELEMENTO = ['COLUMNA', 'VIGA', 'ARRIOSTRE', 'DIAGONAL']
COMPRESION_GRUPOS = ('COLUMNA', 'ARRIOSTRE', 'DIAGONAL')
FAMILIAS_ASD = ('IR', 'CE', 'LI', 'HSS')
FAMILIA_ORDEN = ['IR', 'CE', 'LI', 'HSS']
CHEQUEOS_DEMANDA_BASE = ['T traccion', 'C compresion', 'F flexion eje mayor', 'F flexion eje menor',
                         'V cortante eje mayor', 'V cortante eje menor', 'H1 flexocompresion', 'H1 flexotension',
                         'H2 angulo (H2-1)']
CHEQUEOS_TORSION = ['T torsion (H3.1)', 'H3 interaccion (H3-6)']      # solo HSS (H3)
CHEQUEOS_DEMANDA = CHEQUEOS_DEMANDA_BASE + CHEQUEOS_TORSION
INTERACCION = ('H1 flexocompresion', 'H1 flexotension', 'H2 angulo (H2-1)', 'H3 interaccion (H3-6)')
CHEQUEO_EL = 'EL esbeltez local (B4.1)'
CHEQUEO_EG = 'EG KL/r <= 200 (2.3.2)'
CHEQUEO_D60 = 'D columna ductil KL/r <= 60 (12.1.3.1)'
CHEQUEO_D12 = 'D tabla 12.1.2 ancho/grueso'
CHEQUEOS_ESBELTEZ = [CHEQUEO_EL, CHEQUEO_EG, CHEQUEO_D60, CHEQUEO_D12]
CHEQUEOS_ASD = CHEQUEOS_DEMANDA_BASE + CHEQUEOS_ESBELTEZ
COLS_RESULTADOS = ['ID', 'PERFIL', 'MATERIAL', 'GRUPO', 'CASO QUE GOBIERNA', 'FU TRACCION', 'FU COMPRESION',
                   'FU FLEXION EJE MAYOR', 'FU FLEXION EJE MENOR', 'FU CORTANTE EJE MAYOR', 'FU CORTANTE EJE MENOR',
                   'FU INTERACCION (H1/H2)', 'FU ESBELTEZ LOCAL (B4.1)', 'FU ESBELTEZ GLOBAL (KL/r<=200)',
                   'FU DUCTILIDAD (12.1.2, 12.1.3.1)', 'FU MAXIMO', 'KL/r EJE FUERTE', 'KL/r EJE DEBIL',
                   'ESTADO', 'NOTAS', 'FU TORSION (H3.1)', 'VERIFICACION QUE GOBIERNA', 'FU ESFUERZOS',
                   'ESFUERZO QUE GOBIERNA', 'FU ESBELTEZ', 'ESBELTEZ QUE GOBIERNA']


def _valores_designacion(nombre):
    """Numeros de la designacion comercial: 'IR 6 × 12' -> [6, 12]; 'LI 3 1/2 × 3/8' -> [3.5, 0.375]."""
    partes = [x.strip() for x in seccion_de(nombre).replace('x', '×').split('×')]
    vals = []
    for parte in partes:
        m = re.fullmatch(r'(\d+)\s+(\d+)/(\d+)', parte)
        if m:
            vals.append(int(m[1]) + int(m[2]) / int(m[3]))
            continue
        m = re.fullmatch(r'(\d+)/(\d+)', parte)
        if m:
            vals.append(int(m[1]) / int(m[2]))
            continue
        try:
            vals.append(float(parte))
        except ValueError:
            vals.append(math.inf)
    return vals + [math.inf] * (2 - len(vals))


def clave_orden(nombre):
    """Orden de las listas: tipo (IR, CE, LI), luego la designacion comercial (peralte o ala, luego peso o espesor)
    y la masa como desempate."""
    p = G.PERFILES[nombre]
    a, b = _valores_designacion(nombre)[:2]
    masa = p['masa_kg_m'] if p['masa_kg_m'] is not None else math.inf
    return (FAMILIA_ORDEN.index(p['familia']), a, b, masa, nombre)


def seccion_de(nombre):
    """Seccion sin el tipo: 'IR 10 × 30' -> '10 × 30'."""
    import re
    return re.sub(r'^(IR|CE|LI|HSS)\s*ᴬ?\s*', '', nombre).strip()


def _ductilidad(p, familia, fy, E, grupo, nivel):
    """Tabla 12.1.2 (NTC): I y canales (DN); angulos sencillos: alas (no atiesadas), lambda = b/t <= 0.30 (alta) o
    0.38 (media) sqrt(E/Fy)."""
    import diseno_ntc as DN
    if familia == 'HSS':
        if nivel not in ('ALTA', 'MEDIA'):
            return None
        bf, hw = _hss_planos(p)
        lam = max(bf, hw) / p['t']                       # pared mas esbelta (patin o alma)
        r = math.sqrt(E / fy)
        lda, ldm = NTC_HSS_DUC['ALTA'] * r, NTC_HSS_DUC['MEDIA'] * r
        lim = lda if nivel == 'ALTA' else ldm
        return {'U': lam / lim, 'lambda': lam, 'limite': lim, 'lf': lam, 'lda_f': lda, 'ldm_f': ldm}
    if familia == 'LI':
        if nivel not in ('ALTA', 'MEDIA'):
            return None
        lim = (0.30 if nivel == 'ALTA' else 0.38) * math.sqrt(E / fy)
        return {'U': (p['H'] / p['t']) / lim, 'lambda': p['H'] / p['t'], 'limite': lim}
    forma = {'IR': 'I', 'CE': 'canal'}.get(familia)
    return DN.ductilidad_12_1_2(p, fy, E, forma, grupo, nivel) if forma else None


def _chequear(p, familia, fy, fu, E, nu, Lx, Ly, Lb, kx, ky, grupo, nivel, demandas_list, cb=None,
              e5=False, u_t=1.0):
    """Utilizaciones ASD por demanda (una fila por combinacion) y verificaciones de esbeltez de la barra.
    Devuelve dict: res, klr, dd, el (FU local por elemento), el_det, const (FU que no dependen de la demanda),
    filas (una por demanda) y motivo (si no se puede verificar)."""
    res = resistencias_asd(p, familia, fy, fu, E, nu, Lx, Ly, Lb, kx, ky, max(Lx, Ly), Cb=cb, e5=e5, U=u_t)
    klr = res['klr']
    compr = grupo in COMPRESION_GRUPOS
    fu_el, el_det = esbeltez_local(p, familia, fy, E, compresion=compr)
    const = {CHEQUEO_EL: max(fu_el.values())}
    if compr:
        const[CHEQUEO_EG] = klr / LIMITE_KLR
    dd = _ductilidad(p, familia, fy, E, grupo, nivel)
    if grupo == 'COLUMNA' and nivel in ('ALTA', 'MEDIA'):
        const[CHEQUEO_D60] = klr / 60.0
    if dd is not None and dd.get('U') is not None:
        const[CHEQUEO_D12] = dd['U']
    out = dict(res=res, klr=klr, dd=dd, el=fu_el, el_det=el_det, const=const, filas=[], motivo=res.get('motivo'))
    if res.get('motivo') and any(d['Pr_c'] > 0 for d in demandas_list):
        return out
    for d in demandas_list:
        u = utilizacion_asd(res, d['Pr_t'], d['Pr_c'], d['Mr_x'], d['Mr_y'], d['Vr_x'], d['Vr_y'], d.get('Tr', 0.0))
        u.update(const)
        out['filas'].append(u)
    return out


_NOMBRE_ESFUERZO = {
    'T traccion': 'tracción', 'C compresion': 'compresión', 'F flexion eje mayor': 'flexión eje mayor',
    'F flexion eje menor': 'flexión eje menor', 'V cortante eje mayor': 'cortante eje mayor',
    'V cortante eje menor': 'cortante eje menor', 'H1 flexocompresion': 'interacción H1 flexocompresión',
    'H1 flexotension': 'interacción H1 flexotensión', 'H2 angulo (H2-1)': 'interacción H2 (ángulo)',
    'T torsion (H3.1)': 'torsión H3.1', 'H3 interaccion (H3-6)': 'interacción H3-6',
}


def _valores_esbeltez(ev, c):
    """(lambda, limite, texto) de la verificacion de esbeltez c: esbeltez local, KL/r o ductilidad (12.1.2)."""
    if c == CHEQUEO_EL:
        nombre, (lam, lr) = max(ev['el_det'].items(), key=lambda t: t[1][0] / t[1][1])
        return lam, lr, f'λ {nombre} {lam:.1f}/{lr:.1f}'
    if c in (CHEQUEO_EG, CHEQUEO_D60):
        lim = 60.0 if CHEQUEO_D60 in ev['const'] else LIMITE_KLR
        return ev['klr'], lim, f'KL/r {ev["klr"]:.1f}/{lim:.1f}'
    dd = ev['dd']
    lam = dd['lf'] if 'lf' in dd else dd['lambda']
    lim = lam / dd['U']
    return lam, lim, f'λ ductil {lam:.2f}/{lim:.2f}'


def _resumen_fu(ev, pares):
    """FU por verificacion (maximo sobre combinaciones) y caso que gobierna. pares = [(fila, caso)].
    FU ESFUERZOS: resistencia (tension, compresion, flexion, cortante, interaccion, torsion).
    FU ESBELTEZ: esbeltez local, KL/r y ductilidad (valores de λ o KL/r y su limite en el texto).
    FU MAXIMO: el mayor de ambos (la seccion cumple si FU MAXIMO <= 1)."""
    mx = {}
    for c in CHEQUEOS_DEMANDA:
        vals = [(u[c], caso) for u, caso in pares if u.get(c) is not None]
        if vals:
            mx[c] = max(vals, key=lambda t: t[0])

    def val(c):
        return mx[c][0] if c in mx else None
    inter = [mx[c] for c in INTERACCION if c in mx]
    esf = [(mx[c][0], mx[c][1], c) for c in CHEQUEOS_DEMANDA if c in mx]
    esb = []
    for c, v in ev['const'].items():
        nombre = '(esbeltez local)' if c == CHEQUEO_EL else ('(esbeltez global)' if c == CHEQUEO_EG else '(ductilidad)')
        esb.append((v, nombre, c))
    cands = esf + esb
    fu_max, caso_gob, verif_gob = max(cands, key=lambda t: t[0]) if cands else (None, '', '')
    fu_esf = max(esf, key=lambda t: t[0]) if esf else (None, '', '')
    fu_esb = max(esb, key=lambda t: t[0]) if esb else (None, '', '')
    txt_esb = _valores_esbeltez(ev, fu_esb[2])[2] if esb else '—'
    duct = [ev['const'][c] for c in (CHEQUEO_D60, CHEQUEO_D12) if c in ev['const']]
    return {'CASO QUE GOBIERNA': caso_gob, 'VERIFICACION QUE GOBIERNA': verif_gob,
            'FU ESFUERZOS': fu_esf[0], 'ESFUERZO QUE GOBIERNA': _NOMBRE_ESFUERZO.get(fu_esf[2], fu_esf[2]),
            'FU ESBELTEZ': fu_esb[0], 'ESBELTEZ QUE GOBIERNA': txt_esb,
            'FU TRACCION': val('T traccion'), 'FU COMPRESION': val('C compresion'),
            'FU FLEXION EJE MAYOR': val('F flexion eje mayor'), 'FU FLEXION EJE MENOR': val('F flexion eje menor'),
            'FU CORTANTE EJE MAYOR': val('V cortante eje mayor'), 'FU CORTANTE EJE MENOR': val('V cortante eje menor'),
            'FU INTERACCION (H1/H2)': max(inter, key=lambda t: t[0])[0] if inter else None,
            'FU ESBELTEZ LOCAL (B4.1)': ev['const'].get(CHEQUEO_EL),
            'FU ESBELTEZ GLOBAL (KL/r<=200)': ev['const'].get(CHEQUEO_EG),
            'FU DUCTILIDAD (12.1.2, 12.1.3.1)': max(duct) if duct else None,
            'FU TORSION (H3.1)': val('T torsion (H3.1)'),
            'FU MAXIMO': fu_max}


def _preparar_entradas(materiales_filas, longitudes_df, demandas):
    """Materiales por nombre, longitudes por ID de barra y combinaciones de servicio."""
    mat = {str(f[0]).strip(): dict(fy=_num(f[2]), fu=_num(f[3]), E=_num(f[4]), nu=_num(f[5])) for f in materiales_filas}
    lon = {}
    for _, r in longitudes_df.iterrows():
        id_ = _num(r.get('ID'))
        if id_ is not None:
            lon[int(id_)] = r
    comb = demandas[demandas['TIPO'] == 'COMBINACION']
    return mat, lon, comb


def _entrada_barra(dib, i, id_barra, lon, mat):
    """Datos de entrada ASD de la barra i: longitud, K, Cb, ductilidad y material. Lo no dado se supone y se anota."""
    b = dib.barras[i]
    id_ = id_barra.get(i, i)
    L = float(np.linalg.norm(np.array(dib.nodos[b['n2']]) - np.array(dib.nodos[b['n1']]))) * 100.0
    m = mat.get(str(b.get('material') or '').strip())
    r = lon.get(id_)
    supuestos = []

    def campo(nombre):
        return _num(r.get(nombre)) if r is not None else None
    lx, ly, lb = campo('LX (m)'), campo('LY (m)'), campo('LB (m)')
    kx, ky = campo('KX'), campo('KY')
    cb = _num(r.get('CB')) if r is not None else None
    barra = f'barra {id_}: '
    lx = _req(lx, 'LX (m)', barra)
    ly = _req(ly, 'LY (m)', barra)
    lb = _req(lb, 'LB (m)', barra)
    kx = _req(kx, 'KX', barra)
    ky = _req(ky, 'KY', barra)
    cb = _req(cb, 'CB', barra)
    duct_raw = r.get('DUCTILIDAD') if r is not None else None
    nivel = None
    if duct_raw is not None and not (isinstance(duct_raw, float) and math.isnan(duct_raw)) and str(duct_raw).strip():
        nivel = str(duct_raw).strip().upper()
    lcon = campo('LI LONGITUD CONEXION (m)')
    conexion_raw = _texto(r.get('LI CONEXION')) if r is not None else ''
    sup_li = []
    familia_b = (G.obtener_perfil(b['perfil']) or {}).get('familia')
    if familia_b == 'LI':
        if not conexion_raw:
            raise ErrorAsd(f'{barra}falta LI CONEXION en LONGITUDES (angulo: UNA PATA, DOS PATAS o SOLDADA)')
        if lcon is None:
            raise ErrorAsd(f'{barra}falta LI LONGITUD CONEXION (m) en LONGITUDES (angulo)')
    return dict(m=m, L=L, lx=lx, ly=ly, lb=lb, kx=kx, ky=ky, cb=cb, nivel=nivel, supuestos=supuestos,
                e5=e5_cumple(r), conexion=conexion_raw or None,
                l_li=lcon * 100.0 if lcon is not None else None, sup_li=sup_li)


def perfil_por_grupo(dib):
    """Perfil de cada grupo. Cada grupo usa una sola seccion: si sus barras tienen perfiles distintos, ErrorAsd
    (no se elige uno en silencio). Devuelve {grupo: perfil} para los grupos con barras."""
    import collections
    perfiles, problemas = {}, []
    for g in GRUPOS_ELEMENTO:
        c = collections.Counter(b['perfil'] for b in dib.barras if b['elemento'] == g)
        if len(c) == 1:
            perfiles[g] = next(iter(c))
        elif len(c) > 1:
            detalle = ', '.join(f'{n} ({k} barras)' for n, k in c.most_common())
            problemas.append(f'{g}: {detalle}')
    if problemas:
        raise ErrorAsd('Cada grupo debe tener una sola seccion. Corrija la columna PERFIL de BARRAS en: '
                       + '; '.join(problemas))
    return perfiles


COLS_MATRIZ = ['CLAVE', 'ID', 'PERFIL', 'ELEMENTO DISENO', 'FU MAXIMO', 'FU DEMANDA', 'FU ESBELTEZ LOCAL (B4.1)',
               'FU ESBELTEZ GLOBAL (KL/r<=200)', 'FU DUCTILIDAD (12.1.2, 12.1.3.1)', 'CASO QUE GOBIERNA', 'ESTADO',
               'FU ESFUERZOS', 'FU ESBELTEZ', 'ESBELTEZ QUE GOBIERNA']
FU_DEMANDA_CLAVES = ['FU TRACCION', 'FU COMPRESION', 'FU FLEXION EJE MAYOR', 'FU FLEXION EJE MENOR',
                     'FU CORTANTE EJE MAYOR', 'FU CORTANTE EJE MENOR', 'FU INTERACCION (H1/H2)', 'FU TORSION (H3.1)']


def matriz_fu_por_perfil(dib, materiales_filas, longitudes_df, demandas, id_barra, familias=FAMILIAS_ASD):
    """FU ASD de cada barra con cada perfil del catalogo (IR, CE, LI), con sus combinaciones, longitudes, K y el
    grupo de la barra. Alimenta la hoja FU_MATRIZ: al cambiar el perfil de un grupo, Excel busca estos valores."""
    mat, lon, comb = _preparar_entradas(materiales_filas, longitudes_df, demandas)
    catalogo = sorted([n for n, d in G.PERFILES.items() if d['familia'] in familias], key=clave_orden)
    props = {n: _props_asd(n, G.PERFILES[n]['familia']) for n in catalogo}
    filas = []
    for i, b in enumerate(dib.barras):
        id_ = id_barra.get(i, i)
        e = _entrada_barra(dib, i, id_barra, lon, mat)
        m = e['m']
        f = comb[comb['BARRA'] == id_]
        demandas_list = [_demanda_fila(x) for _, x in f.iterrows()]
        casos = [x['CASO'] for _, x in f.iterrows()]
        pares_base = list(zip(demandas_list, casos))
        for nombre in catalogo:
            p, falta = props[nombre]
            fam = G.PERFILES[nombre]['familia']
            base = [f'{int(id_)}|{nombre}', int(id_), nombre, b['elemento']]
            if p is None or m is None or m['fy'] is None or m['E'] is None or m['nu'] is None:
                motivo = falta if p is None else 'material sin fy, E o nu'
                filas.append(base + [motivo] + ['NO REVISADO'] * 9)
                continue
            if fam == 'LI' and (e['l_li'] is None or e['conexion'] is None):
                # DECISION D20 (abierta): candidatos LI sin datos de conexion quedan NO REVISADO (alternativa: error).
                # candidato angulo: sin LI CONEXION y LI LONGITUD CONEXION en LONGITUDES no se evalua (no se supone)
                filas.append(base + ['falta LI CONEXION o LI LONGITUD CONEXION en LONGITUDES'] + ['NO REVISADO'] * 9)
                continue
            ev = _chequear(p, fam, m['fy'], m['fu'], m['E'], m['nu'], e['lx'] * 100.0, e['ly'] * 100.0,
                           e['lb'] * 100.0, e['kx'], e['ky'], b['elemento'], e['nivel'], demandas_list, cb=e['cb'],
                           e5=(fam == 'LI' and e['e5']),
                           u_t=(factor_u_angulo(p['xc'], e['l_li'], e['conexion']) if fam == 'LI' else 1.0))
            if ev['motivo'] and any(d['Pr_c'] > 0 for d in demandas_list):
                filas.append(base + ['NO REVISADO'] * 9 + ['NO REVISADO'])
                continue
            if not demandas_list:
                el, eg = ev['const'][CHEQUEO_EL], ev['const'].get(CHEQUEO_EG)
                filas.append(base + ['SIN COMBINACIONES', '—', round(el, 3),
                                     round(eg, 3) if eg is not None else '—', '—', '—', 'SIN COMBINACIONES',
                                     '—', round(max(el, eg or 0.0), 3), '—'])
                continue
            fu = _resumen_fu(ev, list(zip(ev['filas'], casos)))
            dem = [fu[c] for c in FU_DEMANDA_CLAVES if fu.get(c) is not None]
            eg = fu['FU ESBELTEZ GLOBAL (KL/r<=200)']
            duct = fu['FU DUCTILIDAD (12.1.2, 12.1.3.1)']
            filas.append(base + [round(fu['FU MAXIMO'], 4), round(max(dem), 4) if dem else '—',
                                 round(fu['FU ESBELTEZ LOCAL (B4.1)'], 4),
                                 round(eg, 4) if eg is not None else '—',
                                 round(duct, 4) if duct is not None else '—',
                                 fu['CASO QUE GOBIERNA'],
                                 'CUMPLE' if fu['FU MAXIMO'] <= 1.0 else 'NO CUMPLE',
                                 _r4(fu['FU ESFUERZOS']), _r4(fu['FU ESBELTEZ']), fu['ESBELTEZ QUE GOBIERNA']])
    return pd.DataFrame(filas, columns=COLS_MATRIZ)


def diseno_asd_desde_planilla(dib, materiales_filas, longitudes_df, demandas, id_barra, ruta_salida):
    """Diseno ASD (AISC 360-22) de las barras IR, CE, LI y HSS con las combinaciones de servicio (COMBINACIONES).
    RESULTADOS: FU por verificacion (demanda/capacidad, <= 1 cumple), esbeltez local (B4.1) y global (KL/r <= 200).
    Las familias fuera de IR, CE, LI y HSS quedan FUERA DE ALCANCE. Devuelve (capacidades, resultados, detalle)."""
    if materiales_filas is None:
        raise ErrorAsd('la planilla no tiene la hoja MATERIALES.')
    mat = {str(f[0]).strip(): dict(fy=_num(f[2]), fu=_num(f[3]), E=_num(f[4]), nu=_num(f[5])) for f in materiales_filas}
    lon = {}
    for _, r in longitudes_df.iterrows():
        id_ = _num(r.get('ID'))
        if id_ is not None:
            lon[int(id_)] = r
    comb = demandas[demandas['TIPO'] == 'COMBINACION']
    if comb.empty:
        raise ErrorAsd('no hay combinaciones de carga: defina COMBINACIONES (cargas de servicio, ASD).')
    caps, res_filas, det_filas = [], [], []
    for i, b in enumerate(dib.barras):
        id_ = id_barra.get(i, i)
        perfil, grupo = b['perfil'], b['elemento']
        fam = G.obtener_perfil(perfil)
        if fam is None:
            motivo = 'perfil no existe en la base Gerdau'
            caps.append({'ID': id_, 'PERFIL': perfil, 'NOTA': motivo})
            res_filas.append(dict(ID=id_, PERFIL=perfil, MATERIAL=b.get('material'), GRUPO=grupo,
                                  ESTADO='NO REVISADO', NOTAS=motivo))
            continue
        familia = fam['familia']
        if familia not in FAMILIAS_ASD:
            motivo = f'fuera del alcance del diseno ({familia}): solo IR, CE, LI y HSS'
            caps.append({'ID': id_, 'PERFIL': perfil, 'FAMILIA': familia, 'NOTA': motivo})
            res_filas.append(dict(ID=id_, PERFIL=perfil, MATERIAL=b.get('material'), GRUPO=grupo,
                                  ESTADO='FUERA DE ALCANCE', NOTAS=motivo))
            continue
        e = _entrada_barra(dib, i, id_barra, lon, mat)
        m, L, lx, ly, lb, kx, ky, cb, nivel, supuestos = (e['m'], e['L'], e['lx'], e['ly'], e['lb'], e['kx'],
                                                          e['ky'], e['cb'], e['nivel'], e['supuestos'])
        p, falta = _props_asd(perfil, familia)
        if p is None or m is None or m['fy'] is None or m['E'] is None or m['nu'] is None:
            motivo = falta if p is None else 'material sin fy, E o nu en MATERIALES'
            caps.append({'ID': id_, 'PERFIL': perfil, 'NOTA': motivo})
            res_filas.append(dict(ID=id_, PERFIL=perfil, MATERIAL=b.get('material'), GRUPO=grupo,
                                  ESTADO='NO REVISADO', NOTAS=motivo))
            continue
        estimados = p.get('_estimados', [])
        es_li = familia == 'LI'
        e5_b = es_li and e['e5']
        u_b = factor_u_angulo(p['xc'], e['l_li'], e['conexion']) if es_li else 1.0
        if es_li:
            supuestos = supuestos + e['sup_li']
        filas_comb = comb[comb['BARRA'] == id_]
        demandas_list = [_demanda_fila(f) for _, f in filas_comb.iterrows()]
        casos = [f['CASO'] for _, f in filas_comb.iterrows()]
        ev = _chequear(p, familia, m['fy'], m['fu'], m['E'], m['nu'], lx * 100.0, ly * 100.0, lb * 100.0,
                       kx, ky, grupo, nivel, demandas_list, cb=cb, e5=e5_b, u_t=u_b)
        res = ev['res']
        if ev['motivo'] and any(d['Pr_c'] > 0 for d in demandas_list):
            motivo = 'NO REVISADO: ' + res['motivo']
            caps.append({'ID': id_, 'PERFIL': perfil, 'FAMILIA': familia, 'NOTA': motivo})
            res_filas.append(dict(ID=id_, PERFIL=perfil, MATERIAL=b.get('material'), GRUPO=grupo,
                                  ESTADO='NO REVISADO', NOTAS=motivo))
            continue
        notas = []
        if familia == 'LI':
            notas.append('LI: ejes principales (u mayor, v menor); H2 con momento resultante; cortante de una pata (G3)')
            notas.append('E5(a) verificado en LONGITUDES: se usa Lc/rz de E5' if e5_b
                         else 'E5 no verificado (LONGITUDES): K L/rz y H2')
            notas.append(f'tension: U = {u_b:.3f} (D3)' if not e['conexion'] == 'DOS PATAS' else 'tension: U = 1.0 (ambas patas)')
        if grupo not in COMPRESION_GRUPOS:
            notas.append('sin verificacion KL/r (elemento sin compresion)')
        cap = {'ID': id_, 'PERFIL': perfil, 'FAMILIA': familia, 'MATERIAL': b.get('material'), 'GRUPO': grupo,
               'L (m)': L / 100.0, 'LX (m)': lx, 'LY (m)': ly, 'LB (m)': lb, 'KX': kx, 'KY': ky, 'CB': cb,
               'DUCTILIDAD': nivel or '(informativo)',
               'Pa traccion (kgf)': res['Pa_t'], 'Pa compresion (kgf)': res['Pa_c'],
               'Ma eje mayor (kgf·cm)': res['Ma_x'], 'Ma eje menor (kgf·cm)': res['Ma_y'],
               'Va eje mayor (kgf)': res['Va_x'], 'Va eje menor (kgf)': res['Va_y'],
               'KL/r eje fuerte': res['KLr_fuerte'], 'KL/r eje debil': res['KLr_debil'],
               'KL/r maximo': ev['klr'], 'Fe gobierna': res['detalle_c'].get('gobierna'),
               'Fn (kgf/cm2)': res['detalle_c'].get('Fn'), 'Ae (cm2)': res['detalle_c'].get('Ae'),
               'EL lambda/lambda_r': '; '.join(f'{n} {lam:.1f}/{lr:.1f}' for n, (lam, lr) in ev['el_det'].items()),
               'FU esbeltez local': max(ev['el'].values()),
               'FU esbeltez global': ev['const'].get(CHEQUEO_EG)}
        cap['E5 aplicado'] = ('SI' if e5_b else 'NO') if es_li else ''
        cap['U tension (LI)'] = round(u_b, 3) if es_li else None
        cap['SUPUESTOS'] = '; '.join(supuestos + ['estimado: ' + s for s in estimados])
        cap['NOTAS'] = '; '.join(notas + ([cap['SUPUESTOS']] if cap['SUPUESTOS'] else []))
        caps.append(cap)
        pares = list(zip(ev['filas'], casos))
        for u, caso in pares:
            det_filas.append([id_, perfil, caso] + [u.get(c) for c in CHEQUEOS_ASD]
                             + [max(v for v in u.values() if v is not None)]
                             + [u.get(c) for c in CHEQUEOS_TORSION])
        if not pares:
            fu = {c: None for c in COLS_RESULTADOS if c.startswith('FU')}
            fu.update({'CASO QUE GOBIERNA': None, 'FU ESBELTEZ LOCAL (B4.1)': ev['const'][CHEQUEO_EL],
                       'FU ESBELTEZ GLOBAL (KL/r<=200)': ev['const'].get(CHEQUEO_EG)})
            estado = 'SIN COMBINACIONES'
        else:
            fu = _resumen_fu(ev, pares)
            estado = 'CUMPLE' if fu['FU MAXIMO'] <= 1.0 else 'NO CUMPLE'
        res_filas.append(dict(ID=id_, PERFIL=perfil, MATERIAL=b.get('material'), GRUPO=grupo, **fu,
                              **{'KL/r EJE FUERTE': res['KLr_fuerte'], 'KL/r EJE DEBIL': res['KLr_debil']},
                              ESTADO=estado, NOTAS=cap['NOTAS']))
    resultados = pd.DataFrame(res_filas, columns=COLS_RESULTADOS)
    detalle = pd.DataFrame(det_filas, columns=['ID', 'PERFIL', 'CASO'] + CHEQUEOS_ASD + ['MAXIMO'] + CHEQUEOS_TORSION)
    capacidades = pd.DataFrame(caps)
    with pd.ExcelWriter(ruta_salida, engine='openpyxl') as w:
        pd.DataFrame([['Metodo', 'ASD con ANSI/AISC 360-22 (Omega: traccion 1.67/2.00, compresion y flexion 1.67, cortante 1.50 o 1.67)'],
                      ['Demandas', 'combinaciones de servicio de COMBINACIONES (ASD); sin factores de carga'],
                      ['FU', 'factor de utilizacion = demanda / capacidad ASD; cumple si FU <= 1.0'],
                      ['FU esbeltez local', 'lambda / lambda_r por elemento (B4.1a en compresion, B4.1b en flexion; angulo: pata)'],
                      ['FU esbeltez global', 'KL/r / 200 en elementos a compresion (2.3.2); vigas sin verificacion de KL/r'],
                      ['Alcance', 'IR, CE, LI (angulos de lados iguales) y HSS (tubulares rectangulares y cuadradas, catalogo ATC). Perfiles macizos (CS, OS, SOL) eliminados de la base; LM no incluido. '
                                  'Ductilidad: 12.1.2 (IR y CE) y 12.1.3.1 (columnas)'],
                      ['Unidades', 'fuerzas kgf; momentos kgf·cm; longitudes cm']],
                     columns=['DATO', 'VALOR']).to_excel(w, sheet_name='RESUMEN', index=False)
        capacidades.to_excel(w, sheet_name='CAPACIDADES', index=False)
        resultados.to_excel(w, sheet_name='RESULTADOS', index=False)
        detalle.to_excel(w, sheet_name='DETALLE', index=False)
    return capacidades, resultados, detalle


def _r4(x):
    return round(x, 4) if x is not None else '—'


def esbeltez_grupo_por_perfil(fu_matriz, grupo):
    """Por perfil: (FU esfuerzos maximo, FU esbeltez maximo, texto de la esbeltez de la barra que mas gobierna),
    con las barras del grupo. None si alguna barra no se puede revisar."""
    fm = fu_matriz[fu_matriz['ELEMENTO DISENO'] == grupo]
    out = {}
    for perfil, sub in fm.groupby('PERFIL'):
        esf = pd.to_numeric(sub['FU DEMANDA'], errors='coerce')
        esb = pd.to_numeric(sub['FU ESBELTEZ'], errors='coerce')
        if esf.isna().any() or esb.isna().any():
            out[perfil] = None
            continue
        k = int(esb.values.argmax())
        out[perfil] = (float(esf.max()), float(esb.max()), str(sub['ESBELTEZ QUE GOBIERNA'].iloc[k]))
    return out


def fu_grupo_por_perfil(fu_matriz, grupo):
    """Por perfil: el FU maximo de las barras del grupo, cada una con su propia demanda. None si alguna barra del
    grupo no se puede revisar (sin Fu, perfil sin datos, sin combinaciones): entonces el perfil no se puede aceptar."""
    fm = fu_matriz[fu_matriz['ELEMENTO DISENO'] == grupo]
    out = {}
    for perfil, sub in fm.groupby('PERFIL'):
        x = pd.to_numeric(sub['FU MAXIMO'], errors='coerce')
        out[perfil] = None if x.isna().any() else float(x.max())
    return out


def listas_desde_matriz(fu_matriz, dib, familias=FAMILIAS_ASD):
    """Perfiles que cumplen por grupo: cada barra del grupo cumple con su propia demanda (FU <= 1 en resistencia,
    esbeltez local y global). Mismo criterio que FU_BARRAS y que el desplegable del notebook. Ordenados por tipo y
    seccion. Un grupo sin barras recibe el catalogo completo, sin verificar."""
    catalogo = sorted([n for n, d in G.PERFILES.items() if d['familia'] in familias], key=clave_orden)
    listas, filas = {}, []
    for g in GRUPOS_ELEMENTO:
        barras_g = [b for b in dib.barras if b['elemento'] == g]
        listas[g] = list(catalogo)
        if not barras_g:
            continue
        import collections
        material = collections.Counter(str(b.get('material') or '').strip() for b in barras_g).most_common(1)[0][0]
        fm = fu_matriz[fu_matriz['ELEMENTO DISENO'] == g].copy()
        for c in ('FU MAXIMO', 'FU ESBELTEZ LOCAL (B4.1)', 'FU ESBELTEZ GLOBAL (KL/r<=200)'):
            fm[c + ' N'] = pd.to_numeric(fm[c], errors='coerce')
        peor = fu_grupo_por_perfil(fu_matriz, g)
        cumplen = []
        for n in catalogo:
            fam = G.PERFILES[n]['familia']
            base = [g, fam, n, seccion_de(n), G.PERFILES[n]['masa_kg_m'], material]
            sub = fm[fm['PERFIL'] == n]
            if sub.empty:
                filas.append(base + [None, None, None, 'sin datos para este grupo'])
                continue
            x = peor.get(n)
            el = sub['FU ESBELTEZ LOCAL (B4.1) N'].max()
            eg = sub['FU ESBELTEZ GLOBAL (KL/r<=200) N'].max()
            eg = None if pd.isna(eg) else round(float(eg), 4)
            if x is None:
                filas.append(base + [None, round(float(el), 4) if not pd.isna(el) else None, eg,
                                     'no revisado: alguna barra del grupo no se puede verificar'])
                continue
            ok = x <= 1.0
            filas.append(base + [round(x, 4), round(float(el), 4) if not pd.isna(el) else None, eg,
                                 'cumple' if ok else 'no cumple'])
            if ok:
                cumplen.append(n)
        listas[g] = sorted(cumplen, key=clave_orden)
    tabla = pd.DataFrame(filas, columns=['GRUPO', 'TIPO', 'PERFIL', 'SECCION', 'MASA (kg/m)', 'MATERIAL',
                                         'FU MAXIMO', 'FU ESBELTEZ LOCAL (B4.1)',
                                         'FU ESBELTEZ GLOBAL (KL/r<=200)', 'RESULTADO'])
    return listas, tabla


def listas_por_grupo_asd(dib, materiales_filas, longitudes_df, demandas, id_barra, familias=FAMILIAS_ASD):
    raise NotImplementedError('Sin uso en la version actual: sus demandas leen solo momentos de extremo (auditoria H-01). No usar.')
    """Perfiles de IR, CE y LI que cumplen en ASD para cada grupo de elementos: FU <= 1 por resistencia, esbeltez
    local y esbeltez global, con la envolvente de las combinaciones, la longitud, K y Cb mas exigentes.
    Ordenados por tipo y seccion. Un grupo sin barras (o sin demanda) recibe el catalogo completo sin verificar.
    Devuelve (listas {grupo: [perfiles]}, tabla DataFrame)."""
    import collections
    mat = {str(f[0]).strip(): dict(fy=_num(f[2]), fu=_num(f[3]), E=_num(f[4]), nu=_num(f[5])) for f in materiales_filas}
    lon = {}
    for _, r in longitudes_df.iterrows():
        id_ = _num(r.get('ID'))
        if id_ is not None:
            lon[int(id_)] = r
    comb = demandas[demandas['TIPO'] == 'COMBINACION']
    catalogo = sorted([n for n, d in G.PERFILES.items() if d['familia'] in familias], key=clave_orden)
    listas, filas_tabla = {}, []
    for g in GRUPOS_ELEMENTO:
        idx = [i for i, b in enumerate(dib.barras) if b['elemento'] == g]
        listas[g] = list(catalogo)
        if not idx:
            continue
        ids = [id_barra.get(i, i) for i in idx]
        f = comb[comb['BARRA'].isin(ids)]
        if f.empty:
            continue
        num = lambda c: f[c].astype(float).fillna(0.0)
        pc = max(num('N I (kgf)').clip(lower=0).max(), (-num('N J (kgf)')).clip(lower=0).max())
        pt = max((-num('N I (kgf)')).clip(lower=0).max(), num('N J (kgf)').clip(lower=0).max())
        mx = max(num('Mz I (kgf·m)').abs().max(), num('Mz J (kgf·m)').abs().max()) * 100.0
        my = max(num('My I (kgf·m)').abs().max(), num('My J (kgf·m)').abs().max()) * 100.0
        vy = max(num('Vy I (kgf)').abs().max(), num('Vy J (kgf)').abs().max())
        vz = max(num('Vz I (kgf)').abs().max(), num('Vz J (kgf)').abs().max())
        base = dict(Mr_x=mx, Mr_y=my, Vr_x=vy, Vr_y=vz)
        sint = [dict(base, Pr_c=pc, Pr_t=0.0), dict(base, Pr_c=0.0, Pr_t=pt)]
        lx = ly = lb = kx = ky = 0.0
        e5_todas, lineas_li = [], []
        cb_min = None
        niveles, materiales = set(), collections.Counter()
        for i in idx:
            r = lon.get(id_barra.get(i, i))
            L = float(np.linalg.norm(np.array(dib.nodos[dib.barras[i]['n2']]) - np.array(dib.nodos[dib.barras[i]['n1']])))

            def campo(nombre):
                return _num(r.get(nombre)) if r is not None else None
            v = {k: campo(k) for k in ('LX (m)', 'LY (m)', 'LB (m)', 'KX', 'KY', 'CB')}
            lx = max(lx, _req(v['LX (m)'], 'LX (m)'))
            ly = max(ly, _req(v['LY (m)'], 'LY (m)'))
            lb = max(lb, _req(v['LB (m)'], 'LB (m)'))
            kx = max(kx, _req(v['KX'], 'KX'))
            ky = max(ky, _req(v['KY'], 'KY'))
            cbi = _req(v['CB'], 'CB')
            cb_min = cbi if cb_min is None else min(cb_min, cbi)
            duct = r.get('DUCTILIDAD') if r is not None else None
            if duct is not None and not (isinstance(duct, float) and math.isnan(duct)) and str(duct).strip():
                niveles.add(str(duct).strip().upper())
            materiales[str(dib.barras[i].get('material') or '').strip()] += 1
            ent = _entrada_barra(dib, i, id_barra, lon, mat)
            e5_todas.append(ent['e5'])
            lineas_li.append((ent['l_li'], ent['conexion']))
        nivel = 'ALTA' if 'ALTA' in niveles else ('MEDIA' if 'MEDIA' in niveles else None)
        e5_grupo = bool(e5_todas) and all(e5_todas)
        nombre_mat = materiales.most_common(1)[0][0]
        m = mat.get(nombre_mat)
        if m is None or m['fy'] is None or m['E'] is None or m['nu'] is None:
            continue
        cumplen = []
        for nombre in catalogo:
            fam = G.PERFILES[nombre]['familia']
            masa = G.PERFILES[nombre]['masa_kg_m']
            base_fila = [g, fam, nombre, seccion_de(nombre), masa, nombre_mat]
            p, falta = _props_asd(nombre, fam)
            if p is None:
                filas_tabla.append(base_fila + [None, None, None, 'no revisado: ' + falta])
                continue
            ev = _chequear(p, fam, m['fy'], m['fu'], m['E'], m['nu'], lx * 100.0, ly * 100.0, lb * 100.0,
                           kx, ky, g, nivel, sint, cb=cb_min,
                           e5=(fam == 'LI' and e5_grupo),
                           u_t=(min(factor_u_angulo(p['xc'], l, c) for l, c in lineas_li)
                                if fam == 'LI' and lineas_li else 1.0))
            if ev['motivo'] and any(d['Pr_c'] > 0 for d in sint):
                filas_tabla.append(base_fila + [None, None, None, 'no revisado: ' + ev['motivo']])
                continue
            umax = max(max(u.values()) for u in ev['filas']) if ev['filas'] else None
            fu_el = ev['const'][CHEQUEO_EL]
            fu_eg = ev['const'].get(CHEQUEO_EG)
            if m['fu'] is None:
                umax, motivo = None, 'no revisado: falta Fu del material'
            else:
                motivo = 'cumple' if umax is not None and umax <= 1.0 else 'no cumple'
            filas_tabla.append(base_fila + [umax, fu_el, fu_eg, motivo])
            if umax is not None and umax <= 1.0:
                cumplen.append(nombre)
        listas[g] = sorted(cumplen, key=clave_orden)
    tabla = pd.DataFrame(filas_tabla, columns=['GRUPO', 'TIPO', 'PERFIL', 'SECCION', 'MASA (kg/m)', 'MATERIAL',
                                                'FU MAXIMO', 'FU ESBELTEZ LOCAL (B4.1)',
                                                'FU ESBELTEZ GLOBAL (KL/r<=200)', 'RESULTADO'])
    return listas, tabla


In [ ]:
%%writefile memoria_asd.py
"""
MEMORIA DE CALCULO del diseno ASD (AISC 360-22) por barra, en HTML imprimible (Archivo > Imprimir > PDF).
Usa el mismo motor que el bloque 10 (diseno_asd.py): cada numero de la memoria sale de los mismos calculos.
Por barra: datos de entrada, propiedades, cada verificacion con su ecuacion, los valores sustituidos y el FU.
Unidades: fuerzas kgf, momentos kgf·cm, longitudes cm, esfuerzos kgf/cm2.
"""
import base64
import copy
import html
import io
import math

import numpy as np
import pandas as pd
import diseno_asd as DA
import perfiles_gerdau as G

CSS = """
body { font-family: Arial, Helvetica, sans-serif; font-size: 10pt; color: #1b1b1b; margin: 24px; }
h1 { font-size: 16pt; margin: 0 0 4px 0; }
h2 { font-size: 12pt; border-bottom: 1.5px solid #1F4E79; color: #1F4E79; margin: 22px 0 8px 0; padding-bottom: 2px; }
h3 { font-size: 10.5pt; color: #1F4E79; margin: 12px 0 4px 0; }
table { border-collapse: collapse; width: 100%; margin: 4px 0 8px 0; font-size: 9pt; }
th, td { border: 1px solid #b9c7d6; padding: 3px 5px; vertical-align: top; text-align: left; }
th { background: #DDEBF7; }
td.num { text-align: right; white-space: nowrap; }
.meta td { border: none; padding: 1px 6px 1px 0; }
.cumple { color: #1e6b33; font-weight: bold; }
.nocumple { color: #a61b1b; font-weight: bold; }
.nota { color: #555; font-size: 8.5pt; }
.barra { page-break-inside: avoid; margin-top: 18px; }
.ec { font-family: 'Courier New', monospace; font-size: 8.5pt; }
@media print { body { margin: 10mm; } h2 { page-break-after: avoid; } }
"""


def _v(x, dec=2):
    """Numero con separador de miles y punto decimal; guion si no hay dato."""
    if x is None:
        return '—'
    if isinstance(x, str):
        return html.escape(x)
    try:
        x = float(x)
    except (TypeError, ValueError):
        return html.escape(str(x))
    if math.isnan(x):
        return '—'
    if abs(x) >= 1000:
        return f'{x:,.0f}'
    return f'{x:,.{dec}f}'


def _fu(x):
    if x is None or isinstance(x, str):
        return '—' if x is None else html.escape(x)
    cls = 'cumple' if x <= 1.0 else 'nocumple'
    return f'<span class="{cls}">{x:.3f}</span>'


def _evaluar(dib, i, id_barra, lon, mat, comb):
    """Evalua la barra i con el motor del bloque 10 y devuelve todo lo necesario para la memoria."""
    b = dib.barras[i]
    id_ = id_barra.get(i, i)
    out = dict(id=id_, perfil=b['perfil'], grupo=b['elemento'], material=b.get('material'))
    fam = G.obtener_perfil(b['perfil'])
    if fam is None:
        out.update(estado='NO REVISADO', motivo='perfil no existe en la base Gerdau')
        return out
    familia = fam['familia']
    out['familia'] = familia
    if familia not in DA.FAMILIAS_ASD:
        out.update(estado='FUERA DE ALCANCE', motivo=f'familia {familia}: no se disena (solo IR, CE, LI y HSS)')
        return out
    e = DA._entrada_barra(dib, i, id_barra, lon, mat)
    m = e['m']
    out.update(L=e['L'], lx=e['lx'], ly=e['ly'], lb=e['lb'], kx=e['kx'], ky=e['ky'], cb=e['cb'],
               nivel=e['nivel'], e5=e['e5'], conexion=e['conexion'], l_li=e['l_li'])
    supuestos = list(e['supuestos'])
    if m is None or m['fy'] is None or m['E'] is None or m['nu'] is None:
        out.update(estado='NO REVISADO', motivo='material sin fy, E o nu en MATERIALES', supuestos=supuestos)
        return out
    out.update(fy=m['fy'], fu=m['fu'], E=m['E'], nu=m['nu'])
    p, falta = DA._props_asd(b['perfil'], familia)
    if p is None:
        out.update(estado='NO REVISADO', motivo=falta, supuestos=supuestos)
        return out
    es_li = familia == 'LI'
    e5_b = es_li and e['e5']
    u_b = DA.factor_u_angulo(p['xc'], e['l_li'], e['conexion']) if es_li else 1.0
    if es_li:
        supuestos += e['sup_li']
    supuestos += ['estimado: ' + s for s in p.get('_estimados', [])]
    out.update(p=p, U=u_b, e5_b=e5_b, supuestos=supuestos)
    f = comb[comb['BARRA'] == id_]
    dem = [DA._demanda_fila(x) for _, x in f.iterrows()]
    casos = [x['CASO'] for _, x in f.iterrows()]
    ev = DA._chequear(p, familia, m['fy'], m['fu'], m['E'], m['nu'], e['lx'] * 100.0, e['ly'] * 100.0,
                      e['lb'] * 100.0, e['kx'], e['ky'], b['elemento'], e['nivel'], dem, cb=e['cb'],
                      e5=e5_b, u_t=u_b)
    if ev['motivo'] and any(d['Pr_c'] > 0 for d in dem):
        out.update(estado='NO REVISADO', motivo='NO REVISADO: ' + ev['motivo'])
        return out
    out.update(ev=ev, res=ev['res'], klr=ev['klr'])
    if not dem:
        out.update(estado='SIN COMBINACIONES', fu_res=None)
        return out
    if familia != 'HSS' and any(d.get('Tr', 0.0) > 0 for d in dem):              # H3: solo HSS
        supuestos.append('torsión de la barra (T) no verificada: H3 aplica solo a HSS en esta versión')
    pares = list(zip(ev['filas'], casos))
    fu = DA._resumen_fu(ev, pares)
    out.update(fu_res=fu, estado='CUMPLE' if fu['FU MAXIMO'] <= 1.0 else 'NO CUMPLE')
    out.update(dem_all=dem, casos_all=casos)
    mejor = None
    for (u, caso), d in zip(pares, dem):
        vals = [u[c] for c in DA.CHEQUEOS_DEMANDA if u.get(c) is not None]
        mx = max(vals) if vals else 0.0
        if mejor is None or mx > mejor[0]:
            mejor = (mx, caso, d)
    out.update(caso_dem=mejor[1], dem=mejor[2])
    return out


def _props_tabla(v):
    p, familia = v['p'], v['familia']
    if familia == 'LI':
        filas = [('Lado b (pata)', p['H'], 'cm'), ('Espesor t', p['t'], 'cm'), ('Area Ag', p['A'], 'cm²'),
                 ('Ix = Iy (catálogo)', p['Ix'], 'cm⁴'), ('Centroide x̄ = ȳ', p['xc'], 'cm'),
                 ('Producto de inercia Ixy', p['Ixy'], 'cm⁴'), ('Iu (eje mayor principal)', p['Iu'], 'cm⁴'),
                 ('Iv (eje menor principal)', p['Iv'], 'cm⁴'), ('ru', p['ru'], 'cm'), ('rz', p['rz'], 'cm'),
                 ('Su (u)', p['Su'], 'cm³'), ('Sv (v)', p['Sv'], 'cm³'), ('J (catálogo)', p.get('J'), 'cm⁴')]
    elif familia == 'HSS':
        filas = [('Peralte H', p['H'], 'cm'), ('Ancho B', p['B'], 'cm'), ('Espesor t (paredes)', p['t'], 'cm'),
                 ('Ancho plano bp = B − 3t', p['B'] - 3.0 * p['t'], 'cm'), ('Peralte plano hp = H − 3t', p['H'] - 3.0 * p['t'], 'cm'),
                 ('Área Ag', p['A'], 'cm²'), ('Ix', p['Ix'], 'cm⁴'), ('Iy', p['Iy'], 'cm⁴'), ('Sx', p['Sx'], 'cm³'),
                 ('Sy', p['Sy'], 'cm³'), ('Zx (sin radios de esquina)', p['Zx'], 'cm³'), ('Zy (sin radios de esquina)', p['Zy'], 'cm³'),
                 ('rx', p['rx'], 'cm'), ('ry', p['ry'], 'cm'), ('J (pared delgada, fórmula)', p['J'], 'cm⁴')]
    else:
        filas = [('Peralte H', p['H'], 'cm'), ('Ancho B', p['B'], 'cm'), ('Espesor del alma e', p['e'], 'cm'),
                 ('Espesor del patín t', p['t'], 'cm'), ('Area Ag', p['A'], 'cm²'), ('Ix', p['Ix'], 'cm⁴'),
                 ('Iy', p['Iy'], 'cm⁴'), ('Sx', p['Sx'], 'cm³'), ('Sy', p['Sy'], 'cm³'), ('Zx', p['Zx'], 'cm³'),
                 ('Zy', p['Zy'], 'cm³'), ('rx', p['rx'], 'cm'), ('ry', p['ry'], 'cm'), ('J', p['J'], 'cm⁴'),
                 ('Cw', p['Cw'], 'cm⁶')]
        if familia == 'CE':
            filas.append(('ro', p.get('ro'), 'cm'))
    body = ''.join(f'<tr><td>{html.escape(n)}</td><td class="num">{_v(x, 3)}</td><td>{u}</td></tr>'
                   for n, x, u in filas)
    if p.get('pagina_manual') is not None:
        body += f'<tr><td colspan="3" class="nota">Catálogo Gerdau rev04, pág. impresa {p["pagina_manual"]}.</td></tr>'
    if p.get('pagina_catalogo_atc'):
        body += (f'<tr><td colspan="3" class="nota">Catálogo ATC, pág. impresa {p["pagina_catalogo_atc"]}. '
                 'J por fórmula de pared delgada; Zx y Zy sin radios de esquina; Cw = 0.</td></tr>')
    if p.get('fuente_completado'):
        body += f'<tr><td colspan="3" class="nota">Completado desde el manual: {html.escape(p["fuente_completado"])}.</td></tr>'
    if any('estimado' in s for s in v.get('supuestos', [])):
        body += '<tr><td colspan="3" class="nota">Valores marcados como estimados en supuestos (geometría).</td></tr>'
    return f'<table><tr><th>Propiedad</th><th>Valor</th><th>Unidad</th></tr>{body}</table>'


def _verificaciones(v):
    """Filas de la tabla de verificaciones: (verificacion, ecuacion, calculo sustituido, capacidad, demanda, FU)."""
    res, fu, p, familia = v['res'], v['fu_res'], v['p'], v['familia']
    fy, fuu, U, A = v['fy'], v['fu'], v['U'], p['A']
    d = v.get('dem') or {}
    filas = []
    # Traccion
    calc_t = f'Fy·Ag/Ωt = {_v(fy, 1)}·{_v(A, 3)}/1.67 = {_v(res["Pa_t_fluencia"], 1)}'
    if fuu:
        calc_t += (f'; Fu·U·Ag/Ωt = {_v(fuu, 1)}·{_v(U, 3)}·{_v(A, 3)}/2.00 = {_v(res["Pa_t_fractura"], 1)}')
    filas.append(('Tracción', '§D2: D2-1 Pn = Fy·Ag (Ωt = 1.67); D2-2 Pn = Fu·Ae (Ωt = 2.00). §D3 Tabla D3.1: Ae = U·Ag (ángulos)' if familia == 'LI' else '§D2: D2-1 Pn = Fy·Ag (Ωt = 1.67); D2-2 Pn = Fu·Ae (Ωt = 2.00), Ae = Ag',
                  calc_t + f'; Pa = mín = {_v(res["Pa_t"], 1)}', res['Pa_t'], d.get('Pr_t'), fu and fu['FU TRACCION']))
    # Compresion
    dc = res['detalle_c']
    if familia == 'LI':
        klr_txt = f'λz = {_v(dc["lam_z"], 1)} (eje z), λu = {_v(dc["lam_u"], 1)} (eje u)'
        ecu_c = '§E3: E3-4 Fe = π²E/(KL/r)², E3-2/E3-3 Fn. §E5: E5-1/E5-2 (Lc/rz) si se verifica. §E7: Ae con patas. Ωc = 1.67'
    else:
        klr_txt = f'KL/rx = {_v(res["KLr_fuerte"], 1)}; KL/ry = {_v(res["KLr_debil"], 1)}'
        ecu_c = '§E3: E3-4 Fe, E3-2/E3-3 Fn. §E4: E4-2, E4-3 (Fex), E4-7. §E7: Ae = Ag − Σ(b − be)·t. Ωc = 1.67'
    if familia == 'HSS':
        ecu_c = ('§E3: E3-4 Fe (Fex, Fey; sección cerrada sin pandeo torsional), E3-2/E3-3 Fn. '
                 '§E7.1: Ae = Ag − Σ(b − be)·t, be por E7-2/E7-3 (c1 = 0.20, c2 = 1.38; λr = 1.40√(E/Fy)). Ωc = 1.67')
    calc_c = (f'{klr_txt}; gobierna {html.escape(str(dc.get("gobierna")))}; '
              + (f'Fe = {_v(dc["Fe"], 1)} kgf/cm²; ' if 'Fe' in dc else '') +
              f'Fn = {_v(dc["Fn"], 1)} kgf/cm²; Ae = {_v(dc["Ae"], 3)} cm²; '
              f'Pn = Fn·Ae = {_v(dc["Fn"] * dc["Ae"], 0)}')
    if fu and fu['FU COMPRESION'] == 0.0:
        calc_c += '; sin compresión en las combinaciones'
    filas.append(('Compresión', ecu_c, calc_c, res['Pa_c'], d.get('Pr_c'), fu and fu['FU COMPRESION']))
    # Flexion eje mayor
    dx = res['detalle_x']
    if familia == 'LI':
        calc_x = (f'My = Fy·Su = {_v(dx["My"], 1)}; Mcr (F10-4) = {_v(dx["Mcr"], 1)}; LTB (F10-2/3) = {_v(dx["LTB"], 1)}; '
                  f'Mn = {_v(dx["Mn"], 1)}')
        ecu_x = '§F10: F10-1 Mn = 1.5·My; F10-2/F10-3 LTB; F10-4 Mcr (βw = 0). §F10.3: F10-6 a F10-8 (pata). Ωb = 1.67'
        mr = math.hypot(d.get('Mr_x', 0.0), d.get('Mr_y', 0.0)) if d else None
    else:
        calc_x = (f'Mp = Fy·Zx = {_v(dx["Mp"], 1)}; LTB (F2) = {_v(dx["LTB"], 1)}; patín (F3) = {_v(dx.get("LFB"), 1)}; '
                  f'Mn = {_v(min(dx["LTB"], dx.get("LFB", dx["LTB"])), 1)}')
        ecu_x = '§F2: F2-1 Mn = Mp; F2-2 LTB inelástico; F2-3 LTB elástico (F2-4 a F2-8). §F3: F3-1, F3-2 (patín). Ωb = 1.67'
        mr = d.get('Mr_x') if d else None
    if familia == 'HSS':
        calc_x = (f'Mp = Fy·Zx = {_v(dx["Mp"], 1)}; patín (F7-2/F7-3) = {_v(dx["LFB"], 1)}; '
                  f'alma (F7-6) = {_v(dx["LWB"], 1)}; LTB (F7-8/F7-9) = {_v(dx["LTB"], 1)}; Mn = {_v(dx["Mn"], 1)}')
        ecu_x = ('§F7: F7-1 Mn = Mp = Fy·Z; F7-2 y F7-3 patín (F7-4: be); F7-6 alma; '
                 '§F7.4: F7-8 y F7-9 LTB (F7-10 y F7-11). Ωb = 1.67')
    filas.append(('Flexión eje mayor', ecu_x, calc_x + f'; Ma = Mn/1.67 = {_v(res["Ma_x"], 1)}',
                  res['Ma_x'], mr, fu and fu['FU FLEXION EJE MAYOR']))
    # Flexion eje menor
    dy = res['detalle_y']
    if familia == 'LI':
        calc_y = f'My = Fy·Sv = {_v(dy["My"], 1)}; Mn = mín(1.5My; LLB) = {_v(dy["Mn"], 1)}'
        ecu_y = '§F10: F10-1 Mn = 1.5·My. §F10.3: F10-6 a F10-8 (pata, eje v). Ωb = 1.67'
        mr_y = mr
    else:
        calc_y = (f'Mp = mín(Fy·Zy; 1.6·Fy·Sy) = {_v(dy["Mp"], 1)}; My = Fy·Sy = {_v(dy["My"], 1)}')
        ecu_y = '§F6: F6-1 Mp = mín(Fy·Zy; 1.6·Fy·Sy); F6-2, F6-3 (patín y alma). Ωb = 1.67'
        mr_y = d.get('Mr_y') if d else None
    if familia == 'HSS':
        calc_y = (f'Mp = Fy·Zy = {_v(dy["Mp"], 1)}; patín (F7-2/F7-3) = {_v(dy["LFB"], 1)}; '
                  f'alma (F7-6) = {_v(dy["LWB"], 1)}; Mn = {_v(dy["Mn"], 1)}')
        ecu_y = ('§F7 (eje menor; las paredes verticales hacen de patín): F7-1, F7-2/F7-3, F7-6. '
                 'Sin pandeo lateral-torsional. Ωb = 1.67')
    filas.append(('Flexión eje menor', ecu_y, calc_y + f'; Ma = {_v(res["Ma_y"], 1)}',
                  res['Ma_y'], mr_y, fu and fu['FU FLEXION EJE MENOR']))
    # Cortante
    dvx, dvy = res['detalle_vx'], res['detalle_vy']
    if familia == 'LI':
        calc_vx = f'Vn = 0.6·Fy·b·t·Cv2 = 0.6·{_v(fy, 1)}·{_v(p["b"], 3)}·{_v(p["t"], 3)}·{_v(dvx["Cv2"], 3)}; Ωv = 1.67'
        ecu_vx = '§G3: G3-1 Vn = 0.6·Fy·b·t·Cv2 (una pata, kv = 1.2). Ωv = 1.67'
        ecu_vy = '§G3: G3-1 (una pata). Ωv = 1.67'
        calc_vy = calc_vx
    else:
        calc_vx = (f'Vn = 0.6·Fy·Aw·Cv1 = 0.6·{_v(fy, 1)}·{_v(p["H"] * p["e"], 2)}·{_v(dvx["Cv1"], 3)}; '
                   f'Ωv = 1.50 si I laminada, 1.67 si no')
        ecu_vx = '§G2.1: G2-1 Vn = 0.6·Fy·Aw·Cv1; G2-2 (I laminada, Ωv = 1.50). §G2.2: G2-3, G2-4 (kv = 5.34). Ωv = 1.67'
        calc_vy = (f'Vn = 2·0.6·Fy·B·t·Cv2 = 2·0.6·{_v(fy, 1)}·{_v(p["B"], 2)}·{_v(p["t"], 3)}·{_v(dvy["Cv2"], 3)}; Ωv = 1.67')
        ecu_vy = '§G6: Vn = 2·0.6·Fy·B·t·Cv2 (dos patines, kv = 1.2). Ωv = 1.67'
    if familia == 'HSS':
        calc_vx = (f'Vn = 0.6·Fy·Aw·Cv2 = 0.6·{_v(fy, 1)}·{_v(dvx["Aw"], 2)}·{_v(dvx["Cv2"], 3)}; '
                   f'Aw = 2·h·t, h/t = {_v(dvx["h_tw"], 2)}; Ωv = 1.67')
        calc_vy = (f'Vn = 0.6·Fy·Aw·Cv2 = 0.6·{_v(fy, 1)}·{_v(dvy["Aw"], 2)}·{_v(dvy["Cv2"], 3)}; '
                   f'Aw = 2·h·t, h/t = {_v(dvy["h_tw"], 2)}; Ωv = 1.67')
        ecu_vx = '§G4: G4-1 Vn = 0.6·Fy·Aw·Cv2, Aw = 2·h·t (h = peralte plano). Cv2 de §G2.2: G2-9 a G2-11 con kv = 5. Ωv = 1.67'
        ecu_vy = '§G4: G4-1 Vn = 0.6·Fy·Aw·Cv2, Aw = 2·h·t (h = ancho plano). Cv2 de §G2.2 con kv = 5. Ωv = 1.67'
    filas.append(('Cortante eje mayor', ecu_vx, calc_vx + f'; Va = {_v(res["Va_x"], 1)}', res['Va_x'],
                  d.get('Vr_x') if d else None, fu and fu['FU CORTANTE EJE MAYOR']))
    filas.append(('Cortante eje menor', ecu_vy, calc_vy + f'; Va = {_v(res["Va_y"], 1)}', res['Va_y'],
                  d.get('Vr_y') if d else None, fu and fu['FU CORTANTE EJE MENOR']))
    # Interaccion
    if familia == 'LI':
        ecu_i = '§H2: H2-1 fra/Fca + frbw/Fcbw + frbz/Fcbz ≤ 1 (Mr = momento resultante sobre cada eje principal)'
        calc_i = (f'Pa compresión = {_v(res["Pa_c"], 1)} kgf; Pa tracción = {_v(res["Pa_t"], 1)} kgf; '
                  f'Mau = {_v(res["Ma_x"], 1)} kgf·cm; Mav = {_v(res["Ma_y"], 1)} kgf·cm')
    else:
        ecu_i = '§H1.1: H1-1a Pr/Pc + 8/9·(Mrx/Mcx + Mry/Mcy) ≤ 1 (Pr/Pc ≥ 0.2); H1-1b Pr/(2Pc) + (Mrx/Mcx + Mry/Mcy) ≤ 1 (Pr/Pc < 0.2)'
        calc_i = f'Pc = {_v(res["Pa_c"], 1)} (compresión) o Pt = {_v(res["Pa_t"], 1)} (tracción)'
    if familia == 'HSS' and res.get('Tc') is not None:                           # H3: torsion y H3.2
        dt = res['detalle_t']
        tr = d.get('Tr', 0.0) if d else 0.0
        calc_t = (f'h = {_v(dt["h"], 3)} cm (lado plano mayor), h/t = {_v(dt["ht"], 2)}, {dt["reg"]}: '
                  f'Fcr = {_v(dt["Fcr"], 1)} kgf/cm²; C = {_v(dt["C"], 2)} cm³; Tn = Fcr·C = {_v(dt["Tn"], 0)} kgf·cm; '
                  f'Tc = Tn/ΩT = {_v(res["Tc"], 0)} kgf·cm')
        filas.append(('Torsión', '§H3.1: H3-1 Tn = Fcr·C; H3-3, H3-4 o H3-5 para Fcr; C = 2(B−t)(H−t)t − 4.5(4−π)t³ '
                      '(nota de usuario). ΩT = 1.67', calc_t, res['Tc'], tr, fu and fu['FU TORSION (H3.1)']))
        activa = tr > 0.2 * res['Tc']
        ecu_i = ('§H3.2: si Tr ≤ 0.2·Tc, interacción H1 y la torsión se descarta. Si Tr > 0.2·Tc: '
                 'H3-6 Pr/Pc + Mrx/Mcx + Mry/Mcy + (Vr/Vc + Tr/Tc)² ≤ 1.0')
        calc_i = (f'Tr/Tc = {_v(tr / res["Tc"], 3)}: ' + ('aplica H3-6' if activa else 'aplica H1 (torsión descartada)')
                  + (f'; H3-6 = {_v(d["Pr_c"] / res["Pa_c"] if d.get("Pr_c", 0) > 0 else 0.0, 3)} (P) + {_v(d["Mr_x"] / res["Ma_x"], 3)} (Mx) + '
                     f'{_v(d["Mr_y"] / res["Ma_y"], 3)} (My) + ...' if activa and d else ''))
    filas.append(('Interacción', ecu_i, calc_i, None, None, fu and fu['FU INTERACCION (H1/H2)']))
    return filas


def _estado_txt(ok):
    if ok is None:
        return '—'
    return f'<span class="{"cumple" if ok else "nocumple"}">{"CUMPLE" if ok else "NO CUMPLE"}</span>'


def _tabla_verif(v):
    filas = _verificaciones(v)
    cuerpo = []
    for nombre, ecu, calc, cap, dem, fu in filas:
        cuerpo.append(f'<tr><td>{nombre}{_refs_verif(nombre, v["familia"])}</td><td class="ec">{html.escape(ecu)}</td>'
                      f'<td class="ec">{calc}</td><td class="num">{_v(cap, 1) if cap is not None else "—"}</td>'
                      f'<td class="num">{_v(dem, 1) if dem is not None else "—"}</td>'
                      f'<td class="num">{_fu(fu)}</td></tr>')
    ev = v['ev']
    for nombre, (lam, lr) in ev['el_det'].items():
        cuerpo.append(f'<tr><td>Esbeltez local: {html.escape(nombre)}{_ref_local(v)}</td>'
                      f'<td class="ec">NTC Tabla 3.2.1 (compresión) / Tabla 3.2.2 (flexión): λ ≤ λr</td>'
                      f'<td class="ec">λ = {_v(lam, 2)}; λr = {_v(lr, 2)}</td>'
                      f'<td class="num">{_v(lr, 2)}</td><td class="num">{_v(lam, 2)}</td>'
                      f'<td class="num">{_estado_txt(lam <= lr)}</td></tr>')
    if v['grupo'] in DA.COMPRESION_GRUPOS:
        lim = 60.0 if (v['grupo'] == 'COLUMNA' and v['nivel'] in ('ALTA', 'MEDIA')) else 200.0
        ecu_g = 'NTC 2.3.2: KL/r ≤ 200 (§E2 recomienda 200)' + ('; columna dúctil 12.1.3.1: KL/r ≤ 60' if lim == 60.0 else '')
        cuerpo.append(f'<tr><td>Esbeltez global{_ref_html("NTC_2_3_2")}'
                      + (_ref_html('NTC_12_1_3_1') if lim == 60.0 else '') + f'</td><td class="ec">{ecu_g}</td>'
                      f'<td class="ec">KL/r máx = {_v(v["klr"], 1)}</td><td class="num">{_v(lim, 0)}</td>'
                      f'<td class="num">{_v(v["klr"], 1)}</td><td class="num">{_estado_txt(v["klr"] <= lim)}</td></tr>')
    else:
        cuerpo.append('<tr><td>Esbeltez global</td><td class="ec">NTC 2.3.2: KL/r ≤ 200 (§E2)</td>'
                      '<td class="ec">no aplica: elemento sin compresión</td><td class="num">—</td>'
                      '<td class="num">—</td><td class="num">—</td></tr>')
    if (v.get('fu_res') or {}).get('FU TRACCION'):
        lim_t = 300.0 if v['grupo'] in ('ARRIOSTRE', 'DIAGONAL') else 240.0
        lmax = max(v['lx'], v['ly']) * 100.0
        rmin = v['p']['rz'] if v['familia'] == 'LI' else min(v['p']['rx'], v['p']['ry'])
        est_t = ('<span class="cumple">dentro de la referencia</span>' if lmax / rmin <= lim_t
                 else '<span class="nocumple">supera la referencia</span>')
        cuerpo.append(f'<tr><td>Esbeltez en tensión (referencia){_ref_html("NTC_2_3_2")}{_ref_html("NTC_4_4")}</td>'
                      f'<td class="ec">NTC 2.3.2: límite de referencia {lim_t:.0f} ({"miembros principales" if lim_t == 240.0 else "contraventeos y secundarios"}); NTC 4.4: sin límite obligatorio</td>'
                      f'<td class="ec">L/r = {_v(lmax, 1)} / {_v(rmin, 2)}</td><td class="num">{lim_t:.0f}</td>'
                      f'<td class="num">{_v(lmax / rmin, 1)}</td><td class="num">{est_t}</td></tr>')
    dd = ev.get('dd')
    if dd is not None and v['nivel']:
        if v['familia'] == 'LI':
            lam_d, lim_d = dd['lambda'], dd['limite']
        else:
            lam_d = dd['lf']
            lim_d = dd['lda_f'] if v['nivel'] == 'ALTA' else dd['ldm_f']
        cuerpo.append(f'<tr><td>Ductilidad ({v["nivel"]}){_ref_html("NTC_12_1_2")}</td><td class="ec">NTC Tabla 12.1.2: ancho/grueso</td>'
                      f'<td class="ec">b/t = {_v(lam_d, 2)}</td><td class="num">{_v(lim_d, 2)}</td>'
                      f'<td class="num">{_v(lam_d, 2)}</td><td class="num">{_estado_txt(lam_d <= lim_d)}</td></tr>')
    cabecera = ('<tr><th>Verificación</th><th>Ecuación</th><th>Cálculo</th><th>Límite / capacidad</th>'
                '<th>Calculado / demanda</th><th>FU o estado</th></tr>')
    return f'<table>{cabecera}{"".join(cuerpo)}</table>'


COLS_GRUPO = ['ID', 'PERFIL', 'FU ESFUERZOS', 'ESFUERZO QUE GOBIERNA', 'FU ESBELTEZ', 'ESBELTEZ QUE GOBIERNA',
              'ELEM. LOCAL CRÍTICO', 'CLASIF. NTC (3.2)', 'λ LOCAL CALC', 'λ LOCAL LÍMITE', 'KL/r CALC', 'KL/r LÍMITE',
              'λ DUCT CALC', 'λ DUCT LÍMITE', 'ESTADO', 'NOTA']


def _interaccion(v):
    """FU de interaccion flexocompresion (H1-1 o H2-1), el maximo sobre combinaciones. Sin carga axial, H1 se reduce
    a Mrx/Mcx + Mry/Mcy: se toma la suma de los FU de flexion."""
    ev = v.get('ev')
    if not ev or not ev.get('filas'):
        return None
    vals = []
    for u in ev['filas']:
        x = u.get('H1 flexocompresion', u.get('H1 flexotension', u.get('H2 angulo (H2-1)')))
        if x is None:
            x = u.get('F flexion eje mayor', 0.0) + u.get('F flexion eje menor', 0.0)
        vals.append(x)
    return max(vals) if vals else None


def _fila_grupo(v):
    """Una fila de la tabla por grupo: FU de interaccion, esbeltez local y global (calculado y limite) y estado."""
    if 'res' not in v or not v.get('fu_res'):
        return [v['id'], v['perfil']] + [None] * 12 + [v['estado'], v.get('motivo', '')]
    ev = v['ev']
    inter = _interaccion(v)
    crit, lam_c, lam_l = max(((n, l, r) for n, (l, r) in ev['el_det'].items()), key=lambda t: t[1] / t[2])
    klr_c = klr_l = None
    if v['grupo'] in DA.COMPRESION_GRUPOS:
        klr_c = round(v['klr'], 2)
        klr_l = 60.0 if (v['grupo'] == 'COLUMNA' and v['nivel'] in ('ALTA', 'MEDIA')) else 200.0
    dd = ev.get('dd')
    lam_d_c = lam_d_l = None
    if dd is not None and v['nivel']:
        lam_d_c = dd['lambda'] if v['familia'] == 'LI' else dd['lf']
        lam_d_l = dd['limite'] if v['familia'] == 'LI' else (dd['lda_f'] if v['nivel'] == 'ALTA' else dd['ldm_f'])
        lam_d_c, lam_d_l = round(lam_d_c, 2), round(lam_d_l, 2)
    nota = []
    if v['familia'] == 'LI' and not v['e5_b'] and v['grupo'] in DA.COMPRESION_GRUPOS:
        nota.append('E5 no verificado: K L/rz y H2')
    if v['grupo'] not in DA.COMPRESION_GRUPOS:
        nota.append('sin compresión: KL/r no aplica')
    clasif = (f'tipo {DA.tipo_seccion_flexion(v["p"], v["familia"], v["fy"], v["E"])}' if v['grupo'] == 'VIGA'
              else DA.clasificacion_compresion(v['p'], v['familia'], v['fy'], v['E']))
    fr = v['fu_res']
    return [v['id'], v['perfil'], round(fr['FU ESFUERZOS'], 3) if fr.get('FU ESFUERZOS') is not None else None,
            fr.get('ESFUERZO QUE GOBIERNA'), round(fr['FU ESBELTEZ'], 3) if fr.get('FU ESBELTEZ') is not None else None,
            fr.get('ESBELTEZ QUE GOBIERNA'), crit, clasif, round(lam_c, 2), round(lam_l, 2), klr_c, klr_l,
            lam_d_c, lam_d_l, v['estado'], '; '.join(nota)]


def resultados_grupo(dib, grupo, perfil, materiales_filas, longitudes_df, demandas, id_barra):
    """Tabla por grupo con un perfil elegido para todas sus barras (no cambia el modelo)."""
    d2 = copy.copy(dib)
    d2.barras = [dict(b, perfil=perfil) if b['elemento'] == grupo else b for b in dib.barras]
    mat, lon, comb = DA._preparar_entradas(materiales_filas, longitudes_df, demandas)
    filas = [_fila_grupo(_evaluar(d2, i, id_barra, lon, mat, comb))
             for i, b in enumerate(d2.barras) if b['elemento'] == grupo]
    return pd.DataFrame(filas, columns=COLS_GRUPO).fillna('—')


def opciones_grupo(fu_matriz, grupo):
    """Opciones del desplegable de un grupo: todas las secciones IR, CE y LI ordenadas por tipo y seccion, con el
    resultado para todas las barras del grupo (cada una con su demanda), segun FU_MATRIZ."""
    catalogo = sorted([n for n, d in G.PERFILES.items() if d['familia'] in DA.FAMILIAS_ASD], key=DA.clave_orden)
    peor = DA.fu_grupo_por_perfil(fu_matriz, grupo)
    det = DA.esbeltez_grupo_por_perfil(fu_matriz, grupo)
    ops = []
    for n in catalogo:
        if n not in peor:
            etiqueta = f'{n}  —  sin datos para este grupo'
        elif peor[n] is None:
            etiqueta = f'{n}  —  no revisado en alguna barra'
        else:
            x = peor[n]
            cumple = 'cumple' if x <= 1.0 else 'no cumple'
            r = det.get(n)
            if r is not None:
                etiqueta = f'{n}  —  {cumple} · FU esfuerzos {r[0]:.2f} · FU esbeltez {r[1]:.2f} ({r[2]})'
            else:
                etiqueta = f'{n}  —  {cumple} (FU máx {x:.2f})'
        ops.append((etiqueta, n))
    return ops


def _maximo_fila(u):
    """FU maximo de una combinacion (todas las verificaciones de demanda)."""
    vals = [u[c] for c in DA.CHEQUEOS_DEMANDA if u.get(c) is not None]
    return max(vals) if vals else None


def _inter_fila(u):
    """FU de interaccion flexocompresion de una combinacion (H1 o H2; sin axial, suma de los FU de flexion)."""
    x = u.get('H1 flexocompresion', u.get('H1 flexotension', u.get('H2 angulo (H2-1)')))
    if x is None:
        x = u.get('F flexion eje mayor', 0.0) + u.get('F flexion eje menor', 0.0)
    return x


def _tabla_combinaciones(v):
    """Demandas y FU de cada combinacion de servicio de la barra. Resalta la combinacion que controla."""
    dem_all, casos, filas = v.get('dem_all'), v.get('casos_all'), v['ev']['filas']
    if not dem_all:
        return '<p class="nota">Sin combinaciones de servicio para esta barra.</p>'
    k_max = max(range(len(filas)), key=lambda k: _maximo_fila(filas[k]) or 0.0)
    cuerpo = []
    for k, (d, caso, u) in enumerate(zip(dem_all, casos, filas)):
        est = ' style="font-weight:bold;background:#fff4d6"' if k == k_max else ''
        cuerpo.append(f'<tr{est}><td>{html.escape(str(caso))}</td><td class="num">{_v(d["Pr_t"], 1)}</td>'
                      f'<td class="num">{_v(d["Pr_c"], 1)}</td><td class="num">{_v(d["Mr_x"], 1)}</td>'
                      f'<td class="num">{_v(d["Mr_y"], 1)}</td><td class="num">{_v(d["Vr_x"], 1)}</td>'
                      f'<td class="num">{_v(d["Vr_y"], 1)}</td><td class="num">{_fu(_inter_fila(u))}</td>'
                      f'<td class="num">{_fu(_maximo_fila(u))}</td></tr>')
    return ('<table><tr><th>Combinación</th><th>Pr tracción (kgf)</th><th>Pr compresión (kgf)</th>'
            '<th>Mrx (kgf·cm)</th><th>Mry (kgf·cm)</th><th>Vrx (kgf)</th><th>Vry (kgf)</th>'
            '<th>FU interacción</th><th>FU máx. combinación</th></tr>' + ''.join(cuerpo) + '</table>')


def _bloque_barra(v):
    """Detalle de una barra: datos, propiedades, combinaciones y verificaciones con ecuaciones y referencias."""
    if 'res' not in v or not v.get('fu_res'):
        return (f'<section class="barra"><h2>Barra {v["id"]} · {html.escape(v["perfil"])} · {v["grupo"]} · '
                f'{v["estado"]}</h2><p>{html.escape(v.get("motivo", ""))}</p></section>')
    cl = 'cumple' if v['estado'] == 'CUMPLE' else 'nocumple'
    dem = v.get('dem') or {}
    datos = (f'<table class="meta"><tr><td><b>Material</b>: {html.escape(str(v["material"]))} · '
             f'Fy = {_v(v["fy"], 0)} kgf/cm² · Fu = {_v(v["fu"], 0)} kgf/cm² · E = {_v(v["E"], 0)} kgf/cm² · '
             f'ν = {_v(v["nu"], 2)}</td></tr>'
             f'<tr><td><b>Longitudes</b>: L = {_v(v["L"], 1)} cm · LX = {_v(v["lx"] * 100, 1)} cm · '
             f'LY = {_v(v["ly"] * 100, 1)} cm · LB = {_v(v["lb"] * 100, 1)} cm · KX = {_v(v["kx"], 2)} · '
             f'KY = {_v(v["ky"], 2)} · Cb = {_v(v["cb"], 2)} · Ductilidad (BARRAS): '
             f'{html.escape(v["nivel"] or "no marcada")}</td></tr>'
             + ('<tr><td><b>Ángulo</b>: E5 ' + ('verificado (LONGITUDES)' if v['e5_b'] else
                                                  'no verificado: se usa K L/rz y H2')
                + f' · conexión {html.escape(v["conexion"])} · U = {_v(v["U"], 3)} · l = {_v(v["l_li"], 1)} cm</td></tr>'
                if v['familia'] == 'LI' else '')
             + '</table>')
    sup = ''.join(f'<li>{html.escape(s)}</li>' for s in v.get('supuestos', []))
    return (f'<section class="barra" id="barra-{v["id"]}">'
            f'<h2>Barra {v["id"]} · {html.escape(v["perfil"])} · {html.escape(v["grupo"])} · '
            f'<span class="{cl}">{v["estado"]}</span> · FU máximo {_v(fu_max(v), 3)}</h2>'
            f'{datos}'
            f'<h3>Propiedades de la sección</h3>{_props_tabla(v)}'
            f'<h3>Combinaciones de servicio de la barra</h3>'
            f'<p class="nota">Cada combinación se revisa con sus propias demandas. Resaltada: la que controla '
            f'(<b>{html.escape(str(v.get("caso_dem")))}</b>).</p>{_tabla_combinaciones(v)}'
            f'<h3>Verificaciones (AISC 360-22 ASD y NTC), combinación que controla</h3>'
            f'<p class="nota">Capacidad = resistencia admisible; demanda = la de la combinación que controla. '
            f'FU = demanda / capacidad (cumple si FU ≤ 1.0). Las filas de esbeltez comparan λ calculado con λ límite (NTC).</p>'
            f'{_tabla_verif(v)}'
            + (f'<h3>Supuestos de esta barra</h3><ul class="nota">{sup}</ul>' if sup else '')
            + '</section>')


def fu_max(v):
    """FU maximo de la barra (todas las verificaciones)."""
    return v['fu_res']['FU MAXIMO']


def _desplazamientos(dib, casos, id_de_indice):
    """Desplazamiento maximo de los nodos por combinacion (mm). casos: [(nombre, tipo, res)], res['nodos'] en cm."""
    filas, mejor = [], None
    for nombre, tipo, res in casos:
        if tipo != 'COMBINACION':
            continue
        d = np.array([res['nodos'][i][:3] for i in range(len(dib.nodos))], float) * 10.0        # cm -> mm
        r = np.linalg.norm(d, axis=1)
        k = int(np.argmax(r))
        fila = dict(combinacion=nombre, ux=float(np.abs(d[:, 0]).max()), uy=float(np.abs(d[:, 1]).max()),
                    uz=float(np.abs(d[:, 2]).max()), resultante=float(r[k]), nodo=id_de_indice[k])
        filas.append(fila)
        if mejor is None or fila['resultante'] > mejor[0]['resultante']:
            mejor = (fila, res)
    return pd.DataFrame(filas), mejor


def _png_deformada(dib, res, titulo):
    """Imagen de la deformada (3D) de un caso: original en gris, deformada en azul. Devuelve (base64, escala)."""
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    P0 = np.array(dib.nodos, float)
    U = np.array([res['nodos'][i][:3] for i in range(len(P0))], float) / 100.0             # cm -> m
    tam = max(float(np.ptp(P0[:, k])) for k in range(3)) or 1.0
    dmax = float(np.max(np.linalg.norm(U, axis=1)))
    escala = 0.1 * tam / dmax if dmax > 0 else 1.0
    P = P0 + escala * U
    fig = Figure(figsize=(7.2, 5.4), dpi=110)
    FigureCanvasAgg(fig)
    ax = fig.add_subplot(111, projection='3d')
    for b in dib.barras:
        a, c = b['n1'], b['n2']
        ax.plot([P0[a, 0], P0[c, 0]], [P0[a, 1], P0[c, 1]], [P0[a, 2], P0[c, 2]], color='#bbbbbb', ls='--', lw=1)
        ax.plot([P[a, 0], P[c, 0]], [P[a, 1], P[c, 1]], [P[a, 2], P[c, 2]], color='#1f77b4', lw=2.2)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title(titulo, fontsize=9)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    return base64.b64encode(buf.getvalue()).decode('ascii'), escala


# Referencias de las verificaciones: apartado y pagina del documento (NTC-Acero CDMX 2020 y AISC 360-22).
REF = {
    'D2': 'AISC 360-22 §D2, ecs. D2-1 y D2-2, pág. 16.1-32 y 16.1-33',
    'D3': 'AISC 360-22 §D3, Tabla D3.1 (factor U), pág. 16.1-34',
    'E3': 'AISC 360-22 §E3, ecs. E3-1 a E3-4, pág. 16.1-40 y 16.1-41',
    'E4': 'AISC 360-22 §E4, ecs. E4-2, E4-3 y E4-7, pág. 16.1-41 y 16.1-42',
    'E5': 'AISC 360-22 §E5 (ángulos sencillos), ecs. E5-1 y E5-2, pág. 16.1-43 y 16.1-44',
    'E7': 'AISC 360-22 §E7 (elementos esbeltos), pág. 16.1-48 y 16.1-49',
    'F2': 'AISC 360-22 §F2, ecs. F2-1 a F2-8, pág. 16.1-53 y 16.1-54',
    'F3': 'AISC 360-22 §F3, ecs. F3-1 y F3-2, pág. 16.1-55',
    'F6': 'AISC 360-22 §F6, ecs. F6-1 a F6-3, pág. 16.1-61 y 16.1-62',
    'F10': 'AISC 360-22 §F10 (ángulos sencillos), ecs. F10-1 a F10-4, pág. 16.1-68 y 16.1-69',
    'F10.3': 'AISC 360-22 §F10.3 (pandeo local de pata), ecs. F10-6 a F10-8, pág. 16.1-69 y 16.1-70',
    'G2': 'AISC 360-22 §G2.1 y §G2.2, ecs. G2-1 a G2-4, pág. 16.1-75 y 16.1-76',
    'G3': 'AISC 360-22 §G3 (ángulo sencillo), ec. G3-1, pág. 16.1-80',
    'G4': 'AISC 360-22 §G4 (HSS rectangulares y cajón), ec. G4-1 con Cv2 de §G2.2 (G2-9 a G2-11, kv = 5), pág. 16.1-80 y 16.1-77',
    'G6': 'AISC 360-22 §G6 (cortante en eje menor), pág. 16.1-81',
    'H1': 'AISC 360-22 §H1.1, ecs. H1-1a y H1-1b, pág. 16.1-82 y 16.1-83',
    'H2': 'AISC 360-22 §H2 (secciones no cubiertas por H1), ec. H2-1, pág. 16.1-84',
    'H3': 'AISC 360-22 §H3.1 (ecs. H3-1 a H3-5) y §H3.2 (ec. H3-6), pág. 16.1-85 a 16.1-87',
    'NTC_2_3_1': 'NTC-Acero CDMX 2020, §2.3.1 (relación de esbeltez K L/r), pág. 40',
    'NTC_2_3_2': 'NTC-Acero CDMX 2020, §2.3.2 (relaciones de esbeltez máximas), pág. 41',
    'NTC_3_2_1_1': 'NTC-Acero CDMX 2020, §3.2.1.1 y Tabla 3.2.1 (compresión), pág. 56 y 57',
    'NTC_3_2_1_2': 'NTC-Acero CDMX 2020, §3.2.1.2 y Tabla 3.2.2 (flexión), pág. 56 y 58',
    'NTC_3_2_5': 'NTC-Acero CDMX 2020, §3.2.5 (anchos efectivos; no aplicados), pág. 61',
    'NTC_4_4': 'NTC-Acero CDMX 2020, §4.4 (relaciones de esbeltez en tensión), pág. 65 y 66',
    'NTC_12_1_2': 'NTC-Acero CDMX 2020, Tabla 12.1.2 (ancho/grueso, ductilidad), pág. 212',
    'NTC_12_1_2_HSS': 'NTC-Acero CDMX 2020, Tabla 12.1.2 (paredes de HSS rectangulares), pág. 212',
    'NTC_3_2_1_HSS': 'NTC-Acero CDMX 2020, Tabla 3.2.1 (paredes de HSS, λr), pág. 58',
    'NTC_3_2_2_HSS': 'NTC-Acero CDMX 2020, Tabla 3.2.2 (patines y almas de HSS), pág. 59',
    'F7': 'AISC 360-22 §F7 (HSS cuadradas y rectangulares), ecs. F7-1 a F7-11, pág. 16.1-63 a 16.1-65',
    'NTC_12_1_3_1': 'NTC-Acero CDMX 2020, §12.1.3.1 (columnas dúctiles, KL/r ≤ 60), pág. 207',
    'NTC_CA_4_1_1': 'NTC Criterios y Acciones CDMX, §4.1.1 b), pág. 8',
    'NTC_CA_4_1_1_A': 'NTC Criterios y Acciones CDMX, §4.1.1 a), pág. 8',
    'NTC_11_4': 'NTC-Acero CDMX 2020, §11.4 (deflexiones: remite a las NTC de Criterios y Acciones), pp. 197 y 198',
    'NTC_11_6': 'NTC-Acero CDMX 2020, §11.6 (desplazamientos laterales: remite al Art. 149 del Reglamento y al Capítulo 1 '
                'de las NTC de Diseño por Sismo), pp. 201 y 202',
}


def _ref_html(k):
    """Linea de referencia (apartado y pagina) bajo el nombre de una verificacion."""
    return f'<br><span class="nota">{REF[k]}</span>'


def _refs_verif(nombre, familia):
    """Referencias de AISC para cada verificacion de la tabla (I y canal, o angulo LI)."""
    li = familia == 'LI'
    mapa = {
        'Tracción': ['D2', 'D3'] if li else ['D2'],
        'Compresión': ['E3', 'E5', 'E7'] if li else ['E3', 'E4', 'E7'],
        'Flexión eje mayor': ['F10', 'F10.3'] if li else ['F2', 'F3'],
        'Flexión eje menor': ['F10', 'F10.3'] if li else ['F6'],
        'Cortante eje mayor': ['G3'] if li else ['G2'],
        'Cortante eje menor': ['G3'] if li else ['G6'],
        'Interacción': ['H2'] if li else ['H1'],
    }
    if familia == 'HSS':
        mapa = {'Tracción': ['D2'], 'Compresión': ['E3', 'E7'], 'Flexión eje mayor': ['F7'], 'Flexión eje menor': ['F7'],
                'Cortante eje mayor': ['G4'], 'Cortante eje menor': ['G4'], 'Torsión': ['H3'], 'Interacción': ['H1', 'H3']}
    return ''.join(_ref_html(k) for k in mapa.get(nombre, []))


def _ref_local(v):
    """Referencia NTC de la esbeltez local: Tabla 3.2.1 en compresion, Tabla 3.2.2 en flexion."""
    if v.get('familia') == 'HSS':
        return _ref_html('NTC_3_2_1_HSS' if v['grupo'] in DA.COMPRESION_GRUPOS else 'NTC_3_2_2_HSS')
    return _ref_html('NTC_3_2_1_1' if v['grupo'] in DA.COMPRESION_GRUPOS else 'NTC_3_2_1_2')


def _nota_normativa():
    """Nota de la memoria: el diseno es AISC 360-22 ASD y las esbelteces se verifican con la NTC-Acero CDMX."""
    filas = [
        ('Esbeltez global en compresión (obligatorio)', REF['NTC_2_3_2'], 'KL/r ≤ 200',
         'COLUMNA, ARRIOSTRE, DIAGONAL'),
        ('Columna dúctil', REF['NTC_12_1_3_1'], 'KL/r ≤ 60', 'COLUMNA con DUCTILIDAD ALTA o MEDIA'),
        ('Esbeltez local en compresión: patín', REF['NTC_3_2_1_1'],
         'b/t ≤ 0.56·√(E/Fy) (I y canal); ala de ángulo b/t ≤ 0.45·√(E/Fy)', 'COLUMNA, ARRIOSTRE, DIAGONAL'),
        ('Esbeltez local en compresión: alma', REF['NTC_3_2_1_1'], 'h/t ≤ 1.49·√(E/Fy)',
         'COLUMNA, ARRIOSTRE, DIAGONAL'),
        ('Esbeltez local en flexión: patín', REF['NTC_3_2_1_2'],
         'b/t ≤ 1.00·√(E/Fy) (tipo 3); 0.38 (tipo 2); 0.30 (tipo 1)', 'VIGA'),
        ('Esbeltez local en flexión: alma', REF['NTC_3_2_1_2'],
         'h/t ≤ 5.70·√(E/Fy) (tipo 3); 3.76 (tipo 2); 2.45 (tipo 1)', 'VIGA'),
        ('Esbeltez local en flexión: ala de ángulo', REF['NTC_3_2_1_2'],
         'b/t ≤ 0.91·√(E/Fy) (tipo 3); 0.54 (tipo 2)', 'VIGA (LI)'),
        ('Tipo de sección en flexión', REF['NTC_3_2_1_2'],
         'tipo 1, 2 o 3 si λ ≤ λr; tipo 4 queda fuera del diseño', 'VIGA'),
        ('Límite de esbeltez en tensión', REF['NTC_2_3_2'] + '. Sin límite obligatorio: ' + REF['NTC_4_4'],
         'L/r ≤ 240 en miembros principales; L/r ≤ 300 en contraventeos y secundarios',
         'barras con tensión'),
        ('Ductilidad: patines y alas', REF['NTC_12_1_2'],
         'b/t ≤ 0.30·√(E/Fy) (alta) o 0.38·√(E/Fy) (media)', 'barras con DUCTILIDAD marcada'),
    ]
    cuerpo = ''.join(f'<tr><td>{a}</td><td>{b}</td><td>{c}</td><td>{d}</td></tr>' for a, b, c, d in filas)
    return ('<h2>Nota normativa: esbelteces con la NTC-Acero CDMX</h2>'
            '<p>Proyecto en México. El diseño de resistencia (tracción, compresión, flexión, cortante e interacción) '
            'se hace con AISC 360-22 (ASD). Las verificaciones de esbeltez local y global se hacen con la '
            'NTC-Acero CDMX (Gaceta Oficial, 7 de julio de 2020), que son obligatorias para este proyecto. Cada límite '
            'indica el apartado y la página del documento.</p>'
            '<table><tr><th>Verificación</th><th>Referencia (documento, apartado y página)</th><th>Límite</th>'
            '<th>Aplica a</th></tr>' + cuerpo + '</table>'
            '<ul class="nota">'
            '<li>Un elemento con λ &gt; λr se considera esbelto (sección tipo 4) y su perfil no se usa. La NTC permite '
            'anchos efectivos (3.2.5); no se aplican, por criterio conservador.</li>'
            '<li>λ = b/t en patines y alas (b = B/2 en patines de I; b = B en canales); h/t en almas, con h = H − 2t.</li>'
            '<li>Tensión: la NTC 4.4 no fija límite obligatorio; los valores 240 y 300 de la NTC 2.3.2 se reportan '
            'como límite de referencia y su cumplimiento es informativo.</li>'
            '</ul>')


_ETIQUETA_CLASE = {'gravitacional': 'Solo D y L', 'sismo': 'Incluye E (sismo)',
                   'viento': 'Incluye W (viento: no aplica al proyecto)', 'otra': 'TIPO no reconocido en ESTADOS'}


def _distorsiones(dib, casos, id_barra):
    """Distorsion horizontal de entrepiso (NTC-CA 4.1.1 b): desplazamiento horizontal relativo entre los extremos de cada
    columna (barra casi vertical) dividido entre su altura. dib.nodos en m; res['nodos'] en cm.
    Devuelve [(combinacion, distorsion_max, columna_que_controla, h_cm, delta_mm)]."""
    cols = []
    for i, b in enumerate(dib.barras):
        p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
        largo = float(np.linalg.norm(p2 - p1))
        dz = abs(p2[2] - p1[2])
        if largo > 0 and dz >= 0.9 * largo:
            bot, top = (b['n1'], b['n2']) if p1[2] < p2[2] else (b['n2'], b['n1'])
            cols.append((id_barra.get(i, i), bot, top, dz * 100.0))
    filas = []
    for nombre, tipo, res in casos:
        if tipo != 'COMBINACION' or not cols:
            continue
        mejor = None
        for idb, bot, top, h_cm in cols:
            ub = np.array(res['nodos'][bot][:2], float)
            ut = np.array(res['nodos'][top][:2], float)
            d_cm = float(np.linalg.norm(ut - ub))
            if mejor is None or d_cm / h_cm > mejor[1]:
                mejor = (nombre, d_cm / h_cm, idb, h_cm, d_cm * 10.0)
        filas.append(mejor)
    return filas


def _cumple_html(ok):
    return f'<span class="{"cumple" if ok else "nocumple"}">{"CUMPLE" if ok else "NO CUMPLE"}</span>'


def _distorsion_texto(rel):
    return '0' if rel <= 0 else f'1 / {1.0 / rel:,.0f}'


def _columnas(dib, id_barra):
    """Columnas (barras casi verticales): [(id, nodo_inferior, nodo_superior, h_cm)]."""
    cols = []
    for i, b in enumerate(dib.barras):
        p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
        largo = float(np.linalg.norm(p2 - p1))
        dz = abs(p2[2] - p1[2])
        if largo > 0 and dz >= 0.9 * largo:
            bot, top = (b['n1'], b['n2']) if p1[2] < p2[2] else (b['n2'], b['n1'])
            cols.append((id_barra.get(i, i), bot, top, dz * 100.0))
    return cols


def _rel(res, bot, top):
    """Desplazamiento horizontal relativo (cm, vector XY) entre los nodos superior e inferior de una columna."""
    return np.array(res['nodos'][top][:2], float) - np.array(res['nodos'][bot][:2], float)


def _dcol(dG, dE, fac, h_cm):
    """Desplazamiento de entrepiso / h de una columna: gravedad dG mas +-fac*dE (la accion sismica es reversible)."""
    return max(float(np.linalg.norm(dG + sg * fac * dE)) for sg in (1.0, -1.0)) / h_cm


def _periodo(dib, cargas, estados, sismos, D_estados, g=9.81):
    """Periodo fundamental aproximado, NTC 6.4.1: T = 2*pi*sqrt(sum(W X^2) / (g*sum(F X))), sumando todos los niveles.
    W de cada nivel: carga D nodal mas la mitad del peso de cada barra en sus extremos. F: fuerza horizontal del nivel
    en la direccion de la fuerza horizontal total del estado sismico. X: desplazamiento medio del nivel en esa direccion (m)."""
    z_niv = sorted({round(float(p[2]), 3) for p in dib.nodos})[1:]
    W_nodo = np.zeros(len(dib.nodos))
    for n in D_estados:
        for idx, v in cargas.get(n, {}).items():
            W_nodo[idx] += -float(v[2])
    for b in dib.barras:
        m = (G.obtener_perfil(b['perfil']) or {}).get('masa') or 0.0
        L = float(np.linalg.norm(np.array(dib.nodos[b['n1']], float) - np.array(dib.nodos[b['n2']], float)))
        W_nodo[b['n1']] += m * L / 2.0
        W_nodo[b['n2']] += m * L / 2.0
    resultado = {}
    for e in sismos:
        cg = cargas.get(e, {})
        H = sum((np.array(v[:2], float) for v in cg.values()), np.zeros(2))
        if np.linalg.norm(H) == 0.0:
            resultado[e] = None
            continue
        Hh = H / np.linalg.norm(H)
        num = den = 0.0
        for z in z_niv:
            nodos_nivel = [i for i, p in enumerate(dib.nodos) if round(float(p[2]), 3) == z]
            F_i = sum((float(np.array(cg[i][:2], float) @ Hh) for i in nodos_nivel if i in cg), 0.0)
            X_i = float(np.mean([np.array(estados[e]['nodos'][i][:2], float) @ Hh for i in nodos_nivel])) / 100.0
            num += float(W_nodo[nodos_nivel].sum()) * X_i ** 2
            den += F_i * X_i
        resultado[e] = 2.0 * np.pi * np.sqrt(num / (g * den)) if den > 0 else None
    return resultado


def _seccion_sismo(dib, casos, id_barra, proyecto):
    """6.3 Distorsion por sismo, metodo estatico (NTC Diseno por Sismo 2023). Todos los valores vienen de la
    configuracion de la planilla (hojas PARAMETROS y NORMA, ver 3.1)."""
    cf = proyecto['conf']
    tipos = getattr(dib, 'estados_tipo', {}) or {}
    estados = {n: res for n, t, res in casos if t == 'ESTADO'}
    sismos = [n for n in estados if tipos.get(n) == 'E']
    if not sismos:
        return ('<h3>6.3 Distorsión por sismo</h3><p class="nota">La planilla no tiene estados sísmicos (TIPO E): '
                'no hay distorsión sísmica que revisar.</p>')
    gravedad = [n for n in estados if tipos.get(n) in ('D', 'L')]
    D_estados = [n for n in estados if tipos.get(n) == 'D']
    cargas = getattr(dib, 'cargas', {}) or {}
    cols = _columnas(dib, id_barra)
    periodos = _periodo(dib, cargas, estados, sismos, D_estados, g=cf['g'])
    T_list = [t for t in periodos.values() if t is not None]
    # DECISION D8 (abierta): k2 con el periodo menor de los estados sismicos (conservador); T segun 6.4.1 con las fuerzas del estado.
    T_ctrl = min(T_list) if T_list else None                  # el periodo menor da el k2 mayor (conservador)
    k2 = 0.0 if (T_ctrl is None or T_ctrl >= cf['ta']) else cf['k2_coef'] * (1.0 - (T_ctrl / cf['ta']) ** cf['k2_exp'])
    R = cf['k1'] * cf['R0'] + k2
    # DECISION D5 (abierta): la carga de la planilla se toma como diseno SV reducido por Q'R'. Distorsion elastica = f * planilla.
    f_el = cf['qp'] * R                                       # LDNE y OI
    f_sv = cf['Q'] * R                                        # Seguridad de Vida (multiplicador QR' de 1.7)
    ks, g_ldne, g_oi, g_sv = cf['ks'], cf['gLDNE'], cf['gOI'], cf['gSV']
    filas, drift_piso = [], {}
    for idb, bot, top, h_cm in cols:
        dG = sum((_rel(estados[g], bot, top) for g in gravedad), np.zeros(2))
        mejor = None
        for e in sismos:
            db = _dcol(dG, _rel(estados[e], bot, top), f_el, h_cm)
            if mejor is None or db > mejor[0]:
                mejor = (db, e)
        db, e = mejor
        dl = max(_dcol(dG, _rel(estados[e2], bot, top), ks * f_el, h_cm) for e2 in sismos)
        dsv = max(_dcol(dG, _rel(estados[e2], bot, top), f_sv, h_cm) for e2 in sismos)
        zb = round(float(dib.nodos[bot][2]), 3)
        drift_piso[zb] = max(drift_piso.get(zb, 0.0), db, dsv)
        marca = lambda ok, g: f'{_cumple_html(ok)} <span class="nota">(γ {g:.4f})</span>'
        filas.append(
            f'<tr><td class="num">{idb}</td><td>{html.escape(str(e))}</td><td class="num">{_v(h_cm, 1)}</td>'
            f'<td class="num">{_v(db * h_cm * 10.0, 2)}</td><td class="num">{"0" if db <= 0 else f"1 / {1.0 / db:,.0f}"}</td>'
            f'<td>{marca(ks * db <= g_ldne, g_ldne)}<br><span class="nota">Ks·δ/h = {ks * db:.5f}</span></td>'
            f'<td>{marca(db <= g_oi, g_oi)}</td><td>{marca(dsv <= g_sv, g_sv)}</td></tr>')
    tabla_ldne = ('<table><tr><th>Columna</th><th>Estado sísmico que controla</th><th>h (cm)</th>'
                  '<th>δ elástica (mm)</th><th>δ/h</th><th>LDNE</th><th>Ocupación Inmediata</th>'
                  '<th>Seguridad de Vida</th></tr>' + ''.join(filas) + '</table>')
    filas_t = ''.join(
        f'<tr><td>{html.escape(str(e))}</td><td class="num">{"—" if periodos[e] is None else f"{periodos[e]:.3f}"}</td></tr>'
        for e in sismos)
    tabla_t = ('<table><tr><th>Estado sísmico</th><th>T aproximado (s), §6.4.1</th></tr>' + filas_t + '</table>')

    zs = sorted({round(float(p[2]), 3) for p in dib.nodos})
    filas2 = []
    for k in range(len(zs) - 1):
        zb, zt = zs[k], zs[k + 1]
        arriba = {i for i, p in enumerate(dib.nodos) if round(float(p[2]), 3) >= zt - 1e-6}
        V = max((float(np.linalg.norm(sum((np.array(v[:2], float) for idx, v in cargas.get(e, {}).items()
                                           if idx in arriba), np.zeros(2)))) for e in sismos), default=0.0)
        W = sum(-float(v[2]) for n in D_estados for idx, v in cargas.get(n, {}).items() if idx in arriba)
        for b in dib.barras:
            if b['n1'] in arriba and b['n2'] in arriba:
                pb = np.array(dib.nodos[b['n1']], float) - np.array(dib.nodos[b['n2']], float)
                masa = (G.obtener_perfil(b['perfil']) or {}).get('masa') or 0.0
                W += masa * float(np.linalg.norm(pb))                          # kgf: kg/m por m
        # DECISION D9 (abierta): V = cortante del sismo; W = carga D sin factor mas peso propio encima; se compara con la mayor distorsion elastica o SV.
        lim = cf['seg2'] * V / W if W > 0 else None
        dmax_piso = drift_piso.get(zb)
        if lim is None or dmax_piso is None:
            estado = '<span class="nota">sin datos</span>'
        else:
            estado = ('<span class="cumple">se pueden despreciar</span>' if dmax_piso <= lim
                      else '<span class="nocumple">requiere análisis de segundo orden</span>')
        filas2.append(
            f'<tr><td>{zb:.2f} a {zt:.2f}</td><td class="num">{_v(V, 1)}</td><td class="num">{_v(W, 1)}</td>'
            f'<td class="num">{"—" if lim is None else f"{lim:.4f}"}</td>'
            f'<td class="num">{"—" if dmax_piso is None else f"{dmax_piso:.4f}"}</td><td>{estado}</td></tr>')
    tabla_2 = ('<table><tr><th>Entrepiso z (m)</th><th>V sismo (kgf)</th><th>W encima (kgf, sin factor)</th>'
               f'<th>{cf["seg2"]:g}·V/W</th><th>δ/h máx (elástica o SV)</th><th>Efectos de 2º orden (2.4.1)</th></tr>'
               + ''.join(filas2) + '</table>') if filas2 else ''

    deform = ('elementos no estructurales deformables o separados: límite 0.004'
              if cf['deformables'] else 'elementos no estructurales no deformables: límite 0.002')
    return (f'<h3>6.3 Distorsión por sismo, método estático (NTC Diseño por Sismo, §1.7, §2.4, §3.2, §3.3, §6.3, §6.4)</h3>'
            f'<p><b>Sistema</b>: {html.escape(cf["sistema"])}. Tabla 4.3.2 (pág. 53): Q = {cf["Q"]:g}, γ<sub>SV</sub> = '
            f'{g_sv:.3f}, γ<sub>OI</sub> = {g_oi:.4f}. R<sub>0</sub> = {cf["R0"]:g} (§3.3.1a).</p>'
            f'<p><b>Factor de sobre-resistencia</b>: k<sub>1</sub> = {cf["k1"]:g} (tres o más crujías resistentes en las dos '
            f'direcciones: {"SÍ" if cf["crujias_ambas"] else "NO"}; §3.3.1a). k<sub>2</sub> = 0.5(1 − (T/Ta)²) si T &lt; Ta, '
            f'0 si T ≥ Ta (§3.3.1b) con T = {"—" if T_ctrl is None else f"{T_ctrl:.3f}"} s y Ta = {cf["ta"]:g} s '
            f'→ k<sub>2</sub> = {k2:.4f}. R = k<sub>1</sub>·R<sub>0</sub> + k<sub>2</sub> = {R:.4f}.</p>'
            f'<p><b>Factores de reducción</b>: Q′ = 1 + (Q − 1)/k = 1 + ({cf["Q"]:g} − 1)/{cf["k_sasid"]:g} = {cf["qp"]:.4f} '
            f'(§3.2, eq. 3.2.1, rama Ta–Tb; k del SASID). Para LDNE y OI: f = Q′·R = {f_el:.4f}. Para Seguridad de Vida: '
            f'f = Q·R = {f_sv:.4f} (§1.7).</p>'
            f'<p><b>Sismo de diseño</b>: el estado sísmico de la planilla es la fuerza de diseño V<sub>0</sub> = c·W<sub>0</sub>/(Q′R′) '
            f'(§6.3, eq. 6.3.1) con c = {cf["c_sasid"]:g}. Por eso la distorsión elástica, sin reducir, es f veces la de la '
            f'planilla.</p>'
            f'<p><b>Limitación de Daño No Estructural (LDNE)</b>: §1.7, pág. 26 y tabla 3.1.1. Usa la intensidad frecuente: '
            f'{html.escape(cf["ks_txt"])}, y gravedad D y L con factor 1, sin factores de carga. Límite γ<sub>LDNE</sub> = '
            f'{g_ldne:.3f} ({deform}).</p>'
            f'{tabla_ldne}'
            f'<p><b>Periodo aproximado (§6.4.1)</b>: T = 2π√(ΣW·X²/(g·ΣF·X)), con fuerzas y desplazamientos del estado '
            f'sísmico, g = {cf["g"]:g} m/s². Se usa el periodo menor para k<sub>2</sub>.</p>'
            f'{tabla_t}'
            f'<p><b>Efectos de segundo orden</b>: §2.4, pág. 32. Se pueden despreciar en un entrepiso si la distorsión no '
            f'excede {cf["seg2"]:g}·V<sub>i</sub>/W<sub>pi</sub> (eq. 2.4.1), con V la cortante del sismo y W el peso encima del '
            f'entrepiso sin factor de carga. Se compara con la mayor distorsión elástica o de SV del entrepiso.</p>'
            f'{tabla_2}')


def _seccion_desplazamientos(dib, casos, id_de_indice, id_barra, clase, proyecto):
    """Seccion 6: desplazamientos por combinacion y deformada de la que controla (6.1); distorsion de entrepiso con
    NTC-CA 4.1.1 b (6.2); distorsion por sismo (6.3) y flecha vertical de vigas (6.4). Valores de la hoja PARAMETROS y NORMA."""
    cf = proyecto['conf']
    tabla, mejor = _desplazamientos(dib, casos, id_de_indice)
    if tabla.empty:
        return '<p>Sin combinaciones de servicio: no hay desplazamientos que reportar.</p>'
    filas = ''.join(
        f'<tr><td>{html.escape(str(r.combinacion))}</td><td class="num">{_v(r.ux, 2)}</td>'
        f'<td class="num">{_v(r.uy, 2)}</td><td class="num">{_v(r.uz, 2)}</td>'
        f'<td class="num">{_v(r.resultante, 2)}</td><td class="num">{r.nodo}</td></tr>'
        for r in tabla.itertuples())
    fila_max, res_ctrl = mejor
    png, escala = _png_deformada(dib, res_ctrl, f'Deformada: combinación que controla {fila_max["combinacion"]}')

    # DECISION D3 (abierta): h/250 sin elementos frageles unidos (NTC-CA 4.1.1 b); distorsion por columna que controla; solo gravitacionales.
    lim_h = cf['lim_h']
    if cf['fragiles']:
        criterio = (f'Límite h/{lim_h:g}: hay elementos no estructurales frágiles unidos a la estructura '
                    f'({REF["NTC_CA_4_1_1"]}).')
    else:
        criterio = (f'Límite h/{lim_h:g}: no hay elementos no estructurales frágiles unidos a la estructura, por lo que '
                    f'aplica el caso de "en otros casos" de {REF["NTC_CA_4_1_1"]}.')
    filas_d = []
    for nombre, rel, idb, h_cm, delta_mm in _distorsiones(dib, casos, id_barra):
        k = clase.get(nombre, 'otra')
        if k == 'gravitacional':
            celda = _cumple_html(rel <= 1.0 / lim_h)
        else:
            celda = '<span class="nota">no evaluada (ver 6.3)</span>'
        filas_d.append(
            f'<tr><td>{html.escape(str(nombre))}</td><td>{_ETIQUETA_CLASE.get(k, k)}</td>'
            f'<td class="num">{_distorsion_texto(rel)}</td><td class="num">{idb}</td>'
            f'<td class="num">{_v(h_cm, 1)}</td><td class="num">{_v(delta_mm, 2)}</td>'
            f'<td class="num">{_v(h_cm / lim_h * 10.0, 2)}</td><td>{celda}</td></tr>')
    if filas_d:
        tabla_dist = ('<table><tr><th>Combinación</th><th>Acciones</th><th>Distorsión máx. δ/h</th><th>Columna</th>'
                      '<th>h (cm)</th><th>δ (mm)</th>' f'<th>Límite h/{lim_h:g} (mm)</th><th>Estado</th></tr>'
                      + ''.join(filas_d) + '</table>')
    else:
        tabla_dist = '<p class="nota">No hay columnas verticales en el modelo para revisar la distorsión.</p>'

    # DECISION D1 (abierta): L/300 justificado con AISC 360-22 Comentario L2 (pag. 16.1-543); la NTC no fija numero. Alternativas: L/240, L/360.
    divisor = cf['flecha_divisor']
    filas_f = []
    for i, r in sorted((getattr(dib, 'flechas', {}) or {}).items()):
        lim = r['L_cm'] * 10.0 / divisor
        filas_f.append(
            f'<tr><td class="num">{id_barra.get(i, i)}</td><td>{html.escape(str(dib.barras[i]["perfil"]))}</td>'
            f'<td class="num">{_v(r["L_cm"], 1)}</td><td class="num">{_v(r["max_mm"], 2)}</td>'
            f'<td>{html.escape(str(r["comb"]))}</td><td class="num">{r["xi"]:.2f}</td>'
            f'<td class="num">{_v(lim, 2)}</td><td>{_cumple_html(r["max_mm"] <= lim)}</td></tr>')
    if filas_f:
        tabla_f = ('<table><tr><th>Barra</th><th>Perfil</th><th>L (cm)</th><th>Flecha máx (mm)</th>'
                   '<th>Combinación que controla</th><th>Posición x/L</th>'
                   f'<th>Límite L/{divisor:g} (mm)</th><th>Estado</th></tr>' + ''.join(filas_f) + '</table>')
    else:
        tabla_f = '<p class="nota">No hay vigas horizontales en el modelo para revisar la flecha.</p>'

    return (f'<h3>6.1 Máximos por combinación</h3>'
            f'<table><tr><th>Combinación</th><th>|UX| máx (mm)</th><th>|UY| máx (mm)</th><th>|UZ| máx (mm)</th>'
            f'<th>Desplazamiento resultante máx (mm)</th><th>Nodo</th></tr>{filas}</table>'
            f'<p><b>Desplazamiento máximo</b>: {_v(fila_max["resultante"], 2)} mm en el nodo {fila_max["nodo"]}, '
            f'combinación <b>{html.escape(str(fila_max["combinacion"]))}</b> (controla).</p>'
            f'<img src="data:image/png;base64,{png}" alt="Deformada" style="max-width:100%">'
            f'<p class="nota">Deformada de la combinación que controla, escala ×{escala:.3g} (el máximo mide 10 % '
            f'de la dimensión del modelo). Gris: sin carga; azul: deformada.</p>'
            f'<h3>6.2 Distorsión horizontal de entrepiso</h3>'
            f'<p>{REF["NTC_CA_4_1_1"]}: el desplazamiento horizontal relativo entre dos niveles sucesivos no debe exceder '
            f'la altura del entrepiso entre 500 (con elementos no estructurales frágiles unidos) o entre 250 (en otros casos). '
            f'Se revisa cada columna: δ es el desplazamiento horizontal relativo entre sus extremos y h su altura; se reporta '
            f'la columna que controla.</p>'
            f'<p><b>Criterio</b>: {criterio}</p>'
            f'{tabla_dist}'
            f'<p class="nota">Solo las combinaciones gravitacionales (D y L) se comparan con este límite. Las combinaciones '
            f'con sismo (E) se revisan en 6.3.</p>'
            f'{_seccion_sismo(dib, casos, id_barra, proyecto)}'
            f'<h3>6.4 Flecha vertical de vigas</h3>'
            f'<p><b>Límite L/{divisor:g}</b>. Justificación: AISC 360-22, Comentario L2 (pág. 16.1-543): la flecha de '
            f'alrededor de 1/300 del claro es visible y puede causar daño arquitectónico; el comentario indica que los límites '
            f'dependen de la estructura (1/360 para pisos y 1/240 para cubiertas son los usuales). La NTC-Acero no fija un '
            f'número: §11.4 (pp. 197 y 198) remite a las NTC de Criterios y Acciones, y {REF["NTC_CA_4_1_1_A"]} remite a las '
            f'normas del material.</p>'
            f'<p>La flecha es la desviación vertical máxima de la viga respecto a la cuerda entre sus extremos, en las '
            f'combinaciones gravitacionales (D y L), sin el movimiento de cuerpo rígido de los apoyos. Se calcula con la '
            f'solución exacta de Euler-Bernoulli entre extremos, a partir de los giros de extremo del modelo y del peso '
            f'propio; no incluye la deformación por cortante.</p>'
            f'{tabla_f}')


def _encabezado_doc(proyecto, dib):
    meta = [('Proyecto', proyecto.get('nombre', '(completar)')),
            ('Elemento / estructura', proyecto.get('elemento', '(completar)')),
            ('Planilla de datos', proyecto.get('archivo', 'planilla_con_diseno.xlsx')),
            ('Fecha', proyecto.get('fecha', '')),
            ('Versión', proyecto.get('version', '(completar)')),
            ('Modelo', f'{len(dib.nodos)} nodos y {len(dib.barras)} barras'),
            ('Método', 'Diseño de resistencia: ASD, ANSI/AISC 360-22. Esbelteces local y global: NTC-Acero CDMX '
                       '(Gaceta Oficial, 7 de julio de 2020).'),
            ('Unidades', 'fuerzas kgf; momentos kgf·cm; longitudes cm; esfuerzos kgf/cm²')]
    filas = ''.join(f'<tr><td><b>{a}</b></td><td>{html.escape(str(b))}</td></tr>' for a, b in meta)
    revision = [('Elaboró', 'elaboro'), ('Revisó', 'reviso'), ('Aprobó', 'aprobo')]
    rev = ''.join(f'<tr><td>{etq}</td><td>{html.escape(str(proyecto.get(clave, "")))}</td><td></td><td></td></tr>'
                  for etq, clave in revision)
    return (f'<table class="meta">{filas}</table>'
            f'<table><tr><th>Cuadro de revisión</th><th>Nombre</th><th>Firma</th><th>Fecha</th></tr>{rev}</table>')


def _criterios_html():
    items = [
        'Diseño de resistencia: AISC 360-22 (ASD). Factores Ω: tracción 1.67 (fluencia) y 2.00 (fractura) (D2); '
        'compresión y flexión 1.67 (E1, F1); cortante 1.67, o 1.50 en I laminada (G1, G2.1).',
        'Demandas: combinaciones de servicio de COMBINACIONES (ASD), sin factores adicionales. Análisis lineal de '
        'primer orden: no incluye efectos P-Δ. Cada barra se revisa con todas sus combinaciones.',
        'Una sola sección por grupo (COLUMNA, VIGA, ARRIOSTRE, DIAGONAL): todas las barras de un grupo usan el mismo '
        'perfil. Cada barra se revisa con sus propias demandas.',
        'Criterio de aceptación: FU ≤ 1.0 en todas las verificaciones; λ ≤ λr (NTC 3.2); KL/r ≤ 200 (NTC 2.3.2); '
        'ductilidad si está marcada (NTC 12.1.2).',
        'Elementos con λ &gt; λr (sección tipo 4) quedan fuera de la lista de perfiles. La NTC permite anchos efectivos '
        '(3.2.5); no se aplican, por criterio conservador.',
        'Perfiles macizos (CS, OS, SOL) eliminados de la base; LM no se incluye.',
        'Fu: de la hoja MATERIALES de la planilla. Un material sin Fu deja sus barras como NO REVISADO.',
        f'Ángulos (LI): E5 solo si LONGITUDES marca las cuatro condiciones ({REF["E5"]}). Si no, K·L/rz y H2 '
        f'({REF["H2"]}). Tracción con Ae = U·Ag, U de D3 ({REF["D3"]}): una pata U = 1 − x̄/l; dos patas U = 1.0.',
        'Estimaciones: IR con Zx, Zy o Cw faltantes se estiman por geometría; LI con propiedades geométricas. Cada '
        'barra lista sus supuestos.',
        'Ductilidad: marcada por barra en BARRAS (ALTA o MEDIA). Vacío = no se verifica la Tabla 12.1.2 ni KL/r ≤ 60.',
        f'Interacción: H1-1 para I y canal ({REF["H1"]}); H2-1 para ángulos ({REF["H2"]}), con el momento resultante '
        'sobre cada eje principal (criterio conservador).',
    ]
    return '<ul>' + ''.join(f'<li>{t}</li>' for t in items) + '</ul>'


def _parametros_html(cf):
    """3.1 Parametros de diseno y de norma usados por el calculo, con su referencia (hojas PARAMETROS y NORMA)."""
    filas_p = ''.join(
        f'<tr><td>{html.escape(str(c))}</td><td>{html.escape(str(k))}</td><td class="num">{html.escape(str(v))}</td>'
        f'<td>{html.escape(str(u))}</td><td>{html.escape(str(r))}</td></tr>'
        for (c, k, v, u, r) in cf.get('par_filas', []))
    filas_n = ''.join(
        f'<tr><td>{html.escape(str(k))}</td><td class="num">{v:g}</td><td>{html.escape(str(u))}</td>'
        f'<td>{html.escape(str(r))}</td></tr>'
        for (k, v, u, r) in cf.get('nor_filas', []))
    resultantes = (
        '<table><tr><th>Valor resultante</th><th>Valor</th><th>Cómo se obtiene</th></tr>'
        f'<tr><td>Sistema sísmico</td><td>{html.escape(cf["sistema"])}</td><td>NORMA, tabla 4.3.2</td></tr>'
        f'<tr><td>Q</td><td class="num">{cf["Q"]:g}</td><td>Tabla 4.3.2</td></tr>'
        f'<tr><td>R0</td><td class="num">{cf["R0"]:g}</td><td>§3.3.1a (2.0 con Q ≥ 3; 1.75 con Q &lt; 3)</td></tr>'
        f'<tr><td>k1</td><td class="num">{cf["k1"]:g}</td><td>§3.3.1a, según crujías: {"SÍ" if cf["crujias_ambas"] else "NO"}</td></tr>'
        f'<tr><td>Q′</td><td class="num">{cf["qp"]:.4f}</td><td>1 + (Q − 1)/k, con k = {cf["k_sasid"]:g} (SASID)</td></tr>'
        f'<tr><td>Ks (LDNE)</td><td class="num">{cf["ks"]:.4f}</td><td>{html.escape(cf["ks_txt"])}</td></tr>'
        f'<tr><td>γ LDNE</td><td class="num">{cf["gLDNE"]:.3f}</td><td>§1.7: 0.004 con elementos deformables o separados; '
        f'0.002 en otro caso</td></tr>'
        f'<tr><td>Límite de distorsión NTC-CA</td><td class="num">h/{cf["lim_h"]:g}</td><td>§4.1.1 b): 500 con elementos frágiles '
        f'unidos; 250 en otro caso</td></tr>'
        f'<tr><td>Límite de flecha de vigas</td><td class="num">L/{cf["flecha_divisor"]:g}</td><td>AISC 360-22, Comentario L2</td></tr>'
        f'<tr><td>Kz (pandeo torsional)</td><td class="num">{cf["kz"]:g}</td><td>AISC 360-22, E4: elegido en PARAMETROS</td></tr>'
        f'<tr><td>Viento en el diseño</td><td>{html.escape(cf["viento"])}</td><td>Esta versión no considera viento</td></tr>'
        '</table>')
    return ('<h3>3.1 Parámetros de diseño y de norma</h3>'
            '<p>Valores que usa el cálculo. Los elige usted en la hoja PARAMETROS del Excel (listas desplegables e '
            'ingresos); los valores de norma están en la hoja NORMA, con su referencia. El código no trae valores de norma '
            'ni supuestos.</p>'
            '<h4>Elecciones e ingresos (PARAMETROS)</h4>'
            '<table><tr><th>Concepto</th><th>Clave</th><th>Valor elegido</th><th>Unidad</th><th>Referencia</th></tr>'
            + filas_p + '</table>'
            '<h4>Valores de norma (NORMA)</h4>'
            '<table><tr><th>Clave</th><th>Valor</th><th>Unidad</th><th>Referencia</th></tr>' + filas_n + '</table>'
            '<h4>Valores resultantes</h4>' + resultantes)


def _datos_entrada_html(dib, materiales_filas, id_barra, lon, mat, equilibrio):
    filas_mat = []
    for f in materiales_filas:
        fu = f[3] if len(f) > 3 else None
        fu_txt = 'sin Fu (barras NO REVISADO)' if fu is None or (isinstance(fu, float) and math.isnan(fu)) \
            else f'{_v(fu, 0)} kgf/cm²'
        filas_mat.append(f'<tr><td>{html.escape(str(f[0]))}</td><td class="num">{_v(f[2], 0)} kgf/cm²</td>'
                         f'<td class="num">{fu_txt}</td><td class="num">{_v(f[4], 0)} kgf/cm²</td>'
                         f'<td class="num">{_v(f[5], 2)}</td></tr>')
    materiales = ('<table><tr><th>Material</th><th>Fy</th><th>Fu</th><th>E</th><th>ν</th></tr>'
                  + ''.join(filas_mat) + '</table>')
    filas_g = []
    for g in DA.GRUPOS_ELEMENTO:
        idx = [i for i, b in enumerate(dib.barras) if b['elemento'] == g]
        if not idx:
            continue
        ents = [DA._entrada_barra(dib, i, id_barra, lon, mat) for i in idx]
        perfiles = sorted({dib.barras[i]['perfil'] for i in idx})
        mats = sorted({str(dib.barras[i].get('material') or '') for i in idx})
        duct = sorted({e['nivel'] or '—' for e in ents})
        filas_g.append(
            f'<tr><td>{g}</td><td>{len(idx)}</td><td>{html.escape(", ".join(perfiles))}</td>'
            f'<td>{html.escape(", ".join(mats))}</td>'
            f'<td class="num">{_v(max(e["lx"] for e in ents) * 100, 1)}</td>'
            f'<td class="num">{_v(max(e["ly"] for e in ents) * 100, 1)}</td>'
            f'<td class="num">{_v(max(e["lb"] for e in ents) * 100, 1)}</td>'
            f'<td class="num">{_v(max(e["kx"] for e in ents), 2)} / {_v(max(e["ky"] for e in ents), 2)}</td>'
            f'<td class="num">{_v(max(e["cb"] for e in ents), 2)}</td><td>{html.escape(", ".join(duct))}</td></tr>')
    grupos = ('<table><tr><th>Grupo</th><th>Barras</th><th>Perfil (una sección)</th><th>Material</th>'
              '<th>LX máx (cm)</th><th>LY máx (cm)</th><th>LB máx (cm)</th><th>KX / KY</th><th>Cb</th>'
              '<th>Ductilidad</th></tr>' + ''.join(filas_g) + '</table>')
    combs = getattr(dib, 'combinaciones', {}) or {}
    cls = getattr(dib, 'clase_combinacion', {}) or {}
    filas_c = ''.join(
        f'<tr><td>{html.escape(str(n))}</td><td>' + ' + '.join(f'{fac:g}·{html.escape(str(e))}'
                                                               for e, fac in fac_d.items()) + '</td></tr>'
        for n, fac_d in combs.items() if cls.get(n) != 'viento')
    combinaciones = ('<table><tr><th>Combinación</th><th>Factores sobre los estados de carga</th></tr>'
                     + (filas_c or '<tr><td colspan="2">Sin definición de combinaciones en la planilla.</td></tr>')
                     + '</table>')
    en_diseno = [str(n) for n in combs if cls.get(n) != 'viento']
    combinaciones += ('<p class="nota"><b>Combinaciones del diseño</b>: ' + (', '.join(en_diseno) or 'ninguna')
                      + '. El viento no se considera en esta versión.</p>')
    estados = ', '.join(html.escape(str(n)) for n, _ in (equilibrio or []))
    return (f'<h3>Materiales</h3>{materiales}'
            f'<h3>Perfil, longitudes y ductilidad por grupo</h3>{grupos}'
            f'<h3>Combinaciones de servicio (ASD)</h3>{combinaciones}'
            f'<h3>Estados de carga</h3><p>{estados or "—"}</p>')


def _modelo_html(dib, equilibrio):
    filas = ''.join(f'<tr><td>{html.escape(str(n))}</td><td class="num">{r:.3g}</td>'
                    f'<td>≈ 0 (equilibrio)</td></tr>' for n, r in (equilibrio or []))
    tabla = ('<table><tr><th>Estado de carga</th><th>Residuo de fuerzas (kgf)</th><th>Criterio</th></tr>'
             + (filas or '<tr><td colspan="3">Sin datos de equilibrio.</td></tr>') + '</table>')
    return (f'<p>El modelo tiene {len(dib.nodos)} nodos y {len(dib.barras)} barras. Se resuelve con OpenSees '
            f'(openseespy) como marco 3D de elementos elásticos lineales (elasticBeamColumn). Cada estado de carga se '
            f'resuelve por separado y las combinaciones de servicio se obtienen por superposición lineal de los '
            f'estados. El peso propio de la sección se aplica cuando ESTADOS lo indica (masa Gerdau). Apoyos y rótulas '
            f'según NODOS y CONEXIONES. Es un análisis de primer orden: no incluye efectos de segundo orden.</p>'
            f'<h3>Verificación de equilibrio</h3>{tabla}'
            f'<p class="nota">El residuo de equilibrio por estado debe ser prácticamente cero; si no lo es, revise la '
            f'planilla antes de usar el diseño.</p>')


def _pendientes_html(evals, materiales_filas, proyecto, dib=None):
    items = []
    no_rev = {}
    for v in evals:
        if v['estado'] == 'NO REVISADO':
            no_rev.setdefault(v.get('motivo', ''), []).append(str(v['id']))
    for motivo, ids in no_rev.items():
        items.append(f'Barras NO REVISADO ({", ".join(ids)}): {html.escape(motivo)}.')
    fuera = [str(v['id']) for v in evals if v['estado'] == 'FUERA DE ALCANCE']
    if fuera:
        items.append(f'Barras fuera del diseño por familia ({", ".join(fuera)}): LM (no incluido).')
    sin_fu = [str(f[0]) for f in materiales_filas
              if len(f) > 3 and (f[3] is None or (isinstance(f[3], float) and math.isnan(f[3])))]
    if sin_fu:
        items.append(f'Material sin Fu en la hoja MATERIALES: {", ".join(sin_fu)}. Cárguelo en la planilla.')
    li_no = [str(v['id']) for v in evals if v.get('familia') == 'LI' and 'res' in v and not v.get('e5_b')]
    if li_no:
        items.append(f'Ángulos sin E5 verificado ({", ".join(li_no)}): se usa K·L/rz y H2. Marque las cuatro '
                     f'condiciones de E5 en LONGITUDES si las cumplen.')
    sin_duct = [str(v['id']) for v in evals if 'res' in v and not v.get('nivel')]
    if sin_duct:
        items.append(f'Barras sin DUCTILIDAD marcada en BARRAS ({len(sin_duct)}): no se verifica la Tabla 12.1.2 '
                     f'de la NTC ni la columna dúctil.')
    if dib is not None and not any(t == 'E' for t in (getattr(dib, 'estados_tipo', {}) or {}).values()):
        items.append('La planilla no tiene estado sísmico (TIPO E): el diseño usa solo combinaciones gravitacionales.')
    items.append("Confirme que el estado sísmico de la planilla se calculó con V<sub>0</sub> = c·W<sub>0</sub>/(Q′R′) "
                 "del método estático (6.3), con el mismo Q, R y c de la sección 3.1.")
    items.append('No se verifican conexiones, placas, soldaduras ni pernos.')
    items.append('Ángulos: el pandeo flexotorsional (E4) no está implementado; si b/t &gt; 0.71·√(E/Fy) la barra queda '
                 'como NO REVISADO.')
    return '<ul>' + ''.join(f'<li>{t}</li>' for t in items) + '</ul>'


def _simbologia_html():
    filas = [('Fy, Fu', 'Esfuerzo de fluencia y de ruptura del acero (kgf/cm²)'),
             ('Ag, Ae, An', 'Área total, efectiva y neta (cm²)'),
             ('U', 'Factor de rezago por cortante diferido (AISC D3)'),
             ('Ωt, Ωc, Ωb, Ωv', 'Factores de seguridad ASD: tracción, compresión, flexión y cortante'),
             ('KL/r', 'Relación de esbeltez (K: factor de longitud efectiva; L: longitud; r: radio de giro)'),
             ('Fe, Fn', 'Esfuerzo de pandeo elástico y nominal (AISC E3)'),
             ('Mp, My, Mn, Mcr', 'Momento plástico, de fluencia, nominal y de pandeo lateral-torsional'),
             ('Zx, Zy, Sx, Sy', 'Módulos plástico y elástico de la sección'),
             ('Cb', 'Factor de gradiente de momento (AISC F1)'),
             ('λ, λr, λp, λpl', 'Relación ancho/grueso del elemento y sus límites (NTC Tablas 3.2.1 y 3.2.2)'),
             ('Pr, Mr, Vr', 'Demandas ASD: carga axial, momento y cortante de la combinación'),
             ('FU', 'Factor de utilización: demanda / capacidad (cumple si FU ≤ 1.0)')]
    cuerpo = ''.join(f'<tr><td>{a}</td><td>{b}</td></tr>' for a, b in filas)
    return f'<table><tr><th>Símbolo</th><th>Significado</th></tr>{cuerpo}</table>'


def memoria_html(dib, materiales_filas, longitudes_df, demandas, id_barra, ruta_salida, proyecto=None,
                 casos=None, id_de_indice=None, equilibrio=None):
    """Escribe la memoria de calculo completa (HTML imprimible) del diseno de todas las barras del modelo."""
    proyecto = dict(proyecto or {})
    if not proyecto.get('conf'):
        raise DA.ErrorAsd('falta la configuracion de PARAMETROS y NORMA: no se genera la memoria')
    DA.perfil_por_grupo(dib)                           # una sola seccion por grupo (error si no)
    mat, lon, comb = DA._preparar_entradas(materiales_filas, longitudes_df, demandas)
    evals = [_evaluar(dib, i, id_barra, lon, mat, comb) for i in range(len(dib.barras))]
    contar = {}
    for v in evals:
        contar[v['estado']] = contar.get(v['estado'], 0) + 1
    bloques_grupo = []
    for g in DA.GRUPOS_ELEMENTO:
        sub = [v for v in evals if v['grupo'] == g]
        if sub:
            df_g = pd.DataFrame([_fila_grupo(v) for v in sub], columns=COLS_GRUPO).fillna('—')
            bloques_grupo.append(f'<h3>{g}</h3>' + df_g.to_html(index=False, na_rep='—', border=0))
    resumen_html = ''.join(bloques_grupo)
    criticas = []
    for g in DA.GRUPOS_ELEMENTO:
        sub = [v for v in evals if v['grupo'] == g and v.get('fu_res')]
        if sub:
            criticas.append(max(sub, key=lambda v: v['fu_res']['FU MAXIMO']))
    desp_html = (_seccion_desplazamientos(dib, casos, id_de_indice, id_barra,
                                          getattr(dib, 'clase_combinacion', {}) or {}, proyecto)
                 if casos is not None else '<p>Sin resultados de desplazamientos.</p>')
    resumen_txt = ' · '.join(f'{k}: {n}' for k, n in sorted(contar.items()))
    cuerpo = f'''<!doctype html><html lang="es"><head><meta charset="utf-8">
<title>Memoria de cálculo</title><style>{CSS}</style></head><body>
<h1>Memoria de cálculo: diseño de soportes de acero</h1>
{_encabezado_doc(proyecto, dib)}
{_nota_normativa()}
<h2>1. Cómo leer esta memoria</h2>
<p>Esta memoria documenta, para cada verificación, la referencia del documento (apartado y página), la ecuación, los
valores sustituidos, la capacidad admisible, la demanda de la combinación que controla y el FU. FU = demanda /
capacidad: cumple si FU ≤ 1.0. El resumen por grupo presenta el FU de interacción flexocompresión y las esbelteces;
el detalle completo de cada elemento está en diseno_asd.xlsx (RESULTADOS).</p>
<h2>2. Criterios de diseño adoptados</h2>{_criterios_html()}
<h2>3. Datos de entrada</h2>{_parametros_html(proyecto['conf'])}{_datos_entrada_html(dib, materiales_filas, id_barra, lon, mat, equilibrio)}
<h2>4. Modelo de análisis</h2>{_modelo_html(dib, equilibrio)}
<h2>5. Resultados por grupo</h2>
<p class="nota">Por grupo: FU de esfuerzos (resistencia, incluida la interacción H1/H2 y la torsión H3) y FU de esbeltez (local, KL/r y ductilidad) por separado, con sus valores calculado y límite; clasificación NTC; λ calculado y λ límite del elemento
que gobierna (esbeltez local, NTC); KL/r calculado y límite (200; 60 en columna dúctil); ductilidad y estado global
(todas las verificaciones).</p>
{('<p>' + resumen_txt + '</p>') if resumen_txt else ''}
{resumen_html}
<h2>6. Desplazamientos máximos por combinación</h2>
{desp_html}
<h2>7. Detalle de barras críticas (FU máximo de cada grupo)</h2>
<p class="nota">Se detalla solo la barra que controla cada grupo. Las demás barras de cada grupo están en la tabla por
grupo (sección 5) y en diseno_asd.xlsx (RESULTADOS).</p>
{"".join(_bloque_barra(v) for v in criticas)}
<h2>8. Pendientes y limitaciones</h2>{_pendientes_html(evals, materiales_filas, proyecto, dib)}
<h2>Anexo. Simbología</h2>{_simbologia_html()}
</body></html>'''
    with open(ruta_salida, 'w', encoding='utf-8') as f:
        f.write(cuerpo)
    return ruta_salida


In [ ]:
# Diseno ASD con ANSI/AISC 360-22 (IR, CE, LI; CS fuera del diseno). Demandas: combinaciones de servicio de COMBINACIONES (bloque 9).
# Resultados POR GRUPO: elija la seccion de cada grupo en su desplegable; la tabla se recalcula al instante.
# "Aplicar" actualiza diseno_asd.xlsx (por elemento) y planilla_con_diseno.xlsx (PERFIL_GRUPO).
import copy
import pandas as pd
from IPython.display import display
import diseno_asd as DA
import memoria_asd as MA
if NOR_LEIDA is not None and dib is not None:
    DA.cargar_norma(NOR_LEIDA['VALORES'], CONF['kz'])   # valores de norma (hoja NORMA) y Kz (PARAMETROS)

RUTA_DISENO = 'diseno_asd.xlsx'
RUTA_PLANILLA_DISENO = 'planilla_con_diseno.xlsx'


def _perfiles_analizados():
    """Perfil con que corrio el analisis, por grupo (hoja BARRAS de resultados_soportes.xlsx, columna PERFIL).
    None si no hay resultados. Si un grupo tiene perfiles distintos, se listan todos."""
    try:
        b = pd.read_excel(RUTA_RESULTADOS, sheet_name='BARRAS', dtype=object)
    except Exception:
        return None
    out = {}
    for g, sub in b.groupby('ELEMENTO'):
        perfiles = sorted({str(p) for p in sub['PERFIL'].dropna()})
        out[str(g)] = perfiles[0] if len(perfiles) == 1 else ' / '.join(perfiles)
    return out


def _escribir_planilla(perfiles, listas, fu_matriz):
    escribir_planilla(RUTA_PLANILLA_DISENO, _filas(nodos, NODOS_COLS), _filas(barras, BARRAS_COLS),
                      filas_cargas=filas_cargas_para_excel(cargas), materiales=materiales_filas,
                      combinaciones=comb_crudas,
                      longitudes=filas_longitudes_para_excel(longitudes_df),
                      estados=[[None if _vacio(v) else v for v in f] for f in estados_df.values.tolist()],
                      listas=listas, grupos_perfil=perfiles, fu_matriz=fu_matriz,
                      perfiles_analizados=_perfiles_analizados())
    guardar_con_valores(RUTA_PLANILLA_DISENO)


def _escribir_diseno(tabla_listas):
    with pd.ExcelWriter(RUTA_DISENO, mode='a', engine='openpyxl', if_sheet_exists='replace') as w:
        tabla_listas.to_excel(w, sheet_name='LISTAS_GRUPO', index=False)


if dib is None or 'bloques' not in globals():
    print('Ejecute primero el bloque 9 (analisis).')
else:
    try:
        perfiles_grupo = DA.perfil_por_grupo(dib)   # una sola seccion por grupo
        capacidades, resultados, detalle = DA.diseno_asd_desde_planilla(
            dib, materiales_leidos, longitudes_df, bloques['DISENO_BARRAS'], dib.id_barra_de_indice, RUTA_DISENO)
        fu_matriz = DA.matriz_fu_por_perfil(dib, materiales_leidos, longitudes_df, bloques['DISENO_BARRAS'],
                                            dib.id_barra_de_indice)
        listas, tabla_listas = DA.listas_desde_matriz(fu_matriz, dib)
    except DA.ErrorAsd as error:
        print('NO se diseno. Corrija lo siguiente y vuelva a ejecutar este bloque:')
        print(error)
    else:
        _escribir_diseno(tabla_listas)
        _escribir_planilla(perfiles_grupo, listas, fu_matriz)
        print(f'Diseno ASD (AISC 360-22). Excel: {RUTA_DISENO} (RESULTADOS por elemento, DETALLE, LISTAS_GRUPO)')
        print(f'Planilla con perfil por grupo: {RUTA_PLANILLA_DISENO} (hoja PERFIL_GRUPO)')
        con_barras = [g for g in DA.GRUPOS_ELEMENTO if any(b['elemento'] == g for b in dib.barras)]
        for g in DA.GRUPOS_ELEMENTO:
            if g not in con_barras:
                print(f'{g}: sin barras en el modelo.')
        try:
            import ipywidgets as W
            selectores, salidas = {}, {}
            for g in con_barras:
                ops = MA.opciones_grupo(fu_matriz, g)
                nombres = [n for _, n in ops]
                if perfiles_grupo[g] not in nombres:
                    ops.insert(0, (f'{perfiles_grupo[g]}  —  perfil actual', perfiles_grupo[g]))
                selectores[g] = W.Dropdown(options=ops, value=perfiles_grupo[g], description=f'{g}:',
                                           layout=W.Layout(width='760px'))
                salidas[g] = W.Output()

            def _mostrar(g):
                with salidas[g]:
                    salidas[g].clear_output(wait=True)
                    tabla = MA.resultados_grupo(dib, g, selectores[g].value, materiales_leidos, longitudes_df,
                                                bloques['DISENO_BARRAS'], dib.id_barra_de_indice)
                    ok = bool((tabla['ESTADO'] == 'CUMPLE').all())
                    print(f'{g}: {selectores[g].value} · {len(tabla)} barras · '
                          f'{"CUMPLE" if ok else "NO CUMPLE"}')
                    display(tabla)

            for g in con_barras:
                selectores[g].observe(lambda ch, g=g: _mostrar(g), names='value')
                _mostrar(g)

            boton = W.Button(description='Aplicar selección a diseno_asd.xlsx y planilla', button_style='primary')
            aviso = W.Output()

            def _aplicar(_):
                with aviso:
                    aviso.clear_output()
                    sel = {g: selectores[g].value for g in con_barras}
                    d2 = copy.copy(dib)
                    d2.barras = [dict(b, perfil=sel[b['elemento']]) if b['elemento'] in sel else b
                                 for b in dib.barras]
                    DA.diseno_asd_desde_planilla(d2, materiales_leidos, longitudes_df, bloques['DISENO_BARRAS'],
                                                 dib.id_barra_de_indice, RUTA_DISENO)
                    _escribir_diseno(tabla_listas)
                    _escribir_planilla(sel, listas, fu_matriz)
                    print('Actualizado con:', sel)

            boton.on_click(_aplicar)
            display(W.VBox([W.HTML('<b>Elija la sección de cada grupo.</b> La lista está ordenada por tipo y sección; '
                                   'la tabla se recalcula al cambiarla. Interacción = FU flexocompresión (H1/H2).')]
                           + [W.VBox([selectores[g], salidas[g]]) for g in con_barras] + [boton, aviso]))
        except ImportError:
            print('Sin ipywidgets: tablas por grupo con el perfil actual.')
            for g in con_barras:
                display(MA.resultados_grupo(dib, g, perfiles_grupo[g], materiales_leidos, longitudes_df,
                                            bloques['DISENO_BARRAS'], dib.id_barra_de_indice))


## Bloque 11. Memoria de cálculo (HTML)

Genera `memoria_calculo.html` con los datos, las propiedades y cada verificación de cada barra, con su ecuación, los valores sustituidos y el FU. Incluye la nota normativa: diseño AISC 360-22 (ASD) y esbelteces local y global con la NTC-Acero CDMX 2020. Para PDF: abra el archivo y use Imprimir > Guardar como PDF. Edite PROYECTO dentro del bloque.

In [ ]:
# Bloque 11: memoria de calculo completa (HTML, imprimible a PDF). Usa el mismo calculo que el bloque 10.
# Datos del proyecto (portada). La configuracion de diseno y de norma sale de las hojas PARAMETROS y NORMA.
import memoria_asd as MA

PROYECTO = {
    'nombre': '(nombre del proyecto)',
    'elemento': '(estructura o elemento)',
    'fecha': '',
    'version': '(completar)',
    'elaboro': '',
    'reviso': '',
    'aprobo': '',
}
RUTA_MEMORIA = 'memoria_calculo.html'
if CONF is not None:
    PROYECTO['conf'] = CONF
if dib is None or 'bloques' not in globals() or 'DISENO_BARRAS' not in bloques:
    print('Ejecute primero el bloque 9 (analisis) y el bloque 10 (diseno).')
else:
    try:
        MA.memoria_html(dib, materiales_leidos, longitudes_df, bloques['DISENO_BARRAS'], dib.id_barra_de_indice,
                        RUTA_MEMORIA, PROYECTO, casos=casos, id_de_indice=id_de_indice, equilibrio=equilibrio)
    except DA.ErrorAsd as error:
        print('NO se genero la memoria. Corrija lo siguiente y vuelva a ejecutar este bloque:')
        print(error)
    else:
        print(f'Memoria de calculo: {RUTA_MEMORIA} (abrir y Imprimir > Guardar como PDF)')
